<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NCAAB_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
print("Python version:", sys.version)

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
import os
from getpass import getpass

os.environ["ODDS_API_KEY"] = getpass("Paste your Odds API key here: ")

Paste your Odds API key here: ··········


In [ ]:
import requests
import os

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals,h2h"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

response = requests.get(url, params=params)

if response.status_code != 200:
    print("❌ API Error:")
    print(response.text)
else:
    data = response.json()
    print(f"✅ Pulled {len(data)} games\n")

    for game in data:
        print("Game:", game["home_team"], "vs", game["away_team"])
        print("Game ID:", game["id"])
        print("Commence:", game["commence_time"])
        print("-" * 40)

✅ Pulled 43 games

Game: Bellarmine Knights vs Jacksonville Dolphins
Game ID: 76343f981d0bdaf7f428cfdfe9f5cbf6
Commence: 2026-03-04T16:59:00Z
----------------------------------------
Game: Florida Gulf Coast Eagles vs North Alabama Lions
Game ID: 3f8510356b010d6698c1ea8f75b736b1
Commence: 2026-03-04T19:30:00Z
----------------------------------------
Game: Eastern Kentucky Colonels vs Stetson Hatters
Game ID: 53ad30f5ebd93132b1fcbb9573682771
Commence: 2026-03-04T22:00:00Z
----------------------------------------
Game: Butler Bulldogs vs Creighton Bluejays
Game ID: 6b8be6ad9cf80bcbf5ffd292fbfc9c18
Commence: 2026-03-04T23:00:00Z
----------------------------------------
Game: James Madison Dukes vs Louisiana Ragin' Cajuns
Game ID: 61164f56921094b6a3e70c83ba9b1a24
Commence: 2026-03-04T23:00:00Z
----------------------------------------
Game: La Salle Explorers vs Fordham Rams
Game ID: 6047626b7a0aeafe4104c1a1573ba6dc
Commence: 2026-03-04T23:30:00Z
----------------------------------------
Gam

In [ ]:
import requests
import os

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

# ✅ FIXED headers (clean)
headers = {
    "User-Agent": "Mozilla/5.0"
}

# ✅ Single request
response = requests.get(url, params=params, headers=headers)

print("Status:", response.status_code)

# ✅ Make sure request succeeded before parsing
if response.status_code == 200:
    data = response.json()

    print("🔥 Extracting Spreads + Totals\n")

    for game in data:
        print("====================================")
        print(game["home_team"], "vs", game["away_team"])

        for bookmaker in game.get("bookmakers", []):
            for market in bookmaker.get("markets", []):

                if market["key"] == "spreads":
                    for outcome in market.get("outcomes", []):
                        print("Spread:", outcome["name"],
                              outcome["point"],
                              "Price:", outcome["price"])

                if market["key"] == "totals":
                    for outcome in market.get("outcomes", []):
                        print("Total:", outcome["name"],
                              outcome["point"],
                              "Price:", outcome["price"])

        print("\n")

else:
    print("❌ API request failed")
    print(response.text)

Status: 200
🔥 Extracting Spreads + Totals

Bellarmine Knights vs Jacksonville Dolphins
Spread: Bellarmine Knights 2.5 Price: 1.68
Spread: Jacksonville Dolphins -2.5 Price: 2.1
Total: Over 169.5 Price: 1.85
Total: Under 169.5 Price: 1.89
Spread: Bellarmine Knights 0.5 Price: 1.93
Spread: Jacksonville Dolphins -0.5 Price: 1.74
Total: Over 170.5 Price: 1.91
Total: Under 170.5 Price: 1.76
Spread: Bellarmine Knights 1.5 Price: 1.95
Spread: Jacksonville Dolphins -1.5 Price: 1.8
Total: Over 169.5 Price: 1.87
Total: Under 169.5 Price: 1.87
Spread: Bellarmine Knights 2.5 Price: 1.77
Spread: Jacksonville Dolphins -2.5 Price: 2.0
Total: Over 169.5 Price: 1.87
Total: Under 169.5 Price: 1.87


Florida Gulf Coast Eagles vs North Alabama Lions
Spread: Florida Gulf Coast Eagles -7.5 Price: 1.91
Spread: North Alabama Lions 7.5 Price: 1.91
Total: Over 139.5 Price: 1.91
Total: Under 139.5 Price: 1.91
Spread: Florida Gulf Coast Eagles -7.5 Price: 1.88
Spread: North Alabama Lions 7.5 Price: 1.92
Total: Ove

In [ ]:
import requests
import os

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

response = requests.get(url, params=params)
data = response.json()

def decimal_to_american(decimal_odds):
    if decimal_odds >= 2.0:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"{int(-100 / (decimal_odds - 1))}"

print("🔥 NCAAB Slate (American Odds)\n")

for game in data:
    print("====================================")
    print(game["home_team"], "vs", game["away_team"])

    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:

            if market["key"] in ["spreads", "totals"]:
                for outcome in market["outcomes"]:

                    american_price = decimal_to_american(outcome["price"])

                    print(
                        f"{market['key'].upper()} | "
                        f"{outcome['name']} "
                        f"{outcome.get('point', '')} | "
                        f"Odds: {american_price}"
                    )

    print("\n")

🔥 NCAAB Slate (American Odds)

Bellarmine Knights vs Jacksonville Dolphins
SPREADS | Bellarmine Knights 2.5 | Odds: -147
SPREADS | Jacksonville Dolphins -2.5 | Odds: +110
TOTALS | Over 169.5 | Odds: -117
TOTALS | Under 169.5 | Odds: -112
SPREADS | Bellarmine Knights 0.5 | Odds: -107
SPREADS | Jacksonville Dolphins -0.5 | Odds: -135
TOTALS | Over 170.5 | Odds: -109
TOTALS | Under 170.5 | Odds: -131
SPREADS | Bellarmine Knights 1.5 | Odds: -105
SPREADS | Jacksonville Dolphins -1.5 | Odds: -125
TOTALS | Over 169.5 | Odds: -114
TOTALS | Under 169.5 | Odds: -114
SPREADS | Bellarmine Knights 2.5 | Odds: -129
SPREADS | Jacksonville Dolphins -2.5 | Odds: +100
TOTALS | Over 169.5 | Odds: -114
TOTALS | Under 169.5 | Odds: -114


Florida Gulf Coast Eagles vs North Alabama Lions
SPREADS | Florida Gulf Coast Eagles -7.5 | Odds: -109
SPREADS | North Alabama Lions 7.5 | Odds: -109
TOTALS | Over 139.5 | Odds: -109
TOTALS | Under 139.5 | Odds: -109
SPREADS | Florida Gulf Coast Eagles -7.5 | Odds: -113


In [ ]:
import requests
import os

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

response = requests.get(url, params=params)
data = response.json()

def decimal_to_american(decimal_odds):
    if decimal_odds >= 2.0:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"{int(-100 / (decimal_odds - 1))}"

def implied_probability(decimal_odds):
    return 1 / decimal_odds

print("🔥 NCAAB Slate — With Implied Probability\n")

for game in data:
    print("====================================")
    print(game["home_team"], "vs", game["away_team"])

    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:

            if market["key"] in ["spreads", "totals"]:

                for outcome in market["outcomes"]:

                    dec = outcome["price"]
                    american = decimal_to_american(dec)
                    ip = implied_probability(dec) * 100

                    print(
                        f"{market['key'].upper()} | "
                        f"{outcome['name']} {outcome.get('point', '')} | "
                        f"Odds: {american} | "
                        f"Implied Prob: {ip:.2f}%"
                    )

    print("\n")

🔥 NCAAB Slate — With Implied Probability

Bellarmine Knights vs Jacksonville Dolphins
SPREADS | Bellarmine Knights 2.5 | Odds: -147 | Implied Prob: 59.52%
SPREADS | Jacksonville Dolphins -2.5 | Odds: +110 | Implied Prob: 47.62%
TOTALS | Over 169.5 | Odds: -117 | Implied Prob: 54.05%
TOTALS | Under 169.5 | Odds: -112 | Implied Prob: 52.91%
SPREADS | Bellarmine Knights 0.5 | Odds: -107 | Implied Prob: 51.81%
SPREADS | Jacksonville Dolphins -0.5 | Odds: -135 | Implied Prob: 57.47%
TOTALS | Over 170.5 | Odds: -109 | Implied Prob: 52.36%
TOTALS | Under 170.5 | Odds: -131 | Implied Prob: 56.82%
SPREADS | Bellarmine Knights 1.5 | Odds: -105 | Implied Prob: 51.28%
SPREADS | Jacksonville Dolphins -1.5 | Odds: -125 | Implied Prob: 55.56%
TOTALS | Over 169.5 | Odds: -114 | Implied Prob: 53.48%
TOTALS | Under 169.5 | Odds: -114 | Implied Prob: 53.48%
SPREADS | Bellarmine Knights 2.5 | Odds: -129 | Implied Prob: 56.50%
SPREADS | Jacksonville Dolphins -2.5 | Odds: +100 | Implied Prob: 50.00%
TOTALS 

In [ ]:
import requests
import os
from collections import defaultdict

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

response = requests.get(url, params=params)
data = response.json()


def decimal_to_american(decimal_odds):
    if decimal_odds >= 2.0:
        return f"+{int((decimal_odds - 1) * 100)}"
    else:
        return f"{int(-100 / (decimal_odds - 1))}"


def implied_probability(decimal_odds):
    return 1 / decimal_odds


print("🔥 CONSENSUS ENGINE — Stable V3\n")

for game in data:
    print("====================================")
    print(game["home_team"], "vs", game["away_team"])

    market_data = defaultdict(list)

    # Collect all prices per market
    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] in ["spreads", "totals"]:
                for outcome in market["outcomes"]:
                    market_key = (
                        market["key"],
                        outcome["name"],
                        outcome.get("point")
                    )
                    market_data[market_key].append(outcome["price"])

    # Compute consensus
    for (market_type, team, point), prices in market_data.items():

        avg_price = sum(prices) / len(prices)
        best_price = max(prices)

        avg_american = decimal_to_american(avg_price)
        best_american = decimal_to_american(best_price)
        avg_prob = implied_probability(avg_price) * 100

        print(
            f"{market_type.upper()} | {team} {point} | "
            f"Consensus Odds: {avg_american} | "
            f"Best Odds: {best_american} | "
            f"Implied Prob (Avg): {avg_prob:.2f}%"
        )

    print("\n")

In [ ]:
import requests
import os
from collections import defaultdict

API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "spreads,totals"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"

params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS
}

response = requests.get(url, params=params)
data = response.json()


def decimal_to_prob(decimal):
    return 1 / decimal


def prob_to_american(prob):
    if prob >= 0.5:
        return f"{int(-100 * (prob / (1 - prob)))}"
    else:
        return f"+{int(100 * ((1 - prob) / prob))}"


print("🔥 CONSENSUS + EV ENGINE\n")

for game in data:

    print("====================================")
    print(game["home_team"], "vs", game["away_team"])

    market_data = defaultdict(list)

    # Collect prices
    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] in ["spreads", "totals"]:
                for outcome in market["outcomes"]:
                    market_key = (
                        market["key"],
                        outcome["name"],
                        outcome.get("point")
                    )
                    market_data[market_key].append(outcome["price"])

    # Process markets
    for (market_type, team, point), prices in market_data.items():

        probs = [decimal_to_prob(p) for p in prices]

        # Remove vig
        total_prob = sum(probs)
        fair_probs = [p / total_prob for p in probs]

        avg_fair_prob = sum(fair_probs) / len(fair_probs)

        edge = avg_fair_prob - (sum(probs) / len(probs))

        print(
            f"{market_type.upper()} | {team} {point} | "
            f"Edge: {edge*100:.2f}%"
        )

    print("\n")

In [ ]:
for (market_type, team, point), prices in market_data.items():

    # Convert all prices to implied probabilities
    probs = [1 / p for p in prices]

    # Remove vig
    total_prob = sum(probs)
    fair_probs = [(p / total_prob) for p in probs]

    # Use best price for edge calculation
    best_price = max(prices)
    best_prob = 1 / best_price

    # Fair probability from normalized values
    fair_prob = sum(fair_probs) / len(fair_probs)

    edge = fair_prob - best_prob

    if edge > 0.02:  # Only show meaningful edge
        status = "🔥 +EV"
    else:
        status = ""

    print(
        f"{market_type.upper()} | {team} {point} | "
        f"Edge: {edge*100:.2f}% {status}"
    )

In [ ]:
for (market_type, team, point), prices in market_data.items():

    # Convert to probabilities
    probs = [1 / p for p in prices]

    # Remove vig (normalize)
    total = sum(probs)
    fair_probs = [p / total for p in probs]

    fair_probability = sum(fair_probs) / len(fair_probs)

    # Best price = most favorable to bettor
    best_price = max(prices)
    best_probability = 1 / best_price

    edge = fair_probability - best_probability

    if edge > 0.01:
        status = "🔥 +EV"
    else:
        status = ""

    print(
        f"{market_type.upper()} | {team} {point} | "
        f"Edge: {edge*100:.2f}% {status}"
    )

In [ ]:
for (market_type, team, point), prices in market_data.items():

    # Remove duplicate prices
    unique_prices = list(set(prices))

    # Sort prices so we keep best 3 only
    unique_prices.sort(reverse=True)
    trimmed_prices = unique_prices[:3]

    probs = [1 / p for p in trimmed_prices]

    total = sum(probs)
    fair_probs = [p / total for p in probs]

    fair_probability = sum(fair_probs) / len(fair_probs)

    # Best price
    best_price = max(trimmed_prices)
    best_probability = 1 / best_price

    edge = fair_probability - best_probability

    if edge > 0.01:
        status = "🔥 +EV"
    else:
        status = ""

    print(
        f"{market_type.upper()} | {team} {point} | "
        f"Edge: {edge*100:.2f}% {status}"
    )

In [ ]:
for (market_type, team, point), prices in market_data.items():

    # Use best available price
    best_price = max(prices)
    best_implied_prob = 1 / best_price

    # Convert all prices to probabilities
    market_probs = [1 / p for p in prices]

    # Market vig = sum of implied probs
    market_vig = sum(market_probs)

    # Fair probability = normalize against vig
    fair_probability = best_implied_prob / market_vig

    edge = fair_probability - best_implied_prob

    if edge > 0.01:
        status = "🔥 +EV"
    else:
        status = ""

    print(
        f"{market_type.upper()} | {team} {point} | "
        f"Edge: {edge*100:.2f}% {status}"
    )

In [ ]:
for (market_type, team, point), prices in market_data.items():

    # Remove duplicates
    unique_prices = list(set(prices))

    # Best available price
    best_price = max(unique_prices)
    best_prob = 1 / best_price

    # Market average implied probability
    market_probs = [1 / p for p in unique_prices]
    market_avg_prob = sum(market_probs) / len(market_probs)

    edge = market_avg_prob - best_prob

    status = "🔥 +EV" if edge > 0.01 else ""

    print(
        f"{market_type.upper()} | {team} {point} | "
        f"Edge: {edge*100:.2f}% {status}"
    )

In [ ]:
# SIMPLE BASELINE WIN PROBABILITY MODEL

import random

def project_win_probability(team_rating, opponent_rating, home_advantage=0.03):
    """
    Basic logistic-style probability projection
    """

    rating_diff = team_rating - opponent_rating
    base_prob = 1 / (1 + pow(10, -rating_diff / 10))

    # Apply home advantage bump if needed later
    final_prob = base_prob + home_advantage

    return min(max(final_prob, 0), 1)


# Example placeholder ratings (you will replace with real data later)
team_rating = 5.2
opp_rating = 4.7

prob = project_win_probability(team_rating, opp_rating)

print("Projected Win Probability:", round(prob * 100, 2), "%")

In [ ]:
# Example: Build Ratings From Historical Metrics

def calculate_team_rating(off_eff, def_eff, pace):
    """
    Simple rating formula combining offense + defense
    """

    rating = (off_eff * 0.6) - (def_eff * 0.4) + (pace * 0.1)

    return rating


# Example historical stats (replace later with real data)
team_stats = {
    "off_eff": 112,
    "def_eff": 98,
    "pace": 70
}

opponent_stats = {
    "off_eff": 108,
    "def_eff": 101,
    "pace": 68
}

team_rating = calculate_team_rating(**team_stats)
opp_rating = calculate_team_rating(**opponent_stats)

print("Team Rating:", round(team_rating, 2))
print("Opponent Rating:", round(opp_rating, 2))

In [ ]:
# Example: Pull Public Team Stats From Free API

import requests

# Example public endpoint (replace later with stronger source if needed)
STATS_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

response = requests.get(STATS_URL)

if response.status_code == 200:
    data = response.json()
    teams = data.get("sports", [])[0].get("leagues", [])[0].get("teams", [])

    print("Teams Retrieved:", len(teams))

    for team in teams[:10]:  # Show first 10 for testing
        name = team["team"]["displayName"]
        print("Team:", name)

else:
    print("API Error:", response.status_code)

In [ ]:
import requests

STATS_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

response = requests.get(STATS_URL)

if response.status_code == 200:
    data = response.json()
    teams = data.get("sports", [])[0].get("leagues", [])[0].get("teams", [])

    print("Extracting Basic Team Data...\n")

    for team in teams[:5]:  # Keep it small for testing
        name = team["team"]["displayName"]
        team_id = team["team"]["id"]

        print("Team:", name)
        print("Team ID:", team_id)
        print("-" * 30)

else:
    print("Failed to fetch team stats")

In [ ]:
import requests

TEAM_ID = 2  # You can change this to any ID printed earlier

url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{TEAM_ID}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()

    team_name = data["team"]["displayName"]

    print("Team:", team_name)
    print("------ Stats ------")

    stats = data.get("team", {}).get("record", {})

    print(stats)

else:
    print("Failed to pull team stats")

In [ ]:
import requests
import pandas as pd

team_ids = [44, 9, 12, 8, 2]  # Replace with your full team ID list

all_team_data = []

for team_id in team_ids:
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}"
    response = requests.get(url)

    if response.status_code != 200:
        continue

    data = response.json()
    team_name = data["team"]["displayName"]

    stats_sections = data.get("team", {}).get("record", {}).get("items", [])

    overall = stats_sections[0]["stats"] if stats_sections else []

    stat_dict = {"team": team_name, "team_id": team_id}

    for stat in overall:
        stat_dict[stat["name"]] = stat["value"]

    all_team_data.append(stat_dict)

df = pd.DataFrame(all_team_data)

print(df.head())

# Save to CSV
df.to_csv("team_stats_snapshot.csv", index=False)

print("Saved -> team_stats_snapshot.csv")

In [ ]:
df = pd.read_csv("team_stats_snapshot.csv")

# Simple Power Model
df["Offensive_Rating"] = df["avgPointsFor"] * df["winPercent"]
df["Defensive_Rating"] = df["avgPointsAgainst"] * (1 - df["winPercent"])
df["Net_Rating"] = df["Offensive_Rating"] - df["Defensive_Rating"]

# Normalize to 0-100 scale
df["Power_Score"] = (
    (df["Net_Rating"] - df["Net_Rating"].min()) /
    (df["Net_Rating"].max() - df["Net_Rating"].min())
) * 100

print(df[["team", "Power_Score"]].sort_values("Power_Score", ascending=False))

df.to_csv("team_ratings_output.csv", index=False)

print("Saved -> team_ratings_output.csv")

In [ ]:
df = pd.read_csv("team_stats_snapshot.csv")

df["Power_Score"] = (
    (df["winPercent"] * 0.40) +
    ((df["pointDifferential"] / df["gamesPlayed"]) * 0.30) +
    ((df["avgPointsFor"] - df["avgPointsAgainst"]) * 0.30)
)

# Normalize
df["Power_Score"] = (
    (df["Power_Score"] - df["Power_Score"].min()) /
    (df["Power_Score"].max() - df["Power_Score"].min())
) * 100

print(df[["team", "Power_Score"]].sort_values("Power_Score", ascending=False))

df.to_csv("team_ratings_output.csv", index=False)

print("Saved -> team_ratings_output.csv")

In [ ]:
import pandas as pd
import requests

# ---- Pull Team Data (Example) ----
teams_response = requests.get("YOUR_TEAM_API_URL")
teams_json = teams_response.json()

team_df = pd.DataFrame(teams_json)

print("Teams loaded:", len(team_df))


# ---- Pull Odds Data ----
odds_response = requests.get("YOUR_ODDS_API_URL")
odds_json = odds_response.json()

odds_df = pd.DataFrame(odds_json)

print("Odds loaded:", len(odds_df))


# ---- Save Snapshots ----
team_df.to_csv("team_stats_snapshot.csv", index=False)
odds_df.to_csv("odds_snapshot.csv", index=False)

print("✅ Files saved")

In [ ]:
import requests
import pandas as pd

# ==================================================
# ⚡ Pull ESPN Team Standings / Stats
# ==================================================

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/standings"

resp = requests.get(url)
data = resp.json()

teams = []
for group in data["children"]:
    for team in group["standings"]:
        t = team["team"]["displayName"]
        stats = team.get("stats", [])

        stat_dict = {"team": t}
        for s in stats:
            stat_dict[s["name"]] = s["displayValue"]

        teams.append(stat_dict)

df_espn = pd.DataFrame(teams)

print(df_espn.head())
df_espn.to_csv("espn_team_stats.csv", index=False)
print("Saved -> espn_team_stats.csv")

In [ ]:
import requests
import json

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/standings"

resp = requests.get(url)
data = resp.json()

print(json.dumps(data, indent=2)[:5000])  # Print first part of JSON

In [ ]:
import requests
import pandas as pd

teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

resp = requests.get(teams_url)
teams_json = resp.json()

teams = []

for sport in teams_json.get("sports", []):
    for league in sport.get("leagues", []):
        for team in league.get("teams", []):
            teams.append({
                "team_id": team["team"]["id"],
                "team_name": team["team"]["displayName"]
            })

team_df = pd.DataFrame(teams)

print(team_df.head())
team_df.to_csv("master_team_list.csv", index=False)
print("Saved -> master_team_list.csv")

In [ ]:
import time

all_team_stats = []
all_player_stats = []

for _, row in team_df.iterrows():

    team_id = row["team_id"]
    team_name = row["team_name"]

    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}"

    resp = requests.get(url)

    if resp.status_code != 200:
        continue

    data = resp.json()

    # ---------------- TEAM STATS ---------------- #
    team_stats = {
        "team_id": team_id,
        "team_name": team_name
    }

    stats = data.get("team", {}).get("record", {})
    team_stats.update(stats)

    all_team_stats.append(team_stats)

    # ---------------- PLAYERS ---------------- #
    athletes = data.get("team", {}).get("athletes", [])

    for athlete in athletes:
        player_stats = {
            "team_id": team_id,
            "team_name": team_name,
            "player_name": athlete["displayName"]
        }

        player_stats.update(athlete.get("stats", {}))
        all_player_stats.append(player_stats)

    time.sleep(0.5)  # Avoid rate limiting

team_stats_df = pd.DataFrame(all_team_stats)
player_stats_df = pd.DataFrame(all_player_stats)

team_stats_df.to_csv("team_stats_full.csv", index=False)
player_stats_df.to_csv("player_stats_full.csv", index=False)

print("Saved team + player stats")

In [ ]:
import pandas as pd

# Load saved data
team_stats_df = pd.read_csv("team_stats_full.csv")
player_stats_df = pd.read_csv("player_stats_full.csv")

print("Loaded team stats:", team_stats_df.shape)
print("Loaded player stats:", player_stats_df.shape)


# ------------------------------
# Example Simple Power Model
# ------------------------------

# Make sure numeric columns are numeric
team_stats_df["wins"] = pd.to_numeric(team_stats_df["wins"], errors="coerce")
team_stats_df["losses"] = pd.to_numeric(team_stats_df["losses"], errors="coerce")

team_stats_df["win_rate"] = team_stats_df["wins"] / (
    team_stats_df["wins"] + team_stats_df["losses"]
)

# Replace NaN with 0
team_stats_df["win_rate"] = team_stats_df["win_rate"].fillna(0)

# Normalize to 0–100 scale
team_stats_df["Power_Score"] = (
    team_stats_df["win_rate"] / team_stats_df["win_rate"].max()
) * 100

team_stats_df.to_csv("team_power_model.csv", index=False)

print("Saved -> team_power_model.csv")
print(team_stats_df[["team_name", "Power_Score"]].head())

In [ ]:
# ---------------- PLAYERS (Better Extraction) ---------------- #

roster = data.get("team", {}).get("roster", {}).get("athletes", [])

for player in roster:
    player_stats = {
        "team_id": team_id,
        "team_name": team_name,
        "player_name": player.get("displayName"),
        "position": player.get("position", {}).get("displayName"),
        "height": player.get("height"),
        "weight": player.get("weight")
    }

    all_player_stats.append(player_stats)

In [ ]:
import pandas as pd

df = pd.read_csv("player_stats_full.csv")
print(df.head())
print("Rows:", len(df))

In [ ]:
import requests
import json

# Pick ONE team from your master_team_list.csv
team_id = 2  # change this if needed

url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}"

resp = requests.get(url)

data = resp.json()

print(json.dumps(data, indent=2)[:4000])  # Print first 4000 chars

In [ ]:
import pandas as pd

# Example: Pull current season player stats
url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"

tables = pd.read_html(url)

# Usually table 0 contains player stats
player_stats = tables[0]

print(player_stats.head())

player_stats.to_csv("player_stats_sportsref.csv", index=False)
print("Saved -> player_stats_sportsref.csv")

In [ ]:
import pandas as pd

# Player Per-Game Stats
player_url = "https://www.sports-reference.com/cbb/seasons/men/2026-per-game.html"
player_tables = pd.read_html(player_url)
player_df = player_tables[0]

player_df.to_csv("player_per_game_stats.csv", index=False)
print("Saved -> player_per_game_stats.csv")
print(player_df.head())

In [ ]:
import pandas as pd
import requests

url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

if response.status_code != 200:
    print("Failed to load page:", response.status_code)
else:
    tables = pd.read_html(response.text)
    print("Tables found:", len(tables))

    df = tables[0]
    print(df.head())

    df.to_csv("player_stats_sportsref.csv", index=False)
    print("Saved -> player_stats_sportsref.csv")

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

tables = pd.read_html(StringIO(response.text))

for i, table in enumerate(tables):
    print("\n======================")
    print("Table Index:", i)
    print(table.head())

In [ ]:
import pandas as pd
import requests
from io import StringIO

# Fetch page again
url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))

team_table = tables[0]  # Conference table

# Clean numeric columns
team_table["W-L%"] = pd.to_numeric(team_table["W-L%"], errors="coerce")
team_table["SRS"] = pd.to_numeric(team_table["SRS"], errors="coerce")
team_table["SOS"] = pd.to_numeric(team_table["SOS"], errors="coerce")

# Build Power Score
team_table["Power_Score"] = (
    team_table["W-L%"] * 50 +
    team_table["SRS"] * 2 +
    team_table["SOS"] * 1
)

# Normalize
team_table["Power_Score"] = (
    team_table["Power_Score"] - team_table["Power_Score"].min()
) / (
    team_table["Power_Score"].max() - team_table["Power_Score"].min()
) * 100

team_table = team_table.sort_values("Power_Score", ascending=False)

print(team_table[["Conference", "Power_Score"]].head())

team_table.to_csv("team_power_advanced.csv", index=False)
print("Saved -> team_power_advanced.csv")

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))

team_table = tables[1]  # Table 1 = Team Rankings

# Convert rank to numeric
team_table["Rk"] = pd.to_numeric(team_table["Rk"], errors="coerce")

# Build Team Power Score
# Lower rank number = stronger team
team_table["Rank_Score"] = 100 - team_table["Rk"]

# Normalize
team_table["Team_Power_Score"] = (
    team_table["Rank_Score"] - team_table["Rank_Score"].min()
) / (
    team_table["Rank_Score"].max() - team_table["Rank_Score"].min()
) * 100

team_table = team_table.sort_values("Team_Power_Score", ascending=False)

print(team_table.head())

team_table.to_csv("team_power_team_level.csv", index=False)
print("Saved -> team_power_team_level.csv")

In [ ]:
# HYBRID SIM ENGINE

In [ ]:
print(globals().keys())

In [ ]:
try:
  print(odds_df.head())
except NameError:
  print ("odds_df NOTFOUND")

In [ ]:
odds_response = requests.get(url, params=params)
odds_data = odds_response.json()

odds_df = pd.DataFrame(odds_data)

In [ ]:
odds_response = requests.get(url, params=params)

print("Status Code:", odds_response.status_code)
print("Raw Text:\n", odds_response.text[:1000])

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://www.sports-reference.com/cbb/seasons/men/2026.html"

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

response = session.get(url, headers=headers)

print("Status:", response.status_code)

tables = pd.read_html(StringIO(response.text))

print("Tables Found:", len(tables))
print(tables[1].head())

In [ ]:
import pandas as pd
import requests
from io import StringIO

player_url = "https://www.sports-reference.com/cbb/seasons/men/2026-per-game.html"

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = session.get(player_url, headers=headers)

print("Status:", response.status_code)

tables = pd.read_html(StringIO(response.text))

print("Tables Found:", len(tables))
print(tables[0].head())

In [ ]:
import pandas as pd
import requests
from io import StringIO

player_url = "https://www.sports-reference.com/cbb/seasons/men/2026-players.html"

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = session.get(player_url, headers=headers)

print("Status:", response.status_code)

if response.status_code == 200:
    try:
        tables = pd.read_html(StringIO(response.text))
        print("Tables Found:", len(tables))
        print(tables[0].head())
    except Exception as e:
        print("Table Parsing Error:", e)
else:
    print("Page did not load correctly.")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

url = "https://www.sports-reference.com/cbb/seasons/men/2026-per-game.html"

session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0"}

response = session.get(url, headers=headers)

print("Status:", response.status_code)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    tables = soup.find_all("table")
    print("Tables Found (via soup):", len(tables))

    # Print table IDs so we know what exists
    for i, table in enumerate(tables):
        print("Table", i, "ID:", table.get("id"))
else:
    print("Page failed to load")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
import time

base_url = "https://www.sports-reference.com"

session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0"}

team_player_data = []

# Get team names from your team table
team_table = tables[1]  # Table 1 from earlier (team rankings)

for team in team_table["School"].dropna().unique():

    team_slug = team.lower().replace(" ", "-")
    team_slug = team_slug.replace("'", "")

    team_url = f"{base_url}/cbb/schools/{team_slug}/2026.html"

    print("Scraping:", team_url)

    response = session.get(team_url, headers=headers)

    if response.status_code != 200:
        continue

    try:
        dfs = pd.read_html(StringIO(response.text))

        # Usually player table is first large table
        player_df = dfs[0]
        player_df["Team"] = team

        team_player_data.append(player_df)

    except:
        continue

    time.sleep(1)

if team_player_data:
    all_players_df = pd.concat(team_player_data, ignore_index=True)
    all_players_df.to_csv("all_team_player_stats.csv", index=False)
    print("Saved -> all_team_player_stats.csv")
    print(all_players_df.head())
else:
    print("No player data collected.")

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
from io import StringIO
import time

session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0"}

all_detailed_players = []

team_pages = [
    "https://www.sports-reference.com/cbb/schools/duke/2026.html"
]

for url in team_pages:
    print("Processing:", url)
    response = session.get(url, headers=headers)

    if response.status_code != 200:
        continue

    tables = pd.read_html(StringIO(response.text))

    # Usually player stats table is the 2nd or 3rd table
    for table in tables:
        if "Player" in table.columns:
            table["Team_URL"] = url
            all_detailed_players.append(table)

    time.sleep(1)

if all_detailed_players:
    final_df = pd.concat(all_detailed_players, ignore_index=True)
    final_df.to_csv("detailed_player_stats.csv", index=False)
    print("Saved -> detailed_player_stats.csv")
    print(final_df.head())
else:
    print("No detailed player tables found")

In [ ]:
import pandas as pd

df = pd.read_csv("detailed_player_stats.csv")

# Clean numeric columns
for col in ["PTS", "TRB", "AST", "STL", "BLK"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create Player Production Score
df["Production_Score"] = (
    df["PTS"] * 1.0 +
    df["TRB"] * 1.2 +
    df["AST"] * 1.5 +
    df["STL"] * 2.0 +
    df["BLK"] * 2.0
)

team_summary = df.groupby("Team")["Production_Score"].sum().reset_index()

team_summary.to_csv("team_production_score.csv", index=False)

print(team_summary.head())
print("Saved -> team_production_score.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("detailed_player_stats.csv")

# Convert numeric columns safely
for col in ["PTS", "TRB", "AST", "STL", "BLK"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create Production Score
df["Production_Score"] = (
    df["PTS"] * 1.0 +
    df["TRB"] * 1.2 +
    df["AST"] * 1.5 +
    df["STL"] * 2.0 +
    df["BLK"] * 2.0
)

# ✅ Group by Team_URL instead (since Team column doesn't exist)
team_summary = df.groupby("Team_URL")["Production_Score"].sum().reset_index()

team_summary.to_csv("team_production_score.csv", index=False)

print(team_summary.head())
print("Saved -> team_production_score.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("detailed_player_stats.csv")

# Extract team name from URL
df["Team"] = df["Team_URL"].str.extract(r"/schools/(.*?)/")[0]
df["Team"] = df["Team"].str.replace("-", " ").str.title()

# Convert numeric safely
for col in ["PTS", "TRB", "AST", "STL", "BLK"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Production Score
df["Production_Score"] = (
    df["PTS"] * 1.0 +
    df["TRB"] * 1.2 +
    df["AST"] * 1.5 +
    df["STL"] * 2.0 +
    df["BLK"] * 2.0
)

team_summary = df.groupby("Team")["Production_Score"].sum().reset_index()

team_summary.to_csv("team_production_score_clean.csv", index=False)

print(team_summary.head())
print("Saved -> team_production_score_clean.csv")

In [ ]:
import pandas as pd

# Load both metrics
power_df = pd.read_csv("team_power_team_level.csv")
production_df = pd.read_csv("team_production_score_clean.csv")

# Clean team names if needed
power_df["School"] = power_df["School"].astype(str)
production_df["Team"] = production_df["Team"].astype(str)

# Merge
merged = power_df.merge(
    production_df,
    left_on="School",
    right_on="Team",
    how="left"
)

# Fill missing production with 0
merged["Production_Score"] = merged["Production_Score"].fillna(0)

# Normalize production to 0-100 scale
merged["Production_Score_Norm"] = (
    merged["Production_Score"] - merged["Production_Score"].min()
) / (
    merged["Production_Score"].max() - merged["Production_Score"].min()
) * 100

# Final Composite Rating
merged["Final_Team_Rating"] = (
    merged["Team_Power_Score"] * 0.6 +
    merged["Production_Score_Norm"] * 0.4
)

merged = merged.sort_values("Final_Team_Rating", ascending=False)

merged.to_csv("final_team_rating_model.csv", index=False)

print(merged[["School", "Final_Team_Rating"]].head())
print("Saved -> final_team_rating_model.csv")

In [ ]:
import pandas as pd

ratings = pd.read_csv("final_team_rating_model.csv")

# Example matchup (you can change teams)
team_a = "Duke"
team_b = "Arizona"

a_rating = ratings.loc[ratings["School"] == team_a, "Final_Team_Rating"].values
b_rating = ratings.loc[ratings["School"] == team_b, "Final_Team_Rating"].values

if len(a_rating) > 0 and len(b_rating) > 0:
    diff = a_rating[0] - b_rating[0]

    # Convert rating difference to projected margin
    projected_margin = diff * 0.25  # scaling factor

    print("Matchup:", team_a, "vs", team_b)
    print("Rating Difference:", diff)
    print("Projected Margin:", projected_margin)
else:
    print("One of the teams not found in ratings.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

ratings = pd.read_csv("final_team_rating_model.csv")

team_a = "Duke"
team_b = "Arizona"

a_rating = ratings.loc[ratings["School"] == team_a, "Final_Team_Rating"].values
b_rating = ratings.loc[ratings["School"] == team_b, "Final_Team_Rating"].values

if len(a_rating) > 0 and len(b_rating) > 0:

    diff = a_rating[0] - b_rating[0]

    # Convert rating diff → projected margin
    projected_margin = diff * 0.25

    # Assume margin variance
    std_dev = 10

    # Win probability
    win_prob = 1 - norm.cdf(0, loc=projected_margin, scale=std_dev)

    print("Projected Margin:", projected_margin)
    print("Win Probability:", round(win_prob * 100, 2), "%")

else:
    print("Team not found")

In [ ]:
import numpy as np

# Example: Market spread from Odds API
market_spread = -8  # <-- Replace with real data from odds_df

projected_margin = 10.625

# Convert spread to implied probability (rough normal approximation)
std_dev = 10

market_win_prob = 1 - np.exp(-(projected_margin - market_spread)**2 / (2 * std_dev**2))

model_prob = 0.856

edge = model_prob - market_win_prob

print("Market Win Prob:", round(market_win_prob, 4))
print("Model Win Prob:", model_prob)
print("Edge:", round(edge, 4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

ratings = pd.read_csv("final_team_rating_model.csv")

results = []

for _, row in odds_df.iterrows():

    game = row["game"]
    team1, team2 = game.split(" vs ")

    a = ratings.loc[ratings["School"] == team1, "Final_Team_Rating"]
    b = ratings.loc[ratings["School"] == team2, "Final_Team_Rating"]

    if len(a) == 0 or len(b) == 0:
        continue

    diff = a.values[0] - b.values[0]
    projected_margin = diff * 0.25
    std_dev = 10

    model_prob = 1 - norm.cdf(0, loc=projected_margin, scale=std_dev)

    # Market implied prob (from odds_df if available)
    market_prob = 0.5  # placeholder — we improve this next

    edge = model_prob - market_prob

    results.append({
        "Game": game,
        "Model_Prob": model_prob,
        "Edge": edge
    })

edge_df = pd.DataFrame(results)
edge_df = edge_df.sort_values("Edge", ascending=False)

print(edge_df.head(10))
edge_df.to_csv("edge_scan_results.csv", index=False)

print("Saved -> edge_scan_results.csv")

In [ ]:
import pandas as pd

odds_rows = []

for game in data:
    game_name = game.get("home_team") + " vs " + game.get("away_team")

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            for outcome in market.get("outcomes", []):
                odds_rows.append({
                    "game": game_name,
                    "market": market["key"],
                    "outcome": outcome.get("name"),
                    "price": outcome.get("price"),
                    "point": outcome.get("point")
                })

odds_df = pd.DataFrame(odds_rows)

print("odds_df recreated")
print(odds_df.head())

In [ ]:
print(type(data))
print(data)

In [ ]:
response = requests.get(url, params=params)

print("Status:", response.status_code)

data = response.json()

print("Type of data:", type(data))
print("Games pulled:", len(data))

In [ ]:
def decimal_to_american(decimal_odds):
    if decimal_odds >= 2:
        return round((decimal_odds - 1) * 100)
    else:
        return round(-100 / (decimal_odds - 1))


print("🔥 Extracting Spreads + Totals (Safe Version)\n")

for game in data:
    print("====================================")

    try:
        print(game["home_team"], "vs", game["away_team"])
    except:
        print("Game name not found")

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):

            if market["key"] == "spreads":
                for outcome in market.get("outcomes", []):
                    american = decimal_to_american(outcome["price"])

                    print("Spread:",
                          outcome["name"],
                          outcome["point"],
                          "| American:",
                          american)

            if market["key"] == "totals":
                for outcome in market.get("outcomes", []):
                    american = decimal_to_american(outcome["price"])

                    print("Total:",
                          outcome["name"],
                          outcome["point"],
                          "| American:",
                          american)

    print("\n")

In [ ]:
import math

def spread_to_probability(spread):
    """
    Rough conversion:
    Uses logistic approximation to convert spread -> win prob
    """
    return 1 / (1 + math.exp(-0.2 * spread))


results = []

print("🔥 Calculating Model Edge vs Market\n")

for game in data:

    home = game["home_team"]
    away = game["away_team"]

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):

            if market["key"] == "spreads":

                for outcome in market.get("outcomes", []):

                    team = outcome["name"]
                    point = outcome["point"]
                    price = outcome["price"]

                    market_prob = 1 / price if price > 0 else 0

                    model_prob = spread_to_probability(point)

                    edge = model_prob - market_prob

                    print("Game:", home, "vs", away)
                    print("Team:", team)
                    print("Spread:", point)
                    print("Market Prob:", round(market_prob,4))
                    print("Model Prob:", round(model_prob,4))
                    print("Edge:", round(edge,4))
                    print("====================================\n")

                    results.append({
                        "game": f"{home} vs {away}",
                        "team": team,
                        "spread": point,
                        "market_prob": market_prob,
                        "model_prob": model_prob,
                        "edge": edge
                    })

In [ ]:
from collections import defaultdict

clean_results = defaultdict(lambda: {"best_edge": -999})

print("🔥 Aggregating Best Spread Opportunities\n")

for game in data:

    home = game["home_team"]
    away = game["away_team"]

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):

            if market["key"] != "spreads":
                continue

            for outcome in market.get("outcomes", []):

                team = outcome["name"]
                point = outcome["point"]
                price = outcome["price"]

                market_prob = 1 / price if price > 0 else 0
                model_prob = 1 / (1 + __import__("math").exp(-0.2 * point))
                edge = model_prob - market_prob

                key = (home, away, team)

                if edge > clean_results[key]["best_edge"]:
                    clean_results[key] = {
                        "spread": point,
                        "price": price,
                        "edge": edge,
                        "model_prob": model_prob,
                        "market_prob": market_prob
                    }

# Print Only Best Per Team
for key, val in clean_results.items():
    print("Game:", key[0], "vs", key[1])
    print("Team:", key[2])
    print("Best Spread:", val["spread"])
    print("Best Price:", val["price"])
    print("Best Edge:", round(val["edge"],4))
    print("====================================\n")

In [ ]:
from collections import defaultdict
import math

clean_results = {}

print("🔥 Aggregating Best Spread Opportunities\n")

for game in data:

    home = game.get("home_team")
    away = game.get("away_team")

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):

            if market["key"] != "spreads":
                continue

            for outcome in market.get("outcomes", []):

                team = outcome.get("name")
                point = outcome.get("point")
                price = outcome.get("price")

                if not price or not point:
                    continue

                market_prob = 1 / price
                model_prob = 1 / (1 + math.exp(-0.2 * point))
                edge = model_prob - market_prob

                key = (home, away, team, point)

                # Keep best edge per line
                if key not in clean_results or edge > clean_results[key]["edge"]:

                    clean_results[key] = {
                        "spread": point,
                        "price": price,
                        "edge": edge,
                        "model_prob": model_prob,
                        "market_prob": market_prob
                    }

# Print Clean Results
for key, val in clean_results.items():
    print("Game:", key[0], "vs", key[1])
    print("Team:", key[2])
    print("Spread:", val["spread"])
    print("Price:", val["price"])
    print("Edge:", round(val["edge"],4))
    print("====================================\n")

In [ ]:
print("🔥 FILTERED +EV SPREADS\n")

filtered = [
    (k, v) for k, v in clean_results.items()
    if v["edge"] > 0.05
]

# Sort by biggest edge
filtered = sorted(filtered, key=lambda x: x[1]["edge"], reverse=True)

for (home, away, team, spread), val in filtered:
    print("Game:", home, "vs", away)
    print("Team:", team)
    print("Spread:", spread)
    print("Edge:", round(val["edge"],4))
    print("Price:", val["price"])
    print("====================================\n")

In [ ]:
import pandas as pd
import numpy as np

# ==============================
# FILTER TOP +EV PLAYS
# ==============================

filtered = []

for game in data:
    home = game["home_team"]
    away = game["away_team"]

    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] == "spreads":

                for outcome in market["outcomes"]:
                    team = outcome["name"]
                    point = outcome["point"]
                    price = outcome["price"]

                    # Convert price to implied prob
                    implied_prob = 1 / price

                    # Your model probability must already exist
                    model_prob = 0.0  # <-- replace with your model probability if available

                    edge = model_prob - implied_prob

                    if edge > 0.15:

                        # Kelly Formula (Fractional Kelly = 0.5)
                        b = price - 1
                        q = model_prob
                        p = 1 - model_prob

                        if price > 1:
                            kelly = (b * q - p) / b
                            kelly = max(kelly, 0)
                            half_kelly = kelly * 0.5
                        else:
                            half_kelly = 0

                        filtered.append({
                            "Game": f"{home} vs {away}",
                            "Team": team,
                            "Spread": point,
                            "Price": price,
                            "Edge": edge,
                            "Half_Kelly_Units": round(half_kelly, 4)
                        })

df_top = pd.DataFrame(filtered)

print("🔥 TOP BETS WITH KELLY")
print(df_top)

df_top.to_csv("top_bets_kelly.csv", index=False)

print("\n✅ Saved -> top_bets_kelly.csv")

In [ ]:
import pandas as pd

# Load your model probabilities or ratings
model_df = pd.read_csv("final_team_rating_model.csv")

print(model_df.head())

# Build dictionary for fast lookup
model_prob_map = dict(zip(model_df["School"], model_df["Final_Team_Rating"]/100))

In [ ]:
import pandas as pd

# Load your rating file
model_df = pd.read_csv("final_team_rating_model.csv")

# Clean team column (important for matching)
model_df["School"] = model_df["School"].str.strip()

# Convert rating → probability (scaling to 0–1)
model_df["Model_Prob"] = model_df["Final_Team_Rating"] / 100.0

print("✅ Model Loaded")
print(model_df.head())

# Create lookup dictionary
model_prob_map = dict(zip(model_df["School"], model_df["Model_Prob"]))

In [ ]:
import pandas as pd

odds_df = pd.read_csv("clean_spread_data.csv")  # or whatever your odds file is called

results = []

for _, row in odds_df.iterrows():

    team = row["team"]
    implied_prob = row["implied_prob"]  # make sure this column exists

    model_prob = model_prob_map.get(team, None)

    if model_prob is None:
        continue

    edge = model_prob - implied_prob

    if edge > 0.05:  # keep only meaningful edges
        results.append({
            "Team": team,
            "Edge": edge,
            "Model_Prob": model_prob,
            "Market_Prob": implied_prob
        })

filtered_df = pd.DataFrame(results)

filtered_df = filtered_df.sort_values("Edge", ascending=False)

filtered_df.to_csv("top_kelly_candidates.csv", index=False)

print("🔥 Saved -> top_kelly_candidates.csv")
print(filtered_df.head(20))

In [ ]:
import os

print("Files in directory:")
print(os.listdir())

In [ ]:
import pandas as pd

for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

In [ ]:
import pandas as pd

# Load your final team ratings
ratings = pd.read_csv("team_ratings_output.csv")

# Load your odds file (replace if needed with correct file name)
odds = pd.read_csv("team_stats_snapshot.csv")

print("Ratings Shape:", ratings.shape)
print("Odds Shape:", odds.shape)

print("\nRatings Preview:")
print(ratings.head())

print("\nOdds Preview:")
print(odds.head())

In [ ]:
import pandas as pd

ratings = pd.read_csv("team_ratings_output.csv")
odds = pd.read_csv("team_stats_snapshot.csv")

# Merge on team name
merged = ratings.merge(odds, on="team", suffixes=("_model", "_market"))

# Calculate edge
merged["Edge"] = merged["Power_Score"] - merged["Power_Score_market"]

# Sort by strongest edge
merged_sorted = merged.sort_values(by="Edge", ascending=False)

print("🔥 Top Edge Opportunities")
print(merged_sorted[["team", "Power_Score", "Power_Score_market", "Edge"]])

merged_sorted.to_csv("edge_comparison.csv", index=False)
print("\n✅ Saved -> edge_comparison.csv")

In [ ]:
import pandas as pd

ratings = pd.read_csv("team_ratings_output.csv")
odds = pd.read_csv("team_stats_snapshot.csv")

print("Ratings Columns:")
print(ratings.columns)

print("\nOdds Columns:")
print(odds.columns)

# Merge
merged = ratings.merge(odds, on="team", suffixes=("_model", "_market"))

print("\nMerged Columns:")
print(merged.columns)

print("\nMerged Preview:")
print(merged.head())

merged.to_csv("merged_debug.csv", index=False)

print("\n✅ Saved -> merged_debug.csv")

In [ ]:
import pandas as pd

# Load merged debug
merged = pd.read_csv("merged_debug.csv")

# -----------------------------
# Create Market Power Score
# -----------------------------
# Example: Normalize winPercent_market to 0-100 scale

merged["Power_Score_market"] = merged["winPercent_market"] * 100


# -----------------------------
# Now Compute Edge Properly
# -----------------------------
merged["Edge"] = merged["Power_Score"] - merged["Power_Score_market"]

# Sort strongest edges
merged = merged.sort_values("Edge", ascending=False)

print("\n🔥 Top Edges")
print(merged[["team", "Power_Score", "Power_Score_market", "Edge"]].head(20))

merged.to_csv("final_edge_output.csv", index=False)

print("\n✅ Saved -> final_edge_output.csv")

In [ ]:
import pandas as pd

# Load merged file you already created
df = pd.read_csv("merged_debug.csv")

# Build a weighted market power score
df["Market_Power"] = (
    (df["winPercent_market"] * 50) +
    (df["pointDifferential_market"] / df["gamesPlayed_market"].replace(0,1)) * 20 +
    (df["avgPointsFor_market"] - df["avgPointsAgainst_market"]) * 10
)

# Normalize to 0–100
df["Market_Power"] = (
    (df["Market_Power"] - df["Market_Power"].min()) /
    (df["Market_Power"].max() - df["Market_Power"].min())
) * 100

df.to_csv("market_power_model.csv", index=False)

print("✅ Saved -> market_power_model.csv")
print(df[["team","Power_Score","Market_Power"]].head())

In [ ]:
import pandas as pd

df = pd.read_csv("market_power_model.csv")

df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

df = df.sort_values("True_Edge", ascending=False)

df.to_csv("true_edge_ranked.csv", index=False)

print("🔥 Top True Edge Bets")
print(df[["team","True_Edge"]].head(15))

In [ ]:
import pandas as pd

df = pd.read_csv("true_edge_ranked.csv")

# Filter strong positive edges
bets = df[df["True_Edge"] > 10]

# Optional: filter minimum win probability
bets = bets[bets["Power_Score"] > 55]

bets["Recommended_Stake"] = "0.5u"

bets.to_csv("filtered_bets_kelly_ready.csv", index=False)

print("🔥 FILTERED +EV BETS")
print(bets[["team","True_Edge","Power_Score","Market_Power"]])

In [ ]:
import pandas as pd

df = pd.read_csv("filtered_bets_kelly_ready.csv")

# Assume implied probability from market power
df["Implied_Prob"] = df["Market_Power"] / 100

# Convert model power to probability
df["Model_Prob"] = df["Power_Score"] / 100

# Kelly Formula
df["Kelly_Fraction"] = (
    (df["Model_Prob"] - (1 - df["Model_Prob"]) / (1/df["Implied_Prob"] - 1))
)

df["Kelly_Fraction"] = df["Kelly_Fraction"].clip(lower=0)

df.to_csv("top_bets_kelly_final.csv", index=False)

print("🔥 KELLY BETS READY")
print(df[["team","Kelly_Fraction"]].head())

In [ ]:
merged_debug.csv

In [ ]:
import pandas as pd

df = pd.read_csv("merged_debug.csv")

print(df.head())
print("Rows:", len(df))

In [ ]:
import pandas as pd

df = pd.read_csv("merged_debug.csv")

# Recalculate edge safely
df["Market_Power"] = (
    (df["winPercent_market"] * 50) +
    ((df["pointsFor_market"] - df["pointsAgainst_market"]) / df["gamesPlayed_market"].replace(0,1)) * 20
)

df["Market_Power"] = (
    (df["Market_Power"] - df["Market_Power"].min()) /
    (df["Market_Power"].max() - df["Market_Power"].min())
) * 100

df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

# Filter strong plays
bets = df[df["True_Edge"] > 10]

bets = bets.sort_values("True_Edge", ascending=False)

print("🔥 FINAL LIVE BETS")
print(bets[["team","Power_Score","Market_Power","True_Edge"]])

In [ ]:
bets.to_csv("live_model_results.csv", index=False)
print("✅ Saved -> live_model_results.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print("Shape:", df.shape)
print(df.sort_values("Edge", ascending=False).head(20))

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print(df.columns)
print("\nRaw contents:")
print(df)

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print("Min True_Edge:", df["True_Edge"].min())
print("Max True_Edge:", df["True_Edge"].max())
print("\nTop 20 True_Edge:")
print(df.sort_values("True_Edge", ascending=False).head(20))

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print(df[["Power_Score", "Market_Power"]].head(10))
print("\nNull Counts:")
print(df[["Power_Score", "Market_Power"]].isna().sum())

In [ ]:
df["Market_Power"] = (
    df.filter(like="_market")
    .select_dtypes(include="number")
    .mean(axis=1)
)

df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

print("Fixed True_Edge Stats:")
print(df["True_Edge"].describe())

df.to_csv("live_model_results_fixed.csv", index=False)
print("✅ Saved -> live_model_results_fixed.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results_fixed.csv")

# Check numeric columns
num_cols = df.select_dtypes(include="number").columns
print("Numeric Columns Used:")
print(num_cols)

print("\nSample Values:")
print(df[num_cols].head())

print("\nAny Nulls?")
print(df[num_cols].isna().sum())

In [ ]:
# Convert market columns to numeric
market_cols = [col for col in df.columns if "_market" in col]

for col in market_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Recalculate Market Power
df["Market_Power"] = df[market_cols].mean(axis=1)

# Recalculate True Edge
df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

print(df[["Power_Score","Market_Power","True_Edge"]].head())

df.to_csv("live_model_results_final.csv", index=False)
print("✅ Saved -> live_model_results_final.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print("Rows:", len(df))
print(df.head())

In [ ]:
print("Ratings teams:")
print(ratings["team"].head(20))

print("\nOdds teams:")
print(odds["team"].head(20))

In [ ]:
print("Ratings team dtype:", ratings["team"].dtype)
print("Odds team dtype:", odds["team"].dtype)

print("\nUnique ratings teams:", ratings["team"].nunique())
print("Unique odds teams:", odds["team"].nunique())

print("\nCommon teams:", set(ratings["team"]).intersection(set(odds["team"])))

In [ ]:
ratings["team"] = ratings["team"].astype(str).str.strip()
odds["team"] = odds["team"].astype(str).str.strip()

ratings["team"] = ratings["team"].str.replace(r"\s+", " ", regex=True)
odds["team"] = odds["team"].str.replace(r"\s+", " ", regex=True)

In [ ]:
merged = ratings.merge(odds, on="team", how="inner", suffixes=("_model", "_market"))

print("Merged rows:", len(merged))
print(merged.head())

In [ ]:
# Compute Power Difference
merged["Market_Power"] = merged[[col for col in merged.columns if col.endswith("_market")]].mean(axis=1)

merged["True_Edge"] = merged["Power_Score"] - merged["Market_Power"]

# Sort strongest edges
final = merged.sort_values("True_Edge", ascending=False)

print("🔥 Top 10 Strongest Edges")
print(final[["team", "Power_Score", "Market_Power", "True_Edge"]].head(10))

final.to_csv("live_model_results_final.csv", index=False)

print("✅ Saved -> live_model_results_final.csv")

In [ ]:
import numpy as np

# Select only numeric market columns
market_cols = [col for col in merged.columns if col.endswith("_market")]

# Normalize market features (scale 0–100)
market_scaled = merged[market_cols].apply(
    lambda x: (x - x.min()) / (x.max() - x.min()) * 100
)

# Compute normalized market power
merged["Market_Power"] = market_scaled.mean(axis=1)

# Now compute real edge
merged["True_Edge"] = merged["Power_Score"] - merged["Market_Power"]

# Sort
final = merged.sort_values("True_Edge", ascending=False)

print("🔥 Cleaned Top 10")
print(final[["team","Power_Score","Market_Power","True_Edge"]].head(10))

final.to_csv("live_model_results_fixed2.csv", index=False)

print("✅ Saved -> live_model_results_fixed2.csv")

In [ ]:
if True_Edge >= 10:
    Signal = "STRONG BET"
elif True_Edge >= 5:
    Signal = "LEAN BET"
elif True_Edge > 0:
    Signal = "MARGINAL EDGE"
else:
    Signal = "NO BET"

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results_fixed2.csv")

# Create bet signal properly using the column
df["Bet_Signal"] = pd.cut(
    df["True_Edge"],
    bins=[-999, 0, 5, 10, 999],
    labels=["NO BET", "MARGINAL EDGE", "LEAN BET", "STRONG BET"]
)

df = df.sort_values("True_Edge", ascending=False)

df.to_csv("auto_bet_recommendations.csv", index=False)

print("🔥 Bet File Created")
print(df[["team","Power_Score","Market_Power","True_Edge","Bet_Signal"]].head(20))

In [ ]:
import pandas as pd

df = pd.read_csv("auto_bet_recommendations.csv")

# Keep only good bets
df = df[df["Bet_Signal"].isin(["STRONG BET", "LEAN BET"])]

df = df.sort_values("True_Edge", ascending=False)

df.to_csv("final_tradeable_bets.csv", index=False)

print("🔥 Final Tradeable Bets")
print(df)

In [ ]:
True_Edge = Power_Score - Market_Power

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

# Remove weak edges
df = df[df["True_Edge"] > 0]

# Keep only top 25% strongest edges
threshold = df["True_Edge"].quantile(0.75)
df = df[df["True_Edge"] >= threshold]

# Rank strongest first
df = df.sort_values("True_Edge", ascending=False)

# Keep top 10
top_bets = df.head(10)

print("🔥 FINAL TOP BETS")
print(top_bets[["team", "Power_Score", "Market_Power", "True_Edge"]])

top_bets.to_csv("final_top_bets.csv", index=False)

print("✅ Saved -> final_top_bets.csv")

In [ ]:
df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

# Calculate edge correctly
df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

print(df[["team","Power_Score","Market_Power","True_Edge"]].head())

In [ ]:
print(df.shape)
print(df.head())

In [ ]:
set(ratings["team"]) - set(odds["team"])

In [ ]:
set(odds["team"]) - set(ratings["team"])

In [ ]:
ratings["team"] = ratings["team"].str.strip().str.lower()
odds["team"] = odds["team"].str.strip().str.lower()

df = ratings.merge(odds, on="team", how="inner")

print(df.shape)

In [ ]:
df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

df = df.sort_values("True_Edge", ascending=False)

print(df[["team","Power_Score","Market_Power","True_Edge"]])

In [ ]:
print(df.columns)

In [ ]:
team
Power_Score
moneyline
spread
total

In [ ]:
import pandas as pd

df = pd.read_csv("merged_debug.csv")  # or your latest merged file

print(df.head())
print("\nColumns:\n")
print(df.columns)

df.to_csv("step1_debug_check.csv", index=False)
print("✅ Saved -> step1_debug_check.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("step1_debug_check.csv")

# Create Market Power from market win percentage
df["Market_Power"] = df["winPercent_market"] * 100

# Now calculate True Edge
df["True_Edge"] = df["Power_Score"] - df["Market_Power"]

# Sort strongest edge first
df = df.sort_values("True_Edge", ascending=False)

print("🔥 Top 10 Strongest Edges")
print(df[["team","Power_Score","Market_Power","True_Edge"]].head(10))

df.to_csv("final_top_bets.csv", index=False)

print("✅ Saved -> final_top_bets.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("final_top_bets.csv")

# 🔥 Filter only strong positive edges
df = df[df["True_Edge"] > 5]

# Sort strongest first
df = df.sort_values("True_Edge", ascending=False)

print("🔥 FINAL BET SIGNALS")
print(df[["team","Power_Score","Market_Power","True_Edge"]])

df.to_csv("filtered_bet_signals.csv", index=False)

print("✅ Saved -> filtered_bet_signals.csv")

In [ ]:
import pandas as pd

# Load your filtered signals
model_df = pd.read_csv("filtered_bet_signals.csv")

# Load your odds file (update filename if needed)
odds_df = pd.read_csv("clean_odds_data.csv")  # <-- change if different name

# Merge on team
merged = model_df.merge(odds_df, on="team", how="left")

# Convert moneyline to implied probability
def moneyline_to_prob(ml):
    if ml > 0:
        return 100 / (ml + 100)
    else:
        return -ml / (-ml + 100)

merged["Implied_Prob"] = merged["moneyline"].apply(moneyline_to_prob)

# Model probability from power score
merged["Model_Prob"] = merged["Power_Score"] / 100

# Calculate true EV
merged["True_EV"] = merged["Model_Prob"] - merged["Implied_Prob"]

merged = merged.sort_values("True_EV", ascending=False)

print("🔥 EV Ranked Bets")
print(merged[["team","True_EV","Model_Prob","Implied_Prob"]])

merged.to_csv("final_ev_bets.csv", index=False)

print("✅ Saved -> final_ev_bets.csv")

In [ ]:
import os

print("Files in directory:")
print(os.listdir())

In [ ]:
import pandas as pd

# Try loading the likely market file
df = pd.read_csv("live_model_results_final.csv")

print(df.head())
print(df.columns)

In [ ]:
import pandas as pd

# Load the file you just printed
df = pd.read_csv("live_model_results_final.csv")

# Remove extreme noise / invalid rows
df = df.dropna(subset=["True_Edge"])

# Sort strongest edges first
df = df.sort_values("True_Edge", ascending=False)

# Filter for positive + meaningful edges
best_bets = df[df["True_Edge"] > 5]  # adjust threshold later

print("🔥 TOP REAL BETS")
print(best_bets[["team", "Power_Score", "Market_Power", "True_Edge"]].head(20))

# Save it
best_bets.to_csv("final_strong_bets.csv", index=False)

print("✅ Saved -> final_strong_bets.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results_final.csv")

df = df.dropna(subset=["True_Edge"])

# Sort strongest edges
df = df.sort_values("True_Edge", ascending=False)

print("🔥 TOP 50 BY EDGE")
print(df[["team","Power_Score","Market_Power","True_Edge"]].head(50))

# Save raw ranked list
df.to_csv("ranked_all_edges.csv", index=False)

print("✅ Saved -> ranked_all_edges.csv")

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("live_model_results.csv")

# Normalize Power_Score and Market_Power
scaler = MinMaxScaler(feature_range=(0,100))

df[["Power_Score_norm","Market_Power_norm"]] = scaler.fit_transform(
    df[["Power_Score","Market_Power"]]
)

# Recalculate True Edge properly
df["True_Edge"] = df["Power_Score_norm"] - df["Market_Power_norm"]

# Sort
df = df.sort_values("True_Edge", ascending=False)

print("🔥 Normalized Top Bets")
print(df[["team","Power_Score_norm","Market_Power_norm","True_Edge"]].head(20))

df.to_csv("true_edge_ranked_normalized.csv", index=False)

print("✅ Saved -> true_edge_ranked_normalized.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("live_model_results.csv")

print("Shape:", df.shape)
print(df.head())

In [ ]:
if df.empty:
    print("🚨 DataFrame is empty. Fix upstream merge first.")
else:
    from sklearn.preprocessing import MinMaxScaler

    scaler = MinMaxScaler(feature_range=(0,100))

    df[["Power_Score_norm","Market_Power_norm"]] = scaler.fit_transform(
        df[["Power_Score","Market_Power"]]
    )

    df["True_Edge"] = df["Power_Score_norm"] - df["Market_Power_norm"]

    df = df.sort_values("True_Edge", ascending=False)

    print(df.head(20))

    df.to_csv("true_edge_ranked_normalized.csv", index=False)

    print("✅ Saved -> true_edge_ranked_normalized.csv")

In [ ]:
print("Ratings rows:", ratings_df.shape)
print("Odds rows:", odds_df.shape)

print("Ratings teams:", ratings_df["team"].unique()[:10])
print("Odds teams:", odds_df["team"].unique()[:10])

In [ ]:
import pandas as pd

ratings_df = pd.read_csv("team_power_team_level.csv")  # or your ratings file
odds_df = pd.read_csv("market_power_model.csv")        # or your market file

print("Ratings:", ratings_df.shape)
print("Odds:", odds_df.shape)

In [ ]:
ratings_df["team"] = ratings_df["team"].str.strip().str.lower()
odds_df["team"] = odds_df["team"].str.strip().str.lower()

In [ ]:
print("Ratings columns:")
print(ratings_df.columns)

print("\nOdds columns:")
print(odds_df.columns)

In [ ]:
# ==============================
# Fix Team Column Before Merge
# ==============================

# Rename School → team
ratings_df = ratings_df.rename(columns={"School": "team"})

# Normalize team names (match odds format)
ratings_df["team"] = ratings_df["team"].str.strip().str.lower()
odds_df["team"] = odds_df["team"].str.strip().str.lower()

print("Updated Ratings Columns:")
print(ratings_df.columns)

print("\nUpdated Odds Columns:")
print(odds_df.columns)

print("\nSample Teams:")
print(ratings_df["team"].head())
print(odds_df["team"].head())

In [ ]:
# ==============================
# Clean Team Names Further
# ==============================

def clean_team_name(x):
    x = x.replace("university", "")
    x = x.replace("state", "")
    x = x.replace("college", "")
    x = x.replace("wildcats", "")
    x = x.replace("tigers", "")
    x = x.replace("eagles", "")
    x = x.replace("bulldogs", "")
    x = x.replace("red", "")
    x = x.replace("blue", "")
    x = x.replace(" ", "")
    return x.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_team_name)
odds_df["team_clean"] = odds_df["team"].apply(clean_team_name)

print("Cleaned Sample:")
print(ratings_df[["team","team_clean"]].head())
print(odds_df[["team","team_clean"]].head())

In [ ]:
# ==============================
# Merge Cleaned DataFrames
# ==============================

merged = ratings_df.merge(
    odds_df,
    on="team_clean",
    suffixes=("_model","_market")
)

print("Merged Rows:", merged.shape[0])
print(merged.head())

merged.to_csv("merged_stage2.csv", index=False)
print("✅ Saved -> merged_stage2.csv")

In [ ]:
from fuzzywuzzy import process

# Create dictionary of market teams
market_teams = odds_df["team_clean"].tolist()

matched_rows = []

for _, row in ratings_df.iterrows():
    team = row["team_clean"]

    match, score = process.extractOne(team, market_teams)

    if score >= 80:  # threshold you can adjust
        market_row = odds_df[odds_df["team_clean"] == match].iloc[0]

        combined = {**row.to_dict(), **market_row.to_dict()}
        combined["fuzzy_score"] = score

        matched_rows.append(combined)

merged = pd.DataFrame(matched_rows)

print("Fuzzy Merged Rows:", merged.shape[0])
print(merged.head())

merged.to_csv("merged_fuzzy.csv", index=False)
print("✅ Saved -> merged_fuzzy.csv")

In [ ]:
!pip install fuzzywuzzy python-Levenshtein

In [ ]:
from rapidfuzz import process, fuzz

market_teams = odds_df["team_clean"].tolist()

matched_rows = []

for _, row in ratings_df.iterrows():
    team = row["team_clean"]

    match, score, _ = process.extractOne(
        team,
        market_teams,
        scorer=fuzz.ratio
    )

    if score >= 80:
        market_row = odds_df[odds_df["team_clean"] == match].iloc[0]
        combined = {**row.to_dict(), **market_row.to_dict()}
        combined["fuzzy_score"] = score
        matched_rows.append(combined)

merged = pd.DataFrame(matched_rows)

print("Merged Rows:", merged.shape)
merged.to_csv("merged_fuzzy.csv", index=False)
print("✅ Saved -> merged_fuzzy.csv")

In [ ]:
from rapidfuzz import process, fuzz

market_teams = odds_df["team_clean"].tolist()

matched_rows = []

for _, row in ratings_df.iterrows():
    team = row["team_clean"]

    match = process.extractOne(
        team,
        market_teams,
        scorer=fuzz.ratio
    )

    if match:
        best_match, score, _ = match

        market_row = odds_df[odds_df["team_clean"] == best_match].iloc[0]

        combined = {**row.to_dict(), **market_row.to_dict()}
        combined["fuzzy_score"] = score

        matched_rows.append(combined)

merged = pd.DataFrame(matched_rows)

print("Merged Rows:", merged.shape)
merged.to_csv("merged_fuzzy_v2.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("merged_fuzzy_v2.csv")

# Calculate True Edge
df["True_Edge"] = df["Team_Power_Score"] - df["Market_Power"]

# Sort strongest edges
df = df.sort_values("True_Edge", ascending=False)

print(df[["team", "Team_Power_Score", "Market_Power", "True_Edge"]].head(20))

df.to_csv("edge_ranked.csv", index=False)

In [ ]:
df_filtered = df[df["True_Edge"] > 10]
df_filtered = df_filtered.sort_values("True_Edge", ascending=False)

print(df_filtered)
df_filtered.to_csv("strong_bets_final.csv", index=False)

In [ ]:
# Aggregate by team and keep strongest edge

df_best = (
    df.sort_values("True_Edge", ascending=False)
      .groupby("team", as_index=False)
      .first()
)

df_best = df_best.sort_values("True_Edge", ascending=False)

print("🔥 Clean Top Bets")
print(df_best[["team","Team_Power_Score","Market_Power","True_Edge"]])

df_best.to_csv("clean_final_bets.csv", index=False)

In [ ]:
df = pd.read_csv("clean_final_bets.csv")

strong = df[df["True_Edge"] > 20]
medium = df[(df["True_Edge"] > 10) & (df["True_Edge"] <= 20)]

print("🔥 STRONG BETS")
print(strong[["team","True_Edge"]])

print("\n⚡ MEDIUM BETS")
print(medium[["team","True_Edge"]])

strong.to_csv("strong_bets.csv", index=False)
medium.to_csv("medium_bets.csv", index=False)

print("✅ Saved strong_bets.csv")
print("✅ Saved medium_bets.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("clean_final_bets.csv")

# Normalize edge into percentage
df["Edge_Percent"] = df["True_Edge"] / df["Market_Power"]

# Simple probability proxy
df["Implied_Probability"] = df["Power_Score"] / 100

# Kelly style unit sizing (scaled)
df["Suggested_Unit"] = df["Edge_Percent"].clip(lower=0) * 1.5

df = df.sort_values("True_Edge", ascending=False)

print(df[["team","True_Edge","Edge_Percent","Suggested_Unit"]])

df.to_csv("bet_engine_v2.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("clean_final_bets.csv")

# Prevent division by zero
df["Market_Power_safe"] = df["Market_Power"].replace(0, np.nan)

df["Edge_Percent"] = df["True_Edge"] / df["Market_Power_safe"]

# Replace NaN / inf with 0
df["Edge_Percent"] = df["Edge_Percent"].replace([np.inf, -np.inf], np.nan).fillna(0)

# Suggested unit = scaled edge
df["Suggested_Unit"] = (df["Edge_Percent"].clip(lower=0) * 2).round(2)

df = df.sort_values("True_Edge", ascending=False)

print(df[["team","True_Edge","Market_Power","Edge_Percent","Suggested_Unit"]])

df.to_csv("bet_engine_clean_v2.csv", index=False)

In [ ]:
Moneyline_Power
Spread_Power
Total_Power

In [ ]:
print(df.columns)

In [ ]:
import numpy as np

# Example placeholder — adjust based on your actual market data
df["Moneyline_Power"] = df["Power_Score"] * 0.4
df["Spread_Power"] = df["Power_Score"] * 0.35
df["Total_Power"] = df["Power_Score"] * 0.25

print(df[["team","Moneyline_Power","Spread_Power","Total_Power"]].head())

In [ ]:
import numpy as np

# Find best market per team
markets = ["Moneyline_Power","Spread_Power","Total_Power"]

df["Best_Market_Value"] = df[markets].max(axis=1)
df["Best_Bet_Type"] = df[markets].idxmax(axis=1)

# Clean up bet type names
df["Best_Bet_Type"] = df["Best_Bet_Type"].str.replace("_Power","")

# Suggested units based on strongest market
df["Suggested_Unit"] = (df["Best_Market_Value"] / 20).clip(lower=0)

df_sorted = df.sort_values("Best_Market_Value", ascending=False)

print(df_sorted[["team","Best_Bet_Type","Best_Market_Value","Suggested_Unit"]])

df_sorted.to_csv("auto_best_bets.csv", index=False)

In [ ]:
df["Suggested_Unit"] = (
    (df["Best_Market_Value"] / df["Best_Market_Value"].max()) * 5
).clip(0.25, 5)

In [ ]:
# ===============================
# 🚀 FULL MODEL RUN
# ===============================

print("🔥 Running Full Betting Model...")

# 1️⃣ Load Clean Ratings + Odds
ratings_df = pd.read_csv("team_ratings_output.csv")
odds_df = pd.read_csv("merged_fuzzy.csv")

# 2️⃣ Merge
df = ratings_df.merge(odds_df, on="team", how="inner")

# 3️⃣ Create Market Power Components
df["Moneyline_Power"] = df["Power_Score"] * 0.4
df["Spread_Power"] = df["Power_Score"] * 0.35
df["Total_Power"] = df["Power_Score"] * 0.25

# 4️⃣ Select Best Market Per Team
markets = ["Moneyline_Power","Spread_Power","Total_Power"]

df["Best_Market_Value"] = df[markets].max(axis=1)
df["Best_Bet_Type"] = df[markets].idxmax(axis=1).str.replace("_Power","")

# 5️⃣ Unit Sizing
df["Suggested_Unit"] = (
    (df["Best_Market_Value"] / df["Best_Market_Value"].max()) * 5
).clip(0.25,5)

# 6️⃣ Sort
df = df.sort_values("Best_Market_Value", ascending=False)

# 7️⃣ Save Output
df.to_csv("FINAL_MODEL_RUN.csv", index=False)

print("✅ MODEL COMPLETE")
print(df[["team","Best_Bet_Type","Best_Market_Value","Suggested_Unit"]])

In [ ]:
print(df.columns)

In [ ]:
df["Moneyline_Power"] = df["Team_Power_Score"] * 0.4

In [ ]:
# Find power column automatically
power_col = [col for col in df.columns if "Power" in col and "market" not in col.lower()]

print("Using power column:", power_col)

if len(power_col) > 0:
    base_power = power_col[0]
else:
    raise Exception("No Power column found!")

df["Moneyline_Power"] = df[base_power] * 0.4
df["Spread_Power"] = df[base_power] * 0.35
df["Total_Power"] = df[base_power] * 0.25

In [ ]:
# ===============================
# 🚀 FULL MODEL RUN
# ===============================

print("🔥 Running Full Betting Model...")

# 1️⃣ Load Clean Ratings + Odds
ratings_df = pd.read_csv("team_ratings_output.csv")
odds_df = pd.read_csv("merged_fuzzy.csv")

# 2️⃣ Merge
df = ratings_df.merge(odds_df, on="team", how="inner")

# 3️⃣ Create Market Power Components
df["Moneyline_Power"] = df["Power_Score"] * 0.4
df["Spread_Power"] = df["Power_Score"] * 0.35
df["Total_Power"] = df["Power_Score"] * 0.25

# 4️⃣ Select Best Market Per Team
markets = ["Moneyline_Power","Spread_Power","Total_Power"]

df["Best_Market_Value"] = df[markets].max(axis=1)
df["Best_Bet_Type"] = df[markets].idxmax(axis=1).str.replace("_Power","")

# 5️⃣ Unit Sizing
df["Suggested_Unit"] = (
    (df["Best_Market_Value"] / df["Best_Market_Value"].max()) * 5
).clip(0.25,5)

# 6️⃣ Sort
df = df.sort_values("Best_Market_Value", ascending=False)

# 7️⃣ Save Output
df.to_csv("FINAL_MODEL_RUN.csv", index=False)

print("✅ MODEL COMPLETE")
print(df[["team","Best_Bet_Type","Best_Market_Value","Suggested_Unit"]])

In [ ]:
print(df.columns)

In [ ]:
# Auto-detect power column
power_cols = [col for col in df.columns if "power" in col.lower() and "market" not in col.lower()]

print("Detected power columns:", power_cols)

if len(power_cols) == 0:
    raise Exception("No power column found!")

base_power = power_cols[0]

df["Moneyline_Power"] = df[base_power] * 0.4
df["Spread_Power"] = df[base_power] * 0.35
df["Total_Power"] = df[base_power] * 0.25

In [ ]:
print(df.columns)

In [ ]:
# ===============================
# 🚀 FULL MODEL RUN
# ===============================

print("🔥 Running Full Betting Model...")

# 1️⃣ Load Clean Ratings + Odds
ratings_df = pd.read_csv("team_ratings_output.csv")
odds_df = pd.read_csv("merged_fuzzy.csv")

# 2️⃣ Merge
df = ratings_df.merge(odds_df, on="team", how="inner")

# 3️⃣ Create Market Power Components
df["Moneyline_Power"] = df["Power_Score"] * 0.4
df["Spread_Power"] = df["Power_Score"] * 0.35
df["Total_Power"] = df["Power_Score"] * 0.25

# 4️⃣ Select Best Market Per Team
markets = ["Moneyline_Power","Spread_Power","Total_Power"]

df["Best_Market_Value"] = df[markets].max(axis=1)
df["Best_Bet_Type"] = df[markets].idxmax(axis=1).str.replace("_Power","")

# 5️⃣ Unit Sizing
df["Suggested_Unit"] = (
    (df["Best_Market_Value"] / df["Best_Market_Value"].max()) * 5
).clip(0.25,5)

# 6️⃣ Sort
df = df.sort_values("Best_Market_Value", ascending=False)

# 7️⃣ Save Output
df.to_csv("FINAL_MODEL_RUN.csv", index=False)

print("✅ MODEL COMPLETE")
print(df[["team","Best_Bet_Type","Best_Market_Value","Suggested_Unit"]])

In [ ]:
import pandas as pd
import numpy as np

print("🔥 EMERGENCY MODEL RUN")

# --------------------------------------------------
# 1️⃣ Load the most recent merged file automatically
# --------------------------------------------------

# Change this if needed — or point to your latest merge
df = pd.read_csv("merged_fuzzy.csv")

print("Columns detected:")
print(df.columns)

# --------------------------------------------------
# 2️⃣ Auto-detect Power column
# --------------------------------------------------

power_cols = [col for col in df.columns if "power" in col.lower() and "market" not in col.lower()]

print("Detected Power Columns:", power_cols)

if len(power_cols) == 0:
    raise Exception("❌ NO POWER COLUMN FOUND — Fix merge.")

base_power = power_cols[0]

# --------------------------------------------------
# 3️⃣ Auto-detect Market Power column
# --------------------------------------------------

market_cols = [col for col in df.columns if "market" in col.lower() and "power" in col.lower()]

if len(market_cols) == 0:
    raise Exception("❌ NO MARKET POWER COLUMN FOUND")

market_power = market_cols[0]

# --------------------------------------------------
# 4️⃣ Compute True Edge
# --------------------------------------------------

df["True_Edge"] = df[base_power] - df[market_power]

# --------------------------------------------------
# 5️⃣ Rank Bets
# --------------------------------------------------

df = df.sort_values("True_Edge", ascending=False)

df["Suggested_Unit"] = (df["True_Edge"] / df["True_Edge"].max() * 5).clip(0.25, 5)

# Keep only strong positive edges
bets = df[df["True_Edge"] > 0]

print("🔥 FINAL BACKED BETS")
print(bets[["team", base_power, market_power, "True_Edge", "Suggested_Unit"]])

bets.to_csv("LIVE_BACKED_BETS.csv", index=False)

print("✅ Saved -> LIVE_BACKED_BETS.csv")

In [ ]:
import pandas as pd
import numpy as np

print("🔥 NORMALIZED MODEL RUN")

df = pd.read_csv("merged_fuzzy.csv")

# Detect power column
power_cols = [col for col in df.columns if "power" in col.lower() and "market" not in col.lower()]
market_cols = [col for col in df.columns if "market_power" in col.lower() or col == "Market_Power"]

if len(power_cols) == 0 or len(market_cols) == 0:
    raise Exception("Missing required columns")

base_power = power_cols[0]
market_power = market_cols[0]

# ---------------------------
# Normalize both metrics
# ---------------------------

df["Power_Norm"] = (df[base_power] - df[base_power].min()) / (
    df[base_power].max() - df[base_power].min() + 1e-9
)

df["Market_Norm"] = (df[market_power] - df[market_power].min()) / (
    df[market_power].max() - df[market_power].min() + 1e-9
)

# True Edge = Normalized Difference
df["True_Edge"] = df["Power_Norm"] - df["Market_Norm"]

# Rank
df = df.sort_values("True_Edge", ascending=False)

# Only positive edges
bets = df[df["True_Edge"] > 0]

bets["Suggested_Unit"] = (bets["True_Edge"] / bets["True_Edge"].max() * 5).clip(0.25,5)

print("🔥 BACKED BETS")
print(bets[["team", "True_Edge", "Suggested_Unit"]])

bets.to_csv("LIVE_BACKED_BETS.csv", index=False)

print("✅ Saved -> LIVE_BACKED_BETS.csv")

In [ ]:
import pandas as pd

print("🔥 FORCE BET EXTRACTION")

df = pd.read_csv("merged_fuzzy.csv")

# Detect power + market power
power_cols = [c for c in df.columns if "power" in c.lower() and "market" not in c.lower()]
market_cols = [c for c in df.columns if "market_power" in c.lower() or c == "Market_Power"]

if not power_cols or not market_cols:
    raise Exception("Missing power columns")

power_col = power_cols[0]
market_col = market_cols[0]

# Raw edge (no normalization)
df["True_Edge"] = df[power_col] - df[market_col]

# RELAXED filter — allow small positive edge
bets = df[df["True_Edge"] > 0]

# If nothing passes — lower threshold slightly
if bets.empty:
    print("⚠ No positive edges. Relaxing threshold to > -5")
    bets = df[df["True_Edge"] > -5]

# Rank strongest first
bets = bets.sort_values("True_Edge", ascending=False)

# Assign aggressive units
bets["Suggested_Unit"] = (
    (bets["True_Edge"] / bets["True_Edge"].abs().max()) * 3
).abs().clip(0.5, 3)

print("🔥 TODAY'S TRADEABLE BETS")
print(bets[["team", power_col, market_col, "True_Edge", "Suggested_Unit"]])

bets.to_csv("FORCED_BETS_NOW.csv", index=False)

print("✅ Saved -> FORCED_BETS_NOW.csv")

In [ ]:
True_Edge = Power_Score - Market_Power

In [ ]:
df["Model_Prob"] = 1 / (1 + np.exp(- (df[power_col] - df[power_col].mean()) / df[power_col].std()))

In [ ]:
df["Market_Prob"] = 1 / (1 + np.exp(- (df[market_col] - df[market_col].mean()) / df[market_col].std()))

In [ ]:
df["True_Edge"] = df["Model_Prob"] - df["Market_Prob"]

In [ ]:
df["Kelly_Fraction"] = df["True_Edge"] / (1 - df["Market_Prob"])
df["Suggested_Unit"] = (df["Kelly_Fraction"] * 5).clip(0.25, 5)

In [ ]:
bets = df[
    (df["True_Edge"] > 0.03) &   # minimum 3% edge
    (df["Model_Prob"] > 0.45)    # avoid extreme garbage probabilities
]

In [ ]:
bets = bets.sort_values("True_Edge", ascending=False)
bets = bets.drop_duplicates(subset=["team"], keep="first")

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("merged_fuzzy.csv")

# Rebuild probabilities using your new method
power_cols = [c for c in df.columns if "power" in c.lower() and "market" not in c.lower()]
market_cols = [c for c in df.columns if "market_power" in c.lower() or c == "Market_Power"]

power_col = power_cols[0]
market_col = market_cols[0]

df["Model_Prob"] = 1 / (1 + np.exp(- (df[power_col] - df[power_col].mean()) / df[power_col].std()))
df["Market_Prob"] = 1 / (1 + np.exp(- (df[market_col] - df[market_col].mean()) / df[market_col].std()))
df["True_Edge"] = df["Model_Prob"] - df["Market_Prob"]

print("🔥 Model Stats")

print("Average Edge:", df["True_Edge"].mean())
print("Positive Edge %:", (df["True_Edge"] > 0).mean() * 100)

print("\nTop 10 Edges:")
print(df.sort_values("True_Edge", ascending=False)[["team","True_Edge"]].head(10))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("merged_fuzzy.csv")

power_cols = [c for c in df.columns if "power" in c.lower() and "market" not in c.lower()]
market_cols = [c for c in df.columns if "market_power" in c.lower() or c == "Market_Power"]

power_col = power_cols[0]
market_col = market_cols[0]

# ---- FIX 1: Drop NaNs first ----
df = df.dropna(subset=[power_col, market_col])

# ---- FIX 2: Add small constant to std to prevent division by zero ----
power_std = df[power_col].std() + 1e-6
market_std = df[market_col].std() + 1e-6

df["Model_Prob"] = 1 / (1 + np.exp(- (df[power_col] - df[power_col].mean()) / power_std))
df["Market_Prob"] = 1 / (1 + np.exp(- (df[market_col] - df[market_col].mean()) / market_std))

df["True_Edge"] = df["Model_Prob"] - df["Market_Prob"]

print("🔥 FIXED MODEL STATS")
print("Average Edge:", df["True_Edge"].mean())
print("Positive Edge %:", (df["True_Edge"] > 0).mean() * 100)

print("\nTop 10:")
print(df.sort_values("True_Edge", ascending=False)[["team","True_Edge"]].head(10))

In [ ]:
import pandas as pd

df = pd.read_csv("merged_fuzzy.csv")

print("Rows:", df.shape)
print(df.dtypes)

print("\nNull Count:")
print(df.isna().sum())

print("\nPower Columns:")
print([c for c in df.columns if "power" in c.lower()])

In [ ]:
merged = pd.merge(
    ratings_df,
    odds_df,
    on="team_clean",
    how="outer"
)

print("Merged Rows:", merged.shape)

In [ ]:
print("Ratings Columns:")
print(ratings_df.columns)

print("\nOdds Columns:")
print(odds_df.columns)

In [ ]:
ratings_df = ratings_df.rename(columns={"Power_Score": "Power_Score_model"})
odds_df = odds_df.rename(columns={"Power_Score": "Power_Score_market"})

In [ ]:
ratings_df["team_clean"] = ratings_df["team"].str.lower().str.strip()
odds_df["team_clean"] = odds_df["team"].str.lower().str.strip()

In [ ]:
merged = pd.merge(
    ratings_df,
    odds_df,
    on="team_clean",
    how="inner"
)

print("Merged Rows:", merged.shape)
print(merged.head())

In [ ]:
merged["True_Edge"] = (
    merged["Power_Score_model"] - merged["Market_Power"]
)

In [ ]:
merged = merged.sort_values("True_Edge", ascending=False)

In [ ]:
print("Ratings Teams:")
print(ratings_df["team_clean"].unique())

print("Odds Teams:")
print(odds_df["team_clean"].unique())

In [ ]:
print("Unmatched Teams:")
print(set(ratings_df["team_clean"]) - set(odds_df["team_clean"]))

In [ ]:
from rapidfuzz import process, fuzz

# Build clean name list from odds
odds_names = odds_df["team_clean"].tolist()

def best_match(team):
    match = process.extractOne(team, odds_names, scorer=fuzz.token_sort_ratio)

    if match and match[1] >= 70:  # Lower threshold = better coverage
        return match[0]
    return None

ratings_df["matched_team"] = ratings_df["team_clean"].apply(best_match)

print("Matched Teams:")
print(ratings_df["matched_team"].notna().sum())

In [ ]:
merged = ratings_df.merge(
    odds_df,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("Merged Rows:", merged.shape)

In [ ]:
import re

def clean_team_name(name):
    name = str(name).lower()
    name = re.sub(r"[^\w\s]", "", name)  # remove punctuation
    name = re.sub(r"\s+", " ", name)     # normalize spaces
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_team_name)
odds_df["team_clean"] = odds_df["team"].apply(clean_team_name)

print("Cleaned teams ready")
print(ratings_df["team_clean"].head())
print(odds_df["team_clean"].head())

In [ ]:
from rapidfuzz import process, fuzz

odds_names = odds_df["team_clean"].tolist()

def best_match(team):
    match = process.extractOne(team, odds_names, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 60:  # lower threshold = better coverage
        return match[0]
    return None

ratings_df["matched_team"] = ratings_df["team_clean"].apply(best_match)

print("Matched teams:")
print(ratings_df["matched_team"].notna().sum())

In [ ]:
merged = ratings_df.merge(
    odds_df,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("Merged Rows:", merged.shape)

In [ ]:
from rapidfuzz import process, fuzz

# Force fresh cleaning again
import re

def clean_team_name(name):
    name = str(name).lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_team_name)
odds_df["team_clean"] = odds_df["team"].apply(clean_team_name)

# --- AUTO MATCH (NO THRESHOLD TO EDIT) ---
odds_names = odds_df["team_clean"].tolist()

matches = []

for team in ratings_df["team_clean"]:
    match = process.extractOne(team, odds_names, scorer=fuzz.token_set_ratio)

    if match:
        matches.append(match[0])
    else:
        matches.append(None)

ratings_df["matched_team"] = matches

# --- MERGE ---
merged = ratings_df.merge(
    odds_df,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("🔥 Merged Rows:", merged.shape)
print(merged.head())

merged.to_csv("merged_auto_fixed.csv", index=False)
print("✅ Saved -> merged_auto_fixed.csv")

In [ ]:
# Calculate True Edge
merged["True_Edge"] = merged["Power_Score"] - merged["Market_Power"]

# Sort strongest edges
merged = merged.sort_values("True_Edge", ascending=False)

print("🔥 Top Edges")
print(merged[["team_x","Power_Score","Market_Power","True_Edge"]].head(20))

merged.to_csv("EDGE_ENGINE_OUTPUT.csv", index=False)
print("✅ Saved -> EDGE_ENGINE_OUTPUT.csv")

In [ ]:
print(merged.columns)

In [ ]:
# Detect power columns automatically
power_cols = [c for c in merged.columns if "Power_Score" in c]
market_cols = [c for c in merged.columns if "Market_Power" in c]

print("Detected Power Columns:", power_cols)
print("Detected Market Columns:", market_cols)

if power_cols and market_cols:

    merged["True_Edge"] = merged[power_cols[0]] - merged[market_cols[0]]

    merged = merged.sort_values("True_Edge", ascending=False)

    print("🔥 TOP 20 EDGES")
    print(merged[["team_x", power_cols[0], market_cols[0], "True_Edge"]].head(20))

    merged.to_csv("EDGE_ENGINE_OUTPUT.csv", index=False)
    print("✅ Saved -> EDGE_ENGINE_OUTPUT.csv")

else:
    print("🚨 Could not detect required columns.")

In [ ]:
print(merged["Market_Power"].describe())

In [ ]:
print(merged["Market_Power"].unique()[:20])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0,100))

merged["Market_Power"] = scaler.fit_transform(
    merged[["Power_Score_market"]]
)

In [ ]:
print(merged["Market_Power"].describe())
print(merged["Market_Power"].head())

In [ ]:
print(merged["team_y"])

In [ ]:
print("Ratings teams:", len(ratings_df))
print("Odds teams:", len(odds_df))

In [ ]:
import os

os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
print(os.getenv("ODDS_API_KEY"))

In [ ]:
import requests
import pandas as pd
import os

API_KEY = os.getenv("ODDS_API_KEY")
SPORT = "basketball_nba"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

data = response.json()

rows = []

for event in data:
    for bookmaker in event["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"]
                    })

odds_nba = pd.DataFrame(rows)

print("NBA Odds Rows:", len(odds_nba))
print(odds_nba.head())

In [ ]:
# Convert odds to numeric just in case
odds_nba["odds"] = pd.to_numeric(odds_nba["odds"])

# Take average line per team
odds_clean = (
    odds_nba
    .groupby("team", as_index=False)
    .agg(avg_odds=("odds", "mean"))
)

print("Unique Teams:", len(odds_clean))
print(odds_clean.head())

In [ ]:
# See ratings team names
print(sorted(ratings_df["team"].unique()))

# See odds team names
print(sorted(odds_clean["team"].unique()))

In [ ]:
merged = ratings_df.merge(
    odds_clean,
    on="team",
    how="inner"
)

print("Merged rows:", len(merged))
print(merged.head())

In [ ]:
import requests
import pandas as pd

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"
sport = "basketball_ncaab"

url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

data = response.json()

In [ ]:
rows = []

for game in data:
    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            for outcome in market["outcomes"]:
                rows.append({
                    "team": outcome["name"],
                    "odds": outcome["price"]
                })

odds_df = pd.DataFrame(rows)

# Average odds per team
odds_clean = (
    odds_df.groupby("team")["odds"]
    .mean()
    .reset_index()
    .rename(columns={"odds": "avg_odds"})
)

print("Unique Teams:", len(odds_clean))
print(odds_clean.head())

In [ ]:
merged = ratings_df.merge(
    odds_clean,
    on="team",
    how="inner"
)

print("Merged rows:", len(merged))

In [ ]:
params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "dateFormat": "iso"
}

response = requests.get(url, params=params)
data = response.json()

# Filter for tomorrow
target_date = "2026-02-25"

rows = []

for game in data:
    if target_date in game["commence_time"]:
        for bookmaker in game["bookmakers"]:
            for market in bookmaker["markets"]:
                if market["key"] == "h2h":
                    for outcome in market["outcomes"]:
                        rows.append({
                            "team": outcome["name"],
                            "odds": outcome["price"]
                        })

odds_df = pd.DataFrame(rows)

print("Filtered Teams:", len(odds_df))
print(odds_df.head())

In [ ]:
odds_clean = odds_df.groupby("team")["odds"].mean().reset_index()

In [ ]:
print(type(ratings_df))
print(ratings_df.head())

In [ ]:
from rapidfuzz import process, fuzz

# Clean names again
import re

def clean_name(name):
    name = str(name).lower()
    name = re.sub(r"university", "", name)
    name = re.sub(r"state", "st", name)
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_name)
odds_clean["team_clean"] = odds_clean["team"].apply(clean_name)

odds_pool = odds_clean["team_clean"].tolist()
used_matches = set()

matched = []

for team in ratings_df["team_clean"]:
    candidates = [t for t in odds_pool if t not in used_matches]

    if not candidates:
        matched.append(None)
        continue

    match = process.extractOne(
        team,
        candidates,
        scorer=fuzz.token_set_ratio
    )

    if match and match[1] >= 55:
        best = match[0]
        matched.append(best)
        used_matches.add(best)
    else:
        matched.append(None)

ratings_df["matched_team"] = matched

print("Unique Matches:", ratings_df["matched_team"].nunique())
print(ratings_df[["team_clean","matched_team"]])

In [ ]:
merged = ratings_df.merge(
    odds_clean,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("Merged Rows:", len(merged))

In [ ]:
merged["Market_Power"] = merged["avg_odds"]  # convert odds to market metric

# Normalize odds so that lower (favorite) = higher market pressure
merged["Market_Power"] = 100 - ((merged["Market_Power"] - merged["Market_Power"].min()) /
                               (merged["Market_Power"].max() - merged["Market_Power"].min() + 1e-9) * 100)

merged["True_Edge"] = merged["Power_Score_model"] - merged["Market_Power"]

merged = merged.sort_values("True_Edge", ascending=False)

print(merged[["team","Power_Score_model","Market_Power","True_Edge"]])

In [ ]:
print(merged.columns)

In [ ]:
# Detect odds column automatically
odds_col = [c for c in merged.columns if "odds" in c.lower()]

print("Detected Odds Columns:", odds_col)

if odds_col:
    merged["Market_Power"] = merged[odds_col[0]]

    # Normalize
    min_val = merged["Market_Power"].min()
    max_val = merged["Market_Power"].max()

    merged["Market_Power"] = 100 - (
        (merged["Market_Power"] - min_val) /
        (max_val - min_val + 1e-9) * 100
    )

    merged["True_Edge"] = merged["Power_Score_model"] - merged["Market_Power"]

    merged = merged.sort_values("True_Edge", ascending=False)

    print(merged[["team","Power_Score_model","Market_Power","True_Edge"]].head(20))
else:
    print("🚨 No odds column found.")

In [ ]:
print("Columns in merged:")
print(merged.columns)

In [ ]:
# Detect team column automatically
team_cols = [c for c in merged.columns if "team" in c.lower()]

print("Detected team columns:", team_cols)

if team_cols:
    team_col = team_cols[0]

    # Sort by edge
    merged = merged.sort_values("True_Edge", ascending=False)

    print(
        merged[[team_col, "Power_Score_model", "Market_Power", "True_Edge"]]
        .head(20)
    )
else:
    print("🚨 No team column found.")

In [ ]:
# Keep only strong positive edges
bet_df = merged[merged["True_Edge"] > 3].copy()

# Calculate Suggested Unit (scaled by edge strength)
bet_df["Suggested_Unit"] = bet_df["True_Edge"] / bet_df["True_Edge"].max() * 5

bet_df = bet_df.sort_values("True_Edge", ascending=False)

print("🔥 FINAL BETS")
print(bet_df[[team_cols[0], "True_Edge", "Suggested_Unit"]])

bet_df.to_csv("NCAAM_TOMORROW_BETS.csv", index=False)
print("✅ Saved -> NCAAM_TOMORROW_BETS.csv")

In [ ]:
import requests
import pandas as pd
import os

API_KEY = os.getenv("ODDS_API_KEY")
sport = "basketball_ncaab"

target_date = "2026-02-25T00:00:00Z"  # 🔥 Force tomorrow

url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": "2026-02-25T00:00:00Z",
    "commenceTimeTo": "2026-02-26T00:00:00Z"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

data = response.json()

rows = []

for game in data:
    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print("Filtered Tomorrow Teams:", len(odds_df))
print(odds_df.head())

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timezone
import os

API_KEY = os.getenv("ODDS_API_KEY")
sport = "basketball_ncaab"

url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": "2026-02-25T00:00:00Z",
    "commenceTimeTo": "2026-02-26T00:00:00Z"
}

response = requests.get(url, params=params)
data = response.json()

now = datetime.now(timezone.utc)

rows = []

for game in data:

    # 🔥 Skip games that already started
    commence = datetime.fromisoformat(game["commence_time"].replace("Z", "+00:00"))

    if commence < now:
        continue

    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print("✅ Clean Tomorrow Teams (No Live Games):", len(odds_df))
print(odds_df.head())

In [ ]:
# Convert odds to numeric
odds_df["odds"] = pd.to_numeric(odds_df["odds"])

# Average odds per team
odds_clean = (
    odds_df.groupby("team", as_index=False)
    .agg(avg_odds=("odds", "mean"))
)

print("Unique Teams After Collapse:", len(odds_clean))
print(odds_clean.head(10))

In [ ]:
from rapidfuzz import process, fuzz

# Clean names again
import re

def clean_name(name):
    name = str(name).lower()
    name = re.sub(r"university", "", name)
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_name)
odds_clean["team_clean"] = odds_clean["team"].apply(clean_name)

odds_pool = odds_clean["team_clean"].tolist()
used = set()

matches = []

for team in ratings_df["team_clean"]:
    candidates = [t for t in odds_pool if t not in used]

    if not candidates:
        matches.append(None)
        continue

    match = process.extractOne(
        team,
        candidates,
        scorer=fuzz.token_set_ratio
    )

    if match and match[1] >= 55:
        best = match[0]
        matches.append(best)
        used.add(best)
    else:
        matches.append(None)

ratings_df["matched_team"] = matches

merged = ratings_df.merge(
    odds_clean,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("🔥 Final Merged Rows:", len(merged))
print(merged.head())

In [ ]:
from rapidfuzz import process, fuzz
import re

def clean(name):
    name = str(name).lower()
    name = re.sub(r"university", "", name)
    name = re.sub(r"college", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean)
odds_clean["team_clean"] = odds_clean["team"].apply(clean)

odds_pool = odds_clean["team_clean"].tolist()

matches = []
used = set()

for team in ratings_df["team_clean"]:

    candidates = [c for c in odds_pool if c not in used]

    if not candidates:
        matches.append(None)
        continue

    match = process.extractOne(
        team,
        candidates,
        scorer=fuzz.token_sort_ratio
    )

    if match and match[1] >= 70:  # 🔥 Raise threshold for accuracy
        best = match[0]
        matches.append(best)
        used.add(best)
    else:
        matches.append(None)

ratings_df["matched_team"] = matches

merged = ratings_df.merge(
    odds_clean,
    left_on="matched_team",
    right_on="team_clean",
    how="inner"
)

print("🔥 Improved Merge Rows:", len(merged))
print(merged[[ "team_x", "Power_Score_model", "avg_odds"]])

In [ ]:
master_teams = odds_clean["team"].unique()

In [ ]:
print("Tomorrow Teams:", len(odds_clean))
print(odds_clean["team"].unique())

In [ ]:
# Use tomorrow teams as master
master_teams = odds_clean["team"].unique()

# Filter ratings to ONLY those teams
filtered_ratings = ratings_df[ratings_df["team"].isin(master_teams)].copy()

print("Filtered Ratings Rows:", len(filtered_ratings))
print(filtered_ratings[["team"]])

In [ ]:
print(ratings_df["team"].unique())

In [ ]:
print("Unique Ratings Teams:")
print(ratings_df["team"].unique())

In [ ]:
# Use only teams that exist in tomorrow slate
tomorrow_teams = odds_clean["team"].unique()

model_df = ratings_df[ratings_df["team"].isin(tomorrow_teams)].copy()

# Simple improved power formula
model_df["Power_Score_model"] = (
    model_df["winPercent"] * 50 +
    model_df["differential"] * 2 +
    model_df["pointsFor"] * 0.1 -
    model_df["pointsAgainst"] * 0.1
)

print("Rebuilt Power Rows:", len(model_df))
print(model_df[["team","Power_Score_model"]].head())

In [ ]:
SLATE_DATE = "2026-02-25"  # 🔥 Change this for each slate

In [ ]:
import requests
from datetime import datetime, timedelta
import pandas as pd
import os

API_KEY = os.getenv("ODDS_API_KEY")
sport = "basketball_ncaab"

start_time = f"{SLATE_DATE}T00:00:00Z"

end_date = (
    datetime.strptime(SLATE_DATE, "%Y-%m-%d")
    + timedelta(days=1)
).strftime("%Y-%m-%d")

end_time = f"{end_date}T00:00:00Z"

url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": start_time,
    "commenceTimeTo": end_time
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

data = response.json()

In [ ]:
# 🔥 Flatten Odds Response
rows = []

for game in data:
    home = game["home_team"]
    away = game["away_team"]

    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:

                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"],
                        "commence_time": game["commence_time"],
                        "bookmaker": bookmaker["title"]
                    })

odds_df = pd.DataFrame(rows)

print("Odds Rows:", len(odds_df))
print(odds_df.head())

In [ ]:
# Convert odds to numeric
odds_df["odds"] = pd.to_numeric(odds_df["odds"], errors="coerce")

# Remove rows with missing odds
odds_df = odds_df.dropna(subset=["odds"])

# Average odds per team
odds_clean = (
    odds_df.groupby("team", as_index=False)
    .agg(avg_odds=("odds", "mean"))
)

print("Unique Teams After Collapse:", len(odds_clean))
print(odds_clean.head(10))

In [ ]:
import re

def clean_team(name):
    name = str(name).lower()
    name = re.sub(r"university", "", name)
    name = re.sub(r"college", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

odds_clean["team_clean"] = odds_clean["team"].apply(clean_team)

print(odds_clean.head())

In [ ]:
# Normalize model team names the same way
import re

def clean_team(name):
    name = str(name).lower()
    name = re.sub(r"university", "", name)
    name = re.sub(r"college", "", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()

ratings_df["team_clean"] = ratings_df["team"].apply(clean_team)

# Merge using cleaned names
merged = ratings_df.merge(
    odds_clean,
    on="team_clean",
    how="inner"
)

print("🔥 Merged Rows:", len(merged))
print(merged.head())

In [ ]:
master_df = odds_clean.copy()
master_df = master_df[["team", "team_clean", "avg_odds"]]

print(master_df.head())

In [ ]:
import numpy as np

master_df["Power_Score_model"] = np.random.uniform(50,100,len(master_df))

print(master_df.head())

In [ ]:
master_df["Market_Power"] = 100 - (
    (master_df["avg_odds"] - master_df["avg_odds"].min()) /
    (master_df["avg_odds"].max() - master_df["avg_odds"].min() + 1e-9)
) * 100

master_df["True_Edge"] = master_df["Power_Score_model"] - master_df["Market_Power"]

master_df = master_df.sort_values("True_Edge", ascending=False)

print(master_df.head(20))

In [ ]:
import numpy as np

# Convert american odds to implied probability
def odds_to_prob(odds):
    odds = float(odds)
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

master_df["implied_prob"] = master_df["avg_odds"].apply(odds_to_prob)

# Convert to 0-100 scale
master_df["Market_Power"] = master_df["implied_prob"] * 100

master_df["True_Edge"] = master_df["Power_Score_model"] - master_df["Market_Power"]

master_df = master_df.sort_values("True_Edge", ascending=False)

print(master_df.head(20))

In [ ]:
from datetime import datetime

# Convert commence_time to datetime
odds_df["commence_time"] = pd.to_datetime(odds_df["commence_time"])

# Extract date
odds_df["game_date"] = odds_df["commence_time"].dt.strftime("%Y-%m-%d")

# Keep ONLY rows that match your slate date
odds_df = odds_df[odds_df["game_date"] == SLATE_DATE]

print("Filtered by Exact Slate Date:")
print(odds_df.head())
print("Rows After Date Filter:", len(odds_df))

In [ ]:
valid_teams = ratings_df["team"].unique()

odds_df = odds_df[odds_df["team"].isin(valid_teams)]

print("After Team Validation:", len(odds_df))

In [ ]:
# Pull odds from API with date filter = TODAY
# Filter where game_date == "2026-02-25"
# Print:
print("Status Code")
print("Rows:", len(odds_df))
print(odds_df.head())

In [ ]:
team_odds = (
    odds_df.groupby("team")["odds"]
    .mean()
    .reset_index()
)

print("Unique Teams:", len(team_odds))

In [ ]:
team_odds["team_clean"] = team_odds["team"].str.lower().str.replace("[^a-z0-9 ]","",regex=True)

In [ ]:
print(team_odds.head())

In [ ]:
ratings_df["team_clean"] = ratings_df["team"].str.lower().str.replace(...)

In [ ]:
ratings_df["team_clean"] = (
    ratings_df["team"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)   # remove special chars
    .str.strip()
)

In [ ]:
team_odds["team_clean"] = (
    team_odds["team"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

In [ ]:
print(ratings_df[["team","team_clean"]].head())
print(team_odds[["team","team_clean"]].head())

In [ ]:
print("Matching Teams:")
print(set(ratings_df.team_clean) & set(team_odds.team_clean))

In [ ]:
master_teams = team_odds["team"].unique()

print("Building Power Scores For:", len(master_teams), "Teams")

In [ ]:
# If odds_df still exists:
print("Odds rows currently loaded:", len(odds_df))

team_odds = (
    odds_df.groupby("team")["odds"]
    .mean()
    .reset_index()
)

print("Unique Teams Rebuilt:", len(team_odds))
print(team_odds.head())

In [ ]:
# Your SLATE_DATE
SLATE_DATE = "2026-02-25"

# Your odds API request block
# (The code we built earlier that returns data)

In [ ]:
print("Status:", response.status_code)
print("Games Returned:", len(data))

In [ ]:
import requests
from datetime import datetime, timedelta
import os

SLATE_DATE = "2026-02-25"

API_KEY = os.getenv("ODDS_API_KEY")

start_time = f"{SLATE_DATE}T00:00:00Z"
end_date = (datetime.strptime(SLATE_DATE, "%Y-%m-%d")
            + timedelta(days=1)).strftime("%Y-%m-%d")
end_time = f"{end_date}T00:00:00Z"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": start_time,
    "commenceTimeTo": end_time
}

response = requests.get(url, params=params)
data = response.json()

print("Status:", response.status_code)
print("Games Returned:", len(data))

In [ ]:
# Build odds_df from data
rows = []

for game in data:
    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"],
                        "commence_time": game["commence_time"],
                        "bookmaker": bookmaker["title"]
                    })

odds_df = pd.DataFrame(rows)

print("Odds Rows:", len(odds_df))

In [ ]:
print(type(data))
print(data[:1])

In [ ]:
import requests
import os
from datetime import datetime, timedelta

SLATE_DATE = "2026-02-25"

API_KEY = os.getenv("ODDS_API_KEY")

start_time = f"{SLATE_DATE}T00:00:00Z"
end_date = (datetime.strptime(SLATE_DATE, "%Y-%m-%d")
            + timedelta(days=1)).strftime("%Y-%m-%d")
end_time = f"{end_date}T00:00:00Z"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": start_time,
    "commenceTimeTo": end_time
}

response = requests.get(url, params=params)
data = response.json()

print("Type:", type(data))
print(data.keys())

In [ ]:
import os

print("Environment Key Value:")
print(os.getenv("ODDS_API_KEY"))

In [ ]:
API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
import os

os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

API_KEY = os.getenv("ODDS_API_KEY")

print("Key Loaded:", API_KEY is not None)

In [ ]:
import requests
from datetime import datetime, timedelta

SLATE_DATE = "2026-02-25"

start_time = f"{SLATE_DATE}T00:00:00Z"

end_date = (
    datetime.strptime(SLATE_DATE, "%Y-%m-%d")
    + timedelta(days=1)
).strftime("%Y-%m-%d")

end_time = f"{end_date}T00:00:00Z"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h",
    "oddsFormat": "american",
    "commenceTimeFrom": start_time,
    "commenceTimeTo": end_time
}

response = requests.get(url, params=params)
data = response.json()

print("Response Type:", type(data))
print(data.keys() if isinstance(data, dict) else "List Returned")

In [ ]:
rows = []

# If API returned dict with "data"
if isinstance(data, dict) and "data" in data:
    games = data["data"]
else:
    games = data

for game in games:
    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"],
                        "commence_time": game["commence_time"],
                        "bookmaker": bookmaker["title"]
                    })

import pandas as pd

odds_df = pd.DataFrame(rows)

print("Odds Rows:", len(odds_df))
print(odds_df.head())

In [ ]:
import pandas as pd

rows = []

for game in data:
    for bookmaker in game.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "team": outcome["name"],
                        "odds": outcome["price"],
                        "commence_time": game["commence_time"],
                        "bookmaker": bookmaker["title"]
                    })

odds_df = pd.DataFrame(rows)

print("Odds Rows:", len(odds_df))
print(odds_df.head())

In [ ]:
# Convert odds to numeric just in case
odds_df["odds"] = pd.to_numeric(odds_df["odds"], errors="coerce")

# Remove rows with missing odds
odds_df = odds_df.dropna(subset=["odds"])

# Collapse to unique teams by averaging odds
team_odds = (
    odds_df.groupby("team", as_index=False)
    .agg(avg_odds=("odds", "mean"))
)

print("Unique Teams After Collapse:", len(team_odds))
print(team_odds.head(15))

In [ ]:
import numpy as np

def odds_to_implied_prob(odds):
    odds = float(odds)
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

team_odds["implied_prob"] = team_odds["avg_odds"].apply(odds_to_implied_prob)

# Convert to 0–100 scale
team_odds["Market_Power"] = team_odds["implied_prob"] * 100

print(team_odds.head())

In [ ]:
import requests
import pandas as pd

# ESPN NCAAM Standings / Stats Endpoint
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

response = requests.get(url)
espn_data = response.json()

print("Status:", response.status_code)
print("Keys:", espn_data.keys())

In [ ]:
teams = []

for team in espn_data["sports"][0]["leagues"][0]["teams"]:
    info = team["team"]

    teams.append({
        "team_name": info["displayName"],
        "team_id": info["id"]
    })

espn_teams_df = pd.DataFrame(teams)

print("Teams Pulled:", len(espn_teams_df))
print(espn_teams_df.head())

In [ ]:
stats_rows = []

for team_id in espn_teams_df["team_id"]:

    stats_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}"

    r = requests.get(stats_url)
    data = r.json()

    try:
        stats = data["team"]["record"]["items"][0]

        stats_rows.append({
            "team_id": team_id,
            "win_percent": stats["stats"][0]["value"],
            "points_for": stats["stats"][1]["value"],
            "points_against": stats["stats"][2]["value"]
        })

    except:
        continue

espn_stats_df = pd.DataFrame(stats_rows)

print(espn_stats_df.head())

In [ ]:
team_odds = team_odds.merge(
    espn_teams_df,
    left_on="team",
    right_on="team_name",
    how="left"
)

team_odds = team_odds.merge(
    espn_stats_df,
    on="team_id",
    how="left"
)

print(team_odds.head())

In [ ]:
import re

# Clean ESPN team names
espn_teams_df["team_clean"] = (
    espn_teams_df["team_name"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# Clean your market teams the same way
team_odds["team_clean"] = (
    team_odds["team"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

print(espn_teams_df.head())
print(team_odds.head())

In [ ]:
from fuzzywuzzy import process

matches = []

for team in team_odds["team_clean"]:
    match = process.extractOne(team, espn_teams_df["team_clean"])

    if match and match[1] > 80:
        matches.append((team, match[0], match[1]))

match_df = pd.DataFrame(matches, columns=["team_clean","espn_match","score"])

print("Matched Teams:", len(match_df))
print(match_df.head())

In [ ]:
!pip install fuzzywuzzy python-Levenshtein

In [ ]:
from fuzzywuzzy import process

In [ ]:
from fuzzywuzzy import process

print("Fuzzy Installed Successfully")

In [ ]:
matches = []

for team in team_odds["team_clean"]:
    match = process.extractOne(team, espn_teams_df["team_clean"])

    if match and match[1] > 80:
        matches.append((team, match[0], match[1]))

match_df = pd.DataFrame(matches, columns=["team_clean","espn_match","score"])

print("Matched Teams:", len(match_df))
print(match_df.head())

In [ ]:
import difflib

matches = []

for team in team_odds["team_clean"]:

    best_match = difflib.get_close_matches(
        team,
        espn_teams_df["team_clean"],
        n=1,
        cutoff=0.75  # stricter threshold
    )

    if best_match:
        matches.append((team, best_match[0]))

match_df = pd.DataFrame(matches, columns=["team_clean","espn_match"])

print("Matched Teams:", len(match_df))
print(match_df.head(20))

In [ ]:
import requests
import pandas as pd

standings_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/standings"

r = requests.get(standings_url)
standings_data = r.json()

teams = []

for group in standings_data["children"]:
    for team in group["standings"]["entries"]:
        info = team["team"]
        stats = team["stats"]

        teams.append({
            "team_name": info["displayName"],
            "win_percent": next((s["value"] for s in stats if s["name"] == "winPercent"), None),
            "wins": next((s["value"] for s in stats if s["name"] == "wins"), None),
            "losses": next((s["value"] for s in stats if s["name"] == "losses"), None),
        })

espn_stats_df = pd.DataFrame(teams)

print("Teams From Standings:", len(espn_stats_df))
print(espn_stats_df.head())

In [ ]:
import requests
import pprint

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/standings"

r = requests.get(url)
standings_data = r.json()

pprint.pprint(standings_data.keys())

In [ ]:
import pprint
pprint.pprint(standings_data)

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.sports-reference.com/cbb/"

r = requests.get(url)
soup = BeautifulSoup(r.text, "html.parser")

print("Page Loaded")

In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://www.sports-reference.com/cbb/schools/"

r = requests.get(base_url)
soup = BeautifulSoup(r.text, "html.parser")

teams = []

for link in soup.find_all("a"):
    href = link.get("href")

    if href and "/cbb/schools/" in href and href.count("/") > 3:
        teams.append("https://www.sports-reference.com" + href)

unique_teams = list(set(teams))

print("Team Pages Found:", len(unique_teams))
print(unique_teams[:10])

In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://www.sports-reference.com/cbb/schools/"

r = requests.get(base_url)
soup = BeautifulSoup(r.text, "html.parser")

teams = []

for link in soup.find_all("a"):
    href = link.get("href")

    if href:
        # Keep only men's pages
        if "/cbb/schools/" in href and "/men/" in href:
            full_url = "https://www.sports-reference.com" + href

            teams.append(full_url)

# Remove duplicates
men_team_pages = list(set(teams))

print("MEN Team Pages Found:", len(men_team_pages))
print(men_team_pages[:10])

In [ ]:
# Get cleaned team names from your market table
market_teams = team_odds["team"].str.lower().tolist()

filtered_team_pages = []

for url in men_team_pages:
    team_name = url.split("/")[-3].replace("-", " ")

    for market_team in market_teams:
        if team_name in market_team or market_team in team_name:
            filtered_team_pages.append(url)
            break

filtered_team_pages = list(set(filtered_team_pages))

print("Filtered Team Pages:", len(filtered_team_pages))
print(filtered_team_pages)

In [ ]:
cleaned_pages = []

for url in filtered_team_pages:
    # Remove season-specific pages
    if ".html" in url:
        continue

    cleaned_pages.append(url)

cleaned_pages = list(set(cleaned_pages))

print("Clean Team Pages:", len(cleaned_pages))
print(cleaned_pages)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

test_url = cleaned_pages[0]

print("Testing:", test_url)

r = requests.get(test_url)
soup = BeautifulSoup(r.text, "html.parser")

# Find the team stats table
tables = pd.read_html(r.text)

print("Tables Found:", len(tables))

# Usually the first table contains season stats
stats_table = tables[0]

print(stats_table.head())

In [ ]:
# Take the first table
df = tables[0]

# Flatten multi-level columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric columns safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

# Take the most recent season (row 0)
latest_season = df.iloc[0]

team_features = {
    "Season": latest_season.get("Unnamed:_1_level_0_Season"),
    "Win%": latest_season.get("Overall_W-L%"),
    "Points_For": latest_season.get("Unnamed:_11_level_0_PS/G"),
    "Points_Against": latest_season.get("Unnamed:_12_level_0_PA/G"),
    "SRS": latest_season.get("Unnamed:_9_level_0_SRS"),
    "SOS": latest_season.get("Unnamed:_10_level_0_SOS"),
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
team_features_df["Power_Score_model"] = (
    team_features_df["Win%"] * 40 +
    team_features_df["SRS"] * 3 -
    team_features_df["Points_Against"]
)

print(team_features_df)

In [ ]:
print(df.columns)

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
# Flatten columns already done
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert everything numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

# Grab most recent season
latest = df.iloc[0]

team_features = {
    "Season": latest["Unnamed:_1_level_0_Season"],
    "Win%": latest["Overall_W-L%"],
    "Points_For": latest["Unnamed:_11_level_0_PS/G"],
    "Points_Against": latest["Unnamed:_12_level_0_PA/G"],
    "SRS": latest["Unnamed:_9_level_0_SRS"],
    "SOS": latest["Unnamed:_10_level_0_SOS"],
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
# Flatten columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric where possible
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    except:
        pass

latest = df.iloc[0]

def get_col_containing(keyword):
    for col in df.columns:
        if keyword in col:
            return latest[col]
    return None

team_features = {
    "Season": get_col_containing("Season"),
    "Win%": get_col_containing("W-L%"),
    "Points_For": get_col_containing("PS/G"),
    "Points_Against": get_col_containing("PA/G"),
    "SRS": get_col_containing("SRS"),
    "SOS": get_col_containing("SOS"),
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
print(df.iloc[0])

In [ ]:
for col in df.columns:
    print(col, ":", df.iloc[0][col])

In [ ]:
# Ensure columns are flattened
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="ignore")

# Remove rows where Season is NaN
season_col = [col for col in df.columns if "Season" in col][0]

df_clean = df.dropna(subset=[season_col])

# Select the FIRST valid season row
latest = df_clean.iloc[0]

def get_col(keyword):
    for col in df.columns:
        if keyword in col:
            return latest[col]
    return None

team_features = {
    "Season": get_col("Season"),
    "Win%": get_col("W-L%"),
    "Points_For": get_col("PS/G"),
    "Points_Against": get_col("PA/G"),
    "SRS": get_col("SRS"),
    "SOS": get_col("SOS"),
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
for col in df.columns:
    if "Unnamed" in col:
        print(col)

In [ ]:
for col in df.columns:
    if "W-L" in col:
        print(col)

In [ ]:
  print(df.columns.tolist())

In [ ]:
# Flatten columns (already flattened but safe to redo)
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by Rank (lower rank = top row = most recent)
rank_col = [col for col in df.columns if "Rk" in col][0]

df_sorted = df.sort_values(by=rank_col, ascending=True)

latest = df_sorted.iloc[0]

def get(keyword):
    for col in df.columns:
        if keyword in col:
            return latest[col]
    return None

team_features = {
    "Win%": get("W-L%"),
    "Points_For": get("PS/G"),
    "Points_Against": get("PA/G"),
    "SRS": get("SRS"),
    "SOS": get("SOS"),
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
# Ensure columns flattened
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Find rank column more safely
rank_col_candidates = [col for col in df.columns if "Rk" in col or "Rank" in col]

if len(rank_col_candidates) == 0:
    raise Exception("No Rank column found!")

rank_col = rank_col_candidates[0]

# Sort by rank
df_sorted = df.sort_values(by=rank_col, ascending=True)

latest = df_sorted.iloc[0]

def get(keyword):
    for col in df.columns:
        if keyword in col:
            return latest[col]
    return None

team_features = {
    "Win%": get("W-L%"),
    "Points_For": get("PS/G"),
    "Points_Against": get("PA/G"),
    "SRS": get("SRS"),
    "SOS": get("SOS"),
}

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
# Flatten columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where Win% is NaN
win_col = [col for col in df.columns if "W-L%" in col][0]

df_clean = df.dropna(subset=[win_col])

# Just take first row (most recent season)
latest = df_clean.iloc[0]

def get(keyword):
    for col in df.columns:
        if keyword in col:
            return latest[col]
    return None

team_features = {
    "Win%": get("W-L%"),
    "Points_For": get("PS/G"),
    "Points_Against": get("PA/G"),
    "SRS": get("SRS"),
    "SOS": get("SOS"),
}

import pandas as pd
team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
# Flatten columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where Season is NaN
df_clean = df.dropna(subset=[df.columns[1]])  # Season column position

# Take most recent season (row 0)
latest = df_clean.iloc[0]

# Extract by column POSITION instead of name
team_features = {
    "Win%": latest[df.columns[5]],      # Overall W-L%
    "Points_For": latest[df.columns[11]],  # PS/G
    "Points_Against": latest[df.columns[12]],  # PA/G
    "SRS": latest[df.columns[9]],      # SRS
    "SOS": latest[df.columns[10]],     # SOS
}

import pandas as pd

team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
# Flatten columns
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert numeric safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Just take first row (most recent)
latest = df.iloc[0]

def get_by_position(position):
    if position < len(df.columns):
        return latest[df.columns[position]]
    return None

team_features = {
    "Win%": get_by_position(5),        # Overall W-L%
    "Points_For": get_by_position(11),  # PS/G
    "Points_Against": get_by_position(12),  # PA/G
    "SRS": get_by_position(9),
    "SOS": get_by_position(10),
}

import pandas as pd
team_features_df = pd.DataFrame([team_features])

print(team_features_df)

In [ ]:
import pandas as pd

all_team_features = []

for url in cleaned_pages:

    try:
        r = requests.get(url)
        tables = pd.read_html(r.text)
        df = tables[0]

        # Flatten
        df.columns = ['_'.join(col).strip() for col in df.columns.values]

        # Convert numeric
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        latest = df.iloc[0]

        def get_by_position(position):
            if position < len(df.columns):
                return latest[df.columns[position]]
            return None

        team_features = {
            "team_url": url,
            "Win%": get_by_position(5),
            "Points_For": get_by_position(11),
            "Points_Against": get_by_position(12),
            "SRS": get_by_position(9),
            "SOS": get_by_position(10),
        }

        all_team_features.append(team_features)

    except Exception as e:
        print("Failed:", url, e)

stats_df = pd.DataFrame(all_team_features)

print("Teams Extracted:", len(stats_df))
print(stats_df.head())

In [ ]:
# Extract team name from URL
stats_df["team"] = stats_df["team_url"].apply(
    lambda x: x.split("/schools/")[1].split("/men/")[0].replace("-", " ")
)

# Drop URL column
stats_df = stats_df.drop(columns=["team_url"])

# Reorder columns
stats_df = stats_df[["team", "Win%", "Points_For", "Points_Against", "SRS", "SOS"]]

print(stats_df.head(20))

In [ ]:
stats_df["Power_Score_model"] = (
    stats_df["Win%"] * 40 +
    stats_df["SRS"] * 2.5 +
    ((stats_df["Points_For"] - stats_df["Points_Against"]) * 0.5)
)

# Normalize to 0-100 scale
stats_df["Power_Score_model"] = (
    100 * (stats_df["Power_Score_model"] - stats_df["Power_Score_model"].min())
    / (stats_df["Power_Score_model"].max() - stats_df["Power_Score_model"].min())
)

print(stats_df.sort_values("Power_Score_model", ascending=False).head(20))

In [ ]:
# Make sure team names are clean and lowercase
stats_df["team_clean"] = stats_df["team"].str.lower().str.strip()

team_odds["team_clean"] = team_odds["team"].str.lower().str.strip()

# Merge model with odds
merged = stats_df.merge(
    team_odds[["team_clean", "avg_odds", "Market_Power"]],
    on="team_clean",
    how="left"
)

# Calculate True Edge
merged["True_Edge"] = merged["Power_Score_model"] - merged["Market_Power"]

# Sort by strongest edge
merged = merged.sort_values("True_Edge", ascending=False)

print(merged[["team", "Power_Score_model", "Market_Power", "True_Edge"]].head(20))

In [ ]:
print(team_odds[["team"]].head(30))

In [ ]:
import re

def clean_team_name(name):
    name = name.lower()
    name = re.sub(r"\b(dukes|patriots|longhorns|wildcats|tigers|knights|eagles|bulldogs|cougars|hawks|raiders|pirates|falcons|broncos)\b", "", name)
    name = re.sub(r"[^a-z\s]", "", name)
    return " ".join(name.split())

stats_df["team_clean"] = stats_df["team"].apply(clean_team_name)
team_odds["team_clean"] = team_odds["team"].apply(clean_team_name)

print(stats_df[["team","team_clean"]].head())
print(team_odds[["team","team_clean"]].head())

In [ ]:
from fuzzywuzzy import process

matches = []

for team in stats_df["team_clean"]:
    match = process.extractOne(team, team_odds["team_clean"])

    if match:
        matches.append({
            "team": team,
            "matched_team": match[0],
            "score": match[1]
        })

match_df = pd.DataFrame(matches)

print("Matched Teams:")
print(match_df.sort_values("score", ascending=False))

In [ ]:
good_matches = match_df[match_df["score"] > 85]

merged = stats_df.merge(
    good_matches,
    left_on="team_clean",
    right_on="team",
    how="inner"
).merge(
    team_odds,
    left_on="matched_team",
    right_on="team_clean",
    how="left"
)

print(merged.head())

In [ ]:
# Keep only important columns from odds
odds_clean = team_odds[["team_clean", "avg_odds", "implied_prob", "Market_Power"]]

# Merge cleanly
final = merged_stats.merge(
    odds_clean,
    left_on="matched_team",
    right_on="team_clean",
    how="left"
)

print(final.head())

In [ ]:
odds_clean = team_odds[["team_clean", "avg_odds", "implied_prob", "Market_Power"]]

final = df.merge(
    odds_clean,
    left_on="matched_team",
    right_on="team_clean",
    how="left"
)

print(final.head())

In [ ]:
print(df.columns)

In [ ]:
left_on="team_clean_x"

In [ ]:
print("Columns in df:")
print(df.columns)

print("Columns in odds:")
print(team_odds.columns)

In [ ]:
import pandas as pd
import requests

all_team_data = []

for url in clean_team_pages:   # your list of team urls
    print("Testing:", url)

    r = requests.get(url)
    tables = pd.read_html(r.text)

    if len(tables) == 0:
        print("No tables found:", url)
        continue

    df = tables[0]

    # ✅ Extract team name from URL
    team_name = url.split("/schools/")[1].split("/men")[0]
    team_name = team_name.replace("-", " ").strip()

    # ✅ Add team column immediately
    df["team_clean"] = team_name

    all_team_data.append(df)

print("Finished scraping")

In [ ]:
clean_team_pages = [
'https://www.sports-reference.com/cbb/schools/duquesne/men/',
'https://www.sports-reference.com/cbb/schools/texas/men/',
'https://www.sports-reference.com/cbb/schools/davidson/men/',
'https://www.sports-reference.com/cbb/schools/mercer/men/',
'https://www.sports-reference.com/cbb/schools/lehigh/men/',
# add the rest of your team urls here
]

print("Team URLs Loaded:", len(clean_team_pages))

In [ ]:
import pandas as pd
import requests

all_team_data = []

for url in clean_team_pages:
    print("Testing:", url)

    r = requests.get(url)
    tables = pd.read_html(r.text)

    if len(tables) == 0:
        print("No tables found:", url)
        continue

    df = tables[0]

    # Extract team name from URL
    team_name = url.split("/schools/")[1].split("/men")[0]
    team_name = team_name.replace("-", " ").strip()

    df["team_clean"] = team_name

    all_team_data.append(df)

print("Scraping Complete")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

all_team_data = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in clean_team_pages:
    print("Testing:", url)

    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        print("Failed Request:", r.status_code)
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    tables = pd.read_html(StringIO(str(soup)))

    if len(tables) == 0:
        print("No tables found:", url)
        continue

    df = tables[0]

    team_name = url.split("/schools/")[1].split("/men")[0]
    team_name = team_name.replace("-", " ").strip()

    df["team_clean"] = team_name

    all_team_data.append(df)

print("Done Scraping")

In [ ]:
stats_df = pd.concat(all_team_data, ignore_index=True)
print(stats_df.head())
print("Total Rows:", len(stats_df))

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

url = clean_team_pages[0]  # test one team

r = requests.get(url, headers=headers)

print("Status Code:", r.status_code)
print("Page Length:", len(r.text))

soup = BeautifulSoup(r.text, "html.parser")

print("Tables Found:", len(soup.find_all("table")))

In [ ]:
tables = pd.read_html(r.text)

for i, table in enumerate(tables):
    print("Table", i)
    print(table.head())

In [ ]:
print(r.status_code)
print(r.url)
print(r.text[:2000])

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}

In [ ]:
r = requests.get(url, headers=headers)

In [ ]:
import random
import time

time.sleep(random.uniform(3,6))

In [ ]:
requests.get(url)

In [ ]:
session = requests.Session()
session.headers.update(headers)

r = session.get(url)

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
})

for url in clean_team_pages:

    time.sleep(random.uniform(3,6))

    print("Testing:", url)

    r = session.get(url)

    if r.status_code != 200:
        print("Blocked:", r.status_code)
        continue

    tables = pd.read_html(r.text)

    # process tables here

In [ ]:
import requests
import pandas as pd
import time
import random

session = requests.Session()

In [ ]:
clean_team_pages

In [ ]:
if len(all_team_data) > 0:
    stats_df = pd.concat(all_team_data, ignore_index=True)
    print("Teams Extracted:", len(stats_df))
    display(stats_df.head())
else:
    print("No data extracted.")

In [ ]:
print(stats_df.shape)
display(stats_df.head())

In [ ]:
len(all_team_data)

In [ ]:
print("Stats Teams:")
print(stats_df["team_clean"].unique())

print("\nOdds Teams:")
print(team_odds["team_clean"].unique())

In [ ]:
team_odds["team_clean"] = (
    team_odds["team_clean"]
    .str.lower()
    .str.replace("the ", "", regex=False)
    .str.split()
    .str[0]
)

In [ ]:
stats_df["team_clean"] = (
    stats_df["team_clean"]
    .str.lower()
    .str.replace("the ", "", regex=False)
    .str.split()
    .str[0]
)

In [ ]:
final_df = stats_df.merge(
    team_odds[["team_clean","avg_odds","implied_prob","Market_Power"]],
    on="team_clean",
    how="inner"
)

print("Final Rows:", len(final_df))
display(final_df.head())

In [ ]:
# Convert model power to probability
final_df["model_prob"] = final_df["Power_Score_model"] / final_df["Power_Score_model"].sum()

In [ ]:
final_df["true_edge"] = final_df["model_prob"] - final_df["implied_prob"]

In [ ]:
final_df = final_df.sort_values("true_edge", ascending=False)

display(final_df[[
    "team",
    "model_prob",
    "implied_prob",
    "true_edge"
]])

In [ ]:
print(team_odds.head(10))
print(team_odds.columns)

In [ ]:
import requests
import pandas as pd

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american"
}

response = requests.get(url, params=params)
games = response.json()

print("Games pulled:", len(games))

In [ ]:
rows = []

for game in games:
    home = game["home_team"]
    away = game["away_team"]

    for book in game["bookmakers"]:
        for market in book["markets"]:

            if market["key"] == "h2h":
                for outcome in market["outcomes"]:
                    rows.append({
                        "game_id": game["id"],
                        "home_team": home,
                        "away_team": away,
                        "team": outcome["name"],
                        "odds": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

odds_df.head()

In [ ]:
def american_to_prob(odds):
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)

odds_df["implied_prob"] = odds_df["odds"].apply(american_to_prob)

In [ ]:
odds_df["team_clean"] = (
    odds_df["team"]
    .str.lower()
    .str.replace("the ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.split()
    .str[0]
)

In [ ]:
odds_df = odds_df.merge(
    stats_df[["team_clean", "Power_Score_model"]],
    on="team_clean",
    how="left"
)

print("Missing Power Scores:", odds_df["Power_Score_model"].isna().sum())
odds_df.head()

In [ ]:
# Sum power per game
game_power = odds_df.groupby("game_id")["Power_Score_model"].transform("sum")

odds_df["model_prob"] = odds_df["Power_Score_model"] / game_power

In [ ]:
odds_df["true_edge"] = odds_df["model_prob"] - odds_df["implied_prob"]

In [ ]:
edges = odds_df.sort_values("true_edge", ascending=False)

display(edges[[
    "team",
    "odds",
    "model_prob",
    "implied_prob",
    "true_edge"
]].head(15))

In [ ]:
missing = odds_df[odds_df["Power_Score_model"].isna()]
print("Teams missing power scores:")
print(missing[["team", "team_clean"]])

In [ ]:
final_edges = odds_df[
    (odds_df["true_edge"] > 0.05) &
    (odds_df["model_prob"] > 0.1)
].sort_values("true_edge", ascending=False)

display(final_edges)

In [ ]:
strong_plays = df[
    (df["true_edge"] > 0.05) &
    (df["model_prob"] > df["implied_prob"])
].sort_values("true_edge", ascending=False)

print("🔥 Strong Plays:")
display(strong_plays)

In [ ]:
# Make sure columns exist first
if "true_edge" not in df.columns:
    df["true_edge"] = df["model_prob"] - df["implied_prob"]

print(df.columns)

In [ ]:
print(df.columns)

In [ ]:
print(df.head())

In [ ]:
df["model_prob"] = df["Power_Score_model"] / df["Power_Score_model"].sum()

In [ ]:
print(df.columns)

In [ ]:
# Clean column names
df.columns = [col.split("_")[-1] for col in df.columns]

# Convert numeric columns
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by Rank (if exists) and take most recent
if "Rk" in df.columns:
    df = df.sort_values("Rk", ascending=True)

latest = df.iloc[0]

stats = {
    "Win%": latest.get("W-L%"),
    "Points_For": latest.get("PS/G"),
    "Points_Against": latest.get("PA/G"),
    "SRS": latest.get("SRS"),
    "SOS": latest.get("SOS"),
}

In [ ]:
# Flatten multi-index columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Convert only object/string columns to numeric
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Flatten columns if needed
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ['_'.join(col).strip() for col in df.columns]

# Safely convert numeric columns
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    except:
        pass

print("Numeric conversion complete")
print(df.dtypes)

In [ ]:
print(df.head())
print(df.columns)

In [ ]:
# Fix duplicate column names
df.columns = [
    f"{col}_{i}" if list(df.columns).count(col) > 1 else col
    for i, col in enumerate(df.columns)
]

print(df.columns)

In [ ]:
# Keep only first W, L, W-L%
df_clean = df.loc[:, ~df.columns.duplicated()]
print(df_clean.head())

In [ ]:
# Sort by Rank (Rk) ascending
df_clean = df_clean.sort_values("Rk")

# Drop rows where Season is NaN
df_clean = df_clean.dropna(subset=["Rk"])

# Take most recent season (row 0 after sort)
latest = df_clean.iloc[0]

print("Latest Season Stats:")
print(latest)

In [ ]:
latest_features = {
    "Season": latest.get("Season"),
    "Win%": latest.filter(like="W-L%").iloc[0],
    "Points_For": latest["PS/G"],
    "Points_Against": latest["PA/G"],
    "SRS": latest["SRS"],
    "SOS": latest["SOS"],
}

print(latest_features)

In [ ]:
win_percent = latest[[col for col in latest.index if "W-L%" in col]].values[0]

clean_stats = {
    "Win%": win_percent,
    "Points_For": latest["PS/G"],
    "Points_Against": latest["PA/G"],
    "SRS": latest["SRS"],
    "SOS": latest["SOS"],
}

print(clean_stats)

In [ ]:
def get_team_power(team_url):

    r = requests.get(team_url, headers={"User-Agent": "Mozilla/5.0"})

    try:
        tables = pd.read_html(StringIO(r.text))
    except:
        return None

    df = tables[0]

    # Clean column names
    df.columns = df.columns.get_level_values(-1)

    # Keep only numeric rows
    df = df[df["Season"].notna()]

    # Sort newest first
    df = df.sort_values("Season", ascending=False)

    # Take top 3 seasons
    df_top = df.head(3)

    weights = [0.6, 0.3, 0.1]
    power_components = []

    for i, row in df_top.iterrows():
        score = (
            float(row["W-L%"]) * 40 +
            float(row["SRS"]) * 3 +
            float(row["SOS"]) * 1
        )
        power_components.append(score)

    weighted_power = sum(w * s for w, s in zip(weights[:len(power_components)], power_components))

    return weighted_power

In [ ]:
team_powers = []

for url in clean_team_pages:
    power = get_team_power(url)

    if power:
        team_name = url.split("/")[6]   # extract team from URL
        team_powers.append({
            "team_url": url,
            "power_score": power
        })

power_df = pd.DataFrame(team_powers)

print(power_df.head())
print("Total Teams Processed:", len(power_df))

In [ ]:
from io import StringIO
import numpy as np

def get_team_power(team_url):

    r = requests.get(team_url, headers={"User-Agent": "Mozilla/5.0"})

    try:
        tables = pd.read_html(StringIO(r.text))
    except:
        return None

    df = tables[0]

    # Flatten multi-level columns if needed
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(-1)

    # Rename columns safely
    df.columns = df.columns.astype(str)

    # Keep rows that actually represent seasons
    df = df[df["Season"].notna()].copy()

    # Convert numeric columns safely
    numeric_cols = ["W-L%", "SRS", "SOS"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sort newest season first
    df = df.sort_values("Season", ascending=False)

    # Take top 3 seasons
    df_top = df.head(3)

    weights = [0.6, 0.3, 0.1]
    power_scores = []

    for i, row in df_top.iterrows():

        w = row.get("W-L%", np.nan)
        srs = row.get("SRS", np.nan)
        sos = row.get("SOS", np.nan)

        if pd.notna(w):
            score = (w * 40) + (srs * 3) + (sos * 1)
            power_scores.append(score)

    if len(power_scores) == 0:
        return None

    weighted_power = sum(
        weights[i] * power_scores[i]
        for i in range(len(power_scores))
    )

    return weighted_power

In [ ]:
team_powers = []

for url in clean_team_pages:
    power = get_team_power(url)

    if power is not None:
        team_name = url.split("/")[6]

        team_powers.append({
            "team_url": url,
            "power_score": power
        })

power_df = pd.DataFrame(team_powers)

print(power_df.head())
print("Teams Processed:", len(power_df))

In [ ]:
for col in numeric_cols:
    matches = [c for c in df.columns if col in str(c)]

    for match in matches:
        df[match] = pd.to_numeric(df[match], errors="coerce")

In [ ]:
# Automatically detect columns that should be numeric
numeric_cols = df.columns

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="ignore")

In [ ]:
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    except:
        pass

In [ ]:
print(df.dtypes)
print(df.head())

In [ ]:
# Sort by Rank (lowest rank = most recent row on sports-reference)
rank_col = [c for c in df.columns if "Rk" in str(c)][0]

df_sorted = df.sort_values(by=rank_col, ascending=True)

# Take only the first row (latest season)
latest_stats = df_sorted.groupby("Rk").head(1)

print(latest_stats.head())
print("Rows After Filter:", len(latest_stats))

In [ ]:
latest_stats = df.iloc[[0]]  # take first row only
print(latest_stats)

In [ ]:
latest = latest_stats.iloc[0]

power_score = (
    float(latest["W-L%_5"]) * 40 +
    float(latest["SRS"]) * 3 +
    float(latest["SOS"]) * 1
)

print("Power Score:", power_score)

In [ ]:
def get_team_power(team_url):
    import requests
    import pandas as pd

    r = requests.get(team_url)
    tables = pd.read_html(r.text)

    if len(tables) == 0:
        return None

    df = tables[0]

    # Clean column names
    df.columns = [str(c).replace(" ", "_") for c in df.columns]

    # Convert numeric columns
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sort by rank (latest season usually top)
    if "Rk" in df.columns:
        df = df.sort_values("Rk", ascending=True)

    latest = df.iloc[0]

    # Compute power
    if "W-L%_5" in df.columns and "SRS" in df.columns and "SOS" in df.columns:
        power = (
            float(latest["W-L%_5"]) * 40 +
            float(latest["SRS"]) * 3 +
            float(latest["SOS"]) * 1
        )
    else:
        return None

    return power

In [ ]:
powers = []

for url in clean_team_pages:
    power = get_team_power(url)

    if power:
        powers.append(power)

print("Teams Processed:", len(powers))

In [ ]:
def get_team_power(team_url):
    import requests
    import pandas as pd

    r = requests.get(team_url)

    try:
        tables = pd.read_html(r.text)
    except:
        return None

    if len(tables) == 0:
        return None

    df = tables[0]

    # Flatten column names
    df.columns = [str(c).split("_")[-1] for c in df.columns]

    # Convert everything numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sort by rank (lowest rank = most recent usually)
    if "Rk" in df.columns:
        df = df.sort_values("Rk", ascending=True)

    latest = df.iloc[0]

    # 🔥 Use clean column names
    if "W-L%" in df.columns and "SRS" in df.columns and "SOS" in df.columns:
        power = (
            float(latest["W-L%"]) * 40 +
            float(latest["SRS"]) * 3 +
            float(latest["SOS"]) * 1
        )
        return power

    return None

In [ ]:
powers = []

for url in clean_team_pages:
    power = get_team_power(url)

    if power is not None:
        powers.append(power)

print("Teams Processed:", len(powers))

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

r = requests.get(team_url, headers=headers)

In [ ]:
import time
import requests

headers = {
    "User-Agent": "Mozilla/5.0"
}

time.sleep(1.5)
r = requests.get(team_url, headers=headers)

print("Status:", r.status_code)

In [ ]:
def get_team_power(team_url):

    import time
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    time.sleep(1.5)

    r = requests.get(team_url, headers=headers)

    if r.status_code != 200:
        print("Failed:", team_url, r.status_code)
        return None

    tables = pd.read_html(r.text)

    if len(tables) == 0:
        print("No tables:", team_url)
        return None

    df = tables[0]

    return df

In [ ]:
for url in clean_team_pages:
    df = get_team_power(url)

    if df is not None:
        print("Success:", url)

In [ ]:
all_team_data = []

for url in clean_team_pages:

    df = get_team_power(url)

    if df is None:
        continue

    # Keep only latest season
    df = df.sort_values("Rk").head(1)

    df["team_url"] = url

    all_team_data.append(df)

stats_df = pd.concat(all_team_data, ignore_index=True)

print("Final Rows:", len(stats_df))
print(stats_df.head())

In [ ]:
# Find column that contains "Rk"
rank_cols = [c for c in df.columns if "Rk" in str(c)]

if len(rank_cols) > 0:
    rank_col = rank_cols[0]
    df = df.sort_values(rank_col).head(1)
else:
    # If no rank column, just take first row
    df = df.head(1)

In [ ]:
print("Columns:", df.columns)

In [ ]:
# If MultiIndex → flatten columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ["_".join([str(i) for i in col if str(i) != "nan"]).strip()
                  for col in df.columns]

print("Flattened Columns:")
print(df.columns)

In [ ]:
rank_cols = [c for c in df.columns if "Rk" in c]

if rank_cols:
    df = df.sort_values(rank_cols[0]).head(1)
else:
    df = df.head(1)

In [ ]:
print(df.columns)

In [ ]:
# Find rank column
rank_col = [c for c in df.columns if "Rk" in c]

if len(rank_col) > 0:
    df = df.sort_values(rank_col[0], ascending=True)
    latest = df.head(1)
else:
    latest = df.head(1)

print("Latest Season:")
print(latest)

In [ ]:
latest = latest.squeeze()

power = (
    latest["Overall_W-L%"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)

print("Power Score:", power)

In [ ]:
def get_col(df, keyword):
    cols = [c for c in df.index if keyword in c]
    if len(cols) > 0:
        return cols[0]
    return None


latest = latest.squeeze()

srs_col = get_col(df, "SRS")
sos_col = get_col(df, "SOS")
win_col = get_col(df, "W-L%")

power = 0

if win_col:
    power += float(latest[win_col]) * 40

if srs_col:
    power += float(latest[srs_col]) * 3

if sos_col:
    power += float(latest[sos_col]) * 1

print("Power Score:", power)

In [ ]:
def get_col(df, keyword):
    cols = [c for c in df.columns if keyword in str(c)]
    if len(cols) > 0:
        return cols[0]
    return None

In [ ]:
latest = latest.squeeze()

srs_col = get_col(df, "SRS")
sos_col = get_col(df, "SOS")
win_col = get_col(df, "W-L%")

power = 0

if win_col:
    power += float(latest[win_col]) * 40

if srs_col:
    power += float(latest[srs_col]) * 3

if sos_col:
    power += float(latest[sos_col]) * 1

print("Power Score:", power)

In [ ]:
df["Power_Score_model"] = (
    df["Power_Score_model"] - df["Power_Score_model"].min()
) / (
    df["Power_Score_model"].max() - df["Power_Score_model"].min()
) * 100

In [ ]:
score_col = [c for c in df.columns if "Power" in str(c)]
print(score_col)

In [ ]:
col = score_col[0]

df["model_prob"] = (
    (df[col] - df[col].min()) /
    (df[col].max() - df[col].min())
)

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.head())

In [ ]:
df.columns = [col.replace("Unnamed: ", "").replace("level_0_", "").strip() for col in df.columns]
print(df.columns)

In [ ]:
print(df.columns)

In [ ]:
latest = df.sort_values("Rk").head(1).squeeze()

power = (
    latest["W-L%"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)

print("Power Score:", power)

In [ ]:
df.columns = ["_".join(col).strip() if isinstance(col, tuple) else col for col in df.columns]
print(df.columns)

In [ ]:
print([col for col in df.columns if "Rk" in col])

In [ ]:
df.columns = df.columns.str.replace(r"^\d+_", "", regex=True)
print(df.columns)

In [ ]:
latest = df.sort_values("Rk").head(1).squeeze()

In [ ]:
power = (
    latest["W-L%"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)
print("Power Score:", power)

In [ ]:
print(df.columns)

In [ ]:
power = (
    latest["Overall_W-L%"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)

print("Power Score:", power)

In [ ]:
latest = df.sort_values("Rk", ascending=True).iloc[0]

In [ ]:
power = (
    float(latest["Overall_W-L%"]) * 40 +
    float(latest["SRS"]) * 3 +
    float(latest["SOS"]) * 1
)

print("Power Score:", power)

In [ ]:
all_powers = []

for url in clean_team_pages:
    try:
        print("Processing:", url)

        power = get_team_power(url)

        if power is not None:
            all_powers.append(power)

    except Exception as e:
        print("Failed:", url, e)

power_df = pd.concat(all_powers, ignore_index=True)

print("Total Teams:", len(power_df))
print(power_df.head())

In [ ]:
power_df["Power_Score_model"] = (
    (power_df["Power_Score_model"] - power_df["Power_Score_model"].min()) /
    (power_df["Power_Score_model"].max() - power_df["Power_Score_model"].min())
) * 100

In [ ]:
print(power_df.columns)

In [ ]:
score_col = [c for c in power_df.columns if "power" in str(c).lower()][0]

power_df[score_col] = (
    (power_df[score_col] - power_df[score_col].min()) /
    (power_df[score_col].max() - power_df[score_col].min())
) * 100

print(power_df.sort_values(score_col, ascending=False).head(10))

In [ ]:
if "Power_Score_model" not in power_df.columns:
    power_df["Power_Score_model"] = 0  # temporary placeholder

print(power_df.columns)

In [ ]:
col = "Power_Score_model"

power_df[col] = (
    (power_df[col] - power_df[col].min()) /
    (power_df[col].max() - power_df[col].min())
) * 100

print(power_df.sort_values(col, ascending=False).head())

In [ ]:
# Flatten multiindex columns
power_df.columns = [
    col[1] if isinstance(col, tuple) else col
    for col in power_df.columns
]

print(power_df.columns)

In [ ]:
numeric_cols = ["W", "L", "W-L%", "SRS", "SOS", "PS/G", "PA/G"]

for col in numeric_cols:
    if col in power_df.columns:
        power_df[col] = pd.to_numeric(power_df[col], errors="coerce")

In [ ]:
# Force flatten columns safely
power_df.columns = [
    str(col[1]).strip() if isinstance(col, tuple) else str(col).strip()
    for col in power_df.columns
]

print(power_df.columns)

In [ ]:
print(power_df.dtypes)

In [ ]:
power_df = power_df.apply(pd.to_numeric, errors="ignore")

In [ ]:
col = "Power_Score_model"

power_df[col] = (
    (power_df[col] - power_df[col].min()) /
    (power_df[col].max() - power_df[col].min())
) * 100

In [ ]:
print(power_df.columns.tolist())

In [ ]:
# Keep only latest season
latest = power_df.sort_values("Rk").head(1).copy()

# Make sure numeric
latest["W-L%"] = pd.to_numeric(latest["W-L%"], errors="coerce")
latest["SRS"] = pd.to_numeric(latest["SRS"], errors="coerce")
latest["SOS"] = pd.to_numeric(latest["SOS"], errors="coerce")

# Recreate power score
latest["Power_Score_model"] = (
    latest["W-L%"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)

print(latest[["W-L%","SRS","SOS","Power_Score_model"]])

In [ ]:
print(latest)
print(type(latest))
print(latest.index)

In [ ]:
print(latest.columns)

In [ ]:
latest.columns = pd.io.parsers.ParserBase({'names':latest.columns})._maybe_dedup_names(latest.columns)
print(latest.columns)

In [ ]:
latest.columns = pd.Index(
    [f"{col}_{i}" if list(latest.columns).count(col) > 1 else col
     for i, col in enumerate(latest.columns)]
)

print(latest.columns)

In [ ]:
cols = pd.Series(latest.columns)

for dup in cols[cols.duplicated()].unique():
    cols[cols[cols == dup].index.tolist()] = [
        f"{dup}_{i}" if i != 0 else dup
        for i in range(sum(cols == dup))
    ]

latest.columns = cols
print(latest.columns)

In [ ]:
latest["W-L%"] = pd.to_numeric(latest["W-L%"], errors="coerce")
latest["SRS"] = pd.to_numeric(latest["SRS"], errors="coerce")
latest["SOS"] = pd.to_numeric(latest["SOS"], errors="coerce")

In [ ]:
print(list(latest.columns))


In [ ]:
def get_col(df, keyword):
    cols = [c for c in df.columns if keyword.lower() in str(c).lower()]
    return cols[0] if cols else None

w_col   = get_col(latest, "W-L")
srs_col = get_col(latest, "SRS")
sos_col = get_col(latest, "SOS")

print("Found columns:", w_col, srs_col, sos_col)

latest[w_col]   = pd.to_numeric(latest[w_col], errors="coerce")
latest[srs_col] = pd.to_numeric(latest[srs_col], errors="coerce")
latest[sos_col] = pd.to_numeric(latest[sos_col], errors="coerce")

In [ ]:
print(w_col, srs_col, sos_col)

In [ ]:
latest["W-L%_5"] = pd.to_numeric(latest["W-L%_5"], errors="coerce")
latest["SRS"]    = pd.to_numeric(latest["SRS"], errors="coerce")
latest["SOS"]    = pd.to_numeric(latest["SOS"], errors="coerce")

power = (
    latest["W-L%_5"] * 40 +
    latest["SRS"] * 3 +
    latest["SOS"] * 1
)

print("Power Score:", float(power))

In [ ]:
df["Power_Score_model"] = (
    (df["power_score"] - df["power_score"].min()) /
    (df["power_score"].max() - df["power_score"].min())
) * 100

In [ ]:
team_df.loc[index, "power_score"] = power

In [ ]:
import pandas as pd
import requests

def get_team_power(team_url):
    tables = pd.read_html(team_url)

    df = tables[0]  # First table (season results)

    # Flatten multi-index if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(-1)

    # Remove duplicate column names safely
    df = df.loc[:, ~df.columns.duplicated()]

    # Keep only rows with numeric Rk
    df = df[pd.to_numeric(df["Rk"], errors="coerce").notna()]

    # Sort by rank (latest season = rank 1)
    df = df.sort_values("Rk")
    latest = df.iloc[0]

    # Convert required columns safely
    wl_col  = [c for c in df.columns if "W-L%" in c][0]
    srs_col = [c for c in df.columns if "SRS" in c][0]
    sos_col = [c for c in df.columns if "SOS" in c][0]

    wl  = float(latest[wl_col])
    srs = float(latest[srs_col])
    sos = float(latest[sos_col])

    power = wl * 40 + srs * 3 + sos * 1

    return power

In [ ]:
test_url = clean_team_pages[0]
print(get_team_power(test_url))

In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://www.sports-reference.com/cbb/schools/"

response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

links = soup.find_all("a")

clean_team_pages = []

for link in links:
    href = link.get("href")
    if href and "/cbb/schools/" in href and href.endswith("/"):
        full_url = "https://www.sports-reference.com" + href
        clean_team_pages.append(full_url)

clean_team_pages = list(set(clean_team_pages))

print("Total team pages found:", len(clean_team_pages))
print(clean_team_pages[:5])

In [ ]:
test_url = clean_team_pages[0]
print(get_team_power(test_url))

In [ ]:
import requests

url = "https://www.sports-reference.com/cbb/schools/"
response = requests.get(url)

print(response.status_code)

In [ ]:
from bs4 import BeautifulSoup
import requests

base_url = "https://www.sports-reference.com"

response = requests.get("https://www.sports-reference.com/cbb/schools/")
soup = BeautifulSoup(response.text, "html.parser")

clean_team_pages = []

for link in soup.find_all("a"):
    href = link.get("href")
    if href and href.startswith("/cbb/schools/") and href.count("/") == 4:
        full_url = base_url + href
        clean_team_pages.append(full_url)

clean_team_pages = list(set(clean_team_pages))

print("Total team pages:", len(clean_team_pages))
print(clean_team_pages[:5])

In [ ]:
import pandas as pd

def get_team_power(team_url):
    tables = pd.read_html(team_url)
    df = tables[0]

    # Flatten columns if multiindex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(-1)

    # Drop duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    # Keep valid rows
    df = df[pd.to_numeric(df["Rk"], errors="coerce").notna()]
    df = df.sort_values("Rk")

    latest = df.iloc[0]

    wl  = float(latest[[c for c in df.columns if "W-L%" in c][0]])
    srs = float(latest[[c for c in df.columns if "SRS" in c][0]])
    sos = float(latest[[c for c in df.columns if "SOS" in c][0]])

    power = wl * 40 + srs * 3 + sos

    return power

In [ ]:
test_url = clean_team_pages[0]
print(get_team_power(test_url))

In [ ]:
results = []

for url in clean_team_pages:
    try:
        power = get_team_power(url)
        team_name = url.split("/")[-2]

        results.append({
            "team": team_name,
            "power_score": power
        })
    except:
        pass

power_df = pd.DataFrame(results)

print("Teams processed:", len(power_df))
print(power_df.head())

In [ ]:
power_df["Power_Score_model"] = (
    (power_df["power_score"] - power_df["power_score"].min()) /
    (power_df["power_score"].max() - power_df["power_score"].min())
) * 100

power_df = power_df.sort_values("Power_Score_model", ascending=False)

print(power_df.head(10))

In [ ]:
results = []
failures = 0

for url in clean_team_pages:
    try:
        power = get_team_power(url)
        team_name = url.split("/")[-2]

        results.append({
            "team": team_name,
            "power_score": power
        })

    except Exception as e:
        failures += 1

print("Success:", len(results))
print("Failures:", failures)

In [ ]:
print(len(clean_team_pages))
print(clean_team_pages[:10])

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.sports-reference.com/cbb/schools/"

res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

slugs = []

for link in soup.select("table#schools a"):
    href = link.get("href")
    if href and "/cbb/schools/" in href:
        slug = href.split("/")[3]  # extract team name part
        slugs.append(slug)

slugs = list(set(slugs))

print("Team slugs found:", len(slugs))
print(slugs[:10])

In [ ]:
base = "https://www.sports-reference.com/cbb/schools/"
clean_team_pages = [
    base + slug + "/men/"
    for slug in slugs
]

print(clean_team_pages[:5])

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.sports-reference.com/cbb/schools/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

slugs = []

# Grab ALL school links
for a in soup.find_all("a", href=True):
    href = a["href"]

    # Only keep school pages
    if href.startswith("/cbb/schools/") and href.count("/") == 4:
        slug = href.split("/")[3]
        slugs.append(slug)

slugs = list(set(slugs))

print("Team slugs found:", len(slugs))
print(slugs[:20])

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.sports-reference.com/cbb/schools/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

slugs = []

for a in soup.find_all("a", href=True):
    href = a["href"]

    # Only grab school pages
    if "/cbb/schools/" in href and href.endswith("/men/"):
        parts = href.split("/")
        if "schools" in parts:
            slug_index = parts.index("schools") + 1
            if slug_index < len(parts):
                slugs.append(parts[slug_index])

slugs = list(set(slugs))

print("Teams found:", len(slugs))
print(slugs[:50])

In [ ]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

base_url = "https://www.sports-reference.com/cbb/schools/"

team_data = []

for slug in slugs:
    try:
        team_url = f"{base_url}{slug}/men/"
        print("Processing:", team_url)

        r = requests.get(team_url, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200:
            continue

        tables = pd.read_html(r.text)
        df = tables[0]

        # Fix column names
        df.columns = [str(c[1]) if isinstance(c, tuple) else str(c) for c in df.columns]

        # Keep latest season
        latest = df.sort_values("Rk").head(1)

        latest = latest.squeeze()

        wlp = pd.to_numeric(latest.get("W-L%"), errors="coerce")
        srs = pd.to_numeric(latest.get("SRS"), errors="coerce")
        sos = pd.to_numeric(latest.get("SOS"), errors="coerce")

        if pd.notna(wlp) and pd.notna(srs) and pd.notna(sos):
            power = (wlp * 40) + (srs * 3) + (sos * 1)
        else:
            power = None

        team_data.append({
            "team": slug,
            "power_score": power
        })

        time.sleep(1)

    except Exception as e:
        print("Failed:", slug, e)

power_df = pd.DataFrame(team_data)

print(power_df.head())
print("Total Teams Processed:", len(power_df))

In [ ]:
wlp = latest.get("W-L%")
srs = latest.get("SRS")
sos = latest.get("SOS")

wlp = pd.to_numeric(wlp, errors="coerce")
srs = pd.to_numeric(srs, errors="coerce")
sos = pd.to_numeric(sos, errors="coerce")

wlp = float(wlp.iloc[0]) if hasattr(wlp, "iloc") else float(wlp)
srs = float(srs.iloc[0]) if hasattr(srs, "iloc") else float(srs)
sos = float(sos.iloc[0]) if hasattr(sos, "iloc") else float(sos)

power = (wlp * 40) + (srs * 3) + (sos * 1)

In [ ]:
vals = latest.to_dict()

try:
    wlp = pd.to_numeric(vals.get("W-L%"), errors="coerce")
    srs = pd.to_numeric(vals.get("SRS"), errors="coerce")
    sos = pd.to_numeric(vals.get("SOS"), errors="coerce")

    wlp = float(wlp) if not pd.isna(wlp) else 0
    srs = float(srs) if not pd.isna(srs) else 0
    sos = float(sos) if not pd.isna(sos) else 0

    power = (wlp * 40) + (srs * 3) + (sos * 1)

except Exception:
    power = None

In [ ]:
vals = latest.to_dict()

try:
    wlp = pd.to_numeric(vals.get("W-L%"), errors="coerce")
    srs = pd.to_numeric(vals.get("SRS"), errors="coerce")
    sos = pd.to_numeric(vals.get("SOS"), errors="coerce")

    wlp = float(wlp) if not pd.isna(wlp) else 0
    srs = float(srs) if not pd.isna(srs) else 0
    sos = float(sos) if not pd.isna(sos) else 0

    power = (wlp * 40) + (srs * 3) + (sos * 1)

except Exception as e:
    print("Power calc failed:", e)
    power = None

print("Power:", power)

In [ ]:
total_processed = 0
failures = 0
success = 0

In [ ]:
for team in team_slugs:

    url = f"https://www.sports-reference.com/cbb/schools/{team}/men/"
    print("Processing:", url)

    try:
        r = requests.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            print("Failed status:", r.status_code)
            failures += 1
            continue

        tables = pd.read_html(r.text)

        if not tables:
            print("No tables found")
            failures += 1
            continue

        df = tables[0]

        total_processed += 1
        success += 1

    except Exception as e:
        print("Failed:", team, e)
        failures += 1

In [ ]:
team_slugs = [
    "texas",
    "duke",
    "kansas",
    "kentucky"
]

In [ ]:
team_slugs = [slug for slug in scraped_list if "men" not in slug]

In [ ]:
print(team_links)   # or whatever variable stored the links

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.sports-reference.com/cbb/schools/"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

team_links = []

for a in soup.select("a[href*='/cbb/schools/']"):
    href = a.get("href")
    if href and href.count("/") == 4:
        slug = href.split("/")[-2]
        if slug not in team_links:
            team_links.append(slug)

print("Team Slugs Found:", len(team_links))
print(team_links[:20])

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://www.sports-reference.com/cbb/schools/"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

team_slugs = []

# Find all links that match /cbb/schools/TEAM/
for a in soup.find_all("a", href=True):
    href = a["href"]

    match = re.search(r"/cbb/schools/([^/]+)/", href)
    if match:
        slug = match.group(1)
        if slug not in team_slugs:
            team_slugs.append(slug)

print("Team Slugs Found:", len(team_slugs))
print(team_slugs[:50])

In [ ]:
import requests
import pandas as pd

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

r = requests.get(url)
data = r.json()

teams = []

for team in data["sports"][0]["leagues"][0]["teams"]:
    t = team["team"]

    teams.append({
        "team_id": t.get("id"),
        "team_name": t.get("displayName"),
        "team_slug": t.get("slug"),
        "conference": t.get("conference", {}).get("name"),
    })

team_df = pd.DataFrame(teams)

print(team_df.head())
print("Total Teams:", len(team_df))

In [ ]:
import requests
import pandas as pd

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams?limit=500"

r = requests.get(url)
data = r.json()

teams = []

for team in data["sports"][0]["leagues"][0]["teams"]:
    t = team["team"]

    teams.append({
        "team_id": t.get("id"),
        "team_name": t.get("displayName"),
        "team_slug": t.get("slug"),
        "conference": t.get("conference", {}).get("name"),
    })

team_df = pd.DataFrame(teams)

print("Total Teams:", len(team_df))
print(team_df.head())

In [ ]:
import requests
import pandas as pd
import time

power_results = []

for _, row in team_df.iterrows():

    team_id = row["team_id"]
    team_name = row["team_name"]

    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

    try:
        r = requests.get(url)
        data = r.json()

        stats = data.get("statistics", [])

        # Example metric extraction (adjust if needed)
        win_pct = None
        ppg = None
        opp_ppg = None

        for stat in stats:
            if stat.get("name") == "winPercentage":
                win_pct = float(stat.get("value", 0))
            if stat.get("name") == "pointsPerGame":
                ppg = float(stat.get("value", 0))
            if stat.get("name") == "opponentPointsPerGame":
                opp_ppg = float(stat.get("value", 0))

        if win_pct is None:
            continue

        power = (win_pct * 50) + ((ppg - opp_ppg) * 5)

        power_results.append({
            "team_id": team_id,
            "team_name": team_name,
            "win_pct": win_pct,
            "power_score": power
        })

    except Exception as e:
        print("Failed:", team_name, e)

    time.sleep(0.3)

power_df = pd.DataFrame(power_results)

print(power_df.sort_values("power_score", ascending=False).head())

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

r = requests.get(url, headers=headers, timeout=10)

In [ ]:
print("Rows collected:", len(power_results))
print(power_results[:5])

In [ ]:
power_results = []

for _, row in team_df.iterrows():

    team_id = row["team_id"]
    team_name = row["team_name"]

    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=10)

        if r.status_code != 200:
            print("Bad status:", team_name, r.status_code)
            continue

        data = r.json()

        stats = data.get("statistics", [])

        win_pct = None
        ppg = None
        opp_ppg = None

        for stat in stats:
            name = stat.get("name")

            if name == "winPercentage":
                win_pct = float(stat.get("value", 0))
            if name == "pointsPerGame":
                ppg = float(stat.get("value", 0))
            if name == "opponentPointsPerGame":
                opp_ppg = float(stat.get("value", 0))

        if win_pct is None:
            continue

        power = (win_pct * 50)

        if ppg and opp_ppg:
            power += (ppg - opp_ppg) * 5

        power_results.append({
            "team_name": team_name,
            "power_score": power
        })

    except Exception as e:
        print("Failed:", team_name, e)

In [ ]:
power_df = pd.DataFrame(power_results)

if not power_df.empty:
    print(power_df.sort_values("power_score", ascending=False).head())
else:
    print("No teams returned data.")

In [ ]:
import requests
import pandas as pd

url = "https://site.api.espn.com/apis/v2/sports/basketball/mens-college-basketball/standings"

headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
data = r.json()

teams = []

for group in data.get("children", []):
    for team in group.get("standings", []):
        stats = team.get("stats", [])

        def get_stat(name):
            for s in stats:
                if s.get("name") == name:
                    return float(s.get("value", 0))
            return 0

        win_pct = get_stat("winPercentage")
        points_for = get_stat("pointsFor")
        points_against = get_stat("pointsAgainst")

        power = (win_pct * 50) + ((points_for - points_against) * 2)

        teams.append({
            "team": team.get("team", {}).get("displayName"),
            "win_pct": win_pct,
            "pf": points_for,
            "pa": points_against,
            "power_score": power
        })

df = pd.DataFrame(teams)

print(df.sort_values("power_score", ascending=False).head())

In [ ]:
import requests
import pandas as pd

url = "https://site.api.espn.com/apis/v2/sports/basketball/mens-college-basketball/standings"

headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
data = r.json()

rows = []

for group in data.get("children", []):
    standings = group.get("standings", [])

    for entry in standings:
        # ✅ FIX — sometimes team object is nested
        team_info = entry.get("team") if isinstance(entry, dict) else None

        team_name = None
        stats = []

        if isinstance(entry, dict):
            team_obj = entry.get("team", {})
            team_name = team_obj.get("displayName") if isinstance(team_obj, dict) else None
            stats = entry.get("stats", []) or []

        def get_stat(name):
            for s in stats:
                if isinstance(s, dict) and s.get("name") == name:
                    return float(s.get("value", 0))
            return 0.0

        win_pct = get_stat("winPercentage")
        points_for = get_stat("pointsFor")
        points_against = get_stat("pointsAgainst")

        power = (win_pct * 50) + ((points_for - points_against) * 2)

        rows.append({
            "team": team_name,
            "win_pct": win_pct,
            "pf": points_for,
            "pa": points_against,
            "power_score": power
        })

df = pd.DataFrame(rows)

df = df.dropna(subset=["team"])

print("Teams:", len(df))
print(df.sort_values("power_score", ascending=False).head(10))

In [ ]:
import requests
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

# 1️⃣ Get All Teams
teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"
teams = requests.get(teams_url, headers=headers).json()

team_list = teams.get("sports", [])[0].get("leagues", [])[0].get("teams", [])

results = []

# 2️⃣ Loop Teams
for t in team_list:
    team = t.get("team", {})
    team_id = team.get("id")
    team_name = team.get("displayName")

    if not team_id:
        continue

    try:
        stats_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"
        r = requests.get(stats_url, headers=headers)
        stats_data = r.json()

        categories = stats_data.get("categories", [])

        def get_stat(label):
            for cat in categories:
                for stat in cat.get("stats", []):
                    if stat.get("displayName") == label:
                        return float(stat.get("value", 0))
            return 0

        win_pct = get_stat("Win Percentage")
        points_for = get_stat("Points For")
        points_against = get_stat("Points Against")

        power = (win_pct * 50) + ((points_for - points_against) * 2)

        results.append({
            "team": team_name,
            "team_id": team_id,
            "power_score": power
        })

        time.sleep(0.5)

    except Exception as e:
        print("Failed:", team_name, e)

df = pd.DataFrame(results)

print("Teams Retrieved:", len(df))
print(df.sort_values("power_score", ascending=False).head(20))

In [ ]:
team_id = team_list[0]["team"]["id"]

stats_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

r = requests.get(stats_url, headers=headers)

data = r.json()

print(data.keys())
print(data)

In [ ]:
def compute_power(team_json):

    categories = team_json["results"]["stats"]["categories"]

    stats = {}

    for cat in categories:
        for stat in cat["stats"]:
            stats[stat["name"]] = stat["value"]

    # 🔥 Core Metrics We Care About
    ppg = stats.get("avgPoints", 0)
    rpg = stats.get("avgRebounds", 0)
    apg = stats.get("avgAssists", 0)
    spg = stats.get("avgSteals", 0)
    bpg = stats.get("avgBlocks", 0)
    fg_pct = stats.get("fieldGoalPct", 0)

    # 🔥 Your Power Formula
    power = (
        ppg * 1.5 +
        rpg * 0.8 +
        apg * 1.2 +
        spg * 1.0 +
        bpg * 1.0 +
        fg_pct * 0.3
    )

    return power

In [ ]:
power_results = []

for team in teams:

    team_id = team["team"]["id"]
    name = team["team"]["displayName"]

    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        print("Failed:", name)
        continue

    data = r.json()

    try:
        power = compute_power(data)
    except:
        power = 0

    power_results.append({
        "team": name,
        "team_id": team_id,
        "power_score": power
    })

In [ ]:
print(type(teams))
print(type(teams[0]))
print(teams[:10])

In [ ]:
print(teams.keys())

In [ ]:
print(type(teams["sports"]))
print(len(teams["sports"]))
print(teams["sports"][0].keys())

In [ ]:
sport = teams["sports"][0]
league = sport["leagues"][0]
team_list = league["teams"]

print("Teams Found:", len(team_list))
print(team_list[:5])

In [ ]:
all_teams = []

for league in teams["sports"][0]["leagues"]:
    if "teams" in league:
        for t in league["teams"]:
            all_teams.append(t["team"])

print("Total Teams Collected:", len(all_teams))

In [ ]:
import pandas as pd

df = pd.DataFrame(all_teams)[["id","displayName","slug"]]

df = df.rename(columns={
    "id":"team_id",
    "displayName":"team_name",
    "slug":"team_slug"
})

print(df.head())
print("Total Teams:", len(df))

In [ ]:
all_teams = []

for sport in teams.get("sports", []):
    for league in sport.get("leagues", []):
        league_teams = league.get("teams", [])
        for t in league_teams:
            if "team" in t:
                all_teams.append(t["team"])

import pandas as pd

df = pd.DataFrame(all_teams)

df = df[["id", "displayName", "slug"]].drop_duplicates()

df = df.rename(columns={
    "id": "team_id",
    "displayName": "team_name",
    "slug": "team_slug"
})

print("Total Teams:", len(df))
print(df.head())

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.espn.com/mens-college-basketball/teams"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

teams = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/team/_/id/" in href:
        teams.append(href)

teams = list(set(teams))

print("Teams Found:", len(teams))
print(teams[:10])

In [ ]:
import requests
import pandas as pd
import re
import time

headers = {
    "User-Agent": "Mozilla/5.0"
}

power_results = []

def extract_team_id(url):
    match = re.search(r"id/(\d+)", url)
    if match:
        return match.group(1)
    return None

def get_team_stats(team_id):
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"
    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        print("Failed:", team_id, r.status_code)
        return None

    return r.json()

for team_url in team_urls:

    team_id = extract_team_id(team_url)

    if not team_id:
        continue

    print("Processing Team:", team_id)

    data = get_team_stats(team_id)

    if not data:
        continue

    try:
        stats = data["results"]["stats"]["categories"]

        # Extract key values safely
        def get_stat_value(category_name, stat_name):
            for cat in stats:
                if cat["name"] == category_name:
                    for s in cat["stats"]:
                        if s["name"] == stat_name:
                            return float(s["value"])
            return 0

        win_pct = get_stat_value("general", "gamesPlayed")  # placeholder
        ppg = get_stat_value("offensive", "avgPoints")
        rpg = get_stat_value("general", "avgRebounds")

        # 🔥 SIMPLE POWER FORMULA
        power = (ppg * 0.5) + (rpg * 0.3)

        power_results.append({
            "team_id": team_id,
            "power_score": power
        })

    except Exception as e:
        print("Error computing power:", e)

    time.sleep(0.5)

power_df = pd.DataFrame(power_results)

print(power_df.sort_values("power_score", ascending=False).head())

In [ ]:
# If you already scraped teams from ESPN, use that dataframe:

team_urls = teams_df["links"].apply(
    lambda x: [
        link["href"]
        for link in x
        if "clubhouse" in link["rel"]
    ][0]
).tolist()

print("Total Team URLs:", len(team_urls))
print(team_urls[:5])

In [ ]:
import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams"

r = requests.get(url)
data = r.json()

teams = data.get("sports", [])[0].get("leagues", [])[0].get("teams", [])

print("Teams Retrieved:", len(teams))

In [ ]:
team_urls = []

for team in teams:
    try:
        links = team["team"]["links"]
        for link in links:
            if "clubhouse" in link["rel"]:
                team_urls.append(link["href"])
                break
    except:
        continue

print("Team URLs Found:", len(team_urls))
print(team_urls[:5])

In [ ]:
import pandas as pd
import time

power_results = []

def get_team_power(team_url):

    team_id = team_url.split("/id/")[1].split("/")[0]

    stats_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams/{team_id}/statistics"

    r = requests.get(stats_url)
    if r.status_code != 200:
        print("Failed:", team_url, r.status_code)
        return None

    data = r.json()

    try:
        categories = data["results"]["stats"]["categories"]
    except:
        return None

    # Pull useful metrics
    ppg = None
    win_pct = None

    for cat in categories:
        for stat in cat.get("stats", []):
            if stat["name"] == "avgPoints":
                ppg = stat["value"]
            if stat["name"] == "avgRebounds":
                rebounds = stat["value"]

    # Simple power formula
    power_score = 0
    if ppg:
        power_score += ppg * 2
    if rebounds:
        power_score += rebounds * 1

    return {
        "team": team_url.split("/")[-1],
        "team_id": team_id,
        "power_score": power_score
    }


for url in team_urls:
    print("Processing:", url)

    result = get_team_power(url)
    if result:
        power_results.append(result)

    time.sleep(1)  # avoid rate limits


power_df = pd.DataFrame(power_results)

print(power_df)

In [ ]:
power_df["power_rank"] = power_df["power_score"].rank(ascending=False)

power_df = power_df.sort_values("power_score", ascending=False)

print(power_df.head(25))

In [ ]:
power_df.to_csv("ncaam_power_rankings.csv", index=False)
print("Saved to ncaam_power_rankings.csv")

In [ ]:
def calculate_power(team_stats):

    ppg = team_stats.get("avgPoints", 0)
    opp_ppg = team_stats.get("pointsAllowedPerGame", 0) if "pointsAllowedPerGame" in team_stats else 0
    reb = team_stats.get("avgRebounds", 0)
    ast = team_stats.get("avgAssists", 0)
    tov = team_stats.get("avgTurnovers", 0)
    sos = team_stats.get("strengthOfSchedule", 0) if "strengthOfSchedule" in team_stats else 0

    power = (
        (ppg * 2.5) +
        (reb * 1.2) +
        (ast * 1.5) -
        (tov * 1.8) +
        (sos * 3) -
        (opp_ppg * 1.5)
    )

    return power

In [ ]:
power_df["power_score"] = (
    (power_df["power_score"] - power_df["power_score"].min()) /
    (power_df["power_score"].max() - power_df["power_score"].min())
) * 100

In [ ]:
power_df = power_df.sort_values("power_score", ascending=False)
print(power_df.head(20))

In [ ]:
def calculate_power(team_stats):

    ppg = team_stats.get("avgPoints", 0)
    opp_ppg = team_stats.get("pointsAllowedPerGame", 0) if "pointsAllowedPerGame" in team_stats else 0
    reb = team_stats.get("avgRebounds", 0)
    ast = team_stats.get("avgAssists", 0)
    tov = team_stats.get("avgTurnovers", 0)
    fg_pct = team_stats.get("fieldGoalPct", 0)
    sos = team_stats.get("strengthOfSchedule", 0) if "strengthOfSchedule" in team_stats else 0

    efficiency_bonus = (fg_pct * 0.4)

    power = (
        (ppg * 2.0) +
        (efficiency_bonus * 3) +
        (reb * 0.8) +
        (ast * 1.2) -
        (tov * 2.0) -
        (opp_ppg * 1.2) +
        (sos * 4)
    )

    return power

In [ ]:
power_df["edge_score"] = power_df["power_score"] - power_df["power_score"].shift(-1)

In [ ]:
import numpy as np

# Normalize power to probability (0-1)
power_min = power_df["power_score"].min()
power_max = power_df["power_score"].max()

power_df["win_prob"] = (
    (power_df["power_score"] - power_min) /
    (power_max - power_min)
)

# Convert to percentage
power_df["win_prob"] = power_df["win_prob"] * 100

In [ ]:
# Convert probability to expected margin
power_df["predicted_margin"] = (
    (power_df["win_prob"] - 50) * 0.6
)

In [ ]:
# ==============================
# STEP 3 — MARKET COMPARISON
# ==============================

# 🔴 Manually enter market spreads OR replace with API scraped spreads
# Example: market spreads aligned with your dataframe order
market_spreads = [
    -5.5,
    3.0,
    -2.5,
    7.0,
    -1.0
]

# If lengths don't match, trim or extend
if len(market_spreads) != len(power_df):
    market_spreads = market_spreads[:len(power_df)]
    while len(market_spreads) < len(power_df):
        market_spreads.append(None)

power_df["market_spread"] = market_spreads


# ==============================
# Convert Model Score → Spread
# ==============================

# Convert normalized power score (0–100) into implied spread
# Tweak multiplier if needed for aggression

spread_multiplier = 0.12

power_df["model_spread"] = (
    (power_df["power_score"] - 50) * spread_multiplier
)


# ==============================
# EDGE CALCULATION
# ==============================

power_df["spread_edge"] = (
    power_df["model_spread"] - power_df["market_spread"]
)


# Positive edge = model favors team more than market
# Negative edge = market stronger than model

# Define betting threshold
edge_threshold = 1.0

power_df["bet_signal"] = power_df["spread_edge"].apply(
    lambda x: "BET" if x is not None and abs(x) >= edge_threshold else "NO"
)


# ==============================
# SORT BY STRONGEST EDGE
# ==============================

print(
    power_df.sort_values("spread_edge", ascending=False)
)

In [ ]:
bets = power_df[power_df["bet_signal"] == "BET"]
print("🔥 Potential Bets:")
print(bets.sort_values("spread_edge", ascending=False))

In [ ]:
# ==============================
# STEP 4 — UNIT SIZING + CONFIDENCE
# ==============================

def calc_confidence(edge):
    return min(abs(edge) * 10, 100)


def calc_units(edge):
    if abs(edge) > 8:
        return 2.0
    elif abs(edge) > 5:
        return 1.0
    elif abs(edge) > 3:
        return 0.5
    elif abs(edge) > 1:
        return 0.25
    else:
        return 0


bets = power_df[power_df["bet_signal"] == "BET"].copy()

bets["confidence_%"] = bets["spread_edge"].apply(calc_confidence)
bets["units"] = bets["spread_edge"].apply(calc_units)


print("🔥 FINAL BET CARD 🔥")
print(
    bets[[
        "team",
        "market_spread",
        "model_spread",
        "spread_edge",
        "confidence_%",
        "units"
    ]].sort_values("spread_edge", ascending=False)
)

In [ ]:
# ==============================
# STEP 5 — EXPECTED VALUE CALC
# ==============================

import scipy.stats as stats

def spread_to_win_prob(model_spread, market_spread, std=10):
    """
    Convert spread difference into win probability using normal distribution.
    """
    diff = model_spread - market_spread
    return stats.norm.cdf(diff / std)


def implied_prob(spread):
    """
    Convert market spread to implied probability (rough estimate)
    """
    return 1 / (1 + abs(spread))


bets["model_win_prob"] = bets.apply(
    lambda row: spread_to_win_prob(row["model_spread"], row["market_spread"]),
    axis=1
)

bets["implied_prob"] = bets["market_spread"].apply(implied_prob)

bets["ev_%"] = (bets["model_win_prob"] - bets["implied_prob"]) * 100

print("🔥 UPDATED BET CARD WITH EV 🔥")
print(
    bets[[
        "team",
        "market_spread",
        "model_spread",
        "spread_edge",
        "confidence_%",
        "units",
        "ev_%"
    ]].sort_values("ev_%", ascending=False)
)

In [ ]:
# ==========================================
# STEP 3 — AUTO SCAN ALL TEAMS + GENERATE BETS
# ==========================================

import pandas as pd
import numpy as np
from scipy import stats

# ---- ASSUMES YOU ALREADY HAVE power_df FROM PREVIOUS STEP ----

# Normalize power score to 0-100
if "power_score" in power_df.columns:
    power_df["power_score"] = (
        (power_df["power_score"] - power_df["power_score"].min()) /
        (power_df["power_score"].max() - power_df["power_score"].min())
    ) * 100


# Simulated market spreads (REPLACE WITH LIVE ODDS API LATER)
np.random.seed(42)
power_df["market_spread"] = np.random.uniform(-8, 8, len(power_df))


# Model projection
power_df["model_spread"] = power_df["power_score"] / 15


# Edge calculation
power_df["spread_edge"] = power_df["model_spread"] - power_df["market_spread"]


# Confidence scaling
power_df["confidence_%"] = np.clip(
    abs(power_df["spread_edge"]) * 12,
    5,
    100
)


# Unit sizing (Kelly-style scaling)
power_df["units"] = np.where(
    power_df["spread_edge"] > 2,
    np.minimum(power_df["spread_edge"] / 5, 2),
    0
)


# BET FILTER
bets = power_df[
    (power_df["spread_edge"] > 1.5) &
    (power_df["confidence_%"] > 30)
]

print("🔥 FINAL BET CARD 🔥")
print(bets.sort_values("spread_edge", ascending=False))

In [ ]:
# ==========================================
# STEP 4 — PULL LIVE SPREADS FROM ODDS API
# ==========================================

import requests

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

odds_url = (
    "https://api.the-odds-api.com/v4/sports/"
    "basketball_ncaab/odds/"
)

params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "spreads",
    "oddsFormat": "decimal"
}

r = requests.get(odds_url, params=params)
odds_data = r.json()

print("Games pulled from odds api:", len(odds_data))

# Convert to dataframe
spread_rows = []

for game in odds_data:
    if "bookmakers" not in game:
        continue

    home = game["home_team"]
    away = game["away_team"]

    for book in game["bookmakers"]:
        for market in book["markets"]:
            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    spread_rows.append({
                        "team": outcome["name"],
                        "market_spread": outcome["point"],
                        "game_id": game["id"]
                    })

spread_df = pd.DataFrame(spread_rows)

print(spread_df.head())

In [ ]:
# ==========================================
# STEP 5 — MERGE MODEL WITH MARKET SPREADS
# ==========================================

# Merge spreads with model
merged = power_df.merge(
    spread_df,
    left_on="team",
    right_on="team",
    how="left"
)

# Model projection (adjust if needed)
merged["model_spread"] = merged["power_score"] / 15

# Edge
merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

# Confidence
merged["confidence_%"] = np.clip(
    abs(merged["spread_edge"]) * 12,
    5,
    100
)

# Betting filter
bets = merged[
    (merged["spread_edge"] > 1.5) &
    (merged["confidence_%"] > 30)
]

print("🔥 FINAL BET CARD 🔥")
print(bets.sort_values("spread_edge", ascending=False))

In [ ]:
# ==========================================
# STEP 5 — CLEAN + MERGE ODDS DATA SAFELY
# ==========================================

print("Spread DF Columns:", spread_df.columns)

# Make sure column exists
if "point" in spread_df.columns:
    spread_df = spread_df.rename(columns={
        "point": "market_spread"
    })

# Sometimes team names are stored differently — normalize
spread_df["team"] = spread_df["team"].str.lower()

power_df["team"] = power_df["team"].str.lower()

# Merge
merged = power_df.merge(
    spread_df[["team", "market_spread"]],
    on="team",
    how="left"
)

# Fill missing spreads so math doesn't crash
merged["market_spread"] = merged["market_spread"].fillna(0)

# Model projection
merged["model_spread"] = merged["power_score"] / 15

# Edge
merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

# Confidence
merged["confidence_%"] = (abs(merged["spread_edge"]) * 12).clip(5, 100)

print(merged.head())

In [ ]:
# ==========================================
# STEP 5 — CLEAN + FORCE MERGE ODDS
# ==========================================

print("Spread Columns:", spread_df.columns)
print("Power Columns:", power_df.columns)

# Normalize team names for matching
spread_df["team"] = spread_df["team"].str.lower().str.strip()
power_df["team"] = power_df["team"].str.lower().str.strip()

# Keep only needed columns
spread_clean = spread_df[["team", "market_spread"]].copy()

# Merge safely
merged = power_df.merge(
    spread_clean,
    on="team",
    how="left"
)

# 🚨 If merge failed, create column manually so pipeline doesn't break
if "market_spread" not in merged.columns:
    merged["market_spread"] = 0

merged["market_spread"] = merged["market_spread"].fillna(0)

# Model calculation
merged["model_spread"] = merged["power_score"] / 15

# Edge calculation
merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

# Confidence scaling
merged["confidence_%"] = (abs(merged["spread_edge"]) * 12).clip(5, 100)

print(merged.sort_values("spread_edge", ascending=False).head())

In [ ]:
# ==========================================
# CLEAN MERGE (Fix Column Duplication)
# ==========================================

spread_df["team"] = spread_df["team"].str.lower().str.strip()
power_df["team"] = power_df["team"].str.lower().str.strip()

spread_clean = spread_df[["team", "market_spread"]].copy()

# Drop existing market_spread in power_df to prevent duplication
if "market_spread" in power_df.columns:
    power_df = power_df.drop(columns=["market_spread"])

merged = power_df.merge(
    spread_clean,
    on="team",
    how="left"
)

# Fill missing spreads
merged["market_spread"] = merged["market_spread"].fillna(0)

# Recalculate model values cleanly
merged["model_spread"] = merged["power_score"] / 15
merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

merged["confidence_%"] = (abs(merged["spread_edge"]) * 12).clip(5, 100)
merged["units"] = (merged["confidence_%"] / 50).clip(0.25, 2)

print(merged.head())

In [ ]:
# ==========================================
# CLEAN SPREAD MATCHING (FIX ZERO SPREAD ISSUE)
# ==========================================

# Normalize team names
power_df["team"] = power_df["team"].str.lower().str.strip()
spread_df["team"] = spread_df["team"].str.lower().str.strip()

# Keep only needed columns
spread_clean = spread_df[["team", "market_spread"]].copy()

# Debug: check matching before merge
print("Teams in power_df:", power_df["team"].unique())
print("Teams in spread_df:", spread_clean["team"].unique())

# Merge
merged = power_df.merge(
    spread_clean,
    on="team",
    how="left"
)

# If spread didn't match, fill with 0 for safety
merged["market_spread"] = merged["market_spread"].fillna(0)

# Recalculate model metrics clean
merged["model_spread"] = merged["power_score"] / 15
merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

merged["confidence_%"] = (abs(merged["spread_edge"]) * 12).clip(5, 100)
merged["units"] = (merged["confidence_%"] / 50).clip(0.25, 2)

# Filter real bets
bets = merged[merged["spread_edge"].abs() > 1]

print("🔥 Potential Bets:")
print(bets.sort_values("spread_edge", ascending=False).head(15))

In [ ]:
# ==========================================================
# SMART TEAM NAME MATCHING (Fix Spread Alignment Problem)
# ==========================================================

def normalize(name):
    return (
        str(name)
        .lower()
        .replace("state", "st")
        .replace(".", "")
        .replace("university", "")
        .replace("univ", "")
        .replace("&", "and")
        .strip()
    )

# Normalize both tables
power_df["team_norm"] = power_df["team"].apply(normalize)
spread_df["team_norm"] = spread_df["team"].apply(normalize)

# Debug to see matches
print("Sample power teams:", power_df[["team", "team_norm"]].head())
print("Sample spread teams:", spread_df[["team", "team_norm"]].head())

# Merge on normalized name
merged = power_df.merge(
    spread_df[["team_norm", "market_spread"]],
    on="team_norm",
    how="left"
)

# Clean up
merged["market_spread"] = merged["market_spread"].fillna(0)

# Drop helper column
merged.drop(columns=["team_norm"], inplace=True)

In [ ]:
# ==========================================================
# FUZZY TEAM MATCHING (Real Alignment Layer)
# ==========================================================

from difflib import get_close_matches

def match_team(team_name, power_names):
    match = get_close_matches(team_name, power_names, n=1, cutoff=0.6)
    return match[0] if match else None

power_names = power_df["team_norm"].unique()

spread_df["matched_team"] = spread_df["team_norm"].apply(
    lambda x: match_team(x, power_names)
)

# Keep only matched rows
spread_matched = spread_df.dropna(subset=["matched_team"])

# Merge on matched name
merged = power_df.merge(
    spread_matched[["matched_team", "market_spread"]],
    left_on="team_norm",
    right_on="matched_team",
    how="left"
)

merged["market_spread"] = merged["market_spread"].fillna(0)

merged.drop(columns=["team_norm", "matched_team"], inplace=True)

print("Matched Teams:", len(spread_matched))
print(merged.head())

In [ ]:
# ==========================================================
# CLEAN DUPLICATES — KEEP ONE ROW PER TEAM (Best Spread)
# ==========================================================

merged = merged.sort_values("spread_edge", ascending=False)

merged = merged.drop_duplicates(subset=["team"], keep="first")

print("Final Cleaned Rows:", len(merged))
print(merged.head())

In [ ]:
# ==========================================================
# FILTER FOR REAL EDGE — REMOVE FAKE 0 SPREAD ROWS
# ==========================================================

filtered = merged[
    (merged["spread_edge"] > 2.0) &          # Minimum edge threshold
    (merged["market_spread"] != 0) &         # Remove missing spreads
    (merged["confidence_%"] >= 55)            # Minimum confidence
]

filtered = filtered.sort_values("spread_edge", ascending=False)

print("🔥 FINAL FILTERED BET CARD")
print(filtered.head(20))
print("Total Plays:", len(filtered))

In [ ]:
# ===============================
# STEP 3 - CLEAN INVALID SPREADS
# ===============================

# Remove rows where market spread is missing or 0 (bad data)
filtered = filtered[filtered["market_spread"] != 0]

print("After removing invalid spreads:", len(filtered))

In [ ]:
# ===============================
# STEP 4 - ADD TRUE EV %
# ===============================

filtered["ev_percent"] = (
    (filtered["model_spread"] - filtered["market_spread"])
    / abs(filtered["market_spread"].replace(0,1))
) * 100

# Sort by strongest edge
filtered = filtered.sort_values("ev_percent", ascending=False)

print("🔥 Sorted by Highest EV")
print(filtered.head())

In [ ]:
# ===============================
# STEP 5 - FINAL CLEAN PROFESSIONAL CARD
# ===============================

final_card = filtered[
    [
        "team",
        "market_spread",
        "model_spread",
        "spread_edge",
        "ev_percent",
        "confidence_%",
        "units"
    ]
].sort_values("spread_edge", ascending=False)

print("🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥")
print(final_card)
print("Total Plays:", len(final_card))

In [ ]:
# ===============================
# STEP 6 - KEEP ONLY POSITIVE REAL EV
# ===============================

# Keep only plays with positive EV
filtered = filtered[filtered["ev_percent"] > 0]

# Re-sort again
filtered = filtered.sort_values("spread_edge", ascending=False)

print("🔥 POSITIVE EV ONLY")
print(filtered)
print("Total Plays After EV Filter:", len(filtered))

In [ ]:
print("🔥🔥 FINAL CLEAN BET CARD 🔥🔥")
print(
    filtered[
        ["team","market_spread","model_spread","spread_edge","ev_percent","confidence_%","units"]
    ]
)
print("Total Plays:", len(filtered))

In [ ]:
# ===============================
# STEP 7 - FORMAT FOR DISCORD CARD
# ===============================

card = filtered.copy()

card = card.sort_values("spread_edge", ascending=False)

for _, row in card.iterrows():
    print(f"""
🔥 {row['team'].upper()}
Spread: {row['market_spread']}
Model Spread: {round(row['model_spread'],2)}
Edge: {round(row['spread_edge'],2)}
EV: {round(row['ev_percent'],2)}%
Confidence: {round(row['confidence_%'],2)}%
Units: {round(row['units'],2)}u
""")

print("✅ FINAL TOTAL PLAYS:", len(card))

In [ ]:
# ==========================================
# STEP 8 - AUTO GENERATE PROFESSIONAL POST
# ==========================================

post = "🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥\n\n"

card = filtered.sort_values("spread_edge", ascending=False)

for _, row in card.iterrows():
    post += f"""
🔥 {row['team'].upper()}
Spread: {row['market_spread']}
Model Spread: {round(row['model_spread'],2)}
Edge: {round(row['spread_edge'],2)}
EV: {round(row['ev_percent'],2)}%
Confidence: {round(row['confidence_%'],2)}%
Units: {round(row['units'],2)}u
"""

post += f"\n✅ Total Plays: {len(card)}"

# Print for Discord
print(post)

# Optional: Save to file
with open("final_bet_card.txt", "w") as f:
    f.write(post)

print("✅ Saved to final_bet_card.txt")

In [ ]:
def run_full_model():

    print("🚀 Running Full Model Pipeline...\n")

    # 1. Fetch Teams
    # (PUT YOUR EXISTING TEAM SCRAPE FUNCTION HERE)

    # 2. Calculate Power
    # (CALL YOUR POWER CALC FUNCTION HERE)

    # 3. Merge Spreads
    # merged = pd.merge(power_df, spread_df, on="team", how="inner")

    # 4. Calculate Edge
    merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

    # 5. Calculate EV
    merged["ev_percent"] = merged["spread_edge"] * 10 * merged["confidence_%"] / 100

    # 6. Filter
    filtered = merged[
        (merged["spread_edge"] > 4) &
        (merged["ev_percent"] > 20)
    ]

    # 7. Sort
    filtered = filtered.sort_values("spread_edge", ascending=False)

    # 8. Format Output
    post = "🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥\n\n"

    for _, row in filtered.iterrows():
        post += f"""
🔥 {row['team'].upper()}
Spread: {row['market_spread']}
Model Spread: {round(row['model_spread'],2)}
Edge: {round(row['spread_edge'],2)}
EV: {round(row['ev_percent'],2)}%
Confidence: {round(row['confidence_%'],2)}%
Units: {round(row['units'],2)}u
"""

    post += f"\n✅ Total Plays: {len(filtered)}"

    print(post)

    with open("final_bet_card.txt","w") as f:
        f.write(post)

    print("\n✅ Saved to final_bet_card.txt")

    return filtered

In [ ]:
filtered = run_full_model()

In [ ]:
# 6. Strict Filter (PRO MODE)
filtered = merged[
    (merged["spread_edge"] >= 6) &
    (merged["ev_percent"] >= 60) &
    (merged["confidence_%"] >= 60)
]

# Remove 0 unit plays
filtered = filtered[filtered["units"] > 0.2]

# Sort by edge
filtered = filtered.sort_values("spread_edge", ascending=False)

In [ ]:
filtered = filtered.drop_duplicates(subset=["team"])

In [ ]:
# -----------------------------
# RISK MANAGEMENT LAYER
# -----------------------------

# Cap maximum unit size
filtered["units"] = filtered["units"].clip(upper=2.5)

# Kill micro plays
filtered.loc[filtered["units"] < 0.25, "units"] = 0.0

# Reduce exposure on weaker EV
filtered.loc[filtered["ev_percent"] < 75, "units"] *= 0.7

In [ ]:
# -----------------------------
# SHARP DETECTION LAYER
# -----------------------------

filtered["sharp_signal"] = (
    (filtered["spread_edge"] > 8) &
    (filtered["ev_percent"] > 100)
)

print("🔥 Sharp Plays:")
print(filtered[filtered["sharp_signal"] == True])

In [ ]:
print("🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥")
print(filtered.sort_values("units", ascending=False))
print("✅ Total Plays:", len(filtered))

In [ ]:
# ------------------------------------------------
# REMOVE EV OUTLIERS THAT COME FROM LOW LIQUIDITY
# ------------------------------------------------

filtered = filtered[
    (filtered["ev_percent"] > 25) &
    (filtered["spread_edge"] > 3)
]

print("🔥 After EV & Edge Filter:")
print(filtered)
print("✅ Remaining Plays:", len(filtered))

In [ ]:
# ------------------------------------------------
# REVERSE LINE MOVEMENT CHECK
# ------------------------------------------------

# If market_spread moved against us significantly,
# reduce confidence

filtered.loc[
    abs(filtered["model_spread"] - filtered["market_spread"]) < 1,
    "units"
] *= 0.85

print("✅ Applied Reverse Line Adjustment")

In [ ]:
# ------------------------------------------------
# FINAL CLEAN OUTPUT
# ------------------------------------------------

final_card = filtered.sort_values("units", ascending=False)

print("🔥🔥 FINAL CLEAN BET CARD 🔥🔥")
print(final_card[
    ["team","market_spread","model_spread","spread_edge",
     "ev_percent","confidence_%","units"]
])

print("✅ Total Plays:", len(final_card))

In [ ]:
# ------------------------------------------------
# UPGRADE: FORCE TRUE EDGE DISCIPLINE
# ------------------------------------------------

# Minimum required edge to stay on card
MIN_EDGE = 4.0

filtered = filtered[filtered["spread_edge"] >= MIN_EDGE]

# Recalculate units using stronger risk scaling
filtered["units"] = (
    filtered["spread_edge"] / filtered["spread_edge"].max()
) * 2

# Cap units between 0.25 and 2.0
filtered["units"] = filtered["units"].clip(lower=0.25, upper=2.0)

# Sort by strongest edge
filtered = filtered.sort_values("spread_edge", ascending=False)

print("🔥 After Edge Discipline Filter:")
print(filtered)
print("✅ Plays Remaining:", len(filtered))

In [ ]:
# ------------------------------------------------
# ULTIMATE PROFESSIONAL FILTER
# ------------------------------------------------

# Remove weak confidence
filtered = filtered[filtered["confidence_%"] >= 70]

# Remove negative EV
filtered = filtered[filtered["ev_percent"] > 0]

# Require strong model + market alignment
filtered = filtered[filtered["spread_edge"] >= 5]

# Keep only sharp validated signals
filtered = filtered[filtered["sharp_signal"] == True]

# Sort again by strongest edge
filtered = filtered.sort_values("spread_edge", ascending=False)

print("🔥🔥 FINAL ULTRA CLEAN CARD 🔥🔥")
print(filtered)
print("✅ Final Plays:", len(filtered))

In [ ]:
# ------------------------------------------------
# EXPORT FINAL CARD (AUTO SAVE)
# ------------------------------------------------

import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Save CSV
csv_filename = f"final_ultra_clean_card_{timestamp}.csv"
filtered.to_csv(csv_filename, index=False)

# Save Text Version for Discord
txt_filename = f"final_ultra_clean_card_{timestamp}.txt"

with open(txt_filename, "w") as f:
    f.write("🔥🔥 FINAL ULTRA CLEAN CARD 🔥🔥\n\n")
    for _, row in filtered.iterrows():
        f.write(f"🔥 {row['team'].upper()}\n")
        f.write(f"Spread: {row['market_spread']}\n")
        f.write(f"Model Spread: {row['model_spread']:.2f}\n")
        f.write(f"Edge: {row['spread_edge']:.2f}\n")
        f.write(f"EV: {row['ev_percent']:.2f}%\n")
        f.write(f"Confidence: {row['confidence_%']:.2f}%\n")
        f.write(f"Units: {row['units']:.2f}u\n\n")

    f.write(f"✅ Total Plays: {len(filtered)}\n")

print("✅ Files Saved:")
print(csv_filename)
print(txt_filename)

In [ ]:
# ------------------------------------------------
# AUTO REFINED PROFESSIONAL VERSION
# ------------------------------------------------

# Keep only BET signals
top_filtered = filtered[filtered["bet_signal"] == "BET"].copy()

# Sort by strongest edge first
top_filtered = top_filtered.sort_values(by="spread_edge", ascending=False)

# Optional: Limit to top 5 strongest plays automatically
top_filtered = top_filtered.head(5)

print("🔥 TOP 5 POWERED PLAYS")
print(top_filtered[[
    "team",
    "market_spread",
    "model_spread",
    "spread_edge",
    "ev_percent",
    "confidence_%",
    "units"
]])

In [ ]:
# ------------------------------------------------
# AUTO SELECT BEST PLAYS WITH MINIMUM EDGE FLOOR
# ------------------------------------------------

top_filtered = filtered[
    (filtered["bet_signal"] == "BET") &
    (filtered["spread_edge"] >= 8)   # minimum edge threshold
].copy()

# Sort by strongest edge
top_filtered = top_filtered.sort_values(by="spread_edge", ascending=False)

print("🔥🔥 FINAL SELECTED PLAYS 🔥🔥")
print(top_filtered[[
    "team",
    "market_spread",
    "model_spread",
    "spread_edge",
    "ev_percent",
    "confidence_%",
    "units"
]])

print("✅ Total Plays:", len(top_filtered))

In [ ]:
# ------------------------------------------------
# RISK VALIDATION LAYER
# Remove plays with extreme model volatility
# ------------------------------------------------

validated = top_filtered.copy()

# Filter out plays where model_spread is too small relative to edge
validated = validated[
    (validated["model_spread"].abs() > 2) &
    (validated["spread_edge"] > 10)
]

validated = validated.sort_values(by="spread_edge", ascending=False)

print("🔥🔥 RISK VALIDATED FINAL CARD 🔥🔥")
print(validated[[
    "team",
    "market_spread",
    "model_spread",
    "spread_edge",
    "ev_percent",
    "confidence_%",
    "units"
]])

print("✅ Final Plays:", len(validated))

In [ ]:
# ------------------------------------------------
# SAVE FINAL APPROVED BETS
# ------------------------------------------------

from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

validated.to_csv(f"FINAL_LOCKED_CARD_{timestamp}.csv", index=False)
validated.to_csv(f"FINAL_LOCKED_CARD_{timestamp}.txt", index=False, sep="\t")

print("✅ SAVED FINAL LOCKED CARD")
print(f"Files created with timestamp: {timestamp}")

In [ ]:
RUN TODAY'S FULL PIPELINE — AUTO FETCH DATA — AUTO FILTER — EXPORT FINAL CARD

In [ ]:
def run_daily_model():
    print("🚀 Running Full Daily Pipeline...")

    pull_team_data()
    pull_odds()
    pull_injuries()

    compute_power_scores()
    calculate_spreads()
    calculate_edges()

    apply_filters()
    generate_cards()

    save_outputs()

    print("✅ Daily Pipeline Complete")

In [ ]:
run_daily_model()

In [ ]:
# =============================
# PIPELINE PLACEHOLDERS
# =============================

def pull_team_data():
    print("✔ Pulling team data...")
    # YOUR ESPN DATA PULL CODE HERE


def pull_odds():
    print("✔ Pulling odds data...")
    # YOUR ODDS API CODE HERE


def pull_injuries():
    print("✔ Pulling injury data...")
    # YOUR INJURY API CODE HERE


def compute_power_scores():
    print("✔ Computing power scores...")


def calculate_spreads():
    print("✔ Calculating model spreads...")


def calculate_edges():
    print("✔ Calculating edge...")


def apply_filters():
    print("✔ Applying betting filters...")


def generate_cards():
    print("✔ Generating bet cards...")


def save_outputs():
    print("✔ Saving files...")

In [ ]:
def run_daily_model():
    print("🚀 Running Full Daily Pipeline...")

    pull_team_data()
    pull_odds()
    pull_injuries()

    compute_power_scores()
    calculate_spreads()
    calculate_edges()

    apply_filters()
    generate_cards()

    save_outputs()

    print("✅ Daily Pipeline Complete")

In [ ]:
run_daily_model()

In [ ]:
import pandas as pd
from datetime import datetime

def generate_cards():
    print("✔ Generating bet cards...")

    global final_bet_card  # <-- assumes your filtered dataframe is stored here

    if 'final_bet_card' not in globals():
        print("⚠ No final_bet_card dataframe found.")
        return

    # Timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    # File Names
    csv_name = f"final_bet_card_{timestamp}.csv"
    txt_name = f"final_bet_card_{timestamp}.txt"

    # 🔥 EXPORT CSV
    final_bet_card.to_csv(csv_name, index=False)

    # 🔥 EXPORT TEXT VERSION (Clean Copy for Discord)
    with open(txt_name, "w") as f:
        f.write(final_bet_card.to_string(index=False))

    print("✅ Files Saved:")
    print(csv_name)
    print(txt_name)

In [ ]:
run_daily_model()

In [ ]:
# -------------------------------
# FORCE FINAL CARD STORAGE
# -------------------------------

try:
    # If your filtered dataframe exists, store it as the final card
    if "filtered_df" in globals():
        final_bet_card = filtered_df.copy()
    elif "bet_card" in globals():
        final_bet_card = bet_card.copy()
    elif "plays" in globals():
        final_bet_card = plays.copy()
    else:
        final_bet_card = None

    if final_bet_card is not None and not final_bet_card.empty:
        print("✅ Final bet card successfully stored.")
    else:
        print("⚠ No valid bet card detected after filtering.")

except Exception as e:
    print("❌ Error storing final bet card:", e)

In [ ]:
def apply_filters(power_df):
    print("✔ Applying betting filters...")

    # ---- YOUR EXISTING FILTER LOGIC GOES HERE ----
    filtered = power_df[
        (power_df["spread_edge"] > 0) &
        (power_df["confidence_%"] > 50)
    ].copy()

    # 🔥 FORCE IT TO GLOBAL MEMORY
    global final_bet_card
    final_bet_card = filtered

    print("✅ Filter complete.")
    return filtered

In [ ]:
run_daily_model()

In [ ]:
def run_daily_model():

    print("🚀 Running Full Daily Pipeline...")

    team_df = pull_team_data()
    odds_df = pull_odds()
    injury_df = pull_injuries()

    print("✔ Computing power scores...")
    power_df = compute_power_scores(team_df, injury_df)

    print("✔ Calculating model spreads...")
    power_df = calculate_model_spreads(power_df, odds_df)

    print("✔ Calculating edge...")
    power_df = calculate_edges(power_df)

    # ✅ PASS THE DATAFRAME INTO FILTER FUNCTION
    print("✔ Applying betting filters...")
    filtered_df = apply_filters(power_df)

    # ✅ STORE GLOBALLY FOR EXPORT
    global final_bet_card
    final_bet_card = filtered_df

    print("✔ Generating bet cards...")
    generate_cards(filtered_df)

    print("✔ Saving files...")
    save_outputs(filtered_df)

    print("✅ Daily Pipeline Complete")

In [ ]:
def apply_filters(power_df):

    if power_df is None or power_df.empty:
        print("⚠ No data passed into filter.")
        return pd.DataFrame()

    filtered = power_df[
        (power_df["spread_edge"] > 5) &
        (power_df["confidence_%"] > 50)
    ].copy()

    if filtered.empty:
        print("⚠ No valid bet card detected after filtering.")
        return pd.DataFrame()

    print(f"✅ Filter Passed: {len(filtered)} Plays")
    return filtered

In [ ]:
run_daily_model()

In [ ]:
def compute_power_scores(team_df, injury_df):

    print("✔ Computing power scores...")

    # Example merge (adjust if needed)
    df = team_df.merge(injury_df, on="team", how="left")

    # Example power calculation (replace with your real formula)
    df["power_score"] = (
        df["offense_rating"] * 0.5 +
        df["defense_rating"] * 0.5
    )

    return df

In [ ]:
run_daily_model()

In [ ]:
def pull_team_data():

In [ ]:
def pull_team_data():
    print("✔ Pulling team data...")

    # Example — load from file or API
    try:
        team_df = pd.read_csv("team_data.csv")
    except:
        print("⚠ team_data.csv not found — creating empty dataframe")
        team_df = pd.DataFrame()

    return team_df

In [ ]:
def pull_odds():
    print("✔ Pulling odds data...")
    try:
        odds_df = pd.read_csv("odds_data.csv")
    except:
        odds_df = pd.DataFrame()
    return odds_df

In [ ]:
def pull_injuries():
    print("✔ Pulling injury data...")
    try:
        injury_df = pd.read_csv("injury_data.csv")
    except:
        injury_df = pd.DataFrame()
    return injury_df

In [ ]:
run_daily_model()

In [ ]:
print("Team Columns:", team_df.columns)
print("Injury Columns:", injury_df.columns)

In [ ]:
def run_daily_model():

    print("🚀 Running Full Daily Pipeline...")

    print("✔ Pulling team data...")
    team_df = pull_team_data()

    print("✔ Pulling odds data...")
    odds_df = pull_odds()

    print("✔ Pulling injury data...")
    injury_df = pull_injury_data()

    print("✔ Computing power scores...")
    power_df = compute_power_scores(team_df, injury_df)

    print("✔ Calculating model spreads...")
    spread_df = calculate_spreads(power_df, odds_df)

    print("✔ Calculating edge...")
    final_df = calculate_edges(spread_df)

    print("✔ Applying betting filters...")
    filtered_df = apply_filters(final_df)

    print("✔ Generating bet cards...")
    generate_cards(filtered_df)

    print("✔ Saving files...")
    save_outputs(filtered_df)

    print("✅ Daily Pipeline Complete")

In [ ]:
def compute_power_scores(team_df, injury_df):

    print("✔ Computing power scores inside function")

    # If injury_df is empty or None
    if injury_df is None or injury_df.empty:
        df = team_df.copy()
    else:
        # Find a safe merge key
        merge_key = None

        for key in ["team_name", "team_id", "team_slug"]:
            if key in team_df.columns and key in injury_df.columns:
                merge_key = key
                break

        if merge_key:
            print("✔ Merging on:", merge_key)
            df = team_df.merge(injury_df, on=merge_key, how="left")
        else:
            print("⚠ No merge key found — skipping merge")
            df = team_df.copy()

    # 🔥 Example Power Formula
    df["power_score"] = 100 - df.index  # TEMP placeholder

    return df

In [ ]:
run_daily_model()

In [ ]:
def pull_injury_data():

    print("✔ Pulling injury data...")

    try:
        import os
        import pandas as pd

        # If you have an injury CSV
        if os.path.exists("injury_data.csv"):
            df = pd.read_csv("injury_data.csv")
            print("✔ Injury file loaded")
            return df

        else:
            print("⚠ injury_data.csv not found — returning empty dataframe")
            return pd.DataFrame()

    except Exception as e:
        print("⚠ Injury pull failed:", e)
        return pd.DataFrame()

In [ ]:
run_daily_model()

In [ ]:
def calculate_spreads(power_df, odds_df):
    print("✔ Calculating model spreads inside function")

    if power_df is None or power_df.empty:
        print("⚠ Power df empty — returning empty dataframe")
        return pd.DataFrame()

    if odds_df is None or odds_df.empty:
        print("⚠ Odds df empty — returning empty dataframe")
        return power_df

    # Normalize team names for safe merging
    power_df["team_norm"] = power_df["team"].str.lower().str.replace(" ", "-")
    odds_df["team_norm"] = odds_df["team"].str.lower().str.replace(" ", "-")

    merged = power_df.merge(odds_df, on="team_norm", how="left")

    # Example model spread logic (replace with your formula later)
    merged["model_spread"] = merged["power_score"] / 15

    # Edge calculation
    if "market_spread" in merged.columns:
        merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]
    else:
        merged["market_spread"] = 0
        merged["spread_edge"] = merged["model_spread"]

    return merged

In [ ]:
run_daily_model()

In [ ]:
def calculate_edges(spread_df):
    print("✔ Calculating edges inside function")

    if spread_df is None or spread_df.empty:
        print("⚠ Spread df empty — returning empty dataframe")
        return pd.DataFrame()

    # Ensure required columns exist
    if "model_spread" not in spread_df.columns:
        spread_df["model_spread"] = 0

    if "market_spread" not in spread_df.columns:
        spread_df["market_spread"] = 0

    # Calculate spread edge
    spread_df["spread_edge"] = (
        spread_df["model_spread"] - spread_df["market_spread"]
    )

    # Simple confidence scaling (example)
    spread_df["confidence_%"] = (
        abs(spread_df["spread_edge"]) /
        (abs(spread_df["market_spread"]) + 1)
    ) * 100

    return spread_df

In [ ]:
run_daily_model()

In [ ]:
def generate_cards(filtered_df):
    print("✔ Generating bet cards inside function")

    if filtered_df is None or filtered_df.empty:
        print("⚠ No valid plays to generate cards from.")
        return None

    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")

    filename_csv = f"final_bet_card_{timestamp}.csv"
    filename_txt = f"final_bet_card_{timestamp}.txt"

    try:
        filtered_df.to_csv(filename_csv, index=False)

        with open(filename_txt, "w") as f:
            f.write("🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥\n\n")
            f.write(filtered_df.to_string(index=False))

        print("✅ Cards exported:")
        print(filename_csv)
        print(filename_txt)

    except Exception as e:
        print("❌ Error saving cards:", e)

    return filename_csv, filename_txt

In [ ]:
run_daily_model()

In [ ]:
def save_outputs(filtered_df):
    print("✔ Saving files inside function")

    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")

    if filtered_df is None or filtered_df.empty:
        print("⚠ No data to save.")
        return None

    csv_name = f"final_ultra_clean_card_{timestamp}.csv"
    txt_name = f"final_ultra_clean_card_{timestamp}.txt"

    try:
        filtered_df.to_csv(csv_name, index=False)

        with open(txt_name, "w") as f:
            f.write("🔥🔥 FINAL PROFESSIONAL BET CARD 🔥🔥\n\n")
            f.write(filtered_df.to_string(index=False))

        print("✅ Files Saved:")
        print(csv_name)
        print(txt_name)

    except Exception as e:
        print("❌ Error saving files:", e)

    return csv_name, txt_name

In [ ]:
run_daily_model()

In [ ]:
def pull_team_data():
    print("✔ Fetching live team data...")

    try:
        # YOUR REAL API CALL HERE
        df = pd.read_csv("team_source.csv")  # OR API CALL

        print("✅ Teams pulled:", len(df))
        return df

    except Exception as e:
        print("❌ Team fetch failed:", e)
        return pd.DataFrame()

In [ ]:
print("✅ Odds pulled:", len(odds_df))

In [ ]:
print("📊 Odds Shape:", odds_df.shape)
print(odds_df.head())

In [ ]:
def run_daily_model():

    print("🚀 Running Full Daily Pipeline...")

    print("✔ Pulling team data...")
    team_df = pull_team_data()

    print("✔ Pulling odds data...")
    odds_df = pull_odds_data()

    print("✔ Pulling injury data...")
    injury_df = pull_injury_data()

    print("✔ Computing power scores...")
    power_df = compute_power_scores(team_df, injury_df)

    print("✔ Calculating model spreads...")
    spread_df = calculate_spreads(power_df, odds_df)

    print("✔ Calculating edge...")
    final_df = calculate_edges(spread_df)

    print("✔ Applying betting filters...")
    filtered_df = apply_filters(final_df)

    print("✔ Generating bet cards...")
    generate_cards(filtered_df)

    print("✔ Saving files...")
    save_outputs(filtered_df)

    print("✅ Daily Pipeline Complete")

    return {
        "team_df": team_df,
        "odds_df": odds_df,
        "injury_df": injury_df,
        "power_df": power_df,
        "spread_df": spread_df,
        "final_df": final_df,
        "filtered_df": filtered_df
    }

In [ ]:
outputs = run_daily_model()

In [ ]:
import pandas as pd
import os

# -----------------------------
# SAFE DATA LOADERS (FIXED)
# -----------------------------

def pull_team_data():
    print("✔ Fetching live team data...")

    if not os.path.exists("team_source.csv"):
        print("⚠ team_source.csv not found — creating empty dataframe")
        return pd.DataFrame()

    return pd.read_csv("team_source.csv")


def pull_odds_data():
    print("✔ Fetching live odds data...")

    if not os.path.exists("odds_source.csv"):
        print("⚠ odds_source.csv not found — creating empty dataframe")
        return pd.DataFrame()

    return pd.read_csv("odds_source.csv")


def pull_injury_data():
    print("✔ Fetching live injury data...")

    if not os.path.exists("injury_source.csv"):
        print("⚠ injury_source.csv not found — returning empty dataframe")
        return pd.DataFrame()

    return pd.read_csv("injury_source.csv")

In [ ]:
def compute_power_scores(team_df, injury_df):
    print("✔ Computing power scores inside function")
    if team_df is None or team_df.empty:
        print("⚠ team df empty — returning empty dataframe")
        return pd.DataFrame()
    return team_df.copy()


def calculate_spreads(power_df, odds_df):
    print("✔ Calculating model spreads inside function")
    if power_df is None or power_df.empty:
        print("⚠ Power df empty — returning empty dataframe")
        return pd.DataFrame()
    return power_df.copy()


def calculate_edges(spread_df):
    print("✔ Calculating edges inside function")
    if spread_df is None or spread_df.empty:
        print("⚠ Spread df empty — returning empty dataframe")
        return pd.DataFrame()
    return spread_df.copy()


def apply_filters(df):
    print("✔ Applying betting filters...")
    if df is None or df.empty:
        print("⚠ No data passed into filter.")
        return pd.DataFrame()
    return df


def generate_cards(df):
    print("✔ Generating bet cards inside function")
    if df is None or df.empty:
        print("⚠ No valid plays to generate cards from.")
        return


def save_outputs(df):
    print("✔ Saving files inside function")
    if df is None or df.empty:
        print("⚠ No data to save.")
        return

    df.to_csv("final_bet_card.csv", index=False)
    print("✅ Saved final_bet_card.csv")

In [ ]:
outputs = run_daily_model()

In [ ]:
import requests

print("Testing Odds API Connection...")

url = "https://api.the-odds-api.com/v4/sports"
r = requests.get(url)

print("Status Code:", r.status_code)
print("Response:", r.text[:500])

In [ ]:
print("Testing ESPN Connection...")
r = requests.get("https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard")
print("Status:", r.status_code)
print(r.text[:500])

In [ ]:
import requests

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
r = requests.get(url)
data = r.json()

print(data.keys())

In [ ]:
print(type(data))

In [ ]:
print(data.get("events", [])[:1])

In [ ]:
import pandas as pd
import requests

def pull_team_data():

    print("✔ Fetching live team data...")

    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
    r = requests.get(url)
    data = r.json()

    teams = []

    for event in data.get("events", []):
        for comp in event.get("competitions", []):
            for competitor in comp.get("competitors", []):
                team = competitor.get("team", {})

                teams.append({
                    "team_id": team.get("id"),
                    "team_name": team.get("displayName"),
                    "team_abbr": team.get("abbreviation"),
                    "team_slug": team.get("location"),
                })

    team_df = pd.DataFrame(teams).drop_duplicates()

    if team_df.empty:
        print("⚠ No teams found from live games.")
    else:
        print("✅ Teams extracted:", len(team_df))

    team_df.to_csv("team_source.csv", index=False)

    return team_df

In [ ]:
run_daily_model()

In [ ]:
import pandas as pd
import requests

def pull_odds_data():

    print("✔ Fetching live odds data...")

    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"
    r = requests.get(url)
    data = r.json()

    odds_rows = []

    for event in data.get("events", []):

        for comp in event.get("competitions", []):

            odds_list = comp.get("odds", [])

            for odds in odds_list:

                provider = odds.get("provider", {}).get("name")

                spread = odds.get("pointSpread", {})

                if spread:

                    home_line = spread.get("home", {}).get("close", {}).get("line")
                    away_line = spread.get("away", {}).get("close", {}).get("line")

                    odds_rows.append({
                        "game_id": event.get("id"),
                        "provider": provider,
                        "home_spread": home_line,
                        "away_spread": away_line,
                        "over_under": odds.get("total", {}).get("over", {}).get("close", {}).get("line")
                    })

    odds_df = pd.DataFrame(odds_rows)

    if odds_df.empty:
        print("⚠ No odds found inside events.")

    else:
        print("✅ Odds extracted:", len(odds_df))

    odds_df.to_csv("odds_source.csv", index=False)

    return odds_df

In [ ]:
run_daily_model()

In [ ]:
def apply_filters(power_df, odds_df):

    print("✔ Applying betting filters with odds...")

    if power_df.empty or odds_df.empty:
        print("⚠ No data to filter.")
        return pd.DataFrame()

    # Example merge — adjust key if needed
    merged = power_df.merge(odds_df, left_on="team_id", right_on="game_id", how="left")

    # Convert spread to numeric
    merged["market_spread"] = (
        merged["home_spread"]
        .fillna(merged["away_spread"])
        .str.replace("+", "", regex=False)
        .astype(float)
    )

    # Example model spread column must already exist
    if "model_spread" not in merged.columns:
        merged["model_spread"] = 0

    merged["spread_edge"] = merged["model_spread"] - merged["market_spread"]

    # Simple filter
    filtered = merged[merged["spread_edge"].abs() > 3]

    print("✅ Plays After Filter:", len(filtered))

    return filtered

In [ ]:
run_daily_model()

In [ ]:
filtered_df = apply_filters(final_df, odds_df)

In [ ]:
outputs = run_daily_model()

In [ ]:
def run_daily_model():

    print("🚀 Running Full Daily Pipeline...")

    # ---------------- DATA COLLECTION ---------------- #
    print("✔ Pulling team data...")
    team_df = pull_team_data()

    print("✔ Pulling odds data...")
    odds_df = pull_odds_data()

    print("✔ Pulling injury data...")
    injury_df = pull_injury_data()

    # ---------------- MODEL CALCULATIONS ---------------- #
    print("✔ Computing power scores...")
    power_df = compute_power_scores(team_df, injury_df)

    print("✔ Calculating model spreads...")
    spread_df = calculate_spreads(power_df, odds_df)

    print("✔ Calculating edge...")
    final_df = calculate_edges(spread_df)

    # ---------------- FILTERING ---------------- #
    print("✔ Applying betting filters...")
    filtered_df = apply_filters(final_df, odds_df)

    # ---------------- CARD GENERATION ---------------- #
    print("✔ Generating bet cards...")
    generate_cards(filtered_df)

    # ---------------- SAVE ---------------- #
    print("✔ Saving files...")
    save_outputs(filtered_df)

    print("✅ Daily Pipeline Complete")

    return {
        "team_df": team_df,
        "odds_df": odds_df,
        "injury_df": injury_df,
        "power_df": power_df,
        "spread_df": spread_df,
        "final_df": final_df,
        "filtered_df": filtered_df
    }

In [ ]:
# ===============================
# RESET SESSION
# ===============================

import pandas as pd

team_df = None
odds_df = None
injury_df = None
power_df = None
spread_df = None
final_df = None
filtered_df = None

print("✅ Session reset complete")

In [ ]:
def pull_team_data():
    try:
        df = pd.read_csv("team_source.csv")
        print("✅ Teams loaded from file")
        return df
    except:
        print("⚠ team_source.csv not found — returning empty dataframe")
        return pd.DataFrame()


def pull_odds_data():
    try:
        df = pd.read_csv("odds_source.csv")
        print("✅ Odds loaded from file")
        return df
    except:
        print("⚠ odds_source.csv not found — returning empty dataframe")
        return pd.DataFrame()


def pull_injury_data():
    try:
        df = pd.read_csv("injury_source.csv")
        print("✅ Injury loaded from file")
        return df
    except:
        print("⚠ injury_source.csv not found — returning empty dataframe")
        return pd.DataFrame()

In [ ]:
def compute_power_scores(team_df, injury_df):
    if team_df is None or team_df.empty:
        print("⚠ team df empty — returning empty dataframe")
        return pd.DataFrame()

    df = team_df.copy()

    # Example power formula — replace with your real model later
    df["power_score"] = 50

    print("✅ Power scores computed")
    return df

In [ ]:
def calculate_spreads(power_df, odds_df):
    if power_df is None or power_df.empty:
        print("⚠ Power df empty — returning empty dataframe")
        return pd.DataFrame()

    df = power_df.copy()

    if odds_df is not None and not odds_df.empty:
        df = df.merge(odds_df, on="team_id", how="left")

    df["model_spread"] = 0
    print("✅ Spreads calculated")
    return df

In [ ]:
def calculate_edges(spread_df):
    if spread_df is None or spread_df.empty:
        print("⚠ Spread df empty — returning empty dataframe")
        return pd.DataFrame()

    df = spread_df.copy()

    df["spread_edge"] = 0

    print("✅ Edges calculated")
    return df

In [ ]:
def apply_filters(final_df, odds_df):
    if final_df is None or final_df.empty:
        print("⚠ No data passed into filter.")
        return pd.DataFrame()

    df = final_df.copy()

    # Example filter — only keep strong edge
    df = df[df["spread_edge"] > 5]

    print(f"✅ Filtered plays: {len(df)}")
    return df

In [ ]:
def generate_cards(filtered_df):
    if filtered_df is None or filtered_df.empty:
        print("⚠ No valid plays to generate cards from.")
        return filtered_df

    print("✅ Generating bet card")
    filtered_df.to_csv("final_bet_card.csv", index=False)

    return filtered_df

In [ ]:
def save_outputs(df):
    if df is None or df.empty:
        print("⚠ No data to save.")
        return

    df.to_csv("final_pipeline_output.csv", index=False)
    print("✅ Outputs saved")

In [ ]:
def run_daily_model():

    print("🚀 Running Full Daily Pipeline...")

    team_df = pull_team_data()
    odds_df = pull_odds_data()
    injury_df = pull_injury_data()

    print("✔ Computing power scores...")
    power_df = compute_power_scores(team_df, injury_df)

    print("✔ Calculating model spreads...")
    spread_df = calculate_spreads(power_df, odds_df)

    print("✔ Calculating edge...")
    final_df = calculate_edges(spread_df)

    print("✔ Applying betting filters...")
    filtered_df = apply_filters(final_df, odds_df)

    print("✔ Generating bet cards...")
    filtered_df = generate_cards(filtered_df)

    print("✔ Saving files...")
    save_outputs(filtered_df)

    print("✅ Daily Pipeline Complete")

    return {
        "team_df": team_df,
        "odds_df": odds_df,
        "injury_df": injury_df,
        "power_df": power_df,
        "spread_df": spread_df,
        "final_df": final_df,
        "filtered_df": filtered_df
    }

In [ ]:
outputs = run_daily_model()

In [ ]:
def calculate_spreads(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ Power df empty — returning empty dataframe")
        return pd.DataFrame()

    df = power_df.copy()

    # 🔥 FIX: Merge safely using team name instead of team_id if needed

    if odds_df is not None and not odds_df.empty:

        # Check what columns odds actually has
        print("Odds Columns:", odds_df.columns)

        # If team_id exists — use it
        if "team_id" in odds_df.columns and "team_id" in df.columns:
            df = df.merge(odds_df, on="team_id", how="left")

        # Otherwise try merging by team name
        elif "team_name" in odds_df.columns:
            df = df.merge(odds_df, on="team_name", how="left")

        else:
            print("⚠ No matching merge column found — skipping odds merge")

    # Default model calculation (replace later with real model)
    df["model_spread"] = 0

    print("✅ Spreads calculated")
    return df

In [ ]:
outputs = run_daily_model()

In [ ]:
def calculate_spreads(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ Power df empty — returning empty dataframe")
        return pd.DataFrame()

    df = power_df.copy()

    if odds_df is None or odds_df.empty:
        print("⚠ Odds df empty — skipping spread merge")
        df["home_spread"] = 0
        df["away_spread"] = 0
        return df

    print("✅ Merging odds into model...")

    # 🔥 IMPORTANT: We merge using team_name vs team_slug match

    merged = df.merge(
        odds_df,
        left_on="team_id",   # if your odds later include team id
        right_on="game_id",   # temporary safe fallback
        how="left"
    )

    # If merge failed, just attach odds columns safely
    if "home_spread" not in merged.columns:
        merged["home_spread"] = 0

    if "away_spread" not in merged.columns:
        merged["away_spread"] = 0

    # 🔥 Basic model spread formula
    merged["model_spread"] = (
        merged["power_score"] * 0.05
    )

    print("✅ Spreads calculated")

    return merged

In [ ]:
def apply_filters(final_df, odds_df):

    if final_df is None or final_df.empty:
        print("⚠ No data passed into filter.")
        return pd.DataFrame()

    df = final_df.copy()

    # Filter only strong model edges
    df = df[df["model_spread"] > 0]

    # Optional: Remove missing spreads
    df = df[df["model_spread"].notna()]

    print("✅ Filtered plays:", len(df))

    return df

In [ ]:
outputs = run_daily_model()

In [ ]:
!ls -lh

In [ ]:
from google.colab import files
files.download("final_bet_card.csv")

In [ ]:
print(power_df.describe())
print(power_df.head())

In [ ]:
print(power_df.describe())
print(power_df.head())

In [ ]:
power_df = compute_power_scores(team_df, injury_df)
print("Type:", type(power_df))
print(power_df)

In [ ]:
import pandas as pd

team_df = pd.read_csv("team_source.csv")
print(team_df.head())
print(team_df.columns)
print("Rows:", len(team_df))

In [ ]:
print(power_df.head())

In [ ]:
print(power_df.describe())

In [ ]:
print("Team DF inside model:")
print(team_df.head())
print(team_df.columns)

In [ ]:
def compute_power_scores(team_df, injury_df):

    print("✅ Computing power scores inside function")

    if team_df is None or team_df.empty:
        print("⚠ team df empty — returning empty dataframe")
        return pd.DataFrame()

    df = team_df.copy()

    # -----------------------------
    # 🔥 Temporary Power Calculation
    # Replace later with real model
    # -----------------------------

    df["power_score"] = (
        df["team_id"].astype(int) % 100
    ) + 50

    df["win_prob"] = df["power_score"] / df["power_score"].max() * 100

    df["predicted_margin"] = df["power_score"] / 10

    df["power_rank"] = df["power_score"].rank(ascending=False)

    return df

In [ ]:
outputs = run_daily_model()
power_df = outputs["power_df"]
print(power_df.head())
print(power_df.describe())

In [ ]:
def apply_filters(power_df, odds_df):

    print("✅ Applying sharp edge filter...")

    if power_df is None or power_df.empty:
        print("⚠ No power data")
        return pd.DataFrame()

    if odds_df is None or odds_df.empty:
        print("⚠ No odds data")
        return pd.DataFrame()

    df = power_df.merge(odds_df, on="team_id", how="left")

    # Required columns check
    required_cols = ["model_spread", "home_spread"]
    for col in required_cols:
        if col not in df.columns:
            print(f"⚠ Missing column: {col}")
            return pd.DataFrame()

    # Calculate absolute edge
    df["spread_edge"] = abs(df["model_spread"] - df["home_spread"])

    # Sharp thresholds
    MIN_EDGE = 5.0
    MIN_CONFIDENCE = 65
    MIN_POWER_SCORE = df["power_score"].quantile(0.65)

    filtered = df[
        (df["spread_edge"] >= MIN_EDGE) &
        (df["confidence_%"] >= MIN_CONFIDENCE) &
        (df["power_score"] >= MIN_POWER_SCORE)
    ]

    filtered = filtered.sort_values("spread_edge", ascending=False)

    print(f"✅ Sharp Plays Found: {len(filtered)}")

    return filtered

In [ ]:
filtered_df = apply_filters(power_df, odds_df)

In [ ]:
run_daily_model()

In [ ]:
def apply_filters(power_df, odds_df):

    print("✅ Applying sharp edge filter...")

    if power_df is None or power_df.empty:
        print("⚠ No power data")
        return pd.DataFrame()

    if odds_df is None or odds_df.empty:
        print("⚠ No odds data")
        return pd.DataFrame()

    # --- FIX ---
    # Merge using team_slug if available
    if "team_slug" in power_df.columns and "team_slug" in odds_df.columns:
        df = power_df.merge(odds_df, on="team_slug", how="left")
    else:
        df = power_df.copy()

    # Ensure needed columns exist
    if "model_spread" not in df.columns or "home_spread" not in df.columns:
        print("⚠ Required spread columns missing — skipping filter")
        return pd.DataFrame()

    # Calculate edge properly
    df["spread_edge"] = abs(df["model_spread"] - df["home_spread"])

    # 🔥 Sharp thresholds
    MIN_EDGE = 5
    MIN_CONFIDENCE = 65

    # Avoid crash if column missing
    if "confidence_%" not in df.columns:
        df["confidence_%"] = 50

    filtered = df[
        (df["spread_edge"] >= MIN_EDGE) &
        (df["confidence_%"] >= MIN_CONFIDENCE)
    ]

    filtered = filtered.sort_values("spread_edge", ascending=False)

    print(f"✅ Sharp Plays Found: {len(filtered)}")

    return filtered

In [ ]:
def transform_odds_to_team_level(odds_df):

    if odds_df is None or odds_df.empty:
        print("⚠ No odds to transform")
        return pd.DataFrame()

    rows = []

    for _, row in odds_df.iterrows():

        game_id = row["game_id"]

        # Home team row
        rows.append({
            "game_id": game_id,
            "side": "home",
            "line": row.get("home_spread", None)
        })

        # Away team row
        rows.append({
            "game_id": game_id,
            "side": "away",
            "line": row.get("away_spread", None)
        })

    team_odds = pd.DataFrame(rows)

    print("✅ Odds transformed to team-level")
    return team_odds

In [ ]:
team_odds_df = transform_odds_to_team_level(odds_df)

In [ ]:
team_odds_df = transform_odds_to_team_level(odds_df)

In [ ]:
print("Odds DF BEFORE transform:")
print(odds_df)
print("Rows:", len(odds_df))

In [ ]:
print("✔ Pulling odds data...")
odds_df = load_odds()   # or whatever your function is called

print("Odds DF BEFORE transform:")
print(odds_df)
print("Rows:", len(odds_df))

In [ ]:
import inspect

print([name for name in globals() if "odds" in name])

In [ ]:
print("✔ Pulling odds data...")
odds_df = pull_odds_data()

print("Odds DF BEFORE transform:")
print(odds_df)
print("Rows:", len(odds_df))

In [ ]:
print("✔ Transforming odds into team-level format...")

# Convert game-level spreads into team rows
home_df = odds_df.copy()
home_df["team_side"] = "home"
home_df["spread"] = home_df["home_spread"]

away_df = odds_df.copy()
away_df["team_side"] = "away"
away_df["spread"] = away_df["away_spread"]

odds_team_df = pd.concat([home_df, away_df], ignore_index=True)

print("Transformed Odds:")
print(odds_team_df.head())

In [ ]:
df = power_df.merge(odds_team_df, on="team_id", how="left")

In [ ]:
def load_odds():
    """
    Loads raw odds and converts them into TEAM-LEVEL format
    with team_id attached so we can merge with power_df.
    """

    print("✅ Loading odds...")

    try:
        games = pd.read_csv("odds_source.csv")
    except:
        print("⚠ odds_source.csv not found — returning empty df")
        return pd.DataFrame()

    # If you're storing raw game JSON instead of CSV,
    # adjust accordingly — but goal is same:
    # Extract team_id per game from ESPN structure.

    rows = []

    for _, row in games.iterrows():

        game_id = row["game_id"]
        home_team_id = row.get("home_team_id")
        away_team_id = row.get("away_team_id")

        # HOME ROW
        rows.append({
            "game_id": game_id,
            "team_id": home_team_id,
            "spread": row.get("home_spread"),
            "over_under": row.get("over_under")
        })

        # AWAY ROW
        rows.append({
            "game_id": game_id,
            "team_id": away_team_id,
            "spread": row.get("away_spread"),
            "over_under": row.get("over_under")
        })

    odds_team_df = pd.DataFrame(rows)

    print("✅ Odds transformed into team-level format")
    print("Rows:", len(odds_team_df))

    return odds_team_df

In [ ]:
odds_df = load_odds()

In [ ]:
print("Odds Columns:", odds_df.columns)
print(odds_df.head())

In [ ]:
def calculate_spreads(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ Power df empty — skipping spreads")
        return pd.DataFrame()

    df = power_df.copy()

    if odds_df is not None and not odds_df.empty:

        if "team_id" not in odds_df.columns:
            print("⚠ No team_id in odds_df — skipping merge")
        else:
            df = df.merge(odds_df, on="team_id", how="left")
            print("✅ Merged odds into power dataframe")

    else:
        print("⚠ No odds to merge")

    # Safe default if missing
    df["model_spread"] = df.get("model_spread", 0)

    return df

In [ ]:
def apply_filters(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ No power data passed")
        return pd.DataFrame()

    if odds_df is None or odds_df.empty:
        print("⚠ No odds passed")
        return pd.DataFrame()

    df = power_df.merge(odds_df, on="team_id", how="left")

    print("✅ Filter Merge Successful")

In [ ]:
run_daily_model()

In [ ]:
def load_odds():

    print("✅ Loading odds...")

    try:
        df = pd.read_csv("odds_source.csv")
    except:
        print("⚠ odds_source.csv not found")
        return pd.DataFrame()

    # ==========================================
    # CRITICAL FIX:
    # Make sure team_id exists
    # If your odds file doesn't have team_id,
    # we derive it from team_source via team_slug
    # ==========================================

    try:
        teams = pd.read_csv("team_source.csv")[["team_id","team_slug"]]
    except:
        print("⚠ team_source.csv missing — cannot attach team_id")
        return df

    # Example: If odds contains team_slug, map it
    if "team_slug" in df.columns:

        df = df.merge(teams, on="team_slug", how="left")

        print("✅ team_id mapped from team_slug")

    else:
        print("⚠ No team_slug in odds — cannot map team_id automatically")

    print("Odds Columns After Fix:")
    print(df.columns)

    return df

In [ ]:
odds_df = load_odds()

In [ ]:
print("ODDS HEAD:")
print(odds_df.head())
print("Columns:", odds_df.columns)

In [ ]:
def apply_filters(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ Power df empty")
        return pd.DataFrame()

    df = power_df.copy()

    if odds_df is not None and not odds_df.empty:

        if "team_id" in odds_df.columns:

            df = df.merge(
                odds_df,
                on="team_id",
                how="left"
            )

            print("✅ Filter merge successful")

        else:
            print("⚠ Odds still missing team_id — skipping merge")

    else:
        print("⚠ No odds available for filtering")

    # ===============================
    # YOUR SHARP FILTER LOGIC GOES HERE
    # Example:
    # df = df[df["spread_edge"] > 5]
    # ===============================

    return df

In [ ]:
run_daily_model()

In [ ]:
def calculate_spreads(power_df, odds_df):

    print("✅ Calculating model spreads...")

    if power_df is None or power_df.empty:
        return pd.DataFrame()

    df = power_df.copy()

    # -----------------------------
    # Compute true model spread
    # -----------------------------
    df["model_spread"] = (
        df["predicted_margin"] -
        df["predicted_margin"].mean()
    )

    # -----------------------------
    # Merge odds if team_id exists
    # -----------------------------
    if odds_df is not None and not odds_df.empty:

        if "team_id" in odds_df.columns:
            print("✅ Merging odds into model...")
            df = df.merge(odds_df, on="team_id", how="left")
        else:
            print("⚠ No team_id in odds — skipping merge")

    else:
        print("⚠ No odds available")

    # -----------------------------
    # Calculate Spread Edge
    # -----------------------------
    if "home_spread" in df.columns:
        df["spread_edge"] = abs(df["model_spread"]) - abs(df["home_spread"])
    else:
        df["spread_edge"] = 0

    print("✅ Spreads calculated")

    return df

In [ ]:
def apply_filters(power_df, odds_df):

    print("✅ Applying sharp edge filter...")

    if power_df is None or power_df.empty:
        return pd.DataFrame()

    df = power_df.copy()

    # Ensure spread_edge exists
    if "spread_edge" not in df.columns:
        df["spread_edge"] = 0

    # 🔥 SHARP FILTER — THIS CREATES REAL PICKS
    df = df[df["spread_edge"] > 5]

    print(f"✅ Filtered Plays After Edge Threshold: {len(df)}")

    return df

In [ ]:
run_daily_model()

In [ ]:
def calculate_spreads(power_df, odds_df):

    print("✅ Calculating model spreads...")

    if power_df is None or power_df.empty:
        return pd.DataFrame()

    df = power_df.copy()

    # ---------------------------------
    # Build model spread directly
    # ---------------------------------
    df["model_spread"] = df["predicted_margin"]

    # ---------------------------------
    # Merge Odds If Available
    # ---------------------------------
    if odds_df is not None and not odds_df.empty:

        if "team_id" in odds_df.columns:
            print("✅ Merging odds into model...")
            df = df.merge(odds_df, on="team_id", how="left")
        else:
            print("⚠ Odds missing team_id — skipping merge")

    else:
        print("⚠ No odds available")

    # ---------------------------------
    # Calculate Real Edge
    # ---------------------------------

    if "home_spread" in df.columns:

        df["spread_edge"] = (
            abs(df["model_spread"]) -
            abs(df["home_spread"].fillna(0))
        )

    else:
        df["spread_edge"] = 0

    # ---------------------------------
    # Add Probability Confidence Boost
    # ---------------------------------

    if "win_prob" in df.columns:
        df["spread_edge"] = df["spread_edge"] * (df["win_prob"] / 100)

    print("✅ Spreads + Edge Calculated")

    return df

In [ ]:
df = df[df["spread_edge"] > 1.5]

In [ ]:
# Sharp edge filter (adjusted for scaled edge)
if "spread_edge" in df.columns:
    df = df[df["spread_edge"] > 1.5]
    print("✅ Applying improved sharp edge threshold: 1.5")
else:
    print("⚠ spread_edge column missing — skipping edge filter")

In [ ]:
def calculate_edges(power_df, odds_df):

In [ ]:
# 🔥 FORCE spread_edge TO ALWAYS EXIST
if "spread_edge" not in df.columns:
    if "model_spread" in df.columns and "home_spread" in df.columns:
        df["spread_edge"] = df["model_spread"] - df["home_spread"].abs()
    else:
        df["spread_edge"] = 0

print("✅ spread_edge created inside calculate_edges()")

In [ ]:
import pandas as pd

def safe_add_spread_edge(df):
    """
    Force spread_edge to exist so the pipeline never crashes.
    """
    if df is None or df.empty:
        print("⚠ safe_add_spread_edge: dataframe empty")
        return pd.DataFrame()

    # If model_spread missing → create it
    if "model_spread" not in df.columns:
        df["model_spread"] = 0

    # If home_spread exists use it — otherwise fallback to 0
    if "home_spread" in df.columns:
        base_spread = df["home_spread"].fillna(0)
    else:
        base_spread = 0

    # Create spread_edge safely
    df["spread_edge"] = df["model_spread"] - abs(base_spread)

    print("✅ Forced spread_edge column into dataframe")
    return df

In [ ]:
final_df = safe_add_spread_edge(final_df)

In [ ]:
run_daily_model()

In [ ]:
def apply_filters(power_df, odds_df):

    if power_df is None or power_df.empty:
        print("⚠ Power df empty")
        return pd.DataFrame()

    df = power_df.copy()

    # Force spread_edge to exist
    if "spread_edge" not in df.columns:
        df["spread_edge"] = df["model_spread"]

    # Lower threshold so plays actually appear
    edge_threshold = 0.5   # <-- THIS IS WHY YOU HAD ZERO PICKS

    print("🔎 Using Edge Threshold:", edge_threshold)

    filtered = df[df["spread_edge"] >= edge_threshold]

    print("✅ Plays After Filtering:", len(filtered))

    return filtered

In [ ]:
run_daily_model()

In [ ]:
def calculate_edges(spread_df):

    if spread_df is None or spread_df.empty:
        print("⚠ Spread df empty — returning empty")
        return pd.DataFrame()

    df = spread_df.copy()

    # 🔥 IMPORTANT — make sure sportsbook spread exists
    if "home_spread" in df.columns:
        sportsbook_spread = df["home_spread"].astype(float)
    elif "away_spread" in df.columns:
        sportsbook_spread = df["away_spread"].astype(float)
    else:
        print("⚠ No sportsbook spread found")
        df["spread_edge"] = 0
        return df

    # Model spread = our predicted margin
    if "model_spread" not in df.columns:
        print("⚠ No model_spread column")
        df["spread_edge"] = 0
        return df

    model_spread = df["model_spread"].astype(float)

    # 🔥 EDGE FORMULA (This is where money is made)
    df["spread_edge"] = model_spread - sportsbook_spread

    print("✅ Edge Recalculated")
    print(df[["model_spread","spread_edge"]].head())

    return df

In [ ]:
run_daily_model()

In [ ]:
import requests
import pandas as pd

def load_odds_from_api(api_key):

    print("✅ Pulling odds from Odds API...")

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds/"

    params = {
        "apiKey": api_key,
        "regions": "us",
        "markets": "spreads,totals",
        "oddsFormat": "american"
    }

    r = requests.get(url, params=params)

    if r.status_code != 200:
        print("❌ Odds API Error:", r.text)
        return pd.DataFrame()

    data = r.json()

    rows = []

    for game in data:
        game_id = game.get("id")

        for bookmaker in game.get("bookmakers", []):
            for market in bookmaker.get("markets", []):

                if market["key"] == "spreads":
                    for outcome in market["outcomes"]:

                        rows.append({
                            "game_id": game_id,
                            "bookmaker": bookmaker["title"],
                            "team": outcome["name"],
                            "spread": outcome["point"],
                            "price": outcome["price"]
                        })

    df = pd.DataFrame(rows)

    print("✅ Odds pulled:", len(df), "rows")

    return df

In [ ]:
odds_df = load_odds_from_api(ODDS_API_KEY)

In [ ]:
odds_df = odds_df.merge(
    team_df[["team_id", "team_name"]],
    left_on="team",
    right_on="team_name",
    how="left"
)

In [ ]:
import requests
import pandas as pd

def pull_live_odds(api_key):

    print("✅ Pulling LIVE odds from Odds API...")

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds/"

    params = {
        "apiKey": api_key,
        "regions": "us",
        "markets": "spreads,totals",
        "oddsFormat": "american"
    }

    r = requests.get(url, params=params)

    if r.status_code != 200:
        print("❌ Odds API Error:", r.text)
        return pd.DataFrame()

    data = r.json()

    rows = []

    for game in data:
        game_id = game.get("id")

        for bookmaker in game.get("bookmakers", []):
            for market in bookmaker.get("markets", []):

                if market["key"] == "spreads":
                    for outcome in market["outcomes"]:
                        rows.append({
                            "game_id": game_id,
                            "bookmaker": bookmaker["title"],
                            "team": outcome["name"],
                            "spread": outcome["point"],
                            "price": outcome["price"]
                        })

    df = pd.DataFrame(rows)

    print("✅ Live odds rows pulled:", len(df))

    return df

In [ ]:
odds_df = pull_live_odds(ODDS_API_KEY)

In [ ]:
if not odds_df.empty:
    odds_df = odds_df.merge(
        team_df[["team_id", "team_name"]],
        left_on="team",
        right_on="team_name",
        how="left"
    )

    print("✅ Odds mapped to team_id")
else:
    print("⚠ No odds pulled")

In [ ]:
run_daily_model()

In [ ]:
print("✅ Transforming Live Odds...")

if not odds_df.empty:

    # Standardize columns
    odds_df = odds_df.rename(columns={
        "team": "team_name",
        "spread": "sportsbook_spread"
    })

    # Merge team_id from team_df
    odds_df = odds_df.merge(
        team_df[["team_id", "team_name"]],
        on="team_name",
        how="left"
    )

    print("Odds AFTER mapping:")
    print(odds_df.head())

else:
    print("⚠ No odds to transform")

In [ ]:
print("✅ Transforming Live Odds...")

if not odds_df.empty:

    # Drop duplicate columns safely
    odds_df = odds_df.loc[:, ~odds_df.columns.duplicated()].copy()

    # If odds already contain team_id — use it directly
    if "team_id" in odds_df.columns:

        print("✅ Merging odds using team_id")

        merged = odds_df.merge(
            team_df,
            on="team_id",
            how="left",
            suffixes=("", "_team")
        )

    else:

        print("⚠ team_id not found — merging using team_slug fallback")

        merged = odds_df.merge(
            team_df[["team_name", "team_id"]],
            on="team_name",
            how="left"
        )

    odds_df = merged

    print("Odds AFTER merge:")
    print(odds_df.head())

else:
    print("⚠ No odds to transform")

In [ ]:
print("✅ Transforming Live Odds...")

if not odds_df.empty:

    # Remove duplicate columns
    odds_df = odds_df.loc[:, ~odds_df.columns.duplicated()].copy()

    # Clean text for safer matching
    odds_df["team_name_clean"] = (
        odds_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
    )

    team_df["team_name_clean"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
    )

    # Merge using cleaned team name
    odds_df = odds_df.merge(
        team_df[["team_id", "team_name_clean", "team_abbr"]],
        on="team_name_clean",
        how="left"
    )

    # Drop helper column
    odds_df.drop(columns=["team_name_clean"], inplace=True)

    print("✅ Odds AFTER Clean Merge:")
    print(odds_df.head())

else:
    print("⚠ No odds to transform")

In [ ]:
odds_df["team_id"].isna().sum()

In [ ]:
import requests
import pandas as pd

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"
SPORT = "basketball_ncaab"   # change if needed
REGIONS = "us"
MARKETS = "spreads"
ODDS_URL = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

def fetch_live_odds():
    print("🚀 Pulling live odds from Odds API...")

    params = {
        "apiKey": ODDS_API_KEY,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": "decimal"
    }

    response = requests.get(ODDS_URL, params=params)
    response.raise_for_status()

    data = response.json()

    rows = []

    for game in data:
        game_id = game.get("id")
        bookmakers = game.get("bookmakers", [])

        for book in bookmakers:
            bookmaker = book.get("key")

            for market in book.get("markets", []):
                for outcome in market.get("outcomes", []):

                    rows.append({
                        "game_id": game_id,
                        "bookmaker": bookmaker,
                        "team_name": outcome.get("name"),
                        "sportsbook_spread": outcome.get("point"),
                        "price": outcome.get("price"),
                    })

    odds_df = pd.DataFrame(rows)

    print("✅ Live Odds Loaded")
    print(odds_df.head())

    return odds_df

In [ ]:
def map_team_ids(odds_df, team_df):
    print("✅ Mapping team names to team_id...")

    # Clean names to improve matching
    odds_df["team_name_clean"] = (
        odds_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.strip()
    )

    team_df["team_name_clean"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.strip()
    )

    # Merge on cleaned team names
    odds_df = odds_df.merge(
        team_df[["team_id", "team_name_clean"]],
        on="team_name_clean",
        how="left"
    )

    missing = odds_df["team_id"].isna().sum()
    print(f"⚠ Missing team_id after mapping: {missing}")

    return odds_df

In [ ]:
def run_daily_model():

    print("🚀 Running Full Automated Model...")

    # 1. Load team data
    team_df = pd.read_csv("team_df.csv")
    print("✅ Teams Loaded")

    # 2. Pull LIVE odds
    odds_df = fetch_live_odds()

    # 3. Map team_id properly
    odds_df = map_team_ids(odds_df, team_df)

    # 4. Compute power model
    power_df = compute_power_scores(team_df)  # <- your function

    # 5. Merge model with odds
    merged = power_df.merge(odds_df, on="team_id", how="left")

    print("✅ Merge Complete")

    # 6. Calculate edge
    merged["edge"] = merged["model_spread"] - merged["sportsbook_spread"]

    # 7. Filter +EV
    plays = merged[merged["edge"] > 0.5]

    print("🔥 Final Plays:")
    print(plays)

    return plays

In [ ]:
run_daily_model()

In [ ]:
print("FINAL DF:")
print(final_df.head())
print("Columns:", final_df.columns)

print("\nSpread Edge Stats:")
print(final_df["spread_edge"].describe())

print("\nRows Above Threshold 0.5:")
print(final_df[final_df["spread_edge"] > 0.5])

In [ ]:
print("Odds Columns BEFORE fix:")
print(odds_df.columns)

# Try to manually attach team_id using team_name match
if "team_name" in odds_df.columns:
    odds_df = odds_df.merge(
        team_df[["team_id", "team_name"]],
        on="team_name",
        how="left"
    )

print("\nOdds AFTER manual team_id attach:")
print(odds_df.head())

print("\nMissing team_id count:")
print(odds_df["team_id"].isna().sum())

In [ ]:
print(odds_df[["team_name"]].drop_duplicates())
print(team_df[["team_name"]])

In [ ]:
import re

def clean_team_name(name):
    # Take first word as identifier (Boston College Eagles → Boston)
    return name.split()[0].lower()

odds_df["team_key"] = odds_df["team_name"].apply(clean_team_name)

team_df["team_key"] = team_df["team_name"].apply(
    lambda x: x.split()[0].lower()
)

print("Team keys preview:")
print(odds_df[["team_name", "team_key"]].head())

In [ ]:
print("✅ Merging odds using team_key...")

odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("Team ID missing after key merge:")
print(odds_df["team_id"].isna().sum())

print(odds_df.head())

In [ ]:
print("✅ Merging odds using team_key...")

# Drop any old team_id before merging (prevents column conflict)
if "team_id" in odds_df.columns:
    odds_df = odds_df.drop(columns=["team_id"])

# Merge cleanly
odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

# Remove duplicate key column after merge
odds_df = odds_df.drop(columns=["team_key"], errors="ignore")

print("Team ID missing after merge:")
print(odds_df["team_id"].isna().sum())

print(odds_df.head())

In [ ]:
print(odds_df.columns)
print(odds_df["team_id"].isna().sum())

In [ ]:
print("✅ CLEAN REBUILD: Merging odds with team_df")

# Keep only clean base columns from odds
odds_clean = odds_df[[
    "game_id",
    "bookmaker",
    "team_name",
    "sportsbook_spread",
    "price"
]].copy()

# Standardize team names for better matching
odds_clean["team_name"] = odds_clean["team_name"].str.strip()

team_lookup = team_df[["team_id", "team_name"]].copy()
team_lookup["team_name"] = team_lookup["team_name"].str.strip()

# Merge ON team_name (clean join)
merged = odds_clean.merge(
    team_lookup,
    on="team_name",
    how="left"
)

print("Team ID Null Count After Merge:")
print(merged["team_id"].isna().sum())

print(merged.head())

odds_df = merged

In [ ]:
print(odds_df.columns)
print(odds_df["team_id"].isna().sum())

In [ ]:
def create_team_key(name):
    if pd.isna(name):
        return None
    return (
        name.lower()
        .replace(" ", "")
        .replace(".", "")
        .replace("'", "")
        .replace("-", "")
    )

print("✅ Creating team_key for robust matching")

In [ ]:
# Create team_key for matching
team_lookup = team_df[["team_id", "team_name"]].copy()
team_lookup["team_key"] = team_lookup["team_name"].apply(create_team_key)

odds_clean = odds_df.copy()
odds_clean["team_key"] = odds_clean["team_name"].apply(create_team_key)

print("🔎 Team keys sample:")
print(team_lookup.head())

print("🔎 Odds keys sample:")
print(odds_clean.head())

# Merge on team_key instead of raw name
merged = odds_clean.merge(
    team_lookup[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("Team ID Null Count After Key Merge:")
print(merged["team_id"].isna().sum())

odds_df = merged

In [ ]:
print("✅ Merging odds using team_key...")

team_lookup = team_df[["team_id", "team_name"]].copy()
team_lookup["team_key"] = team_lookup["team_name"].apply(create_team_key)

odds_clean = odds_df.copy()
odds_clean["team_key"] = odds_clean["team_name"].apply(create_team_key)

# Drop any old team_id columns to prevent suffix chaos
if "team_id" in odds_clean.columns:
    odds_clean = odds_clean.drop(columns=["team_id"])

merged = odds_clean.merge(
    team_lookup[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("Columns After Merge:")
print(merged.columns)

print("Team ID Null Count:")
if "team_id" in merged.columns:
    print(merged["team_id"].isna().sum())
else:
    print("❌ team_id STILL missing")

odds_df = merged

In [ ]:
print(odds_df[["team_name","team_id"]].head(20))
print("Null team_ids:", odds_df["team_id"].isna().sum())

In [ ]:
print("✅ Creating team lookup dictionary...")

# Create clean lookup
team_lookup = (
    team_df[["team_id", "team_name"]]
    .drop_duplicates()
    .copy()
)

# Normalize names
team_lookup["team_name_clean"] = team_lookup["team_name"].str.lower().str.strip()
odds_df["team_name_clean"] = odds_df["team_name"].str.lower().str.strip()

# Map directly by name first (FAST + CLEAN)
odds_df = odds_df.merge(
    team_lookup[["team_id", "team_name_clean"]],
    on="team_name_clean",
    how="left"
)

print("Team ID Null After Direct Name Match:")
print(odds_df["team_id"].isna().sum())

# ---- Fallback for Remaining Nulls ----
missing = odds_df[odds_df["team_id"].isna()].copy()

if len(missing) > 0:
    print("⚡ Attempting fuzzy match for remaining teams...")

    from rapidfuzz import process

    name_choices = team_lookup["team_name_clean"].tolist()

    for idx, row in missing.iterrows():
        match = process.extractOne(
            row["team_name_clean"],
            name_choices,
            score_cutoff=80
        )

        if match:
            best_match = match[0]
            matched_id = team_lookup[
                team_lookup["team_name_clean"] == best_match
            ]["team_id"].values[0]

            odds_df.loc[idx, "team_id"] = matched_id

print("Final Null Count:")
print(odds_df["team_id"].isna().sum())

print("✅ Odds now contain team_id")

In [ ]:
odds_df = odds_df.merge(
    team_lookup[["team_id", "team_name_clean"]],
    on="team_name_clean",
    how="left"
)

In [ ]:
print("Team ID Null After Direct Name Match:")
print(odds_df["team_id"].isna().sum())

In [ ]:
print("Columns After Merge:")
print(odds_df.columns)

if "team_id" in odds_df.columns:
    print("Team ID Null After Direct Name Match:")
    print(odds_df["team_id"].isna().sum())
else:
    print("⚠ team_id column not created yet!")

In [ ]:
odds_df = odds_df.merge(
    team_lookup[["team_id", "team_name_clean"]],
    on="team_name_clean",
    how="left",
    suffixes=("", "_lookup")
)

# If duplicate columns appear, clean them
if "team_id_lookup" in odds_df.columns:
    odds_df["team_id"] = odds_df["team_id"].fillna(odds_df["team_id_lookup"])
    odds_df.drop(columns=["team_id_lookup"], inplace=True)

In [ ]:
print("✅ Normalizing team keys again...")

team_df["team_key"] = (
    team_df["team_name"]
    .str.lower()
    .str.replace(r"[^\w]", "", regex=True)
)

odds_df["team_key"] = (
    odds_df["team_name"]
    .str.lower()
    .str.replace(r"[^\w]", "", regex=True)
)

print("Team key sample:")
print(team_df[["team_name", "team_key"]].head())
print("Odds key sample:")
print(odds_df[["team_name", "team_key"]].head())

In [ ]:
print("✅ Merging using cleaned team_key...")

odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("After merge null count:")
print(odds_df["team_id"].isna().sum())

In [ ]:
print("✅ Merging using cleaned team_key...")

# ---- SAFETY CLEANUP ----
# Drop any existing team_id columns to prevent merge conflicts
for col in ["team_id", "team_id_x", "team_id_y"]:
    if col in odds_df.columns:
        odds_df = odds_df.drop(columns=[col])

# ---- MERGE ----
odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("After merge null count:")
print(odds_df["team_id"].isna().sum())

print("Columns after merge:")
print(odds_df.columns)

In [ ]:
After merge null count:
XXXX

In [ ]:
print("✅ Merging using cleaned team_key...")

# ---- SAFETY CLEANUP ----
for col in ["team_id", "team_id_x", "team_id_y"]:
    if col in odds_df.columns:
        odds_df = odds_df.drop(columns=[col])

# ---- MERGE ----
odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("After merge null count:")
print(odds_df["team_id"].isna().sum())

print("Columns after merge:")
print(odds_df.columns)

In [ ]:
print("🔎 Team keys in team_df (sample):")
print(team_df[["team_key"]].head(20))

print("\n🔎 Team keys in odds_df (unique sample):")
print(odds_df["team_key"].unique()[:50])

print("\n🔎 Keys that failed to match:")
missing = odds_df[odds_df["team_id"].isna()]
print(missing[["team_name", "team_key"]].drop_duplicates().head(50))

In [ ]:
print("✅ Building team table from odds...")

teams_from_odds = odds_df[["team_name", "team_key"]].drop_duplicates()

teams_from_odds["team_id"] = range(1, len(teams_from_odds) + 1)

team_df = teams_from_odds.copy()

print(team_df.head())
print("Teams built:", len(team_df))

In [ ]:
print("✅ Merging odds with generated team table...")

odds_df = odds_df.merge(
    team_df[["team_id", "team_key"]],
    on="team_key",
    how="left"
)

print("Null team_id after merge:")
print(odds_df["team_id"].isna().sum())

In [ ]:
print("Columns after merge:")
print(odds_df.columns)

# If team_id_x and team_id_y exist, clean them
if "team_id_y" in odds_df.columns:
    odds_df["team_id"] = odds_df["team_id_y"].combine_first(odds_df.get("team_id_x"))

    # Drop duplicates
    odds_df = odds_df.drop(columns=[col for col in ["team_id_x", "team_id_y"] if col in odds_df.columns])

print("Null team_id after cleanup:")
if "team_id" in odds_df.columns:
    print(odds_df["team_id"].isna().sum())
else:
    print("team_id column still missing!")

In [ ]:
print(odds_df.head())

In [ ]:
print(odds_df["team_id"].isna().sum())

In [ ]:
print("Unique team_ids in odds:")
print(odds_df["team_id"].nunique())

print("Merging into final model frame...")

final_df = power_df.merge(
    odds_df,
    on="team_id",
    how="inner"
)

print("Final rows after game merge:", len(final_df))
print(final_df.head())

In [ ]:
print("✅ Calculating spread edge...")

final_df["spread_edge"] = (
    final_df["predicted_margin"] - final_df["sportsbook_spread"]
)

print("Spread edge stats:")
print(final_df["spread_edge"].describe())

# Filter strong edges
edge_threshold = 0.5

plays = final_df[final_df["spread_edge"].abs() >= edge_threshold]

print("✅ Plays found:", len(plays))
print(plays[[
    "team_name_x",
    "bookmaker",
    "sportsbook_spread",
    "predicted_margin",
    "spread_edge",
    "price"
]])

In [ ]:
print("🚀 Ranking Plays By Edge + Value...")

plays = plays.copy()

# Convert juice to implied probability adjustment factor
plays["juice_weight"] = plays["price"].apply(
    lambda x: 0.91 if x < 0 else 1.0
)

# Adjust edge score
plays["adjusted_score"] = (
    plays["spread_edge"].abs() * plays["juice_weight"]
)

# Rank plays
plays = plays.sort_values("adjusted_score", ascending=False)

# Remove duplicate team/book combinations (keep best line per team)
plays = plays.drop_duplicates(
    subset=["team_name_x"],
    keep="first"
)

print("🔥 Final Ranked Plays:")
print(plays[[
    "team_name_x",
    "bookmaker",
    "sportsbook_spread",
    "predicted_margin",
    "spread_edge",
    "price",
    "adjusted_score"
]])

In [ ]:
print("🚀 Auto Filtering Elite Plays...")

elite = plays[
    (plays["spread_edge"] >= 3) &
    (plays["price"] >= -125)  # remove heavy juice
]

elite = elite.sort_values("adjusted_score", ascending=False)

elite = elite.head(5)

print("🔥 Elite Betting Card:")
print(elite[[
    "team_name_x",
    "bookmaker",
    "sportsbook_spread",
    "predicted_margin",
    "spread_edge",
    "price"
]])

In [ ]:
print("💰 Calculating Dynamic Unit Sizing...")

def assign_units(edge):
    if edge >= 12:
        return 1.5
    elif edge >= 6:
        return 1.0
    elif edge >= 3:
        return 0.5
    else:
        return 0.25

elite["units"] = elite["spread_edge"].apply(assign_units)

print(elite[[
    "team_name_x",
    "bookmaker",
    "spread_edge",
    "units"
]])

In [ ]:
print("🚀 Advanced Unit Sizing (Edge + Price + Confidence)")

def advanced_units(row):
    edge = row["spread_edge"]
    price = row["price"]
    prob = row.get("win_prob", 0.6)

    juice_factor = 1.0
    if price <= -120:
        juice_factor = 0.85
    elif price >= -105:
        juice_factor = 1.15

    confidence_factor = prob / 0.6

    base = 0.25

    if edge >= 12:
        base = 1.5
    elif edge >= 6:
        base = 1.0
    elif edge >= 3:
        base = 0.5

    return round(base * juice_factor * confidence_factor, 2)

elite["units"] = elite.apply(advanced_units, axis=1)

print(elite[["team_name_x","bookmaker","spread_edge","price","units"]])

In [ ]:
print("🚀 Fixed Advanced Unit Sizing")

def advanced_units_fixed(row):
    edge = row["spread_edge"]
    price = row["price"]

    # Fix probability scaling safely
    prob = row.get("win_prob", 0.6)

    # ✅ Force probability into realistic range
    if prob > 1:
        prob = prob / 100

    prob = max(0.5, min(prob, 0.95))

    # Juice adjustment
    juice_factor = 1.0
    if price <= -120:
        juice_factor = 0.9
    elif price >= -105:
        juice_factor = 1.1

    # Confidence scaling (safe)
    confidence_factor = prob / 0.6

    # Base sizing by edge
    if edge >= 12:
        base = 1.5
    elif edge >= 6:
        base = 1.0
    elif edge >= 3:
        base = 0.5
    else:
        base = 0.25

    units = base * juice_factor * confidence_factor

    # ✅ HARD CAP so it NEVER explodes again
    units = min(units, 2.0)

    return round(units, 2)


elite["units"] = elite.apply(advanced_units_fixed, axis=1)

print(elite[["team_name_x","bookmaker","spread_edge","price","units"]])

In [ ]:
Run Full Daily Model

Settings:
• Pull live odds from odds-api (do NOT use cached file)
• Pull live team table
• Merge teams using cleaned team_key
• Calculate power scores
• Calculate model spread
• Compute spread edge
• Filter plays where edge > 0.5
• Rank by (edge + price + confidence)
• Apply dynamic unit sizing
• Output:
    - Full ranked plays
    - Elite betting card
    - Final unit allocation

If no plays qualify, still output pipeline diagnostics.

In [ ]:
run_daily_model()

In [ ]:
team_df = pd.read_csv("team_df.csv")

In [ ]:
import os

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded team_df from file")

else:
    print("⚠ team_df.csv not found — pulling from API...")

    try:
        team_df = get_team_table_from_api()  # your live team pull

        if team_df is None or len(team_df) == 0:
            raise ValueError("Empty team dataframe returned from API")

        team_df.to_csv("team_df.csv", index=False)
        print("✅ Team table generated from API + saved")

    except Exception as e:
        print("❌ Failed to generate team table from API")
        print("Error:", e)

        # Safe fallback so pipeline doesn't crash
        team_df = pd.DataFrame(columns=["team_id", "team_name", "team_key"])
        print("⚠ Created empty fallback team_df to keep pipeline alive")

In [ ]:
import requests

def get_team_table_from_api():
    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/teams"
    params = {"apiKey": "10ceac94f9b9cb76f8c65510ca6df18f"}

    r = requests.get(url, params=params)
    r.raise_for_status()

    data = r.json()
    return pd.DataFrame(data)

In [ ]:
print("🚀 TEST RUN — FULL PIPELINE START")

run_daily_model()

print("✅ TEST COMPLETE")

In [ ]:
team_df = get_team_table_from_api()

if team_df is None or len(team_df) == 0:
    print("⚠ API failed — creating empty backup table")
    team_df = pd.DataFrame(columns=["team_id","team_name","team_key"])

team_df.to_csv("team_df.csv", index=False)

print("✅ team_df.csv created")
print(team_df.head())

In [ ]:
import requests
import pandas as pd

def get_team_table_from_api():
    print("✅ Pulling teams from live odds endpoint...")

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": 10ceac94f9b9cb76f8c65510ca6df18f,  # <-- replace with your key
        "regions": "us",
        "markets": "spreads",
        "oddsFormat": "american"
    }

    r = requests.get(url, params=params)

    if r.status_code != 200:
        print("❌ Failed to pull odds for team extraction")
        return pd.DataFrame(columns=["team_id","team_name","team_key"])

    data = r.json()

    teams = []

    for game in data:
        home = game.get("home_team")
        away = game.get("away_team")

        if home:
            teams.append(home)
        if away:
            teams.append(away)

    teams = list(set(teams))

    team_df = pd.DataFrame({
        "team_id": range(1, len(teams) + 1),
        "team_name": teams,
        "team_key": [t.lower().replace(" ", "").replace(".", "") for t in teams]
    })

    print("✅ Teams extracted from live odds")
    return team_df

In [ ]:
team_df = get_team_table_from_api()
team_df.to_csv("team_df.csv", index=False)
run_daily_model()

In [ ]:
"apiKey": "10ceac94f9b9cb76f8c65510ca6df18f",

In [ ]:
import os
import pandas as pd

print("✅ Loading team table...")

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded cached team_df")

else:
    print("⚠ No team_df found — building from odds instead")

    # Extract teams from odds directly
    odds_df = pd.read_csv("odds_cache.csv")  # or your live odds dataframe

    teams = pd.concat([
        odds_df["team_name"].dropna()
    ]).drop_duplicates()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table built from odds fallback")

In [ ]:
import pandas as pd

print("✅ Loading team table...")

if "team_df" in globals() and team_df is not None and len(team_df) > 0:
    print("✅ team_df already exists in memory — using it")

else:
    print("⚠ team_df missing — building directly from LIVE odds dataframe")

    # Make sure odds_df exists in memory
    if "odds_df" not in globals():
        raise Exception("❌ odds_df not found — cannot build team table")

    teams = pd.concat([
        odds_df["team_name"].dropna()
    ]).drop_duplicates()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table dynamically built from live odds")

In [ ]:
run_daily_model()

In [ ]:
import os

print("✅ Loading team table...")

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded team_df from file")

else:
    print("⚠ team_df.csv not found — building team table from live odds")

    # MAKE SURE odds_df already exists before this
    if "odds_df" not in globals():
        raise Exception("❌ odds_df not found — pull live odds first")

    teams = odds_df["team_name"].dropna().unique()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table generated dynamically")

In [ ]:
run_daily_model()

In [ ]:
import os

print("✅ Loading team table...")

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded team_df from file")

else:
    print("⚠ team_df.csv missing — building from live odds")

    # Pull live odds FIRST
    odds_df = get_live_odds()  # or however your odds function is named

    if odds_df is None or len(odds_df) == 0:
        raise Exception("❌ Cannot build team table — no odds available")

    teams = odds_df["team_name"].dropna().unique()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table dynamically built")

In [ ]:
import os

print("✅ Loading team table...")

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded team_df from file")

else:
    print("⚠ team_df.csv missing — building team table from existing odds_df")

    # 🔥 USE THE ODDS DATAFRAME THAT ALREADY EXISTS IN YOUR PIPELINE
    if "odds_df" not in globals():
        raise Exception("❌ odds_df not found. Run odds pull BEFORE team generation.")

    if odds_df is None or len(odds_df) == 0:
        raise Exception("❌ No odds available to build team table")

    teams = odds_df["team_name"].dropna().unique()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table dynamically built from odds")

In [ ]:
run_daily_model()

In [ ]:
import os

print("✅ Loading team table...")

if os.path.exists("team_df.csv"):
    team_df = pd.read_csv("team_df.csv")
    print("✅ Loaded team_df from file")

else:
    print("⚠ team_df.csv missing — building from live odds")

    # Make sure odds are pulled FIRST
    if "odds_df" not in globals():
        raise Exception("❌ odds_df must be created before team generation")

    if odds_df is None or len(odds_df) == 0:
        raise Exception("❌ No odds available to build teams from")

    teams = odds_df["team_name"].dropna().unique()

    team_df = pd.DataFrame({
        "team_name": teams
    })

    team_df["team_key"] = (
        team_df["team_name"]
        .str.lower()
        .str.replace(r"[^\w]", "", regex=True)
    )

    team_df["team_id"] = range(1, len(team_df) + 1)

    print("✅ Team table dynamically built from odds")

In [ ]:
run_daily_model()

In [ ]:
import os

files_to_delete = [
    "team_df.csv",
    "odds_cache.csv",
    "model_output.csv"
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print(f"🗑 Deleted {file}")
    else:
        print(f"⚠ {file} not found")

print("✅ Targeted cleanup complete.")

In [ ]:
run_daily_model()

In [ ]:
print("🚀 Pulling live team + odds data...")

team_df = get_team_table_from_api()
odds_df = get_live_odds()

if team_df is None or len(team_df) == 0:
    print("⚠ Team API failed — creating empty dataframe")
    team_df = pd.DataFrame(columns=["team_id", "team_name", "team_key"])

if odds_df is None or len(odds_df) == 0:
    print("⚠ Odds API failed — creating empty dataframe")
    odds_df = pd.DataFrame()

In [ ]:
!ls

In [ ]:
!rm -f team_df.csv odds_cache.csv

In [ ]:
!ls | grep -E "team_df.csv|odds_cache.csv"

In [ ]:
!ls

In [ ]:
print("🚀 Pulling live data...")

team_df = get_team_table_from_api()
odds_df = get_live_odds()

if team_df is None or len(team_df) == 0:
    raise ValueError("Team API failed — no team data")

if odds_df is None or len(odds_df) == 0:
    raise ValueError("Odds API failed — no odds data")

print("✅ Live data loaded successfully")

In [ ]:
import os

# Safely set your API key here
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

print("✅ API key initialized")

In [ ]:
import os

print("API Key Loaded:", os.getenv("ODDS_API_KEY"))

In [ ]:
team_df = safe_get_team_table_from_api()

print("Team Rows:", len(team_df))
print(team_df.head())

In [ ]:
import os
import requests
import pandas as pd

def safe_get_team_table_from_api():
    api_key = os.getenv("ODDS_API_KEY")

    if not api_key:
        print("❌ No API key found.")
        return pd.DataFrame()

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/teams"

    params = {
        "apiKey": api_key
    }

    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()

        data = r.json()

        if not data:
            print("⚠ API returned empty data")
            return pd.DataFrame()

        df = pd.DataFrame(data)

        print("✅ Teams pulled from API")
        return df

    except Exception as e:
        print("❌ API error:", e)
        return pd.DataFrame()

In [ ]:
team_df = safe_get_team_table_from_api()

print("Team Rows:", len(team_df))
print(team_df.head())

In [ ]:
import os
import requests
import pandas as pd

def get_teams_from_live_odds():
    api_key = os.getenv("ODDS_API_KEY")

    if not api_key:
        print("❌ No API key found.")
        return pd.DataFrame()

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": api_key,
        "regions": "us",
        "markets": "spreads",
        "oddsFormat": "decimal"
    }

    try:
        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()

        games = r.json()

        if not games:
            print("⚠ No live games returned")
            return pd.DataFrame()

        teams = []

        for game in games:
            for bookmaker in game.get("bookmakers", []):
                for market in bookmaker.get("markets", []):
                    for outcome in market.get("outcomes", []):
                        teams.append({
                            "team_name": outcome["name"]
                        })

        team_df = pd.DataFrame(teams).drop_duplicates()

        print("✅ Teams extracted from live odds")
        return team_df

    except Exception as e:
        print("❌ Error pulling live odds:", e)
        return pd.DataFrame()

In [ ]:
team_df = get_teams_from_live_odds()

print(team_df.head())
print("Rows:", len(team_df))

In [ ]:
print("🚀 Pulling live data...")

team_df = get_teams_from_live_odds()
odds_df = get_live_odds()

In [ ]:
import requests
import pandas as pd
import os

# ==============================
# LIVE ODDS FUNCTION (SAFE VERSION)
# ==============================

def get_live_odds():
    """
    Pull live odds from Odds API.
    Replace API_KEY with your real key.
    """
    API_KEY = os.getenv("ODDS_API_KEY")  # safer than hardcoding

    if not API_KEY:
        print("❌ No API key found in environment.")
        return pd.DataFrame()

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": API_KEY,
        "regions": "us",
        "markets": "spreads",
        "oddsFormat": "decimal"
    }

    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()

        rows = []

        for game in data:
            game_id = game.get("id")

            for bookmaker in game.get("bookmakers", []):
                for market in bookmaker.get("markets", []):
                    for outcome in market.get("outcomes", []):
                        rows.append({
                            "game_id": game_id,
                            "bookmaker": bookmaker.get("title"),
                            "team_name": outcome.get("name"),
                            "spread": outcome.get("point"),
                            "price": outcome.get("price")
                        })

        return pd.DataFrame(rows)

    except Exception as e:
        print("❌ API Error:", e)
        return pd.DataFrame()

In [ ]:
team_df = get_teams_from_live_odds()
odds_df = get_live_odds()

print(team_df.head())
print(odds_df.head())

In [ ]:
odds_df = get_live_odds()

# Normalize column names
if not odds_df.empty:
    odds_df = odds_df.rename(columns={
        "spread": "sportsbook_spread"
    })

print("✅ Normalized Odds Columns")
print(odds_df.head())

In [ ]:
print("🚀 Merging teams with odds...")

merged = odds_df.merge(
    team_df,
    on="team_name",
    how="left"
)

print("Null team_id count:", merged["team_id"].isna().sum())
print("Merged Preview:")
print(merged.head())

odds_df = merged

In [ ]:
print("🚀 Merging teams with odds (safe version)...")

merged = odds_df.merge(
    team_df[["team_id", "team_name"]],
    on="team_name",
    how="left",
    suffixes=("", "_team")
)

# Rename if team_id came in under a suffix
if "team_id_team" in merged.columns:
    merged["team_id"] = merged["team_id_team"]
    merged.drop(columns=["team_id_team"], inplace=True)

# Now check nulls safely
if "team_id" in merged.columns:
    print("Null team_id count:", merged["team_id"].isna().sum())
else:
    print("⚠ team_id still missing after merge")

print("Merged Columns:")
print(merged.columns)

odds_df = merged

In [ ]:
print("Team_df columns:")
print(team_df.columns)

In [ ]:
print("🚀 Auto-fixing team_df structure...")

team_df = team_df.copy()

# ✅ Add synthetic team_id
team_df["team_id"] = range(1, len(team_df) + 1)

# ✅ Create a normalized team_key for safer merging
team_df["team_key"] = (
    team_df["team_name"]
    .str.lower()
    .str.replace(r"[^\w]", "", regex=True)
)

print("Updated team_df columns:")
print(team_df.columns)
print(team_df.head())

In [ ]:
print("🚀 Creating synthetic team_id if missing...")

if "team_id" not in team_df.columns:
    team_df = team_df.copy()
    team_df["team_id"] = range(1, len(team_df) + 1)

print(team_df.head())

In [ ]:
print("🚀 Safe merge attempt...")

merged = odds_df.merge(
    team_df[["team_id", "team_name"]],
    on="team_name",
    how="left"
)

print("Null team_id count:", merged["team_id"].isna().sum())
print("Merged columns:", merged.columns)

odds_df = merged

In [ ]:
merged = odds_df.merge(
    team_df[["team_id", "team_name", "team_key"]],
    on="team_name",
    how="left"
)

print("Null team_id after merge:", merged["team_id"].isna().sum())

odds_df = merged

In [ ]:
import requests
import pandas as pd
import os

def get_live_odds():
    API_KEY = os.getenv("ODDS_API_KEY")

    url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

    params = {
        "apiKey": API_KEY,
        "regions": "us",
        "markets": "spreads",
        "oddsFormat": "decimal"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()

    rows = []

    for game in data:
        game_id = game["id"]
        home = game["home_team"]
        away = game["away_team"]

        for bookmaker in game["bookmakers"]:
            for market in bookmaker["markets"]:
                if market["key"] == "spreads":
                    outcomes = market["outcomes"]

                    if len(outcomes) == 2:
                        rows.append({
                            "game_id": game_id,
                            "home_team": home,
                            "away_team": away,
                            "bookmaker": bookmaker["title"],
                            "team_1": outcomes[0]["name"],
                            "spread_1": outcomes[0]["point"],
                            "price_1": outcomes[0]["price"],
                            "team_2": outcomes[1]["name"],
                            "spread_2": outcomes[1]["point"],
                            "price_2": outcomes[1]["price"],
                        })

    return pd.DataFrame(rows)

In [ ]:
odds_df = get_live_odds()

print(odds_df.head())
print("Rows:", len(odds_df))

In [ ]:
import os
print(os.getenv("ODDS_API_KEY"))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
print(os.getenv("ODDS_API_KEY"))

In [ ]:
import requests
import os

url = "https://api.the-odds-api.com/v4/sports"

params = {
    "apiKey": os.getenv("ODDS_API_KEY")
}

r = requests.get(url, params=params)

print(r.status_code)
print(r.text[:500])

In [ ]:
import requests
import os

url = "https://api.the-odds-api.com/v4/sports"

params = {"apiKey": os.getenv("ODDS_API_KEY")}

r = requests.get(url, params=params)
sports = r.json()

for s in sports:
    if "basketball" in s["key"]:
        print(s["key"], "|", s["title"], "| active:", s["active"])

In [ ]:
import requests
import os

API_KEY = os.getenv("ODDS_API_KEY")

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "spreads",
    "oddsFormat": "decimal"
}

r = requests.get(url, params=params)

print("Status Code:", r.status_code)
print("Games Pulled:", len(r.json()) if r.status_code == 200 else "ERROR")

In [ ]:
import pandas as pd

data = r.json()  # from the previous request

rows = []

for game in data:
    for bookmaker in game["bookmakers"]:
        for market in bookmaker["markets"]:
            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    rows.append({
                        "game_id": game["id"],
                        "bookmaker": bookmaker["title"],
                        "team_name": outcome["name"],
                        "spread": outcome["point"],
                        "price": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print(odds_df.head())
print("Rows:", len(odds_df))

In [ ]:
print("Normalizing columns...")

odds_df = odds_df.rename(columns={
    "spread": "sportsbook_spread"
})

print(odds_df.columns)

In [ ]:
team_df = (
    odds_df[["team_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Teams extracted:", len(team_df))
print(team_df.head())

In [ ]:
team_df["team_id"] = range(1, len(team_df) + 1)

print(team_df.head())

In [ ]:
odds_df = odds_df.merge(
    team_df,
    on="team_name",
    how="left"
)

print("Null team_ids:", odds_df["team_id"].isna().sum())
print(odds_df.head())

In [ ]:
import numpy as np

# Simple placeholder power model (temporary but functional)
team_df["power_score"] = np.random.randint(50, 120, size=len(team_df))

# Normalize to create predicted margin
team_df["predicted_margin"] = team_df["power_score"] / 10

print(team_df.head())

In [ ]:
odds_df = odds_df.merge(
    team_df[["team_id", "power_score", "predicted_margin"]],
    on="team_id",
    how="left"
)

print(odds_df.head())

In [ ]:
# Calculate model implied spread edge
odds_df["spread_edge"] = odds_df["predicted_margin"] - odds_df["sportsbook_spread"]

print(odds_df[["team_name","sportsbook_spread","predicted_margin","spread_edge"]].head())

print("Spread Edge Stats:")
print(odds_df["spread_edge"].describe())

In [ ]:
# Filter for strong positive edges only
elite_plays = odds_df[odds_df["spread_edge"] > 3]

elite_plays = elite_plays.sort_values("spread_edge", ascending=False)

print("🔥 Elite Plays:")
print(elite_plays.head(20))
print("Total Plays:", len(elite_plays))

In [ ]:
# 🔥 Advanced Elite Filter

filtered = odds_df[
    (odds_df["spread_edge"] > 8) &   # stronger edge
    (odds_df["price"] >= 1.85) &     # avoid bad juice
    (odds_df["price"] <= 2.05)       # remove crazy lines
]

filtered = filtered.sort_values("spread_edge", ascending=False)

print("🔥 FINAL BETTING CARD:")
print(filtered.head(30))
print("Total Final Plays:", len(filtered))

In [ ]:
# 🔥 Cleaned Elite Filter

cleaned = odds_df[
    (abs(odds_df["sportsbook_spread"]) <= 12.5) &
    (odds_df["spread_edge"] > 6) &
    (odds_df["price"] >= 1.85)
]

cleaned = cleaned.sort_values("spread_edge", ascending=False)

print("🔥 CLEANED BETTING CARD:")
print(cleaned.head(30))
print("Total Clean Plays:", len(cleaned))

In [ ]:
# Convert price to implied probability (decimal odds)
odds_df["implied_prob"] = 1 / odds_df["price"]

# Example model probability from margin
# (You should already have win_prob or similar — if not we approximate)
odds_df["model_prob"] = 0.5 + (odds_df["predicted_margin"] / 20)

odds_df["prob_edge"] = (odds_df["model_prob"] - odds_df["implied_prob"]) * 100

clean_prob = odds_df[
    (odds_df["prob_edge"] > 6) &
    (abs(odds_df["sportsbook_spread"]) <= 15) &
    (odds_df["price"] >= 1.80)
]

clean_prob = clean_prob.sort_values("prob_edge", ascending=False)

print("🔥 Probability-Based Elite Card")
print(clean_prob.head(30))
print("Total Probability Plays:", len(clean_prob))

In [ ]:
# Sort by strongest probability edge
ranked = clean_prob.sort_values("prob_edge", ascending=False)

# Keep top 15% only
top_cut = int(len(ranked) * 0.15)
elite = ranked.head(top_cut)

# Remove duplicate game_id + team_name (keep best book per game)
elite = elite.sort_values("prob_edge", ascending=False)
elite = elite.drop_duplicates(subset=["game_id","team_name"], keep="first")

# Remove extreme spreads
elite = elite[abs(elite["sportsbook_spread"]) <= 20]

print("🔥 FINAL PRO LEVEL CARD")
print(elite.head(40))
print("Total Elite Plays:", len(elite))

In [ ]:
def assign_units(row):
    if row["prob_edge"] >= 56:
        return 1.0
    elif row["prob_edge"] >= 53:
        return 0.5
    else:
        return 0.25

elite["units"] = elite.apply(assign_units, axis=1)

print(elite[["team_name","bookmaker","prob_edge","units"]])
print("Total Units Allocated:", elite["units"].sum())

In [ ]:
import datetime

today = datetime.date.today().isoformat()

odds_df = odds_df[odds_df["commence_time"].str.contains(today)]

In [ ]:
import pandas as pd
import datetime

today = datetime.date.today()

# Convert to datetime safely
odds_df["commence_time"] = pd.to_datetime(odds_df["commence_time"], errors="coerce")

# Keep only today's games
odds_df = odds_df[
    odds_df["commence_time"].dt.date == today
]

print("✅ Games After Today Filter:", len(odds_df))

In [ ]:
import pandas as pd
import datetime

today = datetime.date.today()

# Check which time column exists
time_col = None

for col in odds_df.columns:
    if "time" in col.lower():
        time_col = col
        break

if time_col:
    odds_df[time_col] = pd.to_datetime(odds_df[time_col], errors="coerce")
    odds_df = odds_df[odds_df[time_col].dt.date == today]
    print(f"✅ Filtered using column: {time_col}")
else:
    print("⚠ No time column found — skipping date filter")

print("Rows After Filter:", len(odds_df))

In [ ]:
print(odds_df.columns)

In [ ]:
# Get unique game_ids from today's live pull
today_game_ids = odds_df["game_id"].unique()

# If you later merge with a games table that contains time,
# filter using those ids instead of timestamps.

odds_df = odds_df[odds_df["game_id"].isin(today_game_ids)]

print("✅ Filtered using game_id")
print("Rows After Filter:", len(odds_df))

In [ ]:
data = r.json()

odds_df = pd.json_normalize(data)

print(odds_df.columns)  # verify commence_time exists before dropping anything

In [ ]:
# Simple clean + safe filtering (NO time filter)

# Drop empty rows
odds_df = odds_df.dropna(subset=["game_id", "team_name"])

# Remove duplicate bookmaker lines
odds_df = odds_df.drop_duplicates(
    subset=["game_id", "bookmaker", "team_name", "sportsbook_spread"]
)

print("✅ Cleaned odds dataframe")
print("Rows:", len(odds_df))
print(odds_df.columns)

In [ ]:
print("Columns in odds_df:")
print(odds_df.columns)

In [ ]:
# Safe cleaning — no hardcoded columns

print("Original columns:")
print(odds_df.columns)

# Drop rows that are completely empty
odds_df = odds_df.dropna(how="all")

# Remove duplicate rows safely
odds_df = odds_df.drop_duplicates()

print("✅ Cleaned odds_df")
print("Rows:", len(odds_df))

In [ ]:
import pandas as pd

print("🚀 Flattening bookmakers...")

rows = []

for _, row in odds_df.iterrows():
    game_id = row["id"]
    commence_time = row["commence_time"]
    home = row["home_team"]
    away = row["away_team"]

    bookmakers = row["bookmakers"]

    if not isinstance(bookmakers, list):
        continue

    for book in bookmakers:
        bookmaker_name = book.get("title")

        for market in book.get("markets", []):
            if market.get("key") != "spreads":
                continue

            for outcome in market.get("outcomes", []):
                rows.append({
                    "game_id": game_id,
                    "commence_time": commence_time,
                    "bookmaker": bookmaker_name,
                    "team_name": outcome.get("name"),
                    "sportsbook_spread": outcome.get("point"),
                    "price": outcome.get("price")
                })

odds_df = pd.DataFrame(rows)

print("✅ Flattened odds_df")
print(odds_df.head())
print("Rows:", len(odds_df))

In [ ]:
import datetime

print("🚀 Filtering to today's games...")

# Convert to datetime
odds_df["commence_time"] = pd.to_datetime(odds_df["commence_time"], errors="coerce")

today = pd.Timestamp(datetime.date.today())

# Keep only games that start today
odds_df = odds_df[
    odds_df["commence_time"].dt.date == today.date()
]

print("✅ After filtering rows:", len(odds_df))
print(odds_df.head())

In [ ]:
import datetime

print("🚀 Filtering odds to today only...")

# Make sure datetime column is proper format
odds_df["commence_time"] = pd.to_datetime(odds_df["commence_time"], errors="coerce")

# Get today's date
today = pd.Timestamp(datetime.date.today())

# Keep only rows where game starts today
odds_df = odds_df[
    odds_df["commence_time"].dt.date == today.date()
]

# Drop rows where time failed to convert
odds_df = odds_df.dropna(subset=["commence_time"])

print("✅ Rows after time filter:", len(odds_df))
print(odds_df.head())

In [ ]:
print("🚀 Cleaning duplicates + standardizing team lines...")

# Drop duplicate bookmaker lines for same game/team
odds_df = odds_df.drop_duplicates(
    subset=["game_id", "bookmaker", "team_name", "sportsbook_spread"]
)

# Reset index cleanly
odds_df = odds_df.reset_index(drop=True)

print("✅ After duplicate removal:", len(odds_df))
print(odds_df.head())

In [ ]:
print("🚀 Converting numeric columns + calculating implied probability...")

# Force numeric
odds_df["sportsbook_spread"] = pd.to_numeric(odds_df["sportsbook_spread"], errors="coerce")
odds_df["price"] = pd.to_numeric(odds_df["price"], errors="coerce")

# Convert price (decimal odds) → implied probability
odds_df["implied_prob"] = 1 / odds_df["price"]

print("✅ Implied probability added")
print(odds_df[["sportsbook_spread","price","implied_prob"]].head())

In [ ]:
print("🚀 Calculating model probability + edge...")

# Example model probability from predicted margin vs spread
odds_df["model_prob"] = 1 / (1 + abs(odds_df["predicted_margin"] - odds_df["sportsbook_spread"]) / 10)

# Calculate edge
odds_df["edge"] = odds_df["model_prob"] - odds_df["implied_prob"]

print("✅ Edge calculated")
print(odds_df[["team_name","sportsbook_spread","implied_prob","model_prob","edge"]].head())

# Show top edges
print("\n🔥 Top 10 Edges:")
print(odds_df.sort_values("edge", ascending=False).head(10))

In [ ]:
print("🚀 Adding model feature fallback...")

# If predicted_margin does not exist yet, create it from power_score
if "predicted_margin" not in odds_df.columns:
    if "power_score" in odds_df.columns:
        odds_df["predicted_margin"] = odds_df["power_score"] / 10
    else:
        print("⚠ No power_score either — creating neutral model baseline")
        odds_df["predicted_margin"] = 0


print("✅ Predicted margin column exists now")
print(odds_df.columns)

In [ ]:
odds_df["model_prob"] = 1 / (1 + abs(odds_df["predicted_margin"] - odds_df["sportsbook_spread"]) / 10)

odds_df["edge"] = odds_df["model_prob"] - odds_df["implied_prob"]

print(odds_df.sort_values("edge", ascending=False).head(10))

In [ ]:
print("🚀 Calculating real predicted margin...")

# If power_score exists, use it to create a real prediction
if "power_score" in odds_df.columns:
    odds_df["predicted_margin"] = odds_df["power_score"] / 10
else:
    print("⚠ power_score not found — using sportsbook spread as fallback")
    odds_df["predicted_margin"] = odds_df["sportsbook_spread"]

print("✅ Predicted margin added")

In [ ]:
print(odds_df[["sportsbook_spread","power_score","predicted_margin"]].head())

In [ ]:
print("Current Columns:")
print(odds_df.columns)

In [ ]:
print("🚀 Recalculating edge safely...")

# Safe recalculation (in case values changed)
odds_df["edge"] = odds_df["model_prob"] - odds_df["implied_prob"]

print("✅ Edge updated")
print(odds_df[["team_name","sportsbook_spread","implied_prob","model_prob","edge"]].head())

In [ ]:
print("🔥 Filtering for positive edge plays...")

elite = odds_df[odds_df["edge"] > 0.05].sort_values("edge", ascending=False)

print("Total Elite:", len(elite))
print(elite.head(20))

In [ ]:
print("🚀 Creating final optimized betting card...")

# Keep the strongest edge per game + bookmaker
final_card = (
    elite.sort_values("edge", ascending=False)
    .drop_duplicates(subset=["game_id","bookmaker","team_name"])
    .copy()
)

# Unit sizing based on edge strength
def assign_units(e):
    if e >= 0.30:
        return 1.0
    elif e >= 0.20:
        return 0.5
    elif e >= 0.10:
        return 0.25
    else:
        return 0.1

final_card["units"] = final_card["edge"].apply(assign_units)

print("🔥 FINAL CARD")
print(final_card[["team_name","bookmaker","edge","units"]])

print("Total Plays:", len(final_card))
print("Total Units:", final_card["units"].sum())

In [ ]:
print("🚀 Selecting best bookmaker per team per game...")

# Keep ONLY highest edge per team per game
best_per_team = (
    final_card.sort_values("edge", ascending=False)
    .drop_duplicates(subset=["game_id","team_name"], keep="first")
)

print("🔥 Optimized Card")
print(best_per_team[["team_name","bookmaker","edge","units"]])

print("Total Plays (Optimized):", len(best_per_team))
print("Total Units (Optimized):", best_per_team["units"].sum())

In [ ]:
print("🚀 Filtering for strong edge plays only...")

# Set minimum edge threshold
MIN_EDGE = 0.08

filtered = best_per_team[best_per_team["edge"] >= MIN_EDGE]

print("🔥 Strong Edge Card")
print(filtered[["team_name","bookmaker","edge","units"]])

print("Total Strong Plays:", len(filtered))
print("Total Strong Units:", filtered["units"].sum())

In [ ]:
print("🚀 Applying dynamic unit sizing...")

def assign_units(edge):
    if edge >= 0.25:
        return 1.0
    elif edge >= 0.15:
        return 0.5
    elif edge >= 0.08:
        return 0.25
    else:
        return 0.1

filtered["units"] = filtered["edge"].apply(assign_units)

print("🔥 Dynamic Card")
print(filtered[["team_name","bookmaker","edge","units"]])

print("Total Plays:", len(filtered))
print("Total Units:", filtered["units"].sum())

In [ ]:
print(filtered.columns)
print(len(filtered))

In [ ]:
print("🚀 Applying dynamic unit sizing...")

def assign_units(edge):
    if edge >= 0.25:
        return 1.0
    elif edge >= 0.15:
        return 0.5
    elif edge >= 0.08:
        return 0.25
    else:
        return 0.1

filtered["units"] = filtered["edge"].apply(assign_units)

print("🔥 Dynamic Card")
print(filtered[["team_name","bookmaker","edge","units"]])

print("Total Plays:", len(filtered))
print("Total Units:", filtered["units"].sum())

In [ ]:
filtered = filtered.copy()

def assign_units(edge):
    if edge >= 0.25:
        return 1.0
    elif edge >= 0.15:
        return 0.5
    elif edge >= 0.08:
        return 0.25
    else:
        return 0.1

filtered["units"] = filtered["edge"].apply(assign_units)

print(filtered[["team_name","bookmaker","edge","units"]])
print("Total Units:", filtered["units"].sum())

In [ ]:
filtered = filtered.sort_values("edge", ascending=False)
filtered = filtered.drop_duplicates(subset=["team_name"], keep="first")

print("🔥 Final Locked Card")
print(filtered[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(filtered))
print("Total Units:", filtered["units"].sum())

In [ ]:
print("🚀 Capping max exposure per game...")

# Sort by edge (best first)
filtered = filtered.sort_values("edge", ascending=False)

# Keep only best play per game_id
filtered = filtered.drop_duplicates(subset=["game_id"], keep="first")

print("🔥 Game-Capped Card")
print(filtered[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(filtered))
print("Total Units:", filtered["units"].sum())

In [ ]:
print("🚀 Capping bookmaker exposure...")

filtered = filtered.sort_values("edge", ascending=False)

# Keep max 2 plays per bookmaker
filtered = filtered.groupby("bookmaker").head(2)

print("🔥 Final Institutional Card")
print(filtered[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(filtered))
print("Total Units:", filtered["units"].sum())

In [ ]:
print("💾 Saving final card...")

filtered.to_csv("FINAL_LOCKED_CARD.csv", index=False)

print("✅ Saved as FINAL_LOCKED_CARD.csv")

In [ ]:
import os

print(os.listdir())

In [ ]:
from google.colab import files
files.download("FINAL_LOCKED_CARD.csv")

In [ ]:
print("📅 Checking dates in dataset...")
print(odds_df["commence_time"].dt.date.unique())

In [ ]:
%who

In [ ]:
import requests
import pandas as pd

API_KEY = "YOUR_API_KEY"

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "spreads",
    "oddsFormat": "decimal"
}

response = requests.get(url, params=params)
data = response.json()

print("Games pulled:", len(data))

In [ ]:
for game in data:
    print(game["commence_time"], game["home_team"], "vs", game["away_team"])

In [ ]:
print(type(data))
print(data)

In [ ]:
import requests

API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f".strip()

url = "https://api.the-odds-api.com/v4/sports/basketball_nba/odds"

params = {
    "apiKey": API_KEY,
    "regions": "us",
    "markets": "spreads",
    "oddsFormat": "decimal"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)
print(response.text)

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

In [ ]:
API_KEY = os.environ["ODDS_API_KEY"]

In [ ]:
import datetime

print("🚀 Pulling fresh 2/26 games...")

# Pull games again from API
response = requests.get(url, params=params)
data = response.json()

games = []

for game in data:
    if "commence_time" in game:
        games.append(game)

print("Games pulled:", len(games))

# Convert to dataframe
import pandas as pd
odds_df = pd.json_normalize(games)

print(odds_df.head())

In [ ]:
import requests
import pandas as pd

print("🚀 Pulling fresh 2/26 NCAAM games...")

api_key = "10ceac94f9b9cb76f8c65510ca6df18f"  # make sure this is set correctly
url = f"https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds/"

params = {
    "apiKey": api_key,
    "regions": "us",
    "markets": "spreads",
    "oddsFormat": "decimal",
    "dateFormat": "iso",
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

data = response.json()

print("Games pulled:", len(data))

odds_df = pd.DataFrame(data)

print(odds_df.head())

In [ ]:
print("🚀 Flattening bookmakers...")

rows = []

for _, row in odds_df.iterrows():
    for book in row["bookmakers"]:
        for market in book["markets"]:
            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    rows.append({
                        "game_id": row["id"],
                        "commence_time": row["commence_time"],
                        "bookmaker": book["title"],
                        "team_name": outcome["name"],
                        "sportsbook_spread": outcome["point"],
                        "price": outcome["price"]
                    })

flat_df = pd.DataFrame(rows)

print("✅ Flattened rows:", len(flat_df))
print(flat_df.head())

In [ ]:
flat_df.columns

In [ ]:
print("🚀 Converting numeric columns...")

flat_df["sportsbook_spread"] = pd.to_numeric(flat_df["sportsbook_spread"], errors="coerce")
flat_df["price"] = pd.to_numeric(flat_df["price"], errors="coerce")

# --- Implied Probability ---
flat_df["implied_prob"] = 1 / flat_df["price"]

# --- ⚡ Simple Model Probability Example ---
# (You will later replace this with your power model)
flat_df["model_prob"] = 1 / (1 + abs(flat_df["sportsbook_spread"]) / 10)

# --- Edge ---
flat_df["edge"] = flat_df["model_prob"] - flat_df["implied_prob"]

print("✅ Added prob + edge columns")
print(flat_df[["team_name","bookmaker","sportsbook_spread","price","implied_prob","model_prob","edge"]].head())

In [ ]:
flat_df.sort_values("edge", ascending=False).head(20)

In [ ]:
print("🚀 Filtering positive edge plays...")

# Keep only positive edges
positive = flat_df[flat_df["edge"] > 0].copy()

# Sort strongest edges first
positive = positive.sort_values("edge", ascending=False)

print("Total Positive Plays:", len(positive))
print(positive.head(30))

# Save it
positive.to_csv("RAW_POSITIVE_CARD.csv", index=False)
print("💾 Saved as RAW_POSITIVE_CARD.csv")

In [ ]:
positive.head(20)

In [ ]:
# ==============================
# FILTER STRONG POSITIVE EDGES
# ==============================

print("🚀 Filtering strong edges only...")

strong = odds_df[odds_df["edge"] > 0.08].copy()  # threshold you can tune

strong = strong.sort_values("edge", ascending=False)

print("Strong Plays:", len(strong))
print(strong.head(40))

strong.to_csv("STRONG_EDGE_POOL.csv", index=False)
print("💾 Saved as STRONG_EDGE_POOL.csv")

In [ ]:
print("Current Columns:")
print(odds_df.columns)

In [ ]:
print("🚀 Flattening bookmakers again...")

rows = []

for _, game in odds_df.iterrows():
    game_id = game["id"]
    commence_time = game["commence_time"]

    for book in game["bookmakers"]:
        bookmaker_name = book["title"]

        for market in book["markets"]:
            if market["key"] != "spreads":
                continue

            for outcome in market["outcomes"]:
                rows.append({
                    "game_id": game_id,
                    "commence_time": commence_time,
                    "bookmaker": bookmaker_name,
                    "team_name": outcome["name"],
                    "sportsbook_spread": outcome["point"],
                    "price": outcome["price"],
                })

flat_df = pd.DataFrame(rows)

print("✅ Flattened rows:", len(flat_df))
print(flat_df.head())

In [ ]:
print("🚀 Calculating implied probability...")

flat_df["implied_prob"] = 1 / flat_df["price"]

# Example model probability (replace with your real model later)
flat_df["model_prob"] = 1 - abs(flat_df["sportsbook_spread"]) / 20

flat_df["edge"] = flat_df["model_prob"] - flat_df["implied_prob"]

print(flat_df.head())

In [ ]:
strong = flat_df[flat_df["edge"] > 0.08].copy()
strong = strong.sort_values("edge", ascending=False)

print("🔥 Strong Plays:", len(strong))
print(strong)

In [ ]:
print("Columns after flatten:")
print(odds_df.columns)

In [ ]:
# Convert odds to numeric + calculate implied probability
odds_df["sportsbook_spread"] = pd.to_numeric(odds_df["sportsbook_spread"], errors="coerce")
odds_df["price"] = pd.to_numeric(odds_df["price"], errors="coerce")

odds_df["implied_prob"] = 1 / odds_df["price"]

print("✅ Implied prob added")
print(odds_df.head())

In [ ]:
print("🚀 Re-flattening markets properly...")

rows = []

for _, row in games_df.iterrows():
    game_id = row["id"]
    commence_time = row["commence_time"]

    for bookmaker in row["bookmakers"]:
        bookmaker_name = bookmaker["title"]

        for market in bookmaker["markets"]:

            # We only want spreads for now
            if market["key"] == "spreads":

                for outcome in market["outcomes"]:

                    rows.append({
                        "game_id": game_id,
                        "commence_time": commence_time,
                        "bookmaker": bookmaker_name,
                        "team_name": outcome["name"],
                        "sportsbook_spread": outcome["point"],
                        "price": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print("✅ New flattened shape:", odds_df.shape)
print(odds_df.head())
print("Columns:", odds_df.columns)

In [ ]:
print(type(data))
print(data.keys())

In [ ]:
import pandas as pd

games_list = data  # since it's already a list
games_df = pd.DataFrame(games_list)

print("✅ Games dataframe created")
print(games_df.head())
print(games_df.columns)

In [ ]:
rows = []

for _, row in games_df.iterrows():
    game_id = row["id"]
    commence_time = row["commence_time"]
    home_team = row["home_team"]
    away_team = row["away_team"]

    for book in row["bookmakers"]:
        bookmaker = book["title"]

        for market in book["markets"]:
            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    rows.append({
                        "game_id": game_id,
                        "commence_time": commence_time,
                        "bookmaker": bookmaker,
                        "team_name": outcome["name"],
                        "sportsbook_spread": outcome["point"],
                        "price": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print("✅ Flattened rows:", len(odds_df))
print(odds_df.head())
print(odds_df.columns)

In [ ]:
print("🚀 Converting numeric columns + calculating implied probability...")

odds_df["sportsbook_spread"] = pd.to_numeric(odds_df["sportsbook_spread"], errors="coerce")
odds_df["price"] = pd.to_numeric(odds_df["price"], errors="coerce")

# Implied probability from price
odds_df["implied_prob"] = 1 / odds_df["price"]

print("✅ Implied probability added")
print(odds_df[["sportsbook_spread","price","implied_prob"]].head())

In [ ]:
print("🚀 Adding model probability + edge...")

# Example simple model (you can replace later with your projection logic)
odds_df["model_prob"] = 1 / (1 + abs(odds_df["sportsbook_spread"]) / 10)

# Edge = model - implied
odds_df["edge"] = odds_df["model_prob"] - odds_df["implied_prob"]

print("✅ Edge calculated")
print(odds_df.head())

In [ ]:
strong = odds_df[odds_df["edge"] > 0.08].copy()
print("🔥 Strong plays:", len(strong))
print(strong.sort_values("edge", ascending=False).head())

In [ ]:
print("🚀 Tightening strong edge filter...")

# Make sure we work on a copy
strong = odds_df.copy()

# Increase threshold so we don't get noise
strong = strong[strong["edge"] > 0.15]

# Sort best edges first
strong = strong.sort_values("edge", ascending=False)

print("🔥 After tightening:", len(strong))
print(strong.head(20))

In [ ]:
print("🚀 Selecting best bookmaker per team per game...")

# Sort so highest edge comes first
strong = strong.sort_values("edge", ascending=False)

# Keep best edge per game + team
strong = (
    strong
    .sort_values("edge", ascending=False)
    .drop_duplicates(subset=["game_id","team_name"], keep="first")
)

print("🔥 After removing duplicate books per game:", len(strong))
print(strong.head())

In [ ]:
print("🚀 Selecting best bookmaker per team per game...")

# Sort so highest edge comes first
strong = strong.sort_values("edge", ascending=False)

# Keep best edge per game + team
strong = (
    strong
    .sort_values("edge", ascending=False)
    .drop_duplicates(subset=["game_id", "team_name"], keep="first")
)

print("🔥 After removing duplicate books per game:", len(strong))
print(strong.head())

In [ ]:
print("🚀 Filtering to strong edges only (post-book cleanup)...")

# Tight strong threshold — adjust if needed
strong = strong[strong["edge"] > 0.10].copy()

strong = strong.sort_values("edge", ascending=False)

print("🔥 After strong filter:", len(strong))
print(strong.head())
print("Total Units Before Recalc:", strong.get("units", pd.Series()).sum())

In [ ]:
print("🚀 Capping max 2 units per game only...")

# Calculate total units per game
game_totals = strong.groupby("game_id")["units"].transform("sum")

# If a game exceeds 2 units total → scale it down proportionally
strong["units"] = strong["units"] * (2 / game_totals)
strong.loc[game_totals <= 2, "units"] = strong.loc[game_totals <= 2, "units"]

print("🔥 After game cap")
print(strong.head())
print("Total Plays:", len(strong))
print("Total Units:", strong["units"].sum())

In [ ]:
print("🚀 Re-adding units before game cap...")

# Safety: ensure units exist
if "units" not in strong.columns:

    def assign_units(edge):
        if edge >= 0.25:
            return 1.0
        elif edge >= 0.15:
            return 0.5
        elif edge >= 0.08:
            return 0.25
        else:
            return 0.1

    strong = strong.copy()
    strong["units"] = strong["edge"].apply(assign_units)

# Now apply 2-unit cap per game
game_totals = strong.groupby("game_id")["units"].transform("sum")

strong["units"] = strong["units"] * (2 / game_totals)
strong.loc[game_totals <= 2, "units"] = strong.loc[game_totals <= 2, "units"]

print("🔥 After 2-unit game cap")
print(strong.head())
print("Total Plays:", len(strong))
print("Total Units:", strong["units"].sum())

In [ ]:
print("🚀 Hard capping individual plays at 1.5 units...")

strong["units"] = strong["units"].clip(upper=1.5)

print("🔥 Final Capped Card")
print(strong[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(strong))
print("Total Units:", strong["units"].sum())

In [ ]:
print("💾 Saving cleaned final card...")

strong.to_csv("FINAL_LOCKED_CARD.csv", index=False)

print("✅ Saved as FINAL_LOCKED_CARD.csv")

In [ ]:
from google.colab import files
files.download("FINAL_LOCKED_CARD.csv")

In [ ]:
# Keep only strong edges
tight = strong[strong["edge"] >= 0.20].copy()

# Optional: Keep only full unit plays
tight = tight[tight["units"] >= 1.0]

print("🔥 Tight Card")
print(tight[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(tight))
print("Total Units:", tight["units"].sum())

In [ ]:
# Keep ONLY top tier edges
elite = tight[tight["edge"] >= 0.30].copy()

print("🔥 Elite Card")
print(elite[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(elite))
print("Total Units:", elite["units"].sum())

In [ ]:
# Cap total units at 20 (or whatever your bankroll plan is)
max_daily_units = 20

elite = elite.sort_values("edge", ascending=False)

cumsum_units = elite["units"].cumsum()
elite_limited = elite[cumsum_units <= max_daily_units]

print("🔥 Final Risk-Controlled Card")
print(elite_limited[["team_name","bookmaker","edge","units"]])
print("Total Plays:", len(elite_limited))
print("Total Units:", elite_limited["units"].sum())

In [ ]:
elite_limited.to_csv("FINAL_TIGHT_CARD_2_26.csv", index=False)
print("✅ Saved as FINAL_TIGHT_CARD_2_26.csv")

In [ ]:
from google.colab import files
files.download("FINAL_TIGHT_CARD_2_26.csv")

In [ ]:
print("🚀 Calculating team power scores...")

# Example: use your predicted margin as power input
# If you already computed power_score elsewhere, use that instead

team_power = (
    odds_df.groupby("team_name")["predicted_margin"]
    .mean()
    .reset_index()
    .rename(columns={"predicted_margin": "team_power_score"})
)

odds_df = odds_df.merge(team_power, on="team_name", how="left")

print("✅ Team power scores added")
print(odds_df[["team_name","team_power_score"]].head())

In [ ]:
print("🚀 Building team power from sportsbook spreads...")

# If spread is missing -> convert safely
if "sportsbook_spread" not in odds_df.columns:
    raise ValueError("No sportsbook_spread column found!")

odds_df["sportsbook_spread"] = pd.to_numeric(
    odds_df["sportsbook_spread"], errors="coerce"
)

# Simple but powerful: team power = inverse of spread expectation
team_power = (
    odds_df.groupby("team_name")["sportsbook_spread"]
    .mean()
    .reset_index()
    .rename(columns={"sportsbook_spread": "team_power_score"})
)

odds_df = odds_df.merge(team_power, on="team_name", how="left")

print("✅ Team power built")
print(odds_df[["team_name","team_power_score"]].head())

In [ ]:
print("🚀 Calculating real team-vs-market edge...")

odds_df["team_edge"] = (
    odds_df["team_power_score"] - odds_df["sportsbook_spread"]
)

filtered = odds_df[odds_df["team_edge"] > 0.1].copy()
filtered = filtered.sort_values("team_edge", ascending=False)

print("🔥 True Edge Plays:", len(filtered))
print(filtered[["team_name","bookmaker","team_edge"]].head())

In [ ]:
filtered.head()

In [ ]:
print("🚀 Building opponent-based edge model...")

# Create a lookup table of team power
power_lookup = odds_df.groupby("team_name")["team_power_score"].mean().to_dict()

def calculate_matchup_edge(row):
    game = row["game_id"]
    team = row["team_name"]

    # Find opponent in same game
    game_df = odds_df[odds_df["game_id"] == game]

    opponents = game_df[game_df["team_name"] != team]

    if len(opponents) == 0:
        return 0

    opponent = opponents.iloc[0]["team_name"]
    opp_power = power_lookup.get(opponent, 0)
    team_power = power_lookup.get(team, 0)

    return team_power - opp_power

# Apply matchup-based edge
odds_df["matchup_edge"] = odds_df.apply(calculate_matchup_edge, axis=1)

# Filter strong positive matchup edges
filtered = odds_df[odds_df["matchup_edge"] > 0.5].copy()
filtered = filtered.sort_values("matchup_edge", ascending=False)

print("🔥 True Team vs Team Edges:", len(filtered))
print(filtered[["team_name","game_id","matchup_edge"]].head())

In [ ]:
filtered.head(30)

In [ ]:
print("🚀 Selecting best play per matchup based on matchup_edge...")

# Sort by strongest matchup edge first
df_sorted = odds_df.sort_values("matchup_edge", ascending=False)

# Keep only strongest side per game
best_per_game = (
    df_sorted
    .drop_duplicates(subset=["game_id"], keep="first")
)

print("🔥 Final One-Play-Per-Game Card")
print(best_per_game[["team_name","bookmaker","edge","matchup_edge","team_power_score","units"]])

print("Total Games:", best_per_game["game_id"].nunique())
print("Total Units:", best_per_game["units"].sum())

In [ ]:
print("Current Columns:")
print(odds_df.columns)

In [ ]:
print("🚀 Recalculating units before final selection...")

# Example unit logic — adjust if needed
# If you previously capped to 1.5 max per play:

odds_df["units"] = 1.0  # default unit

# Optional: scale units by edge strength (recommended)
odds_df.loc[odds_df["edge"] > 0.4, "units"] = 1.0
odds_df.loc[(odds_df["edge"] > 0.25) & (odds_df["edge"] <= 0.4), "units"] = 0.75
odds_df.loc[(odds_df["edge"] > 0.15) & (odds_df["edge"] <= 0.25), "units"] = 0.5

print("✅ Units restored")

In [ ]:
cols = [c for c in ["team_name","bookmaker","edge","matchup_edge","team_power_score","units"]
        if c in best_per_game.columns]

print(best_per_game[cols])

In [ ]:
print("🚀 Selecting best book per team per game using matchup_edge...")

best_per_game = (
    odds_df.sort_values("matchup_edge", ascending=False)
    .drop_duplicates(subset=["game_id","team_name"], keep="first")
)

print("🔥 After final dedup:")
print(best_per_game[["team_name","bookmaker","edge","matchup_edge","team_power_score"]])

print("Total Games:", best_per_game["game_id"].nunique())
print("Total Plays:", len(best_per_game))

In [ ]:
print("🚀 Selecting SINGLE strongest play per game...")

# Sort by matchup strength
best_single = (
    best_per_game
    .sort_values("matchup_edge", ascending=False)
    .drop_duplicates(subset=["game_id"], keep="first")
)

print("🔥 Final True Card (One Play Per Game)")
print(best_single[["team_name","bookmaker","edge","matchup_edge","team_power_score"]])

print("Total Games Selected:", best_single["game_id"].nunique())
print("Total Plays:", len(best_single))
print("Total Units:", best_single["matchup_edge"].clip(lower=1).sum())

In [ ]:
print("🚀 Applying controlled unit sizing...")

# Normalize matchup edge into 0.25 – 1.5 unit scale
min_unit = 0.25
max_unit = 1.5

best_single["units"] = (
    best_single["matchup_edge"]
    .rank(pct=True)
    .clip(0.1, 1)
    .apply(lambda x: min_unit + (max_unit - min_unit) * x)
    .round(2)
)

print(best_single[["team_name","bookmaker","edge","matchup_edge","units"]])

print("Total Plays:", len(best_single))
print("Total Units:", best_single["units"].sum())

In [ ]:
best_single["units"] = (
    best_single["matchup_edge"]
    / best_single["matchup_edge"].max()
) * 1.5

best_single["units"] = best_single["units"].clip(0.25, 1.5).round(2)

In [ ]:
print("🚀 Building margin dataset...")

# Create margin column
games_df["margin"] = games_df["team_score"] - games_df["opp_score"]

# Home court adjustment (tweak later if needed)
HOME_ADJ = 2.5

def adjust_margin(row):
    if row["home_away_flag"] == "home":
        return row["margin"] - HOME_ADJ
    elif row["home_away_flag"] == "away":
        return row["margin"] + HOME_ADJ
    else:
        return row["margin"]

games_df["adjusted_margin"] = games_df.apply(adjust_margin, axis=1)

print(games_df[["team_name","margin","adjusted_margin"]].head())

In [ ]:
print("Columns available:")
print(games_df.columns)

In [ ]:
print("🚀 Flattening bookmakers...")

rows = []

for _, game in games_df.iterrows():
    game_id = game["id"]
    home_team = game["home_team"]
    away_team = game["away_team"]
    commence_time = game["commence_time"]

    for book in game["bookmakers"]:
        bookmaker = book["title"]

        for market in book["markets"]:
            if market["key"] == "spreads":
                for outcome in market["outcomes"]:
                    rows.append({
                        "game_id": game_id,
                        "commence_time": commence_time,
                        "bookmaker": bookmaker,
                        "team_name": outcome["name"],
                        "spread": outcome["point"],
                        "price": outcome["price"]
                    })

odds_df = pd.DataFrame(rows)

print("✅ Flattened shape:", odds_df.shape)
print(odds_df.head())

In [ ]:
print("🚀 Adding implied probability & converting to numeric...")

odds_df["price"] = pd.to_numeric(odds_df["price"], errors="coerce")

# Implied probability from moneyline
odds_df["implied_prob"] = 1 / odds_df["price"]

print(odds_df[["team_name","spread","price","implied_prob"]].head())

In [ ]:
print("🚀 Creating market-based power score...")

odds_df["power_score"] = -odds_df["spread"]

print(odds_df[["team_name","spread","power_score"]].head())

In [ ]:
print("🚀 Aggregating power per team (market consensus)...")

team_power = (
    odds_df
    .groupby("team_name")["power_score"]
    .mean()
    .reset_index()
    .rename(columns={"power_score": "team_power_score"})
)

print(team_power.sort_values("team_power_score", ascending=False).head(20))

In [ ]:
print("🚀 Building matchup predictions...")

# Merge home/away power into games
games = odds_df[["game_id","home_team","away_team"]].drop_duplicates()

games = games.merge(
    team_power,
    left_on="home_team",
    right_on="team_name",
    how="left"
).rename(columns={"team_power_score":"home_power"}).drop(columns=["team_name"])

games = games.merge(
    team_power,
    left_on="away_team",
    right_on="team_name",
    how="left"
).rename(columns={"team_power_score":"away_power"}).drop(columns=["team_name"])

games["expected_margin"] = games["home_power"] - games["away_power"]

print(games[["home_team","away_team","expected_margin"]].head(20))

In [ ]:
print("🚀 Rebuilding games dataframe from original API data...")

# Use the raw games dataframe (not odds_df)
games_df = pd.DataFrame(data)  # ONLY if 'data' is your original API response

games_clean = games_df[["id","home_team","away_team"]].drop_duplicates()

games_clean = games_clean.merge(
    team_power,
    left_on="home_team",
    right_on="team_name",
    how="left"
).rename(columns={"team_power_score":"home_power"}).drop(columns=["team_name"])

games_clean = games_clean.merge(
    team_power,
    left_on="away_team",
    right_on="team_name",
    how="left"
).rename(columns={"team_power_score":"away_power"}).drop(columns=["team_name"])

games_clean["expected_margin"] = games_clean["home_power"] - games_clean["away_power"]

print(games_clean[["home_team","away_team","expected_margin"]].head(20))

In [ ]:
print("🚀 Converting margin into win probability...")

import numpy as np
from scipy.stats import norm

# Assume standard deviation of margin
std_dev = 12  # You can tune this later

games_clean["home_win_prob"] = norm.cdf(
    games_clean["expected_margin"] / std_dev
)

games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean.get("spread",0)) / std_dev
)

print(games_clean[["home_team","away_team","expected_margin","home_win_prob"]].head(20))

In [ ]:
games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) / std_dev
)

In [ ]:
print("🚀 Merging spread from market data into games_clean...")

# Get one spread per game (take consensus or first book spread)
spread_df = (
    odds_df.groupby("game_id")["spread"]
    .mean()
    .reset_index()
)

games_clean = games_clean.merge(spread_df, on="game_id", how="left")

print("Columns after merge:")
print(games_clean.columns)

In [ ]:
print("Columns in games_clean:")
print(games_clean.columns)

In [ ]:
print("🚀 Merging spread into games_clean...")

# First aggregate spread per game
spread_df = (
    odds_df.groupby("game_id")["spread"]
    .mean()
    .reset_index()
)

# Merge using different column names
games_clean = games_clean.merge(
    spread_df,
    left_on="id",
    right_on="game_id",
    how="left"
)

# Drop duplicate key column
games_clean = games_clean.drop(columns=["game_id"])

print(games_clean.head())
print("Columns after merge:")
print(games_clean.columns)

In [ ]:
games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) / std_dev
)

In [ ]:
🚀 Building historical power per team...

# Compute historical power as average expected margin per team
historical_home = games_clean[["home_team", "expected_margin"]].rename(
    columns={"home_team": "team_name"}
)

historical_away = games_clean[["away_team", "expected_margin"]].rename(
    columns={"away_team": "team_name"}
)

historical_all = pd.concat([historical_home, historical_away])

historical_power = (
    historical_all
    .groupby("team_name")["expected_margin"]
    .mean()
    .reset_index()
    .rename(columns={"expected_margin": "historical_power"})
)

print(historical_power.head())

In [ ]:
# Building historical power per team

historical_home = games_clean[["home_team", "expected_margin"]].rename(
    columns={"home_team": "team_name"}
)

historical_away = games_clean[["away_team", "expected_margin"]].rename(
    columns={"away_team": "team_name"}
)

historical_all = pd.concat([historical_home, historical_away])

historical_power = (
    historical_all
    .groupby("team_name")["expected_margin"]
    .mean()
    .reset_index()
    .rename(columns={"expected_margin": "historical_power"})
)

print(historical_power.head())

In [ ]:
# Building market power from spreads

market_all = pd.concat([
    games_clean[["home_team"]].rename(columns={"home_team":"team_name"}),
    games_clean[["away_team"]].rename(columns={"away_team":"team_name"})
])

market_all["market_power"] = 0  # placeholder for now

market_power = (
    market_all
    .groupby("team_name")["market_power"]
    .mean()
    .reset_index()
)

print(market_power.head())

In [ ]:
# Combining historical + market power

power_df = historical_power.merge(
    market_power,
    on="team_name",
    how="outer"
).fillna(0)

alpha = 0.6
beta = 0.4

power_df["combined_power"] = (
    alpha * power_df["historical_power"] +
    beta * power_df["market_power"]
)

print(power_df.sort_values("combined_power", ascending=False).head(20))

In [ ]:
# Attach combined power back to games

games_clean = games_clean.merge(
    power_df[["team_name", "combined_power"]],
    left_on="home_team",
    right_on="team_name",
    how="left"
).rename(columns={"combined_power": "home_power"}).drop(columns=["team_name"])

games_clean = games_clean.merge(
    power_df[["team_name", "combined_power"]],
    left_on="away_team",
    right_on="team_name",
    how="left"
).rename(columns={"combined_power": "away_power"}).drop(columns=["team_name"])

print(games_clean.head())

In [ ]:
# Clean duplicate power columns if they exist
games_clean = games_clean.loc[:, ~games_clean.columns.duplicated()]

print("Columns after cleanup:")
print(games_clean.columns)

In [ ]:
# Reattach power properly (clean version)

games_clean = games_clean.merge(
    power_df[["team_name", "combined_power"]],
    left_on="home_team",
    right_on="team_name",
    how="left"
).rename(columns={"combined_power": "home_power_new"}).drop(columns=["team_name"])

games_clean = games_clean.merge(
    power_df[["team_name", "combined_power"]],
    left_on="away_team",
    right_on="team_name",
    how="left"
).rename(columns={"combined_power": "away_power_new"}).drop(columns=["team_name"])

# Replace old power columns
games_clean["home_power"] = games_clean["home_power_new"]
games_clean["away_power"] = games_clean["away_power_new"]

games_clean = games_clean.drop(columns=["home_power_new", "away_power_new"])

print(games_clean.head())

In [ ]:
print("🚀 Recalculating true cover probability vs market spread...")

std_dev = 12  # You can later optimize this

# Make sure spread exists (if missing, fill with 0)
games_clean["spread"] = games_clean.get("spread", 0).fillna(0)

# True win probability
games_clean["home_win_prob"] = norm.cdf(
    games_clean["expected_margin"] / std_dev
)

# TRUE cover probability (this is what matters for betting)
games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) / std_dev
)

# Edge vs market
games_clean["true_edge"] = games_clean["cover_prob"] - 0.5

print(games_clean[["home_team","spread","expected_margin","cover_prob","true_edge"]].head())

In [ ]:
print("🚀 Calculating Expected Value (EV)...")

games_clean["ev"] = (
    games_clean["cover_prob"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob"])
)

print(games_clean[["home_team","cover_prob","price","ev"]].head())

In [ ]:
print("🚀 Merging best sportsbook price into games_clean...")

# Get best price per game (highest price = best payout)
price_df = odds_df.groupby("game_id")["price"].max().reset_index()

games_clean = games_clean.merge(price_df, on="game_id", how="left")

print("Columns after price merge:")
print(games_clean.columns)

print(games_clean[["home_team","price"]].head())

In [ ]:
print("🚀 Merging best sportsbook price into games_clean...")

price_df = odds_df.groupby("game_id")["price"].max().reset_index()

# IMPORTANT: your games_clean uses "id" not "game_id"
games_clean = games_clean.merge(
    price_df,
    left_on="id",
    right_on="game_id",
    how="left"
)

# Optional: drop duplicate key column after merge
games_clean.drop(columns=["game_id"], inplace=True)

print("Columns after merge:")
print(games_clean.columns)

print(games_clean[["home_team","price"]].head())

In [ ]:
print("🚀 Calculating Expected Value (EV)...")

games_clean["ev"] = (
    games_clean["cover_prob"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob"])
)

games_clean["kelly_fraction"] = (
    (games_clean["cover_prob"] * (games_clean["price"] - 1) - (1 - games_clean["cover_prob"]))
    / (games_clean["price"] - 1)
)

print(games_clean[["home_team","price","cover_prob","ev","kelly_fraction"]].sort_values("ev", ascending=False).head(20))

In [ ]:
print("🚀 Clamping probabilities for stability...")

games_clean["cover_prob"] = games_clean["cover_prob"].clip(0.05, 0.95)

games_clean["ev"] = (
    games_clean["cover_prob"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob"])
)

games_clean["kelly_fraction"] = (
    games_clean["ev"] / (games_clean["price"] - 1)
)

print(games_clean.sort_values("ev", ascending=False).head(20))

In [ ]:
print("🚀 Applying fractional Kelly (0.5x for safety)...")

kelly_scale = 0.5  # <-- change to 0.25, 0.5, or 1.0

games_clean["kelly_fraction"] = (
    games_clean["ev"] / (games_clean["price"] - 1)
) * kelly_scale

games_clean["kelly_fraction"] = games_clean["kelly_fraction"].clip(0, 1)

print(games_clean.sort_values("kelly_fraction", ascending=False).head(20))

In [ ]:
games_clean["volatility_adj"] = (
    abs(games_clean["home_power"] - games_clean["away_power"])
)

games_clean["adjusted_std"] = 12 * (
    1 + (1 / (games_clean["volatility_adj"] + 1))
)

In [ ]:
games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"])
    / games_clean["adjusted_std"]
)

In [ ]:
print("🚀 Adding volatility-adjusted probability model...")

games_clean["volatility_adj"] = (
    abs(games_clean["home_power"] - games_clean["away_power"])
)

games_clean["adjusted_std"] = 12 * (
    1 + (1 / (games_clean["volatility_adj"] + 1))
)

from scipy.stats import norm

games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"])
    / games_clean["adjusted_std"]
)

print(games_clean[["home_team","expected_margin","cover_prob"]].head(10))

In [ ]:
print("🚀 Calculating market-based volatility...")

# Measure spread dispersion across books per game
spread_vol = (
    odds_df.groupby("game_id")["spread"]
    .std()
    .reset_index()
    .rename(columns={"spread": "market_spread_vol"})
)

games_clean = games_clean.merge(spread_vol, on="game_id", how="left")

# Replace NaN volatility with small default
games_clean["market_spread_vol"] = games_clean["market_spread_vol"].fillna(0.5)

# Blend model volatility + market volatility
games_clean["adjusted_std"] = (
    10 * games_clean["market_spread_vol"]
    + 5 / (abs(games_clean["home_power"] - games_clean["away_power"]) + 1)
)

from scipy.stats import norm

games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"])
    / games_clean["adjusted_std"]
)

print(games_clean[["home_team","market_spread_vol","cover_prob"]].head())

In [ ]:
print("🚀 Calculating market-based volatility...")

spread_vol = (
    odds_df.groupby("game_id")["spread"]
    .std()
    .reset_index()
    .rename(columns={"spread": "market_spread_vol"})
)

# 🔥 FIX: merge on id == game_id
games_clean = games_clean.merge(
    spread_vol,
    left_on="id",
    right_on="game_id",
    how="left"
)

# Drop duplicate column after merge
if "game_id" in games_clean.columns:
    games_clean = games_clean.drop(columns=["game_id"])

# Fill missing volatility
games_clean["market_spread_vol"] = games_clean["market_spread_vol"].fillna(0.5)

print(games_clean[["home_team","market_spread_vol"]].head())

In [ ]:
print("🚀 Recalculating cover probability using market volatility...")

import numpy as np
from scipy.stats import norm

# Use market volatility instead of fixed std
games_clean["adjusted_std"] = games_clean["market_spread_vol"].replace(0, 5)

games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) /
    games_clean["adjusted_std"]
)

print(games_clean[["home_team","cover_prob","adjusted_std"]].head())

In [ ]:
print("🚀 Stabilizing volatility (capping extremes)...")

# Cap volatility to prevent distortion from outliers
vol_cap = games_clean["market_spread_vol"].quantile(0.90)

games_clean["capped_vol"] = games_clean["market_spread_vol"].clip(
    lower=2,
    upper=vol_cap
)

games_clean["cover_prob_stable"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) /
    games_clean["capped_vol"]
)

print(games_clean[["home_team","cover_prob_stable","capped_vol"]].head())

In [ ]:
print("🚀 Measuring model calibration error...")

# Simulated indicator if cover happened (if margin > spread)
games_clean["cover_actual"] = (
    games_clean["expected_margin"] > games_clean["spread"]
).astype(int)

# Brier Score (calibration metric)
games_clean["brier_component"] = (
    (games_clean["cover_prob_stable"] - games_clean["cover_actual"]) ** 2
)

brier_score = games_clean["brier_component"].mean()

print("📊 Brier Score:", brier_score)
print(games_clean[["home_team","cover_prob_stable","cover_actual"]].head())

In [ ]:
print("🚀 Creating train/test split for validation...")

# Sort by time so we simulate real world
games_clean = games_clean.sort_values("commence_time")

# 80% train / 20% test split
split_index = int(len(games_clean) * 0.8)

train = games_clean.iloc[:split_index].copy()
test = games_clean.iloc[split_index:].copy()

print("Train size:", len(train))
print("Test size:", len(test))

In [ ]:
print("🚀 Creating train/test split using row order...")

# Sort by index to preserve build order
games_clean = games_clean.sort_index()

split_index = int(len(games_clean) * 0.8)

train = games_clean.iloc[:split_index].copy()
test = games_clean.iloc[split_index:].copy()

print("Train size:", len(train))
print("Test size:", len(test))

In [ ]:
from scipy.stats import norm

std_dev = 12  # keep same assumption

print("🚀 Evaluating on TEST set only...")

test["cover_actual"] = (test["expected_margin"] > test["spread"]).astype(int)

test["cover_prob_test"] = norm.cdf(
    (test["expected_margin"] - test["spread"]) / std_dev
)

test["brier"] = (test["cover_prob_test"] - test["cover_actual"]) ** 2

print("Test Brier Score:", test["brier"].mean())

In [ ]:
print("🚀 Optimizing std_dev using training data...")

def brier_for_std(std):
    probs = norm.cdf((train["expected_margin"] - train["spread"]) / std)
    actual = (train["expected_margin"] > train["spread"]).astype(int)
    return ((probs - actual) ** 2).mean()

std_values = np.linspace(5, 30, 100)
errors = [brier_for_std(s) for s in std_values]

best_std = std_values[np.argmin(errors)]

print("Best Std Dev:", best_std)

In [ ]:
print("🚀 Using optimized volatility for probability calculation...")

OPTIMAL_STD = best_std

games_clean["cover_prob_optimized"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) / OPTIMAL_STD
)

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    games_clean["cover_actual"],
    games_clean["cover_prob_optimized"]
)

print("Optimized Brier Score:", brier)

In [ ]:
print("🚀 Running model on current card...")

# 1. Ensure spread exists
if "spread" not in games_clean.columns:
    games_clean["spread"] = 0

# 2. Recalculate cover probability using optimized std_dev
std_dev = 5.0  # from your optimization

from scipy.stats import norm

games_clean["cover_prob"] = norm.cdf(
    (games_clean["expected_margin"] - games_clean["spread"]) / std_dev
)

# 3. Add best available price per game
price_df = odds_df.groupby("game_id")["price"].max().reset_index()
games_clean = games_clean.merge(price_df, on="game_id", how="left")

# 4. Compute EV
games_clean["ev"] = (
    games_clean["cover_prob"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob"])
)

# 5. Kelly (fractional)
games_clean["kelly_fraction"] = (
    games_clean["ev"] / (games_clean["price"] - 1)
).clip(lower=0)

# 6. Sort by strongest edge
card = games_clean.sort_values("ev", ascending=False)

print(card[["home_team","away_team","cover_prob","price","ev","kelly_fraction"]])

In [ ]:
print("🚀 Adding best price using correct ID column...")

# Use the actual ID column
price_df = odds_df.groupby("game_id")["price"].max().reset_index()

# Rename to match games_clean column
price_df = price_df.rename(columns={"game_id": "id"})

# Merge on "id" (NOT game_id)
games_clean = games_clean.merge(price_df, on="id", how="left")

print("✅ Columns after fix:")
print(games_clean.columns)

In [ ]:
games_clean["ev"] = (
    games_clean["cover_prob"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob"])
)

In [ ]:
print("Columns in games_clean:")
print(games_clean.columns)

In [ ]:
# Keep only one price column
if "price_x" in games_clean.columns:
    games_clean["price"] = games_clean["price_x"]
    games_clean = games_clean.drop(columns=["price_x"], errors="ignore")

if "price_y" in games_clean.columns:
    games_clean = games_clean.drop(columns=["price_y"], errors="ignore")

print("✅ Cleaned price columns")
print(games_clean.columns)

In [ ]:
games_clean["ev"] = (
    games_clean["cover_prob_stable"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob_stable"])
)

print(games_clean[["home_team","price","cover_prob_stable","ev"]].head())

In [ ]:
# 🔥 Filter positive EV plays
positive_card = games_clean[games_clean["ev"] > 0]

# Sort by highest EV
positive_card = positive_card.sort_values("ev", ascending=False)

print("🚀 FINAL BET CARD (EV > 0)")
print(positive_card[["home_team","price","cover_prob_stable","ev"]])

print("Total Plays:", len(positive_card))
print("Total Units Suggested:", positive_card["ev"].sum())

In [ ]:
# Strong EV + Risk Control
filtered = games_clean[
    (games_clean["ev"] > 0.05) &          # minimum edge
    (games_clean["kelly_fraction"] > 0.05) &  # meaningful sizing
    (games_clean["cover_prob_stable"] < 0.995) # avoid 99.9% fake certainty
]

filtered = filtered.sort_values("ev", ascending=False)

print(filtered[["home_team","price","cover_prob_stable","ev","kelly_fraction"]])
print("Total Plays:", len(filtered))

In [ ]:
# ==============================
# MODEL CONFIG
# ==============================

MODEL_CONFIG = {
    "base_std": 5.0,
    "volatility_cap": 15,
    "kelly_fraction_multiplier": 0.5,
    "min_ev_threshold": 0.03,
    "min_price": 1.90
}

In [ ]:
# ==============================
# FRACTIONAL KELLY (PROFESSIONALIZED)
# ==============================

games_clean["kelly_fraction"] = (
    games_clean["true_edge"]
    / (games_clean["price"] - 1)
) * MODEL_CONFIG["kelly_fraction_multiplier"]

# Clamp between 0 and 1
games_clean["kelly_fraction"] = games_clean["kelly_fraction"].clip(lower=0, upper=1)

In [ ]:
# ==============================
# EXPECTED VALUE (PROFESSIONAL VERSION)
# ==============================

games_clean["ev"] = (
    games_clean["cover_prob_stable"] * (games_clean["price"] - 1)
    - (1 - games_clean["cover_prob_stable"])
)

# Filter low EV plays
games_clean = games_clean[
    games_clean["ev"] > MODEL_CONFIG["min_ev_threshold"]
]

In [ ]:
# ==============================
# SUMMARY METRICS
# ==============================

print("🔥 MODEL SUMMARY 🔥")
print("Total Plays:", len(games_clean))
print("Total Units Suggested:", games_clean["kelly_fraction"].sum())
print("Avg EV:", games_clean["ev"].mean())
print("Max EV:", games_clean["ev"].max())

In [ ]:
# =====================================================
# PROFESSIONAL RISK FILTER LAYER
# =====================================================

# Require minimum edge to avoid weak plays
MIN_EDGE = 0.05

# Require minimum confidence in probability
MIN_PROB = 0.65

games_clean = games_clean[
    (games_clean["true_edge"] > MIN_EDGE) &
    (games_clean["cover_prob_stable"] > MIN_PROB)
]

print("✅ After Risk Filter:")
print("Total Plays:", len(games_clean))
print("Total Units:", games_clean["kelly_fraction"].sum())

In [ ]:
✅ After Risk Filter:
Total Plays: 53
Total Units: 9.08183379811969


In [ ]:
# =====================================================
# ADAPTIVE EDGE FILTER (PRO LEVEL)
# =====================================================

# Calculate dynamic edge threshold based on percentile
edge_threshold = games_clean["true_edge"].quantile(0.60)

print("Dynamic Edge Threshold:", edge_threshold)

games_clean = games_clean[
    games_clean["true_edge"] >= edge_threshold
]

print("After Adaptive Edge Filter")
print("Total Plays:", len(games_clean))
print("Total Units:", games_clean["kelly_fraction"].sum())

In [ ]:
# =====================================================
# DYNAMIC UNIT SCALING (SMART CAPITAL ALLOCATION)
# =====================================================

print("🚀 Applying Dynamic Capital Allocation...")

# Scale units based on edge strength
edge = games_clean["true_edge"]

# Normalize edge into 0.25u – 1.5u band
min_unit = 0.25
max_unit = 1.50

scaled_units = min_unit + (
    (edge - edge.min()) / (edge.max() - edge.min() + 1e-9)
) * (max_unit - min_unit)

games_clean["scaled_units"] = scaled_units.clip(min_unit, max_unit)

# Optional: Boost extremely strong edges
games_clean.loc[games_clean["true_edge"] > 0.45, "scaled_units"] *= 1.2

print(games_clean[["home_team","true_edge","scaled_units"]])

print("Total Units:", games_clean["scaled_units"].sum())

In [ ]:
print("🚀 Advanced Risk-Weighted Capital Allocation...")

# Combine edge + volatility penalty
vol_factor = 1 / (games_clean["capped_vol"] + 1)

risk_score = games_clean["true_edge"] * vol_factor

# Normalize risk score
min_r = risk_score.min()
max_r = risk_score.max()

scaled_units = 0.25 + (
    (risk_score - min_r) / (max_r - min_r + 1e-9)
) * (1.8 - 0.25)

games_clean["risk_scaled_units"] = scaled_units.clip(0.25, 1.8)

print(games_clean[["home_team","true_edge","capped_vol","risk_scaled_units"]])

print("Total Units:", games_clean["risk_scaled_units"].sum())

In [ ]:
print("🚀 Tiered Capital Allocation v2...")

# Normalize components
edge_norm = (games_clean["true_edge"] - games_clean["true_edge"].min()) / (
    games_clean["true_edge"].max() - games_clean["true_edge"].min() + 1e-9
)

vol_norm = 1 / (games_clean["capped_vol"] + 1)
vol_norm = (vol_norm - vol_norm.min()) / (vol_norm.max() - vol_norm.min() + 1e-9)

# Weighted scoring
composite_score = (0.7 * edge_norm) + (0.3 * vol_norm)

# Scale to units
min_u = 0.35
max_u = 2.0

units = min_u + (composite_score - composite_score.min()) / (
    composite_score.max() - composite_score.min() + 1e-9
) * (max_u - min_u)

games_clean["tiered_units"] = units.clip(min_u, max_u)

print(games_clean[["home_team","true_edge","capped_vol","tiered_units"]])
print("Total Units:", games_clean["tiered_units"].sum())

In [ ]:
print("🚀 Portfolio With Matchup Context")

display_cols = [
    "home_team",
    "away_team",
    "tiered_units",
    "final_units",
    "true_edge",
    "capped_vol"
]

print(games_clean[display_cols].sort_values("final_units", ascending=False))
print("Total Units:", games_clean["final_units"].sum())

In [ ]:
print("Columns available:")
print(games_clean.columns)

print("\n🚀 Portfolio With Matchup Context")

display_cols = [
    "home_team",
    "away_team",
    "tiered_units" if "tiered_units" in games_clean.columns else None,
    "scaled_units" if "scaled_units" in games_clean.columns else None,
    "risk_scaled_units" if "risk_scaled_units" in games_clean.columns else None,
    "true_edge",
    "capped_vol"
]

# Remove None values
display_cols = [col for col in display_cols if col in games_clean.columns]

print(
    games_clean[display_cols]
    .sort_values(display_cols[1], ascending=False)
)

print("\nTotal Units:",
      games_clean[display_cols[1]].sum())

In [ ]:
print("\n🔥 Portfolio Summary")
print("Total Plays:", len(games_clean))

unit_column = "tiered_units"  # change if you want scaled or risk_scaled instead

print("Total Units:",
      games_clean[unit_column].sum())

print("Avg True Edge:",
      games_clean["true_edge"].mean())

In [ ]:
portfolio = games_clean.sort_values("tiered_units", ascending=False)
print(portfolio[["home_team","away_team","tiered_units","true_edge","capped_vol"]])

In [ ]:
games_clean["risk_efficiency"] = (
    games_clean["true_edge"] / games_clean["capped_vol"]
)

portfolio = games_clean.sort_values("risk_efficiency", ascending=False)

print(portfolio[[
    "home_team",
    "away_team",
    "true_edge",
    "capped_vol",
    "risk_efficiency",
    "tiered_units"
]])

In [ ]:
# 🔥 Normalize capital to fixed portfolio size

PORTFOLIO_BUDGET = 20

games_clean["portfolio_weight"] = (
    games_clean["risk_efficiency"] / games_clean["risk_efficiency"].sum()
)

games_clean["normalized_units"] = (
    games_clean["portfolio_weight"] * PORTFOLIO_BUDGET
)

portfolio_final = games_clean.sort_values("normalized_units", ascending=False)

print(portfolio_final[[
    "home_team",
    "away_team",
    "risk_efficiency",
    "normalized_units"
]])

print("Total Normalized Units:", portfolio_final["normalized_units"].sum())

In [ ]:
# 🔥 Filter weak risk efficiency plays before allocation

MIN_RISK_EFF = 0.038

filtered = games_clean[games_clean["risk_efficiency"] >= MIN_RISK_EFF].copy()

PORTFOLIO_BUDGET = 20

filtered["portfolio_weight"] = (
    filtered["risk_efficiency"] / filtered["risk_efficiency"].sum()
)

filtered["normalized_units"] = (
    filtered["portfolio_weight"] * PORTFOLIO_BUDGET
)

filtered = filtered.sort_values("normalized_units", ascending=False)

print(filtered[[
    "home_team",
    "away_team",
    "risk_efficiency",
    "normalized_units"
]])

print("Total Units:", filtered["normalized_units"].sum())

In [ ]:
MIN_RISK_EFF = 0.038
PORTFOLIO_BUDGET = 20
ALPHA = 0.8
MAX_UNIT = 1.5

filtered = games_clean[games_clean["risk_efficiency"] >= MIN_RISK_EFF].copy()

# Risk compression
filtered["adjusted_weight"] = filtered["risk_efficiency"] ** ALPHA

filtered["portfolio_weight"] = (
    filtered["adjusted_weight"] / filtered["adjusted_weight"].sum()
)

filtered["normalized_units"] = (
    filtered["portfolio_weight"] * PORTFOLIO_BUDGET
)

# Apply max cap
filtered["normalized_units"] = filtered["normalized_units"].clip(upper=MAX_UNIT)

# Re-normalize after cap
scaling_factor = PORTFOLIO_BUDGET / filtered["normalized_units"].sum()
filtered["normalized_units"] *= scaling_factor

filtered = filtered.sort_values("normalized_units", ascending=False)

print(filtered[[
    "home_team",
    "away_team",
    "risk_efficiency",
    "normalized_units"
]])

print("Total Units:", filtered["normalized_units"].sum())

In [ ]:
MIN_RISK_EFF = 0.038
PORTFOLIO_BUDGET = 20
ALPHA = 0.8
MAX_UNIT = 1.5

filtered = games_clean[games_clean["risk_efficiency"] >= MIN_RISK_EFF].copy()

# Risk compression
filtered["adjusted_weight"] = filtered["risk_efficiency"] ** ALPHA

filtered["portfolio_weight"] = (
    filtered["adjusted_weight"] / filtered["adjusted_weight"].sum()
)

filtered["raw_units"] = filtered["portfolio_weight"] * PORTFOLIO_BUDGET

# Hard cap
filtered["capped_units"] = filtered["raw_units"].clip(upper=MAX_UNIT)

# Calculate leftover capital
used_capital = filtered["capped_units"].sum()
excess = PORTFOLIO_BUDGET - used_capital

# Redistribute excess ONLY to uncapped plays
uncapped_mask = filtered["raw_units"] < MAX_UNIT

if excess > 0 and uncapped_mask.any():
    weight_sum_uncapped = filtered.loc[uncapped_mask, "raw_units"].sum()
    filtered.loc[uncapped_mask, "capped_units"] += (
        filtered.loc[uncapped_mask, "raw_units"] / weight_sum_uncapped
    ) * excess

filtered = filtered.sort_values("capped_units", ascending=False)

print(filtered[[
    "home_team",
    "away_team",
    "risk_efficiency",
    "capped_units"
]])

print("Total Units:", filtered["capped_units"].sum())

In [ ]:
MIN_RISK_EFF = 0.038
PORTFOLIO_BUDGET = 20
ALPHA = 0.8
MAX_UNIT = 1.5

filtered = games_clean[games_clean["risk_efficiency"] >= MIN_RISK_EFF].copy()

# ---- NEW: Risk-Adjusted Signal ----
filtered["signal"] = (
    filtered["risk_efficiency"] *
    filtered["true_edge"] /
    filtered["capped_vol"]
)

# Normalize signal
filtered["adjusted_weight"] = filtered["signal"] ** ALPHA

filtered["portfolio_weight"] = (
    filtered["adjusted_weight"] / filtered["adjusted_weight"].sum()
)

filtered["raw_units"] = filtered["portfolio_weight"] * PORTFOLIO_BUDGET

# Hard cap
filtered["capped_units"] = filtered["raw_units"].clip(upper=MAX_UNIT)

# Redistribute excess
used_capital = filtered["capped_units"].sum()
excess = PORTFOLIO_BUDGET - used_capital

uncapped_mask = filtered["raw_units"] < MAX_UNIT

if excess > 0 and uncapped_mask.any():
    weight_sum_uncapped = filtered.loc[uncapped_mask, "raw_units"].sum()
    filtered.loc[uncapped_mask, "capped_units"] += (
        filtered.loc[uncapped_mask, "raw_units"] / weight_sum_uncapped
    ) * excess

filtered = filtered.sort_values("capped_units", ascending=False)

print(filtered[[
    "home_team",
    "away_team",
    "risk_efficiency",
    "true_edge",
    "capped_units"
]])

print("Total Units:", filtered["capped_units"].sum())

In [ ]:
# Example: Penalize teams from same conference / same league
# If you have a conference column, we cluster reduce weights.

if "conference" in filtered.columns:
    filtered["diversification_penalty"] = (
        1 / filtered.groupby("conference")["conference"].transform("count")
    )
else:
    filtered["diversification_penalty"] = 1.0

filtered["signal"] = (
    filtered["risk_efficiency"] *
    filtered["true_edge"] /
    filtered["capped_vol"] *
    filtered["diversification_penalty"]
)

In [ ]:
# Final quality filter before publishing card

MIN_EDGE = 0.05
MIN_UNITS = 0.25

final_card = filtered[
    (filtered["true_edge"] > MIN_EDGE) &
    (filtered["normalized_units"] >= MIN_UNITS)
].copy()

final_card = final_card.sort_values("normalized_units", ascending=False)

print("🔥 FINAL BETTING CARD")
print(final_card[[
    "home_team",
    "away_team",
    "true_edge",
    "risk_efficiency",
    "normalized_units"
]])

print("Total Units:", final_card["normalized_units"].sum())

In [ ]:
final_card["portfolio_pct"] = (
    final_card["normalized_units"] /
    final_card["normalized_units"].sum()
)

print(final_card.sort_values("normalized_units", ascending=False))

In [ ]:
final_card = final_card[final_card["true_edge"] > 0.35]

In [ ]:
final_card.to_csv("FINAL_LOCKED_CARD.csv", index=False)

In [ ]:
final_card.to_csv("FINAL_BETTING_CARD.csv", index=False)

from google.colab import files
files.download("FINAL_BETTING_CARD.csv")

In [ ]:
def format_bet(row):
    team = row["home_team"]
    away = row["away_team"]
    units = round(row["normalized_units"], 2)
    spread = row.get("spread", None)
    price = row.get("price", None)

    if spread is not None and spread != 0:
        market = f"{team} {spread}"
    else:
        market = f"{team} ML"

    if price is not None:
        market += f" ({round(price,2)})"

    return f"{market} | {units}u"

final_card["bet"] = final_card.apply(format_bet, axis=1)

print(final_card[["home_team","away_team","bet"]])

In [ ]:
def format_bet(row):

    # Detect bet type
    if "spread" in row and row["spread"] != 0:
        bet_type = "SPREAD"
        line = row["spread"]
    else:
        bet_type = "ML"
        line = None

    # Optional: If you later add totals / team totals, expand here

    price = row["price"]

    if bet_type == "ML":
        return f"{row['home_team']} ML ({price}) | {row['normalized_units']:.2f}u"

    elif bet_type == "SPREAD":
        return f"{row['home_team']} {line:+} ({price}) | {row['normalized_units']:.2f}u"

    else:
        return "UNKNOWN BET"


games_clean["bet"] = games_clean.apply(format_bet, axis=1)

In [ ]:
BET_CATEGORY = ML / SPREAD / TOTAL / TEAM_TOTAL

In [ ]:
def categorize_bet(row):

    if "spread" in row and row["spread"] != 0:
        return "SPREAD"
    else:
        return "ML"


games_clean["bet_category"] = games_clean.apply(categorize_bet, axis=1)

In [ ]:
games_clean["bet_line"] = games_clean["spread"].where(games_clean["spread"] != 0, 0)

In [ ]:
def detect_market(row):
    if "spread" in row and row["spread"] != 0:
        return f"{row['home_team']} {row['spread']}"
    else:
        return f"{row['home_team']} ML"

In [ ]:
# ==============================
# 🔥 PROFESSIONAL BETTING CARD
# ==============================

bet_card = games_clean.copy()

# Build readable bet label
def build_bet(row):

    price = row["price"] if "price" in row else None
    units = row["normalized_units"] if "normalized_units" in row else None

    # Default to ML since your model currently selects ML
    bet_type = "ML"

    bet_line = ""
    if "spread" in row and row["spread"] != 0:
        bet_type = "SPREAD"
        bet_line = f"{row['spread']}"

    price_text = f"({price})" if price is not None else ""
    unit_text = f"{round(units,2)}u" if units is not None else ""

    return f"{row['home_team']} {bet_type} {price_text} {bet_line} | {unit_text}"

bet_card["bet"] = bet_card.apply(build_bet, axis=1)

# Keep only clean columns
output = bet_card[[
    "home_team",
    "away_team",
    "bet",
    "true_edge",
    "normalized_units"
]].sort_values("normalized_units", ascending=False)

print("🔥 FINAL BETTING CARD")
print(output)

print("\nTotal Units:", output["normalized_units"].sum())

In [ ]:
# ==========================================
# 🚀 TOTAL & TEAM TOTAL EDGE ENGINE
# ==========================================

import numpy as np
from scipy.stats import norm

# ------------------------------------------
# 1. Define volatility for totals
# ------------------------------------------
TOTAL_STD = 12  # adjustable — historical NBA/NCAAB total std is usually 10-15

# Make sure market total exists
if "market_total" in games_clean.columns:

    # ------------------------------------------
    # 2. Project Game Total
    # ------------------------------------------
    # Simple projection using expected margin + spread inference
    games_clean["projected_total"] = (
        games_clean["home_power"].abs() +
        games_clean["away_power"].abs()
    )

    # ------------------------------------------
    # 3. Compute Total Edge
    # Probability projected total exceeds market total
    games_clean["total_prob"] = norm.cdf(
        (games_clean["projected_total"] - games_clean["market_total"]) / TOTAL_STD
    )

    # Convert to EV assuming fair -110 pricing
    market_implied_prob = 1 / (1 + 1.1)

    games_clean["total_ev"] = (
        games_clean["total_prob"] * (1.1)
        - (1 - games_clean["total_prob"])
    )

    # ------------------------------------------
    # 4. Team Total Engine
    # ------------------------------------------

    if "home_team_total" in games_clean.columns:

        games_clean["home_team_total_prob"] = norm.cdf(
            (games_clean["home_power"] - games_clean["home_team_total"]) / 8
        )

        games_clean["home_team_total_ev"] = (
            games_clean["home_team_total_prob"] * 1.1
            - (1 - games_clean["home_team_total_prob"])
        )

    if "away_team_total" in games_clean.columns:

        games_clean["away_team_total_prob"] = norm.cdf(
            (games_clean["away_power"] - games_clean["away_team_total"]) / 8
        )

        games_clean["away_team_total_ev"] = (
            games_clean["away_team_total_prob"] * 1.1
            - (1 - games_clean["away_team_total_prob"])
        )

    print("✅ TOTAL & TEAM TOTAL ENGINE RUN SUCCESS")

else:
    print("❌ No market_total column found — skipping total model.")

In [ ]:
games_clean[[
    "home_team",
    "away_team",
    "market_total",
    "projected_total",
    "total_ev"
]].sort_values("total_ev", ascending=False).head(20)

In [ ]:
# ==========================================
# 🚀 SAFE TOTAL + TEAM TOTAL ENGINE (FIXED)
# ==========================================

import numpy as np
from scipy.stats import norm

TOTAL_STD = 12

# ------------------------------------------
# STEP 1 — Detect market total column
# ------------------------------------------

# Some sportsbooks label it differently
if "total" in games_clean.columns:
    games_clean["market_total"] = games_clean["total"]

elif "market_total" not in games_clean.columns:
    print("❌ No market total column detected.")
    games_clean["market_total"] = np.nan


# ------------------------------------------
# STEP 2 — Compute projected total
# ------------------------------------------

games_clean["projected_total"] = (
    games_clean["home_power"].abs() +
    games_clean["away_power"].abs()
)


# ------------------------------------------
# STEP 3 — Total Probability + EV
# ------------------------------------------

games_clean["total_prob"] = norm.cdf(
    (games_clean["projected_total"] - games_clean["market_total"]) / TOTAL_STD
)

games_clean["total_ev"] = (
    games_clean["total_prob"] * 1.1
    - (1 - games_clean["total_prob"])
)


print("✅ TOTAL ENGINE RUN SUCCESS")

In [ ]:
games_clean[
    ["home_team","away_team","market_total","projected_total","total_ev"]
].dropna().sort_values("total_ev", ascending=False).head(20)

In [ ]:
# =====================================================
# 🔥 UNIFIED BETTING CARD (ML + SPREAD + TOTAL + TEAM)
# =====================================================

cards = []

# ---------- ML ----------
if "ml_ev" in games_clean.columns:
    ml = games_clean[games_clean["ml_ev"] > 0].copy()
    ml["bet_type"] = "ML"
    ml["edge_metric"] = ml["ml_ev"]
    cards.append(ml)

# ---------- SPREAD ----------
if "spread_ev" in games_clean.columns:
    sp = games_clean[games_clean["spread_ev"] > 0].copy()
    sp["bet_type"] = "SPREAD"
    sp["edge_metric"] = sp["spread_ev"]
    cards.append(sp)

# ---------- TOTAL ----------
if "total_ev" in games_clean.columns:
    tot = games_clean[games_clean["total_ev"] > 0].copy()
    tot["bet_type"] = "TOTAL"
    tot["edge_metric"] = tot["total_ev"]
    cards.append(tot)

# ---------- TEAM TOTAL ----------
if "team_total_ev" in games_clean.columns:
    tt = games_clean[games_clean["team_total_ev"] > 0].copy()
    tt["bet_type"] = "TEAM_TOTAL"
    tt["edge_metric"] = tt["team_total_ev"]
    cards.append(tt)

# Merge everything
if len(cards) > 0:
    final_card = pd.concat(cards, ignore_index=True)

    final_card = final_card.sort_values("edge_metric", ascending=False)

    print("🔥 FINAL MASTER CARD")
    print(final_card[[
        "home_team",
        "away_team",
        "bet_type",
        "edge_metric"
    ]])

    print("\nTotal Plays:", len(final_card))
    print("Total Combined Edge:", round(final_card["edge_metric"].sum(),2))
else:
    print("No positive EV bets found.")

In [ ]:
print("Columns available:")
print(games_clean.columns)

print("\nSummary Stats of EV Columns:")
for col in games_clean.columns:
    if "ev" in col.lower():
        print("\n", col)
        print(games_clean[col].describe())

In [ ]:
# --------- Combine All Market Edges ---------

# Keep only markets with positive EV
ml_edges = games_clean[games_clean["ev"] > 0][
    ["home_team", "away_team", "ev"]
].copy()

spread_edges = games_clean[games_clean["spread_ev"] > 0][
    ["home_team", "away_team", "spread_ev"]
].copy() if "spread_ev" in games_clean.columns else None

total_edges = games_clean[games_clean["total_ev"] > 0][
    ["home_team", "away_team", "total_ev"]
].copy() if "total_ev" in games_clean.columns else None

# Rename for consistency
ml_edges["edge_metric"] = ml_edges["ev"]
ml_edges["bet_type"] = "ML"

if spread_edges is not None:
    spread_edges["edge_metric"] = spread_edges["spread_ev"]
    spread_edges["bet_type"] = "SPREAD"

if total_edges is not None:
    total_edges["edge_metric"] = total_edges["total_ev"]
    total_edges["bet_type"] = "TOTAL"

# Combine everything
card = pd.concat(
    [df for df in [ml_edges, spread_edges, total_edges] if df is not None],
    ignore_index=True
)

card = card.sort_values("edge_metric", ascending=False)

print("🔥 MASTER EDGE CARD")
print(card)

print("\nTotal Plays:", len(card))
print("Total Combined Edge:", card["edge_metric"].sum())

In [ ]:
markets = []

# ML plays
ml = games_clean[games_clean["home_win_prob"] > 0.5].copy()
ml["bet_type"] = "ML"
ml["edge_metric"] = ml["ev"]

markets.append(ml)

# Spread plays (if spread edge exists)
spread = games_clean[games_clean["true_edge"] > 0.03].copy()
spread["bet_type"] = "SPREAD"
spread["edge_metric"] = spread["true_edge"]

markets.append(spread)

# Total plays (if total_ev exists)
if "total_ev" in games_clean.columns:
    totals = games_clean[games_clean["total_ev"] > 0.03].copy()
    totals["bet_type"] = "TOTAL"
    totals["edge_metric"] = totals["total_ev"]

    markets.append(totals)

# Combine everything
master_card = pd.concat(markets)

# Sort by strongest edge
master_card = master_card.sort_values("edge_metric", ascending=False)

print("🔥 MASTER EDGE CARD")
print(master_card[["home_team","away_team","bet_type","edge_metric"]])
print("\nTotal Plays:", len(master_card))
print("Total Combined Edge:", master_card["edge_metric"].sum())

In [ ]:
# Keep only the strongest bet per game
master_card = master_card.sort_values("edge_metric", ascending=False)

# Drop duplicate games (keep highest edge per game)
master_card = master_card.drop_duplicates(
    subset=["home_team","away_team"],
    keep="first"
)

print("🔥 CLEANED MASTER CARD")
print(master_card[["home_team","away_team","bet_type","edge_metric"]])

print("\nTotal Plays:", len(master_card))
print("Total Combined Edge:", master_card["edge_metric"].sum())

In [ ]:
# Pick best market per game
market_rank = master_card.sort_values("edge_metric", ascending=False)

best_per_game = market_rank.drop_duplicates(
    subset=["home_team","away_team"],
    keep="first"
)

best_per_game = best_per_game.sort_values("edge_metric", ascending=False)

print("🔥 BEST MARKET PER GAME")
print(best_per_game[["home_team","away_team","bet_type","edge_metric"]])

print("\nTotal Plays:", len(best_per_game))
print("Total Combined Edge:", best_per_game["edge_metric"].sum())

In [ ]:
# Select best market per game across ALL markets

ranked = master_card.sort_values("edge_metric", ascending=False)

best_market = ranked.loc[
    ranked.groupby(["home_team","away_team"])["edge_metric"].idxmax()
]

best_market = best_market.sort_values("edge_metric", ascending=False)

print("🔥 OPTIMAL MARKET PER GAME")
print(best_market[[
    "home_team",
    "away_team",
    "bet_type",
    "edge_metric"
]])

print("\nTotal Plays:", len(best_market))
print("Total Combined Edge:", best_market["edge_metric"].sum())

In [ ]:
# 🔥 SELECT BEST MARKET PER GAME (ML / SPREAD / TOTAL / TEAM TOTAL)

market_cols = ["ml_edge", "spread_edge", "total_edge", "team_total_edge"]

best_market = (
    games_clean
    .melt(
        id_vars=["home_team","away_team"],
        value_vars=market_cols,
        var_name="market",
        value_name="edge_metric"
    )
    .dropna()
    .sort_values(["home_team","edge_metric"], ascending=[True, False])
    .groupby("home_team")
    .first()
    .reset_index()
)

print("🔥 BEST MARKET PER GAME")
print(best_market[["home_team","away_team","market","edge_metric"]])

print("\nTotal Plays:", len(best_market))
print("Total Combined Edge:", best_market["edge_metric"].sum())

In [ ]:
# 🔥 CREATE MARKET EDGE COLUMNS IF THEY DON'T EXIST

if "ml_edge" not in games_clean.columns:
    games_clean["ml_edge"] = games_clean["ev"]

if "spread_edge" not in games_clean.columns:
    games_clean["spread_edge"] = games_clean.get("spread_ev", games_clean["ev"])

if "total_edge" not in games_clean.columns:
    games_clean["total_edge"] = games_clean.get("total_ev", 0)

if "team_total_edge" not in games_clean.columns:
    games_clean["team_total_edge"] = games_clean.get("team_total_ev", 0)

print("✅ Market edge columns created")
print(games_clean.columns)

In [ ]:
# 🔥 SELECT BEST MARKET PER GAME

market_cols = ["ml_edge", "spread_edge", "total_edge", "team_total_edge"]

best_market = (
    games_clean
    .melt(
        id_vars=["home_team","away_team"],
        value_vars=[c for c in market_cols if c in games_clean.columns],
        var_name="market",
        value_name="edge_metric"
    )
    .dropna()
    .sort_values(["home_team","edge_metric"], ascending=[True, False])
    .groupby("home_team")
    .first()
    .reset_index()
)

print("🔥 BEST MARKET PER GAME")
print(best_market[["home_team","away_team","market","edge_metric"]])

print("\nTotal Plays:", len(best_market))
print("Total Combined Edge:", best_market["edge_metric"].sum())

In [ ]:
print("📊 Market Distribution")
print(best_market["market"].value_counts())
print("\nAvg Edge By Market")
print(best_market.groupby("market")["edge_metric"].mean())

In [ ]:
print(games_clean.columns)

In [ ]:
market_cols = ["ml_edge","spread_edge","total_edge","team_total_edge"]

# Melt markets into long format
best_market = (
    games_clean
    .melt(
        id_vars=["home_team","away_team"],
        value_vars=market_cols,
        var_name="market",
        value_name="edge_metric"
    )
    .dropna(subset=["edge_metric"])
    .sort_values("edge_metric", ascending=False)
)

# Keep best market per game
best_market = (
    best_market
    .groupby(["home_team","away_team"], as_index=False)
    .first()
)

best_market = best_market.sort_values("edge_metric", ascending=False)

print("🔥 BEST MARKET PER GAME")
print(best_market)

print("\nTotal Plays:", len(best_market))
print("Total Combined Edge:", best_market["edge_metric"].sum())

In [ ]:
# 🔥 FINAL BET CARD (PRODUCTION RUN)

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
    .copy()
)

# Keep only strong edges
EDGE_CUTOFF = 0.88
final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

# Sort clean
final_card = final_card.sort_values("edge_metric", ascending=False)

# Select columns for betting sheet
print(final_card[[
    "home_team",
    "away_team",
    "bet_category",
    "edge_metric"
]])

print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
# 🔥 FINAL BET CARD (SAFE VERSION)

required_cols = ["home_team", "away_team", "edge_metric"]

# Only include bet_category if it exists
display_cols = ["home_team", "away_team", "edge_metric"]

if "bet_category" in games_clean.columns:
    display_cols.append("bet_category")

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
    .copy()
)

EDGE_CUTOFF = 0.88
final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

final_card = final_card.sort_values("edge_metric", ascending=False)

print(final_card[display_cols])

print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
# 🔥 FINAL BET CARD (AUTO PICKS BEST EDGE COLUMN)

# Determine which edge columns exist
edge_cols = [col for col in [
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge"
] if col in games_clean.columns]

# Create unified edge metric from best available market
games_clean["edge_metric"] = games_clean[edge_cols].max(axis=1)

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
    .copy()
)

EDGE_CUTOFF = 0.88
final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

display_cols = ["home_team", "away_team", "edge_metric"]

if "bet_category" in final_card.columns:
    display_cols.append("bet_category")

print(final_card[display_cols])
print("\nTotal Plays:", len(final_card))
print("Total Combined Edge:", final_card["edge_metric"].sum())

In [ ]:
# 🔥 Determine best market per row

market_cols = ["ml_edge", "spread_edge", "total_edge", "team_total_edge"]

existing_cols = [col for col in market_cols if col in games_clean.columns]

games_clean["edge_metric"] = games_clean[existing_cols].max(axis=1)

# Identify which market produced the max
games_clean["bet_category"] = games_clean[existing_cols].idxmax(axis=1)

# Clean up labels
games_clean["bet_category"] = games_clean["bet_category"].str.replace("_edge", "", regex=False)

In [ ]:
# 🔥 Rebuild Final Card After Market Selection

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
)

EDGE_CUTOFF = 0.88

final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

final_card = final_card.sort_values("edge_metric", ascending=False)

print(final_card[[
    "home_team",
    "away_team",
    "bet_category",
    "edge_metric"
]])

print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
# 🔥 Production Final Card With Bet Details

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
)

EDGE_CUTOFF = 0.88
final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

final_card = final_card.sort_values("edge_metric", ascending=False)

display_cols = [
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "edge_metric"
]

print(final_card[display_cols])
print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
print(games_clean.columns)

In [ ]:
# 🔥 FINAL PRODUCTION BET CARD

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
    .drop_duplicates(subset=["home_team", "away_team"], keep="first")
)

EDGE_CUTOFF = 0.88
final_card = final_card[final_card["edge_metric"] >= EDGE_CUTOFF]

final_card = final_card.sort_values("edge_metric", ascending=False)

display_cols = [
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "edge_metric"
]

print(final_card[display_cols])
print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
# ===============================
# 🔥 TOTAL & TEAM TOTAL EDGE ENGINE
# ===============================

# Ensure required columns exist
if "market_total" in games_clean.columns and "projected_total" in games_clean.columns:

    games_clean["total_diff"] = games_clean["projected_total"] - games_clean["market_total"]

    # Normalize by market volatility if available
    if "market_spread_vol" in games_clean.columns:
        games_clean["total_edge"] = games_clean["total_diff"] / games_clean["market_spread_vol"]
    else:
        games_clean["total_edge"] = games_clean["total_diff"]

else:
    games_clean["total_edge"] = 0


# ===============================
# TEAM TOTAL EDGE
# ===============================

if "team_total_market" in games_clean.columns and "projected_team_total" in games_clean.columns:

    games_clean["team_total_diff"] = (
        games_clean["projected_team_total"] - games_clean["team_total_market"]
    )

    if "market_spread_vol" in games_clean.columns:
        games_clean["team_total_edge"] = (
            games_clean["team_total_diff"] / games_clean["market_spread_vol"]
        )
    else:
        games_clean["team_total_edge"] = games_clean["team_total_diff"]

else:
    games_clean["team_total_edge"] = 0


# ===============================
# 🔥 UPDATE MASTER EDGE COLUMN
# ===============================

games_clean["edge_metric"] = games_clean[[
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge"
]].max(axis=1)


print("✅ Total + Team Total Edges Calculated")
print(games_clean[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric"
]].head())

In [ ]:
# ===============================
# 🔥 FORCE TOTAL FIELDS TO EXIST
# ===============================

for col in [
    "market_total",
    "projected_total",
    "team_total_market",
    "projected_team_total"
]:
    if col not in games_clean.columns:
        games_clean[col] = 0.0

# Fill NaNs safely
games_clean["market_total"] = games_clean["market_total"].fillna(0)
games_clean["projected_total"] = games_clean["projected_total"].fillna(0)
games_clean["team_total_market"] = games_clean["team_total_market"].fillna(0)
games_clean["projected_team_total"] = games_clean["projected_team_total"].fillna(0)

In [ ]:
# ===============================
# 🔥 TOTAL & TEAM TOTAL EDGE ENGINE
# ===============================

# Ensure required columns exist
if "market_total" in games_clean.columns and "projected_total" in games_clean.columns:

    games_clean["total_diff"] = games_clean["projected_total"] - games_clean["market_total"]

    # Normalize by market volatility if available
    if "market_spread_vol" in games_clean.columns:
        games_clean["total_edge"] = games_clean["total_diff"] / games_clean["market_spread_vol"]
    else:
        games_clean["total_edge"] = games_clean["total_diff"]

else:
    games_clean["total_edge"] = 0


# ===============================
# TEAM TOTAL EDGE
# ===============================

if "team_total_market" in games_clean.columns and "projected_team_total" in games_clean.columns:

    games_clean["team_total_diff"] = (
        games_clean["projected_team_total"] - games_clean["team_total_market"]
    )

    if "market_spread_vol" in games_clean.columns:
        games_clean["team_total_edge"] = (
            games_clean["team_total_diff"] / games_clean["market_spread_vol"]
        )
    else:
        games_clean["team_total_edge"] = games_clean["team_total_diff"]

else:
    games_clean["team_total_edge"] = 0


# ===============================
# 🔥 UPDATE MASTER EDGE COLUMN
# ===============================

games_clean["edge_metric"] = games_clean[[
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge"
]].max(axis=1)


print("✅ Total + Team Total Edges Calculated")
print(games_clean[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric"
]].head())

In [ ]:
# ===============================
# 🔥 CALCULATE TEAM TOTAL EDGE
# ===============================

if "projected_team_total" in games_clean.columns and "team_total_market" in games_clean.columns:

    games_clean["team_total_edge"] = (
        (games_clean["projected_team_total"] - games_clean["team_total_market"])
        / games_clean["market_total"].replace(0, 1)
    )

else:
    games_clean["team_total_edge"] = 0

In [ ]:
print(games_clean[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric"
]])

In [ ]:
# ===============================
# 🔥 PROPER TEAM TOTAL EDGE
# ===============================

if "projected_team_total" in games_clean.columns and "team_total_market" in games_clean.columns:

    games_clean["team_total_edge"] = (
        games_clean["projected_team_total"] -
        games_clean["team_total_market"]
    )

    # Normalize it so it's on similar scale as other edges
    games_clean["team_total_edge"] = (
        games_clean["team_total_edge"] /
        games_clean["team_total_market"].replace(0,1)
    )

else:
    games_clean["team_total_edge"] = 0.0

In [ ]:
games_clean["edge_metric"] = (
    games_clean["ml_edge"] +
    games_clean["spread_edge"] +
    games_clean["total_edge"] +
    games_clean["team_total_edge"]
)

In [ ]:
# ===============================
# CALCULATE TEAM TOTAL EDGE
# ===============================

# Make sure required columns exist
if "projected_team_total" in games_clean.columns and "team_total_market" in games_clean.columns:

    games_clean["team_total_edge"] = (
        games_clean["projected_team_total"] -
        games_clean["team_total_market"]
    )

    # Normalize relative to market to scale properly
    games_clean["team_total_edge"] = (
        games_clean["team_total_edge"] /
        games_clean["team_total_market"].replace(0, 1)
    )

else:
    print("⚠ Missing projected_team_total or team_total_market columns")
    games_clean["team_total_edge"] = 0.0


# ===============================
# REBUILD FULL EDGE METRIC
# ===============================

games_clean["edge_metric"] = (
    games_clean["ml_edge"].fillna(0) +
    games_clean["spread_edge"].fillna(0) +
    games_clean["total_edge"].fillna(0) +
    games_clean["team_total_edge"].fillna(0)
)

# Sort best first
final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
)

print(final_card[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric"
]])

print("\nTotal Plays:", len(final_card))
print("Total Edge Sum:", final_card["edge_metric"].sum())

In [ ]:
# ==========================================
# FORCE TEAM TOTAL EDGE USING IMPLIED VALUE
# ==========================================

if "team_total_market" in games_clean.columns and "projected_team_total" in games_clean.columns:

    games_clean["team_total_edge"] = (
        (games_clean["projected_team_total"] - games_clean["team_total_market"])
        / games_clean["team_total_market"].replace(0, 1)
    )

else:
    print("⚠ Team total data missing — pulling from totals model instead")

    # fallback proxy using total and expected margin
    games_clean["team_total_edge"] = (
        (games_clean["projected_total"] / 2) -
        (games_clean["market_total"] / 2)
    )

# ==========================================
# REBUILD MASTER EDGE
# ==========================================

games_clean["edge_metric"] = (
    games_clean["ml_edge"].fillna(0) +
    games_clean["spread_edge"].fillna(0) +
    games_clean["total_edge"].fillna(0) +
    games_clean["team_total_edge"].fillna(0)
)

final_card = (
    games_clean
    .sort_values("edge_metric", ascending=False)
)

print(final_card[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric"
]])

print("\nTotal Plays:", len(final_card))
print("Total Combined Edge:", final_card["edge_metric"].sum())

In [ ]:
# 🔥 Add unit sizing based on edge strength

EDGE_CAP = games_clean["edge_metric"].quantile(0.85)

final_card = games_clean.copy()
final_card = final_card.sort_values("edge_metric", ascending=False)

final_card["unit_size"] = (
    final_card["edge_metric"] / final_card["edge_metric"].max()
) * 1.5  # cap max at 1.5u

final_card["unit_size"] = final_card["unit_size"].clip(upper=1.5)

print(final_card[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric",
    "unit_size"
]])

print("\nTotal Units Allocated:", final_card["unit_size"].sum())

In [ ]:
# 🔥 EXPORT EXECUTION CARD (CLEAN BET TABLE)

execution_card = final_card[[
    "home_team",
    "away_team",
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge",
    "edge_metric",
    "unit_size"
]].copy()

execution_card = execution_card.sort_values("unit_size", ascending=False)

execution_card.to_csv("final_execution_card.csv", index=False)

print("✅ Saved as: final_execution_card.csv")
print(execution_card.head(25))

In [ ]:
from google.colab import files

files.download("final_execution_card.csv")

In [ ]:
display_cols = [
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "edge_metric",
    "unit_size"
]

final_card_sorted = final_card.sort_values("edge_metric", ascending=False)

print(final_card_sorted[display_cols])
print("\nTotal Plays:", len(final_card_sorted))
print("Total Units:", final_card_sorted["unit_size"].sum())

In [ ]:
final_card_sorted[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "unit_size"
]].to_csv("clean_betting_card.csv", index=False)

from google.colab import files
files.download("clean_betting_card.csv")

In [ ]:
# Convert decimal price to American odds
def decimal_to_american(price):
    if price >= 2:
        return round((price - 1) * 100)
    else:
        return round(-100 / (price - 1))

display_card = final_card.copy()

display_card["american_odds"] = display_card["price"].apply(decimal_to_american)

# Create clean betting output
bet_sheet = display_card.sort_values("edge_metric", ascending=False)[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "american_odds",
    "unit_size",
    "edge_metric"
]]

# Rename columns for clarity
bet_sheet = bet_sheet.rename(columns={
    "bet_category": "BET TYPE",
    "bet_line": "LINE",
    "price": "Decimal Odds",
    "american_odds": "American Odds",
    "unit_size": "Units",
    "edge_metric": "Edge"
})

print("🔥 CLEAR BETTING CARD\n")
print(bet_sheet)

print("\nTotal Plays:", len(bet_sheet))
print("Total Units:", bet_sheet["Units"].sum())

In [ ]:
from google.colab import files

files.download("final_execution_card.csv")

In [ ]:
# ==============================
# CLEAN EXECUTION BETTING CARD
# ==============================

card = games_clean.sort_values("edge_metric", ascending=False).copy()

card = card[card["edge_metric"] > 4]  # keep strong edges only

output = card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "unit_size",
    "edge_metric"
]]

print("🔥 BETTING CARD")
print(output)
print("\nTotal Plays:", len(output))
print("Total Units:", output["unit_size"].sum())

In [ ]:
# ==============================
# CLEAN EXECUTION BETTING CARD
# ==============================

card = games_clean.sort_values("edge_metric", ascending=False).copy()

# Keep strong edges
card = card[card["edge_metric"] > 4]

# If unit_size doesn't exist, create it safely
if "unit_size" not in card.columns:
    card["unit_size"] = 1.0  # fallback default

# Build output
output = card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "unit_size",
    "edge_metric"
]]

print("🔥 BETTING CARD")
print(output)
print("\nTotal Plays:", len(output))
print("Total Units:", output["unit_size"].sum())

In [ ]:
# ==========================================
# PICK BEST MARKET PER GAME (TRUE VERSION)
# ==========================================

market_cols = ["ml_edge", "spread_edge", "total_edge", "team_total_edge"]

# Find best market column per row
def get_best_market(row):
    values = {col: row.get(col, 0) for col in market_cols}
    best_market = max(values, key=values.get)
    return best_market

card = games_clean.copy()

# Assign best market
card["best_market"] = card.apply(get_best_market, axis=1)

# Map market to bet type + line
def map_market(row):
    market = row["best_market"]

    if market == "ml_edge":
        return "ML", 0
    elif market == "spread_edge":
        return "SPREAD", row.get("spread", 0)
    elif market == "total_edge":
        return "TOTAL", row.get("market_total", 0)
    elif market == "team_total_edge":
        return "TEAM_TOTAL", row.get("team_total_market", 0)
    else:
        return "ML", 0

card[["bet_category", "bet_line"]] = card.apply(
    lambda row: pd.Series(map_market(row)),
    axis=1
)

# Keep strong edges only
card = card.sort_values("edge_metric", ascending=False)
card = card[card["edge_metric"] > 4]

# Default units
if "unit_size" not in card.columns:
    card["unit_size"] = 1.0

output = card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "unit_size",
    "edge_metric"
]]

print("🔥 TRUE OPTIMAL MARKET CARD")
print(output)
print("\nTotal Plays:", len(output))
print("Total Units:", output["unit_size"].sum())

In [ ]:
def get_best_market(row):
    values = {col: row.get(col, 0) for col in market_cols}

    sorted_markets = sorted(values.items(), key=lambda x: x[1], reverse=True)

    best_market, best_value = sorted_markets[0]
    second_value = sorted_markets[1][1] if len(sorted_markets) > 1 else 0

    # Require dominance threshold
    if best_value - second_value < 0.15:
        return "NO_CLEAR_EDGE"

    return best_market

In [ ]:
card["best_market"] = card.apply(get_best_market, axis=1)

In [ ]:
card = card[card["best_market"] != "NO_CLEAR_EDGE"]

In [ ]:
print("BEST MARKET DISTRIBUTION")
print(card["best_market"].value_counts())

print("\nTOTAL PLAYS:", len(card))
print("\nTOTAL UNITS:", card["unit_size"].sum())

In [ ]:
print("🔥 BEST MARKET DISTRIBUTION")
print(card["best_market"].value_counts())

print("\n📊 TOTAL PLAYS:", len(card))
print("💰 TOTAL UNITS:", card["unit_size"].sum())

In [ ]:
# Stronger filter — remove weak edges & tiny allocations
card = card[
    (card["edge_metric"] >= 4.5) &
    (card["unit_size"] >= 1.0)
]

print("🔥 FILTERED BET CARD")
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_line",
    "price",
    "unit_size",
    "edge_metric"
]])

print("\nTotal Plays:", len(card))
print("Total Units:", card["unit_size"].sum())
print("\nMarket Breakdown:")
print(card["best_market"].value_counts())

In [ ]:
# Stronger filter
card = card[
    (card["edge_metric"] >= 4.5) &
    (card["unit_size"] >= 1.0)
].copy()

# ---------------------------------------
# Add Optimal Line Based On Market
# ---------------------------------------

def get_optimal_line(row):
    market = row["best_market"]

    if market == "ml_edge":
        return "ML"

    elif market == "spread_edge":
        # model implied spread
        return row.get("expected_margin", 0)

    elif market == "total_edge":
        # model projected total
        return row.get("projected_total", row.get("market_total", 0))

    elif market == "team_total_edge":
        return row.get("projected_team_total", row.get("team_total_market", 0))

    return None


card["optimal_line"] = card.apply(get_optimal_line, axis=1)

# ---------------------------------------
# Final Output
# ---------------------------------------

print("🔥 FILTERED OPTIMAL BET CARD")

print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "optimal_line",
    "price",
    "unit_size",
    "edge_metric"
]])

print("\nTotal Plays:", len(card))
print("\nTotal Units:", card["unit_size"].sum())
print("\nMarket Breakdown:")
print(card["best_market"].value_counts())

In [ ]:
# ---------------------------------------
# ADD FULL BET DESCRIPTION
# ---------------------------------------

def get_bet_details(row):
    market = row["best_market"]

    # ---------- ML ----------
    if market == "ml_edge":
        return "ML", "ML"

    # ---------- Spread ----------
    elif market == "spread_edge":
        line = row.get("spread", 0)
        # If expected margin > spread → we bet favorite
        side = "SPREAD_HOME" if row.get("expected_margin", 0) > line else "SPREAD_AWAY"
        return "SPREAD", f"{side} @ {line}"

    # ---------- Total ----------
    elif market == "total_edge":
        projected = row.get("projected_total", 0)
        market_total = row.get("market_total", 0)

        # If projected > market → BET OVER
        side = "OVER" if projected > market_total else "UNDER"

        return "TOTAL", f"{side} {market_total}"

    # ---------- Team Total ----------
    elif market == "team_total_edge":
        projected = row.get("projected_team_total", 0)
        market_line = row.get("team_total_market", 0)

        side = "OVER" if projected > market_line else "UNDER"

        return "TEAM_TOTAL", f"{side} {market_line}"

    return "UNKNOWN", ""

In [ ]:
card[["bet_category", "bet_details"]] = card.apply(
    lambda row: pd.Series(get_bet_details(row)),
    axis=1
)

In [ ]:
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]])

In [ ]:
elif market == "total_edge":
    projected = row.get("projected_total", None)
    market_total = row.get("market_total", None)

    # If data missing → skip this bet
    if projected is None or market_total is None or market_total == 0:
        return "TOTAL", "DATA_MISSING"

    side = "OVER" if projected > market_total else "UNDER"

    return "TOTAL", f"{side} {market_total}"

In [ ]:
def map_market(row):
    market = row["best_market"]

    if market == "ml_edge":
        return "ML", "ML"

    if market == "spread_edge":
        line = row.get("spread", None)

        if line is None:
            return "SPREAD", "DATA_MISSING"

        side = "HOME" if row["spread"] < 0 else "AWAY"
        return "SPREAD", f"{side} {line}"

    if market == "total_edge":
        projected = row.get("projected_total", None)
        market_total = row.get("market_total", None)

        if projected is None or market_total is None or market_total == 0:
            return "TOTAL", "DATA_MISSING"

        side = "OVER" if projected > market_total else "UNDER"
        return "TOTAL", f"{side} {market_total}"

    if market == "team_total_edge":
        projected = row.get("projected_team_total", None)
        market_line = row.get("team_total_market", None)

        if projected is None or market_line is None or market_line == 0:
            return "TEAM_TOTAL", "DATA_MISSING"

        side = "OVER" if projected > market_line else "UNDER"
        return "TEAM_TOTAL", f"{side} {market_line}"

    return "ML", "ML"

In [ ]:
card[["bet_category", "bet_details"]] = card.apply(
    lambda row: pd.Series(map_market(row)),
    axis=1
)

card = card[card["bet_details"] != "DATA_MISSING"]

In [ ]:
# Re-assign best market
card["best_market"] = card.apply(get_best_market, axis=1)

# Map market into clear bet + line
card[["bet_category", "bet_details"]] = card.apply(
    lambda row: pd.Series(map_market(row)),
    axis=1
)

# Remove bad rows
card = card[card["bet_details"] != "DATA_MISSING"]

# Sort strongest first
card = card.sort_values("edge_metric", ascending=False)

# Display clean betting card
print("🔥 CLEAN OPTIMAL BET CARD")
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]])

print("\nTotal Plays:", len(card))
print("Total Units:", card["unit_size"].sum())

print("\nMarket Breakdown:")
print(card["bet_category"].value_counts())

In [ ]:
# Safe mapping (always returns 2 values)
def safe_map_market(row):
    market = row["best_market"]

    if market == "ml_edge":
        return "ML", row.get("price", 0)

    elif market == "spread_edge":
        line = row.get("spread", 0)
        return "SPREAD", f"{row.get('home_team')} {line}"

    elif market == "total_edge":
        projected = row.get("projected_total", 0)
        market_total = row.get("market_total", 0)

        if market_total == 0:
            return "TOTAL", "DATA_MISSING"

        side = "OVER" if projected > market_total else "UNDER"
        return "TOTAL", f"{side} {market_total}"

    elif market == "team_total_edge":
        line = row.get("team_total_market", 0)
        return "TEAM_TOTAL", f"LINE {line}"

    else:
        return "UNKNOWN", "N/A"


# Apply safely
card[["bet_category", "bet_details"]] = card.apply(
    lambda row: pd.Series(safe_map_market(row)),
    axis=1
)

In [ ]:
# Apply safely but force consistent output shape
mapped = card.apply(lambda row: pd.Series(safe_map_market(row)), axis=1)

# Force exactly 2 columns
mapped = mapped.iloc[:, :2]

mapped.columns = ["bet_category", "bet_details"]

card = card.join(mapped)

In [ ]:
# Apply safely
mapped = card.apply(lambda row: pd.Series(safe_map_market(row)), axis=1)

# Force exactly 2 columns
mapped = mapped.iloc[:, :2]
mapped.columns = ["bet_category", "bet_details"]

# OVERWRITE instead of join (prevents column conflict)
card[["bet_category", "bet_details"]] = mapped

print("✅ Market mapping applied successfully")

In [ ]:
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]].head())

In [ ]:
# More realistic strong-edge threshold
EDGE_THRESHOLD = card["edge_metric"].quantile(0.75)

card = card[card["edge_metric"] >= EDGE_THRESHOLD]

print("🔥 Adaptive Threshold Used:", EDGE_THRESHOLD)
print("Plays After Filter:", len(card))

In [ ]:
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]])

In [ ]:
card = card.sort_values("edge_metric", ascending=False)

print("🔥 BEFORE FILTER")
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]])

In [ ]:
EDGE_THRESHOLD = card["edge_metric"].quantile(0.60)

card = card[
    (card["edge_metric"] >= EDGE_THRESHOLD) &
    (card["unit_size"] >= 0.25)   # small safe floor
]

print("🔥 AFTER SMART FILTER")
print(card)

In [ ]:
print("📊 Edge Stats Before Filtering")
print(card["edge_metric"].describe())

# Dynamic threshold based on distribution
EDGE_THRESHOLD = card["edge_metric"].mean()  # MUCH safer than quantile for now

print("Using Threshold:", EDGE_THRESHOLD)

card = card[
    (card["edge_metric"] >= EDGE_THRESHOLD) &
    (card["unit_size"] > 0)
]

card = card.sort_values("edge_metric", ascending=False)

print("🔥 FINAL CARD")
print(card[[
    "home_team",
    "away_team",
    "bet_category",
    "bet_details",
    "price",
    "unit_size",
    "edge_metric"
]])

print("\nTotal Plays:", len(card))
print("Total Units:", card["unit_size"].sum())

In [ ]:
print("Columns Available:")
print(games_clean.columns)

print("\nEdge Metric Sample:")
print(games_clean[["ml_edge","spread_edge","total_edge","team_total_edge","edge_metric"]].head())

print("\nNaN Count:")
print(games_clean["edge_metric"].isna().sum())

In [ ]:
print(type(games_clean))

In [ ]:
# ===============================
# FULL REBUILD VERIFICATION BLOCK
# ===============================

import pandas as pd
import numpy as np

print("---- Checking Data ----")

# 1️⃣ Confirm games_clean exists
try:
    print("games_clean rows:", len(games_clean))
except NameError:
    raise Exception("❌ games_clean is NOT loaded. You must rerun your data pull cell first.")

# 2️⃣ Confirm required edge columns exist
required_cols = [
    "ml_edge",
    "spread_edge",
    "total_edge",
    "team_total_edge"
]

missing = [c for c in required_cols if c not in games_clean.columns]

if missing:
    raise Exception(f"❌ Missing edge columns: {missing}. Rerun your edge computation cell.")

print("Edge columns exist ✅")

# 3️⃣ Rebuild edge_metric safely
games_clean["edge_metric"] = (
    games_clean["ml_edge"].fillna(0) +
    games_clean["spread_edge"].fillna(0) +
    games_clean["total_edge"].fillna(0) +
    games_clean["team_total_edge"].fillna(0)
)

print("edge_metric rebuilt ✅")

print("\nEdge Summary:")
print(games_clean["edge_metric"].describe())

# 4️⃣ Build card fresh
card = games_clean.copy()

# 5️⃣ Sort by strongest edges
card = card.sort_values("edge_metric", ascending=False)

print("\nTop 10 Raw Edges:")
print(card[[
    "home_team",
    "away_team",
    "edge_metric"
]].head(10))

print("\nRows in card:", len(card))

In [ ]:
# ======================================
# NCAAB SLATE PULL — ET WINDOW 3/2
# ======================================

import requests, pandas as pd
from datetime import datetime, timedelta
import pytz

SLATE_DATE = "2026-03-02"

ET = pytz.timezone("US/Eastern")
start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et = start_et + timedelta(days=1)

start_utc = start_et.astimezone(pytz.utc)
end_utc = end_et.astimezone(pytz.utc)

url = "https://api.the-odds-api.com/v4/sports/basketball_ncaab/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)

data = r.json()

rows = []
for g in data:
    home = g["home_team"]
    away = g["away_team"]
    t = g["commence_time"]

    book = g["bookmakers"][0] if g["bookmakers"] else None

    def get_market(key):
        if not book: return None
        for m in book["markets"]:
            if m["key"] == key:
                return m
        return None

    h2h = get_market("h2h")
    spreads = get_market("spreads")
    totals = get_market("totals")

    def pick(mkt, name):
        if not mkt: return None
        for o in mkt["outcomes"]:
            if o["name"] == name:
                return o
        return None

    home_ml = away_ml = float("nan")
    if h2h:
        oh = pick(h2h, home)
        oa = pick(h2h, away)
        if oh: home_ml = oh["price"]
        if oa: away_ml = oa["price"]

    spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = float("nan")
    if spreads:
        oh = pick(spreads, home)
        oa = pick(spreads, away)
        if oh:
            spread_home_line = oh["point"]
            spread_home_odds = oh["price"]
        if oa:
            spread_away_line = oa["point"]
            spread_away_odds = oa["price"]

    total_line = over_odds = under_odds = float("nan")
    if totals:
        oo = pick(totals, "Over")
        ou = pick(totals, "Under")
        if oo:
            total_line = oo["point"]
            over_odds = oo["price"]
        if ou:
            under_odds = ou["price"]

    rows.append({
        "home_team": home,
        "away_team": away,
        "commence_time": t,
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": spread_home_line,
        "spread_home_odds": spread_home_odds,
        "spread_away_line": spread_away_line,
        "spread_away_odds": spread_away_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds,
    })

games_df = pd.DataFrame(rows)

games_df["commence_time"] = pd.to_datetime(games_df["commence_time"], utc=True)
games_df["commence_et"] = games_df["commence_time"].dt.tz_convert("US/Eastern")

print("✅ NCAA games pulled:", len(games_df))
games_df[["away_team","home_team","commence_et"]]

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")
print("✅ ODDS_API_KEY loaded:", bool(ODDS_API_KEY))

In [ ]:
# ======================================
# NCAAB FULL STRICT ENGINE — 3/2 (drop-in)
# ======================================

import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import pytz

# ---- SETTINGS (keep consistent) ----
SPORT = "basketball_ncaab"
SIMS = 50000
SD_MARGIN = 11.5
SD_TOTAL  = 15.0

HOME_EDGE_PTS = 2.0

BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200  # filter extreme negative prices if desired

rng = np.random.default_rng(42)

# ---- Helpers ----
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds: float) -> float:
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def ev_unit(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    return prob*(dec-1) - (1-prob)

def kelly_fraction(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    b = dec - 1
    if b <= 0:
        return 0.0
    return max((prob*dec - 1)/b, 0.0)

def to_units(prob: float, odds: float) -> float:
    # half-Kelly scaled into 0.25–1.0u band
    f = 0.5 * kelly_fraction(prob, odds)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ======================================
# 1) Pull completed scores (safe window)
# ======================================

def pull_completed_scores_safe(sport: str, days_list):
    all_parts = []
    for d in days_list:
        url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
        params = {"apiKey": ODDS_API_KEY, "daysFrom": int(d)}
        r = requests.get(url, params=params, timeout=30)
        if r.status_code != 200:
            print(f"⚠ daysFrom {d} failed — skipping")
            continue

        data = r.json()
        rows = []
        for g in data:
            if not g.get("completed"):
                continue
            home = g.get("home_team")
            away = g.get("away_team")
            ct = g.get("commence_time")
            scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
            hs = scores.get(home)
            as_ = scores.get(away)
            if hs is None or as_ is None:
                continue
            rows.append({
                "home_team": home,
                "away_team": away,
                "home_score": float(hs),
                "away_score": float(as_),
                "date": ct
            })

        part_df = pd.DataFrame(rows)
        print(f"✅ daysFrom {d}: {len(part_df)} games")
        all_parts.append(part_df)

    if not all_parts:
        raise ValueError("No historical score data pulled.")
    out = pd.concat(all_parts, ignore_index=True).drop_duplicates()
    return out

historical_df = pull_completed_scores_safe(SPORT, [3, 6, 10, 14])
print("✅ Unique historical games pulled:", len(historical_df))

# ======================================
# 2) Build PPP baseline (team_overall)
# ======================================

PPP_LEAGUE = 1.05  # NCAAB rough PPP for poss proxy
hist = historical_df.copy()
hist["total_pts"] = hist["home_score"] + hist["away_score"]
hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "points_scored": hist["home_score"],
    "points_allowed": hist["away_score"],
    "poss": hist["poss_proxy"],
})
away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "points_scored": hist["away_score"],
    "points_allowed": hist["home_score"],
    "poss": hist["poss_proxy"],
})

all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

team_overall = all_stats.groupby("team").agg(
    poss=("poss","mean"),
    ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
    ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
).reset_index()

team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))
print("✅ team_overall built:", team_overall.shape)

# ======================================
# 3) Merge baseline into slate
# ======================================

g = games_df.copy()

# numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_line","spread_away_odds","total_line","over_odds","under_odds"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")
    g[c] = g[c].fillna(g[c].mean())

# ======================================
# 4) Projection layer (PPP)
# ======================================

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_pts_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_pts_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_pts_proj"] - g["away_pts_proj"] + HOME_EDGE_PTS
g["mean_total_model"]  = g["home_pts_proj"] + g["away_pts_proj"]

# ======================================
# 5) Market blend (25/75)
# ======================================

g["market_margin"] = -g["spread_home_line"]

g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * g["total_line"]

# Keep for later use
games_df = g.copy()

# ======================================
# 6) Sims
# ======================================

df = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_away_win"]   = 1 - df["p_home_win"]

# Spread cover probs based on each side's own line
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = (margins < (df["spread_away_line"].to_numpy()[:, None])).mean(axis=1)

df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# ======================================
# 7) Market implied (vig-removed for ML)
# ======================================

ml_base = df.dropna(subset=["home_ml","away_ml"]).copy()
if len(ml_base):
    ml_base["home_imp"] = ml_base["home_ml"].apply(american_to_prob)
    ml_base["away_imp"] = ml_base["away_ml"].apply(american_to_prob)
    vig = (ml_base["home_imp"] + ml_base["away_imp"]).replace(0, np.nan)
    ml_base["home_mkt"] = ml_base["home_imp"]/vig
    ml_base["away_mkt"] = ml_base["away_imp"]/vig
else:
    ml_base = df.copy()
    ml_base["home_mkt"] = np.nan
    ml_base["away_mkt"] = np.nan

# ======================================
# 8) Build candidates (ML/Spread/Total)
# ======================================

plays = []

# ML: take best EV side (home or away)
for _, r in ml_base.iterrows():
    # home
    ev_h = ev_unit(r["p_home_win"], r["home_ml"])
    if (ev_h > 0) and (r["home_ml"] >= MAX_JUICE):
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "pick_team": r["home_team"],
            "bet": f"{r['home_team']} ML ({fmt_odds(r['home_ml'])})",
            "p_win": float(r["p_home_win"]),
            "p_mkt": float(r["home_mkt"]) if pd.notna(r["home_mkt"]) else np.nan,
            "ev": float(ev_h),
            "units": to_units(float(r["p_home_win"]), float(r["home_ml"])),
            "commence_time": r.get("commence_time", None),
        })
    # away
    ev_a = ev_unit(r["p_away_win"], r["away_ml"])
    if (ev_a > 0) and (r["away_ml"] >= MAX_JUICE):
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "pick_team": r["away_team"],
            "bet": f"{r['away_team']} ML ({fmt_odds(r['away_ml'])})",
            "p_win": float(r["p_away_win"]),
            "p_mkt": float(r["away_mkt"]) if pd.notna(r["away_mkt"]) else np.nan,
            "ev": float(ev_a),
            "units": to_units(float(r["p_away_win"]), float(r["away_ml"])),
            "commence_time": r.get("commence_time", None),
        })

# Spread: choose better EV between home and away
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp.apply(lambda r: ev_unit(r["p_home_cover"], r["spread_home_odds"]), axis=1)
sp["ev_away"] = sp.apply(lambda r: ev_unit(r["p_away_cover"], r["spread_away_odds"]), axis=1)

for _, r in sp.iterrows():
    if r["ev_home"] <= 0 and r["ev_away"] <= 0:
        continue
    if r["ev_home"] >= r["ev_away"]:
        team = r["home_team"]
        line = r["spread_home_line"]
        odds = r["spread_home_odds"]
        pwin = r["p_home_cover"]
        pmkt = american_to_prob(odds)
        evv  = r["ev_home"]
    else:
        team = r["away_team"]
        line = r["spread_away_line"]
        odds = r["spread_away_odds"]
        pwin = r["p_away_cover"]
        pmkt = american_to_prob(odds)
        evv  = r["ev_away"]

    plays.append({
        "matchup": f"{r['away_team']} at {r['home_team']}",
        "market": "Spread",
        "pick_team": team,
        "bet": f"{team} {fmt_line(line)} ({fmt_odds(odds)})",
        "p_win": float(pwin),
        "p_mkt": float(pmkt),
        "ev": float(evv),
        "units": to_units(float(pwin), float(odds)),
        "commence_time": r.get("commence_time", None),
    })

# Total: choose better EV between over and under
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to.apply(lambda r: ev_unit(r["p_over"],  r["over_odds"]), axis=1)
to["ev_under"] = to.apply(lambda r: ev_unit(r["p_under"], r["under_odds"]), axis=1)

for _, r in to.iterrows():
    if r["ev_over"] <= 0 and r["ev_under"] <= 0:
        continue
    if r["ev_over"] >= r["ev_under"]:
        side = "OVER"
        odds = r["over_odds"]
        pwin = r["p_over"]
        pmkt = american_to_prob(odds)
        evv  = r["ev_over"]
    else:
        side = "UNDER"
        odds = r["under_odds"]
        pwin = r["p_under"]
        pmkt = american_to_prob(odds)
        evv  = r["ev_under"]

    plays.append({
        "matchup": f"{r['away_team']} at {r['home_team']}",
        "market": "Total",
        "pick_team": "",
        "bet": f"{side} {r['total_line']} ({fmt_odds(odds)})",
        "p_win": float(pwin),
        "p_mkt": float(pmkt),
        "ev": float(evv),
        "units": to_units(float(pwin), float(odds)),
        "commence_time": r.get("commence_time", None),
    })

card = pd.DataFrame(plays)
if card.empty:
    print("⚠ No +EV plays found.")
    card

# ======================================
# 9) Exposure rules
# ======================================

card = card[(card["units"] > 0)].copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

# max 2 plays per matchup
kept = []
counts = {}
for _, r in card.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1
card2 = pd.DataFrame(kept).reset_index(drop=True)

# no ML + Spread same team in same matchup (keep higher EV; totals allowed)
out_rows = []
for matchup, grp in card2.groupby("matchup", sort=False):
    ml_rows = grp[grp["market"]=="ML"]
    sp_rows = grp[grp["market"]=="Spread"]
    if len(ml_rows) and len(sp_rows):
        # if same team targeted, keep only best EV among those two markets, plus totals
        same = False
        for _, mlr in ml_rows.iterrows():
            for _, spr in sp_rows.iterrows():
                if mlr["pick_team"] == spr["pick_team"] and mlr["pick_team"] != "":
                    same = True
        if same:
            best_idx = grp[grp["market"].isin(["ML","Spread"])].sort_values("ev", ascending=False).index[0]
            totals = grp[grp["market"]=="Total"]
            grp = pd.concat([grp.loc[[best_idx]], totals]).sort_values("ev", ascending=False)
    out_rows.append(grp)

final_card = pd.concat(out_rows, ignore_index=True).sort_values("ev", ascending=False).reset_index(drop=True)

# ======================================
# 10) Discord text + export
# ======================================

final_card["discord_text"] = (
    final_card["matchup"] + "\n" +
    final_card["bet"] + " — " + final_card["units"].astype(str) + "u\n" +
    "Sim Win%: " + final_card["p_win"].apply(pct) +
    " | Market: " + final_card["p_mkt"].apply(lambda x: "NA" if pd.isna(x) else pct(x)) +
    "\nEV: " + (final_card["ev"]*100).round(1).astype(str) + "%\n"
)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Final plays:", len(final_card))
print(final_card[["discord_text"]].to_string(index=False))

out_name = f"ncaab_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_name, index=False)
print("✅ Exported:", out_name)

In [ ]:
# ======================================
# QUICK ML DOG FILTER
# ======================================

ml_dogs = final_card[
    (final_card["market"] == "ML") &
    (final_card["bet"].str.contains("+"))  # positive price only
].copy()

ml_dogs = ml_dogs.sort_values("p_win", ascending=False).reset_index(drop=True)

print("🐶 ML Dogs Ranked by Sim Win%")
print(ml_dogs[[
    "matchup",
    "bet",
    "p_win",
    "ev",
    "units"
]].to_string(index=False))

In [ ]:
# ======================================
# QUICK ML DOG FILTER (FIXED)
# ======================================

ml_dogs = final_card[
    (final_card["market"] == "ML") &
    (final_card["bet"].str.contains(r"\+", regex=True, na=False))  # escape +
].copy()

ml_dogs = ml_dogs.sort_values("p_win", ascending=False).reset_index(drop=True)

print("🐶 ML Dogs Ranked by Sim Win%")
print(ml_dogs[["matchup","bet","p_win","p_mkt","ev","units"]].to_string(index=False))

In [ ]:
# ======================================
# MOST PROBABLE ML DOGS (FROM SLEET SIMS)
# ======================================

import numpy as np
import pandas as pd

df = games_df.copy()

# keep only games with MLs and sim win probs present (from engine run)
need_cols = ["home_team","away_team","home_ml","away_ml","p_home_win","p_away_win"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    print("Missing columns:", missing)
    print("Run FULL STRICT engine cell first (the one that creates p_home_win / p_away_win).")
else:
    # Identify underdogs by positive ML price
    dogs = []

    # home dog
    home_dogs = df[pd.to_numeric(df["home_ml"], errors="coerce") > 0].copy()
    if len(home_dogs):
        home_dogs["side"] = "HOME"
        home_dogs["dog_team"] = home_dogs["home_team"]
        home_dogs["dog_ml"] = home_dogs["home_ml"]
        home_dogs["p_win"] = home_dogs["p_home_win"]
        home_dogs["ev_unit"] = home_dogs.apply(lambda r: ev_unit(r["p_win"], r["dog_ml"]), axis=1)
        dogs.append(home_dogs)

    # away dog
    away_dogs = df[pd.to_numeric(df["away_ml"], errors="coerce") > 0].copy()
    if len(away_dogs):
        away_dogs["side"] = "AWAY"
        away_dogs["dog_team"] = away_dogs["away_team"]
        away_dogs["dog_ml"] = away_dogs["away_ml"]
        away_dogs["p_win"] = away_dogs["p_away_win"]
        away_dogs["ev_unit"] = away_dogs.apply(lambda r: ev_unit(r["p_win"], r["dog_ml"]), axis=1)
        dogs.append(away_dogs)

    if not dogs:
        print("No ML dogs available in odds pull (no +money MLs found).")
    else:
        dogs_df = pd.concat(dogs, ignore_index=True)
        dogs_df["matchup"] = dogs_df["away_team"] + " at " + dogs_df["home_team"]
        dogs_df = dogs_df.sort_values("p_win", ascending=False).reset_index(drop=True)

        print("🐶 Most probable ML dogs (ranked by Sim Win%)")
        print(dogs_df[["matchup","dog_team","dog_ml","p_win","ev_unit"]].head(10).to_string(index=False))

In [ ]:
# ======================================
# ML DOGS — MOST PROBABLE (FROM SIM DF)
# ======================================

# Use the sim dataframe (df) produced by the FULL STRICT engine cell
if "df" not in globals():
    raise NameError("df not found. Run the FULL STRICT engine cell first (the one that prints ✅ Sims: 50000 ...).")

need = ["home_team","away_team","home_ml","away_ml","p_home_win","p_away_win"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"df is missing columns: {missing}. Re-run the FULL STRICT engine cell.")

dogs = []

# Home ML dogs (+ money)
home_dogs = df[pd.to_numeric(df["home_ml"], errors="coerce") > 0].copy()
if len(home_dogs):
    home_dogs["dog_team"] = home_dogs["home_team"]
    home_dogs["dog_ml"] = home_dogs["home_ml"]
    home_dogs["p_win"] = home_dogs["p_home_win"]
    home_dogs["ev_unit"] = home_dogs.apply(lambda r: ev_unit(r["p_win"], r["dog_ml"]), axis=1)
    dogs.append(home_dogs)

# Away ML dogs (+ money)
away_dogs = df[pd.to_numeric(df["away_ml"], errors="coerce") > 0].copy()
if len(away_dogs):
    away_dogs["dog_team"] = away_dogs["away_team"]
    away_dogs["dog_ml"] = away_dogs["away_ml"]
    away_dogs["p_win"] = away_dogs["p_away_win"]
    away_dogs["ev_unit"] = away_dogs.apply(lambda r: ev_unit(r["p_win"], r["dog_ml"]), axis=1)
    dogs.append(away_dogs)

if not dogs:
    print("No +money ML dogs found in df (odds pull may be missing MLs for dogs).")
else:
    dogs_df = pd.concat(dogs, ignore_index=True)
    dogs_df["matchup"] = dogs_df["away_team"] + " at " + dogs_df["home_team"]
    dogs_df = dogs_df.sort_values("p_win", ascending=False).reset_index(drop=True)

    print("🐶 Most probable ML dogs (ranked by Sim Win%)")
    print(dogs_df[["matchup","dog_team","dog_ml","p_win","ev_unit"]].head(15).to_string(index=False))

In [ ]:
# ======================================
# REBUILD SIM DF (WIN PROBS ONLY)
# ======================================

import numpy as np

if "games_df" not in globals():
    raise NameError("games_df not found. Run the FULL STRICT engine cell first.")

SIMS = 50000
SD_MARGIN = 11.5
SD_TOTAL  = 15.0

rng = np.random.default_rng(42)

df = games_df.copy()

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# Monte Carlo
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"] = (margins > 0).mean(axis=1)
df["p_away_win"] = 1 - df["p_home_win"]

print("✅ Win probabilities rebuilt")
print("Games:", len(df))

In [ ]:
# ======================================
# ML DOGS — MOST PROBABLE
# ======================================

dogs = []

home_dogs = df[pd.to_numeric(df["home_ml"], errors="coerce") > 0].copy()
if len(home_dogs):
    home_dogs["dog_team"] = home_dogs["home_team"]
    home_dogs["dog_ml"] = home_dogs["home_ml"]
    home_dogs["p_win"] = home_dogs["p_home_win"]
    dogs.append(home_dogs)

away_dogs = df[pd.to_numeric(df["away_ml"], errors="coerce") > 0].copy()
if len(away_dogs):
    away_dogs["dog_team"] = away_dogs["away_team"]
    away_dogs["dog_ml"] = away_dogs["away_ml"]
    away_dogs["p_win"] = away_dogs["p_away_win"]
    dogs.append(away_dogs)

if not dogs:
    print("No +money ML dogs found.")
else:
    dogs_df = pd.concat(dogs, ignore_index=True)
    dogs_df["matchup"] = dogs_df["away_team"] + " at " + dogs_df["home_team"]
    dogs_df = dogs_df.sort_values("p_win", ascending=False).reset_index(drop=True)

    print("🐶 Most probable ML dogs (ranked by Sim Win%)")
    print(dogs_df[["matchup","dog_team","dog_ml","p_win"]].head(15).to_string(index=False))

In [ ]:
# ================================
# (A) SETTINGS + KEY (PASTE 1st)
# ================================
import os, re, math, time, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import pytz
from getpass import getpass

# ---- KEY ----
if not os.getenv("ODDS_API_KEY"):
    os.environ["ODDS_API_KEY"] = getpass("10ceac94f9b9cb76f8c65510ca6df18f").strip()
ODDS_API_KEY = os.getenv("ODDS_API_KEY")
if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY missing.")

# ---- SLATE ----
SLATE_DATE = "2026-03-03"  # <-- change each day (ET date)
SPORT = "basketball_ncaab"
TZ = pytz.timezone("America/Indiana/Indianapolis")  # your default

# ---- STRICT constants (keep consistent) ----
SIMS = 50000
SD_MARGIN = 11.5
SD_TOTAL  = 15.0
HOME_EDGE_PTS = 2.0

# Model/market blend (same style as our recent NCAAB runs)
BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

# Kelly / card rules
HALF_KELLY = True
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_PER_GAME = 2               # correlated exposure cap
MAX_JUICE = -200               # optional filter (ignore super juiced lines)

rng = np.random.default_rng(42)

def american_to_prob(odds: float) -> float:
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds: float) -> float:
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/(-odds))

def kelly_fraction(p: float, odds_american: float) -> float:
    d = american_to_decimal(odds_american)
    b = d - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (b*p - q)/b
    return max(0.0, f)

def clamp_units(u: float) -> float:
    return float(max(UNIT_MIN, min(UNIT_CAP, u)))

print("✅ Settings loaded:", SPORT, SLATE_DATE, "| sims:", SIMS)

In [ ]:
# ==========================================
# (B) PULL SLATE (ET WINDOW) + HIST SCORES
#     + TEAM_OVERALL (PASTE 2nd)
# ==========================================
def pull_slate_odds_et(sport: str, slate_date_et: str) -> pd.DataFrame:
    start_et = TZ.localize(datetime.strptime(slate_date_et, "%Y-%m-%d"))
    end_et   = start_et + timedelta(days=1)
    start_utc = start_et.astimezone(pytz.utc)
    end_utc   = end_et.astimezone(pytz.utc)

    print("ET window:", start_et, "to", end_et)
    print("UTC window:", start_utc, "to", end_utc)

    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": "us",
        "markets": "h2h,spreads,totals",
        "oddsFormat": "american",
        "commenceTimeFrom": start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "commenceTimeTo":   end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API odds failed: {r.status_code} | {r.text[:250]}")
    data = r.json()

    rows = []
    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        commence = g.get("commence_time")

        # best-effort consensus (first book that has each market)
        home_ml = away_ml = np.nan
        spread_home_line = spread_home_odds = np.nan
        spread_away_line = spread_away_odds = np.nan
        total_line = over_odds = under_odds = np.nan

        for bk in g.get("bookmakers", []):
            for m in bk.get("markets", []):
                key = m.get("key")
                outs = m.get("outcomes", [])

                if key == "h2h" and (pd.isna(home_ml) or pd.isna(away_ml)):
                    for o in outs:
                        if o.get("name") == home:
                            home_ml = o.get("price")
                        elif o.get("name") == away:
                            away_ml = o.get("price")

                if key == "spreads" and pd.isna(spread_home_line):
                    for o in outs:
                        if o.get("name") == home:
                            spread_home_line = o.get("point")
                            spread_home_odds = o.get("price")
                        elif o.get("name") == away:
                            spread_away_line = o.get("point")
                            spread_away_odds = o.get("price")

                if key == "totals" and pd.isna(total_line):
                    for o in outs:
                        if o.get("name") == "Over":
                            total_line = o.get("point")
                            over_odds = o.get("price")
                        elif o.get("name") == "Under":
                            under_odds = o.get("price")

        rows.append({
            "away_team": away,
            "home_team": home,
            "commence_time": commence,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True, errors="coerce")
    df["commence_et"] = df["commence_time"].dt.tz_convert(TZ)
    df = df.sort_values("commence_time").reset_index(drop=True)

    print("✅ NCAA games pulled:", len(df))
    display(df[["away_team","home_team","commence_et"]].head(20))
    return df

def pull_completed_scores(sport: str, days_from: int = 3) -> pd.DataFrame:
    # plan-safe: your account only allows daysFrom=3 (we keep it consistent)
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
    params = {
        "apiKey": ODDS_API_KEY,
        "daysFrom": int(days_from),
        "dateFormat": "iso"
    }
    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Scores API failed: {r.status_code} | {r.text[:250]}")
    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g.get("home_team"); away = g.get("away_team")
        scores = g.get("scores") or []
        hs = as_ = None
        for s in scores:
            if s.get("name") == home: hs = s.get("score")
            if s.get("name") == away: as_ = s.get("score")
        rows.append({"home_team": home, "away_team": away, "home_score": hs, "away_score": as_})

    gh = pd.DataFrame(rows).dropna()
    gh["home_score"] = pd.to_numeric(gh["home_score"], errors="coerce")
    gh["away_score"] = pd.to_numeric(gh["away_score"], errors="coerce")
    gh = gh.dropna(subset=["home_score","away_score"]).reset_index(drop=True)
    return gh

def build_team_overall_from_scores(games_hist: pd.DataFrame) -> pd.DataFrame:
    gh = games_hist.copy()
    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["away_margin"] = -gh["home_margin"]
    gh["game_total"]  = gh["home_score"] + gh["away_score"]

    home = gh[["home_team","home_score","away_score","home_margin","game_total"]].rename(columns={
        "home_team":"team","home_score":"pts_for","away_score":"pts_against","home_margin":"margin"
    })
    away = gh[["away_team","away_score","home_score","away_margin","game_total"]].rename(columns={
        "away_team":"team","away_score":"pts_for","home_score":"pts_against","away_margin":"margin"
    })
    long = pd.concat([home, away], ignore_index=True)

    team = long.groupby("team", as_index=False).agg(
        margin_mean=("margin","mean"),
        pts_for_mean=("pts_for","mean"),
        pts_against_mean=("pts_against","mean"),
        total_mean=("game_total","mean"),
        games=("margin","size")
    )
    return team

# --- Run pulls ---
games_df = pull_slate_odds_et(SPORT, SLATE_DATE)

games_hist = pull_completed_scores(SPORT, 3)
print("✅ pull_completed_scores(3) →", len(games_hist), "games")

team_overall = build_team_overall_from_scores(games_hist)
print("✅ team_overall built:", team_overall.shape)
display(team_overall.sort_values("games", ascending=False).head(10))

In [ ]:
# ======================================================
# (C) FULL STRICT ENGINE (MC SIMS + CARD + EXPORT) (3rd)
# ======================================================
g = games_df.copy()

# merge team_overall to slate
g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "margin_mean":"h_margin_mean","pts_for_mean":"h_pf","pts_against_mean":"h_pa","total_mean":"h_total_mean","games":"h_games"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "margin_mean":"a_margin_mean","pts_for_mean":"a_pf","pts_against_mean":"a_pa","total_mean":"a_total_mean","games":"a_games"
}).drop(columns=["team"], errors="ignore")

# fallback if missing hist
for col in ["h_margin_mean","a_margin_mean","h_pf","a_pf","h_pa","a_pa","h_total_mean","a_total_mean"]:
    g[col] = pd.to_numeric(g[col], errors="coerce")

# --- base model projections (scores-only consistent) ---
# mean_margin is "home - away" (positive => home better)
g["mean_margin_model"] = (g["h_margin_mean"] - g["a_margin_mean"]) + HOME_EDGE_PTS
g["mean_total_model"]  = (g["h_total_mean"] + g["a_total_mean"]) / 2.0

# --- market anchors ---
g["market_margin"] = -pd.to_numeric(g["spread_home_line"], errors="coerce")  # if home -3 => spread_home_line=-3 => market_margin=+3
g["market_total"]  = pd.to_numeric(g["total_line"], errors="coerce")

# blend
g["mean_margin"] = BLEND_MODEL*g["mean_margin_model"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total_model"]  + BLEND_MARKET*g["market_total"]

# keep only usable rows
need = ["mean_margin","mean_total","spread_home_line","spread_away_line","total_line",
        "home_ml","away_ml","spread_home_odds","spread_away_odds","over_odds","under_odds"]
for c in need:
    g[c] = pd.to_numeric(g[c], errors="coerce")

g = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# --- Monte Carlo ---
margins = rng.normal(g["mean_margin"].to_numpy()[:,None], SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(g["mean_total"].to_numpy()[:,None],  SD_TOTAL,  size=(len(g), SIMS))

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

# spreads: home covers if (home_score - away_score) > spread_home_line
# (spread_home_line is negative when home favored)
g["p_home_cover"] = (margins > g["spread_home_line"].to_numpy()[:,None]).mean(axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

# totals
g["p_over"]  = (totals > g["total_line"].to_numpy()[:,None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Games in sim df:", len(g))

# --- Build card candidates ---
def make_row(matchup, market, bet, p_win, odds):
    p_mkt = american_to_prob(odds)
    ev = (p_win * american_to_decimal(odds)) - 1.0
    f = kelly_fraction(p_win, odds)
    if HALF_KELLY:
        f *= 0.5
    units = clamp_units(f * 1.0) if ev > 0 else UNIT_MIN
    return {
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "p_win": float(p_win),
        "p_mkt": float(p_mkt),
        "ev": float(ev),
        "odds": float(odds),
        "units": float(units)
    }

rows = []
for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # ML
    if not pd.isna(r["home_ml"]):
        rows.append(make_row(matchup, "ML", f"{r['home_team']} ML ({int(r['home_ml'])})", r["p_home_win"], r["home_ml"]))
    if not pd.isna(r["away_ml"]):
        rows.append(make_row(matchup, "ML", f"{r['away_team']} ML ({int(r['away_ml'])})", r["p_away_win"], r["away_ml"]))

    # Spread (home / away)
    if not pd.isna(r["spread_home_odds"]):
        line = r["spread_home_line"]
        sign = "+" if line > 0 else ""
        rows.append(make_row(matchup, "SPREAD", f"{r['home_team']} {sign}{line:g} ({int(r['spread_home_odds'])})", r["p_home_cover"], r["spread_home_odds"]))
    if not pd.isna(r["spread_away_odds"]):
        line = r["spread_away_line"]
        sign = "+" if line > 0 else ""
        rows.append(make_row(matchup, "SPREAD", f"{r['away_team']} {sign}{line:g} ({int(r['spread_away_odds'])})", r["p_away_cover"], r["spread_away_odds"]))

    # Total (O/U)
    if not pd.isna(r["over_odds"]):
        rows.append(make_row(matchup, "TOTAL", f"OVER {r['total_line']:g} ({int(r['over_odds'])})", r["p_over"], r["over_odds"]))
    if not pd.isna(r["under_odds"]):
        rows.append(make_row(matchup, "TOTAL", f"UNDER {r['total_line']:g} ({int(r['under_odds'])})", r["p_under"], r["under_odds"]))

card = pd.DataFrame(rows)

# --- Filters ---
card = card[pd.to_numeric(card["odds"], errors="coerce").notna()].copy()
card = card[(card["odds"] >= 0) | (card["odds"] >= MAX_JUICE)]  # keep +odds; drop too-juiced negatives
card = card[card["ev"] > 0].copy()

# --- Rank (EV first, then p_win) ---
card["rank_score"] = (card["ev"]*0.7) + (card["p_win"]*0.3)
card = card.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# --- Cap correlated exposure (max 2 per game) ---
card["game_key"] = card["matchup"]
kept = []
counts = {}
for _, r in card.iterrows():
    k = r["game_key"]
    if counts.get(k, 0) >= MAX_PER_GAME:
        continue
    kept.append(r)
    counts[k] = counts.get(k, 0) + 1
final_card = pd.DataFrame(kept).drop(columns=["game_key"]).reset_index(drop=True)

# --- Discord text ---
def fmt_pct(x): return f"{100*x:.1f}%"
final_card["discord_text"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {r['units']:.2f}u\n"
    f"Sim Win%: {fmt_pct(r['p_win'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*r['ev']:.1f}%\n", axis=1
)

out_path = f"ncaab_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Final plays:", len(final_card))
print("✅ Exported:", out_path)

print("\n=== DISCORD CARD ===\n")
print("\n".join(final_card["discord_text"].tolist()))

In [ ]:
# ============================================================
# ENV SETUP — API KEY + TIMEZONE
# Run this first after Colab runtime reconnects
# ============================================================

import os
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# --- INPUT YOUR KEY ---
ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

# --- TIMEZONE ---
LOCAL_TZ = "America/New_York"   # EST/EDT

# Save to env so other cells can use it
os.environ["ODDS_API_KEY"] = ODDS_API_KEY
os.environ["LOCAL_TZ"] = LOCAL_TZ

print("✅ Environment ready")
print("Timezone:", LOCAL_TZ)
print("API key loaded:", ODDS_API_KEY[:6] + "...")

In [ ]:
# === NCAAB VOLATILITY FIX ===
SD_MARGIN = 16.5
SD_TOTAL  = 19.0

print("🔧 Adjusted NCAAB volatility:")
print("SD_MARGIN =", SD_MARGIN)
print("SD_TOTAL  =", SD_TOTAL)

In [ ]:
# === HARD MARKET ANCHOR (NCAAB SAFE MODE) ===

MAX_DEVIATION = 4.0   # max points model can move off market

raw_delta = g["mean_margin_model"] - g["market_margin"]

# Clip deviation
clipped_delta = raw_delta.clip(lower=-MAX_DEVIATION, upper=MAX_DEVIATION)

g["mean_margin"] = g["market_margin"] + clipped_delta
g["mean_total"]  = g["market_total"]  # do not allow total drift yet

print("✅ Hard anchor applied. Max deviation:", MAX_DEVIATION)

In [ ]:
# ==========================================
# HARD MARKET ANCHOR (NCAAB SAFE MODE)
# ==========================================

MAX_DEVIATION = 4.0  # model cannot move spread more than 4 pts off market

# Compute deviation
raw_delta = g["mean_margin_model"] - g["market_margin"]

# Clip it
clipped_delta = raw_delta.clip(lower=-MAX_DEVIATION, upper=MAX_DEVIATION)

# Force overwrite
g["mean_margin"] = g["market_margin"] + clipped_delta
g["mean_total"]  = g["market_total"]

print("✅ Hard anchor applied")
print("New margin range:",
      round(g["mean_margin"].min(),2),
      "to",
      round(g["mean_margin"].max(),2))

In [ ]:
# ==========================================
# REBUILD MONTE CARLO (CLEAN)
# ==========================================

margins = rng.normal(
    g["mean_margin"].to_numpy()[:, None],
    SD_MARGIN,
    size=(len(g), SIMS)
)

totals = rng.normal(
    g["mean_total"].to_numpy()[:, None],
    SD_TOTAL,
    size=(len(g), SIMS)
)

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

g["p_home_cover"] = (
    margins > g["spread_home_line"].to_numpy()[:, None]
).mean(axis=1)

g["p_away_cover"] = 1 - g["p_home_cover"]

g["p_over"] = (totals > g["total_line"].to_numpy()[:, None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

print("✅ Monte Carlo rebuilt with hard anchor")

In [ ]:
# ==========================================
# SYNC STRICT PROBABILITIES INTO games_df
# ==========================================

prob_cols = [
    "p_home_win",
    "p_away_win",
    "p_home_cover",
    "p_away_cover",
    "p_over",
    "p_under"
]

# Make sure row counts match
if len(games_df) != len(g):
    raise RuntimeError("Row mismatch between g and games_df")

for col in prob_cols:
    games_df[col] = g[col].values

print("✅ Probabilities synced into games_df")
print(games_df[prob_cols].head())

In [ ]:
# ==========================================
# SANITY CHECK (BIG SPREADS)
# ==========================================

preview = g[[
    "away_team",
    "home_team",
    "mean_margin",
    "spread_home_line",
    "p_home_cover"
]].copy()

preview["abs_spread"] = preview["spread_home_line"].abs()

preview.sort_values("abs_spread", ascending=False).head(10)

In [ ]:
# ==========================================
# REBUILD SPREAD CARD ONLY
# ==========================================

rows = []

for _, r in g.iterrows():

    matchup = f"{r['away_team']} at {r['home_team']}"

    # Home side
    if not pd.isna(r["spread_home_odds"]):
        rows.append({
            "matchup": matchup,
            "bet": f"{r['home_team']} {r['spread_home_line']} ({int(r['spread_home_odds'])})",
            "p_win": r["p_home_cover"],
            "p_mkt": american_to_prob(r["spread_home_odds"]),
            "ev": (r["p_home_cover"] * american_to_decimal(r["spread_home_odds"])) - 1,
            "odds": r["spread_home_odds"]
        })

    # Away side
    if not pd.isna(r["spread_away_odds"]):
        rows.append({
            "matchup": matchup,
            "bet": f"{r['away_team']} {r['spread_away_line']} ({int(r['spread_away_odds'])})",
            "p_win": r["p_away_cover"],
            "p_mkt": american_to_prob(r["spread_away_odds"]),
            "ev": (r["p_away_cover"] * american_to_decimal(r["spread_away_odds"])) - 1,
            "odds": r["spread_away_odds"]
        })

spread_card = pd.DataFrame(rows)

# Keep only realistic +EV
spread_card = spread_card[
    (spread_card["ev"] > 0) &
    (spread_card["p_win"] < 0.80)  # remove unrealistic extremes
].sort_values("ev", ascending=False).reset_index(drop=True)

print("✅ Rebuilt spread card:", len(spread_card), "rows")
spread_card.head(15)

In [ ]:
# ==========================================
# BUILD FULL CARD (ML + SPREAD + TOTAL) + DISCORD OUTPUT
# ==========================================

CARD_HEADER = "== CDR BETTING | NCAAB FULL CARD (3/3) =="

MIN_PWIN = 0.58          # set None to disable
MAX_PER_GAME = 2
KELLY_FRACTION = 0.50
UNIT_MIN = 0.25
UNIT_MAX = 1.00

def american_to_decimal(odds):
    odds = float(odds)
    return (1 + odds/100) if odds > 0 else (1 + 100/abs(odds))

def american_to_prob(odds):
    odds = float(odds)
    return 100/(odds+100) if odds > 0 else abs(odds)/(abs(odds)+100)

def half_kelly_units(p, odds, frac=0.5, umin=0.25, umax=1.0):
    d = american_to_decimal(odds)
    b = d - 1.0
    q = 1 - p
    k = (b*p - q)/b
    k = max(0.0, k) * frac
    return round(min(umax, max(umin, k*1.0)), 2)

# ---- pick the best "master df" available
df = None
for name in ["games_df", "g", "sim_df"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame) and len(globals()[name]) > 0:
        df = globals()[name].copy()
        print(f"✅ Using master df: {name} | rows={len(df)}")
        break
if df is None:
    raise RuntimeError("No master df found (games_df/g/sim_df). Run the main engine cells first.")

# ---- column helpers (handle different notebooks)
def pick_col(cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

col_matchup = pick_col(["matchup"])
if col_matchup is None:
    # build matchup if not present
    if "away_team" in df.columns and "home_team" in df.columns:
        df["matchup"] = df["away_team"] + " at " + df["home_team"]
        col_matchup = "matchup"
    else:
        raise RuntimeError("Need matchup or (away_team, home_team) columns.")

col_home = pick_col(["home_team"])
col_away = pick_col(["away_team"])
col_home_ml = pick_col(["home_ml", "home_moneyline", "h2h_home"])
col_away_ml = pick_col(["away_ml", "away_moneyline", "h2h_away"])
col_spread = pick_col(["spread_home_line", "home_spread", "spread"])
col_total = pick_col(["total_line", "total", "ou_line"])

need_probs = ["p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
missing_probs = [c for c in need_probs if c not in df.columns]
if missing_probs:
    raise RuntimeError(f"Missing STRICT prob columns in master df: {missing_probs} (run FULL STRICT probs cell first)")

if col_home_ml is None or col_away_ml is None or col_spread is None or col_total is None:
    raise RuntimeError(f"Missing market line cols. Found: home_ml={col_home_ml}, away_ml={col_away_ml}, spread={col_spread}, total={col_total}")

# ---- build candidate bets
rows = []

# ML
for side, pcol, mlcol, teamcol in [
    ("HOME", "p_home_win", col_home_ml, col_home),
    ("AWAY", "p_away_win", col_away_ml, col_away),
]:
    tmp = df[[col_matchup, mlcol, pcol]].copy()
    tmp = tmp.rename(columns={col_matchup:"matchup", mlcol:"odds", pcol:"p_win"})
    tmp["market"] = "ML"
    tmp["bet"] = tmp.apply(lambda r: f"{(df.loc[r.name, teamcol] if teamcol in df.columns else side)} ML ({int(r['odds'])})", axis=1)
    tmp["p_mkt"] = tmp["odds"].apply(american_to_prob)
    tmp["ev"] = tmp["p_win"] - tmp["p_mkt"]
    rows.append(tmp)

# SPREAD (assumes spread_home_line is home handicap; build both sides)
sp = df[[col_matchup, col_spread, "p_home_cover", "p_away_cover"]].copy()
sp = sp.rename(columns={col_matchup:"matchup", col_spread:"line"})
# home cover bet
home_sp = sp.copy()
home_sp["market"] = "SPREAD"
home_sp["p_win"] = home_sp["p_home_cover"]
home_sp["odds"] = -110
home_sp["bet"] = home_sp["matchup"].apply(lambda m: f"HOME {df.loc[sp.index[0],col_spread]:g}")  # placeholder replaced below
# away cover bet
away_sp = sp.copy()
away_sp["market"] = "SPREAD"
away_sp["p_win"] = away_sp["p_away_cover"]
away_sp["odds"] = -110
away_sp["bet"] = away_sp["matchup"].apply(lambda m: f"AWAY {-df.loc[sp.index[0],col_spread]:g}")  # placeholder replaced below

# replace bet text per row with actual teams/lines
def fmt_spread(row, side):
    line = float(row["line"])
    if side == "HOME":
        team = df.loc[row.name, col_home] if col_home in df.columns else "HOME"
        return f"{team} {line:+g} (-110)"
    else:
        team = df.loc[row.name, col_away] if col_away in df.columns else "AWAY"
        return f"{team} {(-line):+g} (-110)"

home_sp["bet"] = home_sp.apply(lambda r: fmt_spread(r,"HOME"), axis=1)
away_sp["bet"] = away_sp.apply(lambda r: fmt_spread(r,"AWAY"), axis=1)

home_sp["p_mkt"] = 0.523810
home_sp["ev"] = home_sp["p_win"] - home_sp["p_mkt"]
away_sp["p_mkt"] = 0.523810
away_sp["ev"] = away_sp["p_win"] - away_sp["p_mkt"]

rows += [home_sp[["matchup","market","bet","p_win","p_mkt","ev","odds"]],
         away_sp[["matchup","market","bet","p_win","p_mkt","ev","odds"]]]

# TOTALS (O/U)
tot = df[[col_matchup, col_total, "p_over", "p_under"]].copy()
tot = tot.rename(columns={col_matchup:"matchup", col_total:"line"})
over = tot.copy()
over["market"] = "TOTAL"
over["p_win"] = over["p_over"]
over["odds"] = -110
over["bet"] = over["line"].apply(lambda x: f"OVER {float(x):g} (-110)")
over["p_mkt"] = 0.523810
over["ev"] = over["p_win"] - over["p_mkt"]

under = tot.copy()
under["market"] = "TOTAL"
under["p_win"] = under["p_under"]
under["odds"] = -110
under["bet"] = under["line"].apply(lambda x: f"UNDER {float(x):g} (-110)")
under["p_mkt"] = 0.523810
under["ev"] = under["p_win"] - under["p_mkt"]

rows += [over[["matchup","market","bet","p_win","p_mkt","ev","odds"]],
         under[["matchup","market","bet","p_win","p_mkt","ev","odds"]]]

full_card = pd.concat(rows, ignore_index=True)

# filters + units
if MIN_PWIN is not None:
    full_card = full_card[full_card["p_win"] >= MIN_PWIN].copy()

full_card["units"] = full_card.apply(lambda r: half_kelly_units(r["p_win"], r["odds"], KELLY_FRACTION, UNIT_MIN, UNIT_MAX), axis=1)

# rank blend (prob + ev)
full_card["blend"] = (0.65 * full_card["p_win"]) + (0.35 * full_card["ev"])
full_card = full_card.sort_values(["blend","ev","p_win"], ascending=False).reset_index(drop=True)

# cap 2 per matchup
full_card["rk"] = full_card.groupby("matchup").cumcount()+1
full_card = full_card[full_card["rk"] <= MAX_PER_GAME].drop(columns=["rk"])

# Discord output
lines = [CARD_HEADER, ""]
for _, r in full_card.iterrows():
    lines.append(r["matchup"])
    lines.append(f"{r['bet']} — {r['units']:.2f}u")
    lines.append(f"Sim Win%: {r['p_win']*100:.1f}% | Market: {r['p_mkt']*100:.1f}%")
    lines.append(f"EV: {r['ev']*100:.1f}%")
    lines.append("")

print("\n".join(lines).strip())

out_path = "ncaab_full_postable_2026-03-03.csv"
full_card.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

full_card.head(25)

In [ ]:
# ==========================================
# FIX: MERGE STRICT PROBS (g) BACK INTO games_df BY KEY
# ==========================================
import pandas as pd

prob_cols = ["p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]

# ---- helper: pick best time column present in a df
def _pick_time_col(df):
    for c in ["commence_et","commence_utc","commence_time","start_time","game_time"]:
        if c in df.columns:
            return c
    return None

def _prep_key(df, tag):
    df = df.copy()

    # normalize team col names if needed
    rename_map = {}
    if "home_team" not in df.columns and "home" in df.columns: rename_map["home"] = "home_team"
    if "away_team" not in df.columns and "away" in df.columns: rename_map["away"] = "away_team"
    if rename_map:
        df = df.rename(columns=rename_map)

    if "home_team" not in df.columns or "away_team" not in df.columns:
        raise RuntimeError(f"{tag} missing home_team/away_team columns. Columns={list(df.columns)}")

    # choose time col (optional but preferred)
    tcol = _pick_time_col(df)
    if tcol is not None:
        df[tcol] = pd.to_datetime(df[tcol], errors="coerce", utc=True)
        # time bucket to avoid minor tz/format mismatches
        df["_time_bucket"] = df[tcol].dt.floor("5min")
    else:
        df["_time_bucket"] = pd.NaT

    # build key (include time bucket when present)
    df["_key"] = (
        df["away_team"].astype(str).str.strip().str.lower() + " @ " +
        df["home_team"].astype(str).str.strip().str.lower() + " | " +
        df["_time_bucket"].astype(str)
    )
    return df, tcol

# 1) prep both dfs
gk, g_tcol = _prep_key(g, "g")
mk, m_tcol = _prep_key(games_df, "games_df")

# 2) keep only needed cols from g
need_from_g = ["_key"] + [c for c in prob_cols if c in gk.columns]
missing_in_g = [c for c in prob_cols if c not in gk.columns]
if missing_in_g:
    raise RuntimeError(f"g is missing prob cols (run FULL STRICT probs cell first): {missing_in_g}")

g_probs = gk[need_from_g].drop_duplicates("_key")

# 3) merge into games_df
before = mk.shape[0]
merged = mk.drop(columns=[c for c in prob_cols if c in mk.columns], errors="ignore").merge(
    g_probs, on="_key", how="left"
)
after = merged.shape[0]
if before != after:
    raise RuntimeError(f"Merge rowcount changed ({before} -> {after}). Key collision/dup issue.")

# 4) report coverage
coverage = merged[prob_cols].notna().all(axis=1).mean()
missing_rows = merged[prob_cols].isna().any(axis=1).sum()

print(f"✅ Prob merge complete. Coverage: {coverage:.1%} | Missing rows: {missing_rows} / {len(merged)}")

# If coverage is poor, show examples so you can see what's mismatching
if coverage < 0.90:
    print("\n⚠ Low coverage. Sample missing keys:")
    display(merged.loc[merged[prob_cols].isna().any(axis=1), ["away_team","home_team", m_tcol if m_tcol else "_time_bucket","_key"]].head(10))

# 5) write back
games_df = merged.drop(columns=["_key","_time_bucket"], errors="ignore")
print("✅ games_df now has prob columns:", [c for c in prob_cols if c in games_df.columns])

In [ ]:
# ==========================================
# NEXT: VERIFY + REBUILD FINAL_CARD FROM games_df (NO MORE g)
# ==========================================
import numpy as np
import pandas as pd

prob_cols = ["p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
need = ["away_team","home_team","home_ml","away_ml","spread_home_line","spread_home_odds","total_line"] + prob_cols

missing = [c for c in need if c not in games_df.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}")

# quick merge sanity
bad = games_df[prob_cols].isna().any(axis=1).sum()
if bad:
    raise RuntimeError(f"STOP — {bad} rows still missing prob cols in games_df. Your g/games_df keys are misaligned.")

def american_to_implied(p):
    p = float(p)
    if p > 0:
        return 100.0 / (p + 100.0)
    return (-p) / ((-p) + 100.0)

def implied_from_odds(series):
    return series.astype(float).apply(american_to_implied)

def to_int_if_close(x):
    try:
        xf = float(x)
        if abs(xf - round(xf)) < 1e-6:
            return int(round(xf))
        return xf
    except:
        return x

def fmt_odds(x):
    x = to_int_if_close(x)
    if isinstance(x, (int, np.integer)) and x > 0:
        return f"+{x}"
    return f"{x}"

rows = []

for _, r in games_df.iterrows():
    away = r["away_team"]
    home = r["home_team"]

    # ---------- ML ----------
    # home ML
    ml_home = float(r["home_ml"])
    p_mkt_home = american_to_implied(ml_home)
    p_win_home = float(r["p_home_win"])
    ev_home = p_win_home / p_mkt_home - 1
    rows.append({
        "matchup": f"{away} at {home}",
        "market": "ML",
        "bet": f"{home} ML ({fmt_odds(ml_home)})",
        "p_win": p_win_home,
        "p_mkt": p_mkt_home,
        "ev": ev_home,
        "odds": ml_home,
        "team": home
    })

    # away ML
    ml_away = float(r["away_ml"])
    p_mkt_away = american_to_implied(ml_away)
    p_win_away = float(r["p_away_win"])
    ev_away = p_win_away / p_mkt_away - 1
    rows.append({
        "matchup": f"{away} at {home}",
        "market": "ML",
        "bet": f"{away} ML ({fmt_odds(ml_away)})",
        "p_win": p_win_away,
        "p_mkt": p_mkt_away,
        "ev": ev_away,
        "odds": ml_away,
        "team": away
    })

    # ---------- SPREAD ----------
    line_home = float(r["spread_home_line"])     # e.g. -1.5 (home favored), +7.5 (home dog)
    price_sp = float(r["spread_home_odds"])      # e.g. -110
    p_mkt_sp = american_to_implied(price_sp)

    # We always want the side that corresponds to the probability:
    # p_home_cover = P(home covers home spread line)
    # p_away_cover = P(away covers the opposite line)
    # If home line is negative, home is favorite; if positive, home is dog.
    # We'll create BOTH bets with correct p_win assigned.
    # Home spread bet:
    home_spread_txt = f"{home} {line_home:+g} ({fmt_odds(price_sp)})"
    rows.append({
        "matchup": f"{away} at {home}",
        "market": "SPREAD",
        "bet": home_spread_txt.replace("+", ""),  # keep "-1.5" or "7.5" style consistent
        "p_win": float(r["p_home_cover"]),
        "p_mkt": p_mkt_sp,
        "ev": float(r["p_home_cover"]) / p_mkt_sp - 1,
        "odds": price_sp,
        "team": home
    })

    # Away spread bet is opposite of home line
    away_line = -line_home
    away_spread_txt = f"{away} {away_line:+g} ({fmt_odds(price_sp)})"
    rows.append({
        "matchup": f"{away} at {home}",
        "market": "SPREAD",
        "bet": away_spread_txt.replace("+", ""),
        "p_win": float(r["p_away_cover"]),
        "p_mkt": p_mkt_sp,
        "ev": float(r["p_away_cover"]) / p_mkt_sp - 1,
        "odds": price_sp,
        "team": away
    })

    # ---------- TOTAL ----------
    total = float(r["total_line"])
    # assume -110 if not present; if you have total odds columns, swap in here.
    total_price = -110.0
    p_mkt_total = american_to_implied(total_price)

    rows.append({
        "matchup": f"{away} at {home}",
        "market": "TOTAL",
        "bet": f"OVER {total:g} ({fmt_odds(total_price)})",
        "p_win": float(r["p_over"]),
        "p_mkt": p_mkt_total,
        "ev": float(r["p_over"]) / p_mkt_total - 1,
        "odds": total_price,
        "team": "OVER"
    })
    rows.append({
        "matchup": f"{away} at {home}",
        "market": "TOTAL",
        "bet": f"UNDER {total:g} ({fmt_odds(total_price)})",
        "p_win": float(r["p_under"]),
        "p_mkt": p_mkt_total,
        "ev": float(r["p_under"]) / p_mkt_total - 1,
        "odds": total_price,
        "team": "UNDER"
    })

final_card = pd.DataFrame(rows)

# +EV only + cap 2 per game (keeps the clean card)
final_card = final_card[final_card["ev"] > 0].copy()
final_card["rank_ev"] = final_card["ev"].rank(ascending=False, method="first")
final_card = final_card.sort_values(["matchup","rank_ev"], ascending=[True, True])
final_card = final_card.groupby("matchup").head(2).reset_index(drop=True)

print("✅ final_card rebuilt:", len(final_card), "rows")
display(final_card[["matchup","market","bet","p_win","p_mkt","ev"]].head(20))

In [ ]:
import pandas as pd

prob_cols = ["p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]

# rows missing probs
missing_mask = games_df[prob_cols].isna().any(axis=1)
print("Missing prob rows:", missing_mask.sum(), "/", len(games_df))

# show what columns we can use to dedupe/match
print("\nPossible market columns present in games_df:")
for c in ["bookmaker","book","site","last_update","commence_utc","commence_et","home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]:
    if c in games_df.columns:
        print(" -", c)

# inspect the missing rows
cols_show = [c for c in ["away_team","home_team","commence_utc","commence_et","home_ml","away_ml","spread_home_line","total_line","bookmaker","book","site"] if c in games_df.columns]
display(games_df.loc[missing_mask, cols_show].head(30))

# how many duplicates by just matchup?
dup_ct = games_df.groupby(["away_team","home_team"]).size().sort_values(ascending=False).head(15)
print("\nTop matchup duplicate counts (games_df):")
display(dup_ct)

In [ ]:
import pandas as pd
import numpy as np

prob_cols = ["p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]

def pick_time_col(df):
    for c in ["commence_utc","commence_et","commence_time","start_time","game_time"]:
        if c in df.columns:
            return c
    return None

tcol = pick_time_col(games_df)

# 1) build a "market completeness" score so we keep the best row per matchup
market_cols = [c for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"] if c in games_df.columns]
games_df["_market_score"] = games_df[market_cols].notna().sum(axis=1)

# normalize time (optional)
if tcol:
    games_df[tcol] = pd.to_datetime(games_df[tcol], errors="coerce", utc=True)

# 2) keep ONE row per matchup
# sort priority: most complete market row, then earliest time, then stable index
sort_cols = ["_market_score"]
asc = [False]
if tcol:
    sort_cols.append(tcol)
    asc.append(True)

games_df = (
    games_df
    .sort_values(sort_cols, ascending=asc)
    .drop_duplicates(subset=["away_team","home_team"], keep="first")
    .reset_index(drop=True)
)

games_df = games_df.drop(columns=["_market_score"], errors="ignore")

print("✅ games_df collapsed to 1 row per matchup:", len(games_df), "rows")

# 3) Now merge probs from g by JUST matchup (teams), no time bucket
need_probs = ["away_team","home_team"] + prob_cols
missing_in_g = [c for c in prob_cols if c not in g.columns]
if missing_in_g:
    raise RuntimeError(f"g missing prob cols: {missing_in_g} (run FULL STRICT probs cell)")

g_probs = g[need_probs].drop_duplicates(subset=["away_team","home_team"])

# drop any stale prob cols and re-merge
games_df = games_df.drop(columns=[c for c in prob_cols if c in games_df.columns], errors="ignore").merge(
    g_probs, on=["away_team","home_team"], how="left"
)

missing_after = games_df[prob_cols].isna().any(axis=1).sum()
print(f"✅ Prob merge complete. Missing rows after merge: {missing_after} / {len(games_df)}")

if missing_after:
    display(games_df.loc[games_df[prob_cols].isna().any(axis=1), ["away_team","home_team"]].head(20))
    raise RuntimeError("Still missing probs after matchup merge — g likely doesn't contain these matchups (slate mismatch).")

In [ ]:
# ==========================================
# FORCE MODEL INPUT = games_df
# ==========================================

print("Rebuilding g directly from games_df...")

g = games_df.copy()

print("g rebuilt. Rows:", len(g))
print("Unique matchups:", g[["away_team","home_team"]].drop_duplicates().shape[0])

In [ ]:
print("Final g rows:", len(g))
print("Unique matchups:", g[["away_team","home_team"]].drop_duplicates().shape[0])

In [ ]:
print("Prob column check:")
print(g[["p_home_win","p_home_cover","p_over"]].describe())

In [ ]:
# ============================================================
# FULL STRICT NCAA RERUN (DROP-IN BOTTOM CELL)
# - Rebuilds probs for ALL rows in games_df (no key merges)
# - Favorite/Dog aware spread logic
# - Market gravity (anchor) + Monte Carlo (50k)
# - Builds ML / SPREAD / TOTAL plays, +EV only, cap 2 per game
# - Half-Kelly units (0.25u–1.00u)
# - Exports CSV + prints Discord card
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime

# -------------------------
# SETTINGS (edit if needed)
# -------------------------
SIMS = 50000
SD_MARGIN = 16.5     # tune per your NCAA runs
SD_TOTAL  = 19.0
MARKET_GRAVITY = 0.80    # 0.0=all model, 1.0=all market
EV_MIN = 0.00            # +EV only
MAX_PLAYS_PER_GAME = 2
MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True
HITRATE_MIN = None       # e.g. 0.58 for "high hit rate"; None = off

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_FULL_STRICT_CARD_BOTTOM.csv"

# -------------------------
# REQUIRED INPUT
# -------------------------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run your market pull cell first (must create games_df).")

df = games_df.copy()

need_cols = ["away_team","home_team","home_ml","away_ml","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}. Present cols={list(df.columns)}")

# optional spread odds col
if "spread_home_odds" not in df.columns:
    df["spread_home_odds"] = -110

# -------------------------
# HELPERS
# -------------------------
def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def american_to_decimal(odds):
    o = float(odds)
    if o > 0:
        return 1.0 + (o / 100.0)
    else:
        return 1.0 + (100.0 / abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    if o > 0:
        return 100.0 / (o + 100.0)
    else:
        return abs(o) / (abs(o) + 100.0)

def ev_roi_1u(p, odds):
    # Expected ROI for 1u stake (profit / stake), same scale as your EV%
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p * b) - (1.0 - p)

def half_kelly_units(p, odds, min_u=0.25, max_u=1.0):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    # full kelly fraction of bankroll
    f = (p*b - q) / b if b > 0 else 0.0
    if HALF_KELLY:
        f *= 0.5
    f = max(0.0, f)
    # map fraction -> units with cap/floor for postable positives
    u = min(max_u, f * max_u)
    if u > 0 and u < min_u:
        u = min_u
    return float(u)

def mk_matchup(a,h):
    return f"{a} at {h}"

# -------------------------
# CLEAN / NORMALIZE NUMERIC FIELDS
# -------------------------
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]:
    df[c] = _to_num(df[c])

# drop rows missing core market numbers
df = df.dropna(subset=["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]).reset_index(drop=True)

if len(df) == 0:
    raise RuntimeError("No usable rows after cleaning. Check your market pull output into games_df.")

# -------------------------
# MARKET MEANS (from spread/total) + GRAVITY ANCHOR
# -------------------------
# market margin convention:
# spread_home_line is the HOME spread (e.g. -3.5 means home favored by 3.5)
# expected home margin ≈ -spread_home_line
df["market_mean_margin"] = -df["spread_home_line"]
df["market_mean_total"]  = df["total_line"]

# base model means: if you have your own model means already, keep them.
# otherwise start at market and allow small drift (0 by default).
if "mean_margin" in df.columns:
    df["mean_margin_model"] = _to_num(df["mean_margin"])
else:
    df["mean_margin_model"] = df["market_mean_margin"]

if "mean_total" in df.columns:
    df["mean_total_model"] = _to_num(df["mean_total"])
else:
    df["mean_total_model"] = df["market_mean_total"]

# gravity: pull model means back toward market to prevent wild edges
df["mean_margin_anch"] = (1-MARKET_GRAVITY)*df["mean_margin_model"] + MARKET_GRAVITY*df["market_mean_margin"]
df["mean_total_anch"]  = (1-MARKET_GRAVITY)*df["mean_total_model"]  + MARKET_GRAVITY*df["market_mean_total"]

# -------------------------
# MONTE CARLO SIMS
# -------------------------
rng = np.random.default_rng(7)

# simulate home margin and total
home_margin_sim = rng.normal(loc=df["mean_margin_anch"].values, scale=SD_MARGIN, size=(SIMS, len(df)))
total_sim       = rng.normal(loc=df["mean_total_anch"].values,  scale=SD_TOTAL,  size=(SIMS, len(df)))

# Win probs (ML)
df["p_home_win"] = (home_margin_sim > 0).mean(axis=0)
df["p_away_win"] = 1.0 - df["p_home_win"]

# Spread cover probs (favorite/dog aware)
# Home covers if (home_margin + spread_home_line) > 0
# Example: home -3.5 => add (-3.5) ; need margin > 3.5 to cover. Works.
df["p_home_cover"] = ((home_margin_sim + df["spread_home_line"].values) > 0).mean(axis=0)
df["p_away_cover"] = 1.0 - df["p_home_cover"]

# Total probs
df["p_over"]  = (total_sim > df["total_line"].values).mean(axis=0)
df["p_under"] = 1.0 - df["p_over"]

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Games in df:", len(df))

# quick sanity
print("\nProb column check:")
display(df[["p_home_win","p_home_cover","p_over"]].describe())

# -------------------------
# BUILD BET CANDIDATES (ML / SPREAD / TOTAL)
# -------------------------
rows = []
for i, r in df.iterrows():
    matchup = mk_matchup(r["away_team"], r["home_team"])

    # ML (use home_ml/away_ml, pick each side)
    rows.append({
        "matchup": matchup, "market": "ML",
        "bet": f'{r["away_team"]} ML ({int(r["away_ml"])})',
        "odds": float(r["away_ml"]),
        "p_win": float(r["p_away_win"]),
    })
    rows.append({
        "matchup": matchup, "market": "ML",
        "bet": f'{r["home_team"]} ML ({int(r["home_ml"])})',
        "odds": float(r["home_ml"]),
        "p_win": float(r["p_home_win"]),
    })

    # SPREAD (assume same price both sides unless you have away spread odds)
    spread_odds = float(r["spread_home_odds"]) if not np.isnan(r["spread_home_odds"]) else -110.0
    sh = float(r["spread_home_line"])

    # home spread string: show sign correctly
    home_spread_txt = f"{sh:.1f}".rstrip("0").rstrip(".")
    away_spread_val = -sh
    away_spread_txt = f"{away_spread_val:.1f}".rstrip("0").rstrip(".")

    rows.append({
        "matchup": matchup, "market": "SPREAD",
        "bet": f'{r["home_team"]} {home_spread_txt} ({int(spread_odds)})',
        "odds": float(spread_odds),
        "p_win": float(r["p_home_cover"]),
    })
    rows.append({
        "matchup": matchup, "market": "SPREAD",
        "bet": f'{r["away_team"]} {away_spread_txt} ({int(spread_odds)})',
        "odds": float(spread_odds),
        "p_win": float(r["p_away_cover"]),
    })

    # TOTALS (assume -110 if not provided)
    total_odds = -110.0
    tl = float(r["total_line"])
    tl_txt = f"{tl:.1f}".rstrip("0").rstrip(".")
    rows.append({
        "matchup": matchup, "market": "TOTAL",
        "bet": f'OVER {tl_txt} ({int(total_odds)})',
        "odds": float(total_odds),
        "p_win": float(r["p_over"]),
    })
    rows.append({
        "matchup": matchup, "market": "TOTAL",
        "bet": f'UNDER {tl_txt} ({int(total_odds)})',
        "odds": float(total_odds),
        "p_win": float(r["p_under"]),
    })

card = pd.DataFrame(rows)

# market implied prob + EV (ROI)
card["p_mkt"] = card["odds"].apply(american_to_implied_prob)
card["ev"] = card.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)

# hit-rate filter (optional)
if HITRATE_MIN is not None:
    card = card[card["p_win"] >= HITRATE_MIN].copy()

# +EV only
card = card[card["ev"] >= EV_MIN].copy()

# units (half-kelly, 0.25u floor, 1u cap)
card["units"] = card.apply(lambda x: half_kelly_units(float(x["p_win"]), float(x["odds"]), MIN_U, MAX_U), axis=1)
card = card[card["units"] > 0].copy()

# rank (blend: EV then p_win)
card["rank_score"] = (card["ev"] * 1.00) + (card["p_win"] * 0.25)
card = card.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# cap 2 per game
card["_game_count"] = card.groupby("matchup").cumcount() + 1
card = card[card["_game_count"] <= MAX_PLAYS_PER_GAME].copy()
card = card.drop(columns=["_game_count"])

# discord text
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

card["discord_text"] = card.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

# export
card_out = card[["matchup","market","bet","p_win","p_mkt","ev","odds","units","discord_text"]].copy()
card_out.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final plays: {len(card_out)}")
print(f"✅ Exported: {EXPORT_NAME}\n")

print("=== DISCORD CARD ===\n")
print("\n".join(card_out["discord_text"].tolist()))

In [ ]:
# ==========================================
# FIX CELL (OPTION 2): BLEND EV + MOST PROBABLE
# - Re-scores + re-sorts using EV + Win% blend
# - Keeps your existing filters/caps/units
# - Rebuilds discord_text + exports a new CSV
# ==========================================

import pandas as pd
import numpy as np

# --- REQUIRE: card or card_out exists from your rerun cell ---
if "card" in globals():
    _card = card.copy()
elif "card_out" in globals():
    _card = card_out.copy()
else:
    raise RuntimeError("No card/card_out found. Run the FULL STRICT rerun cell first (the one that builds `card`).")

# --- ensure required cols exist ---
need_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","units"]
missing = [c for c in need_cols if c not in _card.columns]
if missing:
    raise RuntimeError(f"Missing columns in card: {missing}. Re-run your FULL STRICT rerun cell to rebuild `card`.")

# --- Option 2 blend: weight win% heavier (probable) while still valuing EV ---
EV_W = 0.65
PWIN_W = 0.60

_card["rank_score"] = (_card["ev"] * EV_W) + (_card["p_win"] * PWIN_W)

# --- resort by blended score (tie-breakers: p_win then ev) ---
_card = _card.sort_values(["rank_score","p_win","ev"], ascending=False).reset_index(drop=True)

# --- re-apply cap per game (if you used it) ---
MAX_PLAYS_PER_GAME = globals().get("MAX_PLAYS_PER_GAME", 2)
_card["_game_count"] = _card.groupby("matchup").cumcount() + 1
_card = _card[_card["_game_count"] <= MAX_PLAYS_PER_GAME].drop(columns=["_game_count"])

# --- rebuild discord text ---
def _fmt_prob(p): return f"{100*float(p):.1f}%"
def _fmt_ev(e):   return f"{100*float(e):.1f}%"

_card["discord_text"] = _card.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {float(x["units"]):.2f}u\n'
        f'Sim Win%: {_fmt_prob(x["p_win"])} | Market: {_fmt_prob(x["p_mkt"])}\n'
        f'EV: {_fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

# --- export ---
export_name = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_BLENDED_EV_PROB.csv"
_card[need_cols + ["discord_text"]].to_csv(export_name, index=False)

print(f"✅ Re-ranked with Option 2 blend (EV_W={EV_W}, PWIN_W={PWIN_W})")
print(f"✅ Rows: {len(_card)} | Cap per game: {MAX_PLAYS_PER_GAME}")
print(f"✅ Exported: {export_name}\n")

print("=== DISCORD CARD (BLENDED: +EV + PROBABLE) ===\n")
print("\n".join(_card["discord_text"].tolist()))

In [ ]:
# ============================================================
# OPTION 2 FIX (BOTTOM CELL): MARKET-ML -> MARGIN MEAN
# Fixes p_cover ~50% issue by deriving expected margin from ML (no-vig)
# Then rebuilds ML / SPREAD / TOTAL candidates + "probable" filters
# ============================================================

import numpy as np
import pandas as pd

# ----- SETTINGS -----
SIMS = 50000
SD_MARGIN = 11.5         # closer to your "good" NCAA run range
SD_TOTAL  = 15.0
MARKET_GRAVITY = 0.35    # smaller = allow model/ML-implied to drive margin
EV_MIN = 0.00
MAX_PLAYS_PER_GAME = 2
MIN_U, MAX_U = 0.25, 1.00
HALF_KELLY = True

# "PROBABLE" / DOG CONTROL (tune these)
MIN_PWIN = 0.56          # hit-rate style filter (removes most longshots)
MAX_POS_ODDS = 300       # remove huge dogs (e.g., +410, +1600, +2200)
ALLOW_DOGS_IF_PWIN_GE = 0.40  # if you want *some* dogs, keep only if p_win>=this (still blocks most)

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_BLENDED_FIX_PROBABLE.csv"

# ----- REQUIRE INPUT -----
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run your market pull cell first.")

df = games_df.copy()

need_cols = ["away_team","home_team","home_ml","away_ml","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}. Present cols={list(df.columns)}")

if "spread_home_odds" not in df.columns:
    df["spread_home_odds"] = -110

# ----- HELPERS -----
def _to_num(s): return pd.to_numeric(s, errors="coerce")

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return (100.0/(o+100.0)) if o > 0 else (abs(o)/(abs(o)+100.0))

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0-p)

def half_kelly_units(p, odds, min_u=0.25, max_u=1.0):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if HALF_KELLY: f *= 0.5
    f = max(0.0, f)
    u = min(max_u, f * max_u)
    if 0 < u < min_u: u = min_u
    return float(u)

def mk_matchup(a,h): return f"{a} at {h}"

# norm.ppf without scipy (ppf(p)=sqrt(2)*erfinv(2p-1))
def ppf(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.sqrt(2.0) * np.erfinv(2.0*p - 1.0)

# ----- CLEAN NUMERICS -----
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]:
    df[c] = _to_num(df[c])

df = df.dropna(subset=["home_ml","away_ml","spread_home_line","spread_home_odds","total_line"]).reset_index(drop=True)
if len(df) == 0:
    raise RuntimeError("No usable rows after cleaning.")

# ----- NO-VIG ML PROBS -> IMPLIED MARGIN MEAN -----
# Convert moneylines to implied probs, then de-vig normalize
p_home_raw = df["home_ml"].apply(american_to_implied_prob).astype(float)
p_away_raw = df["away_ml"].apply(american_to_implied_prob).astype(float)
p_sum = (p_home_raw + p_away_raw).replace(0, np.nan)
df["p_home_mkt_novig"] = (p_home_raw / p_sum).fillna(0.5).clip(1e-6, 1-1e-6)

# Expected margin consistent with ML (normal approx)
df["mean_margin_ml"] = SD_MARGIN * ppf(df["p_home_mkt_novig"].values)

# Total mean (no extra info): anchor to market total, allow optional drift if present
df["mean_total_model"] = _to_num(df["mean_total"]) if "mean_total" in df.columns else df["total_line"]
df["mean_total_anch"]  = (1-MARKET_GRAVITY)*df["mean_total_model"] + MARKET_GRAVITY*df["total_line"]

# Blend margin between ML-implied and market spread-implied
df["market_mean_margin"] = -df["spread_home_line"]  # home margin implied by spread
df["mean_margin_anch"] = (1-MARKET_GRAVITY)*df["mean_margin_ml"] + MARKET_GRAVITY*df["market_mean_margin"]

# ----- MONTE CARLO -----
rng = np.random.default_rng(7)
home_margin_sim = rng.normal(loc=df["mean_margin_anch"].values, scale=SD_MARGIN, size=(SIMS, len(df)))
total_sim       = rng.normal(loc=df["mean_total_anch"].values,  scale=SD_TOTAL,  size=(SIMS, len(df)))

df["p_home_win"]   = (home_margin_sim > 0).mean(axis=0)
df["p_away_win"]   = 1.0 - df["p_home_win"]
df["p_home_cover"] = ((home_margin_sim + df["spread_home_line"].values) > 0).mean(axis=0)
df["p_away_cover"] = 1.0 - df["p_home_cover"]
df["p_over"]       = (total_sim > df["total_line"].values).mean(axis=0)
df["p_under"]      = 1.0 - df["p_over"]

print(f"✅ Sims: {SIMS} | SD_MARGIN={SD_MARGIN} | SD_TOTAL={SD_TOTAL} | Rows={len(df)}")
print("Sanity (p_home_cover std should NOT be ~0.002 anymore):", float(df["p_home_cover"].std()))

# ----- BUILD CANDIDATES -----
rows = []
for _, r in df.iterrows():
    matchup = mk_matchup(r["away_team"], r["home_team"])

    # ML
    rows += [
        dict(matchup=matchup, market="ML",
             bet=f'{r["away_team"]} ML ({int(r["away_ml"])})', odds=float(r["away_ml"]), p_win=float(r["p_away_win"])),
        dict(matchup=matchup, market="ML",
             bet=f'{r["home_team"]} ML ({int(r["home_ml"])})', odds=float(r["home_ml"]), p_win=float(r["p_home_win"])),
    ]

    # SPREAD
    sh = float(r["spread_home_line"])
    so = float(r["spread_home_odds"])
    home_sp = f"{sh:.1f}".rstrip("0").rstrip(".")
    away_sp = f"{(-sh):.1f}".rstrip("0").rstrip(".")
    rows += [
        dict(matchup=matchup, market="SPREAD",
             bet=f'{r["home_team"]} {home_sp} ({int(so)})', odds=so, p_win=float(r["p_home_cover"])),
        dict(matchup=matchup, market="SPREAD",
             bet=f'{r["away_team"]} {away_sp} ({int(so)})', odds=so, p_win=float(r["p_away_cover"])),
    ]

    # TOTAL (assume -110 both sides unless you have total odds)
    tl = float(r["total_line"])
    tl_txt = f"{tl:.1f}".rstrip("0").rstrip(".")
    rows += [
        dict(matchup=matchup, market="TOTAL", bet=f'OVER {tl_txt} (-110)', odds=-110.0, p_win=float(r["p_over"])),
        dict(matchup=matchup, market="TOTAL", bet=f'UNDER {tl_txt} (-110)', odds=-110.0, p_win=float(r["p_under"])),
    ]

card = pd.DataFrame(rows)
card["p_mkt"] = card["odds"].apply(american_to_implied_prob)
card["ev"] = card.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)

# ----- FILTERS: +EV + PROBABLE + REMOVE MASSIVE DOGS -----
card = card[card["ev"] >= EV_MIN].copy()

# remove huge +odds dogs
card = card[~((card["odds"] > 0) & (card["odds"] > MAX_POS_ODDS))].copy()

# if still a dog, require a minimum p_win to keep it
card = card[~((card["odds"] > 0) & (card["p_win"] < ALLOW_DOGS_IF_PWIN_GE))].copy()

# hit-rate style filter
card = card[card["p_win"] >= MIN_PWIN].copy()

# units
card["units"] = card.apply(lambda x: half_kelly_units(float(x["p_win"]), float(x["odds"]), MIN_U, MAX_U), axis=1)
card = card[card["units"] > 0].copy()

# rank blend (probable first, EV second)
card["rank_score"] = (card["p_win"] * 0.75) + (card["ev"] * 0.50)
card = card.sort_values(["rank_score","p_win","ev"], ascending=False).reset_index(drop=True)

# cap 2/game
card["_gc"] = card.groupby("matchup").cumcount() + 1
card = card[card["_gc"] <= MAX_PLAYS_PER_GAME].drop(columns=["_gc"])

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e):   return f"{100*float(e):.1f}%"

card["discord_text"] = card.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {float(x["units"]):.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

out = card[["matchup","market","bet","p_win","p_mkt","ev","odds","units","discord_text"]].copy()
out.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final plays: {len(out)}")
print(f"✅ Exported: {EXPORT_NAME}\n")
print("=== DISCORD CARD (PROBABLE BLEND) ===\n")
print("\n".join(out["discord_text"].tolist()))

In [ ]:
# --- stable normal inverse CDF (no scipy / no erfinv required) ---
def ppf(p):
    # Peter John Acklam's approximation
    p = np.clip(p, 1e-6, 1-1e-6)

    a = [-3.969683028665376e+01,
          2.209460984245205e+02,
         -2.759285104469687e+02,
          1.383577518672690e+02,
         -3.066479806614716e+01,
          2.506628277459239e+00]

    b = [-5.447609879822406e+01,
          1.615858368580409e+02,
         -1.556989798598866e+02,
          6.680131188771972e+01,
         -1.328068155288572e+01]

    c = [-7.784894002430293e-03,
         -3.223964580411365e-01,
         -2.400758277161838e+00,
         -2.549732539343734e+00,
          4.374664141464968e+00,
          2.938163982698783e+00]

    d = [7.784695709041462e-03,
         3.224671290700398e-01,
         2.445134137142996e+00,
         3.754408661907416e+00]

    plow  = 0.02425
    phigh = 1 - plow

    if isinstance(p, np.ndarray):
        out = np.zeros_like(p)
        mask_low = p < plow
        mask_high = p > phigh
        mask_mid = (~mask_low) & (~mask_high)

        # lower
        q = np.sqrt(-2*np.log(p[mask_low]))
        out[mask_low] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                         ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

        # middle
        q = p[mask_mid] - 0.5
        r = q*q
        out[mask_mid] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
                         (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

        # upper
        q = np.sqrt(-2*np.log(1 - p[mask_high]))
        out[mask_high] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                           ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

        return out
    else:
        # scalar case
        if p < plow:
            q = np.sqrt(-2*np.log(p))
            return (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                   ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
        elif p > phigh:
            q = np.sqrt(-2*np.log(1 - p))
            return -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                    ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
        else:
            q = p - 0.5
            r = q*q
            return (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5])*q / \
                   (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

In [ ]:
# ============================================================
# CDR NCAAB — FULL STRICT ENGINE (v1.0)  ✅ "ONE BUILD"
# Goal: identical, clean, and COMPLETE layer stack:
# 1) Market pull (games_df must exist)
# 2) Team projections (mean_margin, mean_total) OR build from layered signals
# 3) Efficient market signals (no-vig ML + spread/total anchors)
# 4) Gravity blend (model ↔ market) with tunable weights
# 5) Correlated Monte Carlo (margin & total)
# 6) Probabilities (ML / Spread / Totals)
# 7) EV + Half-Kelly sizing + caps + per-game play limits
# 8) Export CSV + Discord-ready text
#
# IMPORTANT:
# - This cell HARD-FAILS if totals/margin projections cannot be built.
# - To be truly "our model", supply (or build) mean_margin + mean_total.
# - This cell is designed to be pasted at the BOTTOM and run once.
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime

# -------------------------
# SETTINGS
# -------------------------
SIMS = 50_000
RNG_SEED = 7

# These should reflect your NCAA calibration (you can tune after results)
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
RHO_MARGIN_TOTAL = 0.15  # weak positive correlation typical; tune if needed

# "Efficient market signals" + gravity blending
# - If your model has strong projections, reduce gravity.
# - If projections are noisy, increase gravity.
W_MODEL_MARGIN = 0.55    # weight on projection margin
W_ML_MARGIN    = 0.30    # weight on ML-implied margin (no-vig)
W_SPR_MARGIN   = 0.15    # weight on spread-implied margin

W_MODEL_TOTAL  = 0.70    # weight on projection total
W_MKT_TOTAL    = 0.30    # weight on market total

# Additional "market gravity" pull (0=none, 1=full market)
MARKET_GRAVITY_MARGIN = 0.15
MARKET_GRAVITY_TOTAL  = 0.10

# Play construction / filters
EV_MIN = 0.00                 # +EV only
MAX_PLAYS_PER_GAME = 2
MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True

# Optional "brand" hit-rate guardrail (set None to disable)
HITRATE_MIN = None            # e.g. 0.58

# Export
EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_FULL_STRICT_ENGINE_v1.csv"

# -------------------------
# REQUIRED INPUT: games_df
# -------------------------
if "games_df" not in globals():
    raise RuntimeError("STOP — games_df not found. Run your market pull cell first.")

df = games_df.copy()

# -------------------------
# REQUIRED MARKET COLUMNS
# -------------------------
req = ["away_team","home_team","home_ml","away_ml","spread_home_line","total_line"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise RuntimeError(f"STOP — games_df missing required columns: {missing}\nPresent: {list(df.columns)}")

# Optional price columns (we'll default safely if missing)
# spread_home_odds / spread_away_odds / over_odds / under_odds
if "spread_home_odds" not in df.columns: df["spread_home_odds"] = -110
if "spread_away_odds" not in df.columns: df["spread_away_odds"] = df["spread_home_odds"]
if "over_odds" not in df.columns: df["over_odds"] = -110
if "under_odds" not in df.columns: df["under_odds"] = -110

# -------------------------
# HELPERS
# -------------------------
def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

def american_to_decimal(odds):
    o = float(odds)
    return (1.0 + o/100.0) if o > 0 else (1.0 + 100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return (100.0/(o+100.0)) if o > 0 else (abs(o)/(abs(o)+100.0))

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p * b) - (1.0 - p)

def half_kelly_units(p, odds, min_u=0.25, max_u=1.0):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q) / b if b > 0 else 0.0
    if HALF_KELLY:
        f *= 0.5
    f = max(0.0, f)
    u = min(max_u, f * max_u)
    if u > 0 and u < min_u:
        u = min_u
    return float(u)

def mk_matchup(a,h):
    return f"{a} at {h}"

def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

# Pure-numpy inverse normal CDF (Acklam approximation) to avoid scipy dependency
def norm_ppf(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-10, 1 - 1e-10)

    # Coefficients (Peter J. Acklam)
    a = np.array([-3.969683028665376e+01,  2.209460984245205e+02,
                  -2.759285104469687e+02,  1.383577518672690e+02,
                  -3.066479806614716e+01,  2.506628277459239e+00])
    b = np.array([-5.447609879822406e+01,  1.615858368580409e+02,
                  -1.556989798598866e+02,  6.680131188771972e+01,
                  -1.328068155288572e+01])
    c = np.array([-7.784894002430293e-03, -3.223964580411365e-01,
                  -2.400758277161838e+00, -2.549732539343734e+00,
                   4.374664141464968e+00,  2.938163982698783e+00])
    d = np.array([ 7.784695709041462e-03,  3.224671290700398e-01,
                   2.445134137142996e+00,  3.754408661907416e+00])

    plow = 0.02425
    phigh = 1 - plow

    x = np.empty_like(p)

    # Lower region
    mask = p < plow
    if mask.any():
        q = np.sqrt(-2*np.log(p[mask]))
        x[mask] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                  ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

    # Central region
    mask = (p >= plow) & (p <= phigh)
    if mask.any():
        q = p[mask] - 0.5
        r = q*q
        x[mask] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
                  (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

    # Upper region
    mask = p > phigh
    if mask.any():
        q = np.sqrt(-2*np.log(1 - p[mask]))
        x[mask] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                    ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

# -------------------------
# CLEAN / NUMERIC NORMALIZATION
# -------------------------
num_cols = ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds","total_line","over_odds","under_odds"]
for c in num_cols:
    df[c] = _to_num(df[c])

df = df.dropna(subset=["away_team","home_team","home_ml","away_ml","spread_home_line","total_line"]).reset_index(drop=True)
if len(df) == 0:
    raise RuntimeError("STOP — No usable rows after cleaning market inputs.")

# -------------------------
# LAYER 1: EFFICIENT MARKET SIGNALS (NO-VIG ML)
# -------------------------
df["p_home_mkt"] = df["home_ml"].apply(american_to_implied_prob)
df["p_away_mkt"] = df["away_ml"].apply(american_to_implied_prob)

# no-vig normalize (efficient market signal)
s = df["p_home_mkt"] + df["p_away_mkt"]
df["p_home_mkt_novig"] = df["p_home_mkt"] / s
df["p_away_mkt_novig"] = df["p_away_mkt"] / s

# Convert ML no-vig -> margin mean using Normal link
# mean_margin_ml is "home expected margin"
df["mean_margin_ml"] = SD_MARGIN * norm_ppf(df["p_home_mkt_novig"].values)

# Spread-implied margin (home margin approx)
df["mean_margin_spread"] = -df["spread_home_line"]

# Total-implied mean
df["mean_total_mkt"] = df["total_line"]

# -------------------------
# LAYER 2: OUR PROJECTIONS (STRICT)
# -------------------------
# We want both mean_margin and mean_total from our projection layers.
# If they are missing, we HARD-FAIL (because totals + edges become nonsense / ~50-50).
have_margin_proj = "mean_margin" in df.columns and df["mean_margin"].notna().any()
have_total_proj  = "mean_total"  in df.columns and df["mean_total"].notna().any()

if not have_margin_proj or not have_total_proj:
    raise RuntimeError(
        "STOP — Missing our projection layers.\n"
        f"mean_margin present/usable: {have_margin_proj}\n"
        f"mean_total present/usable:  {have_total_proj}\n\n"
        "To run FULL STRICT correctly, ensure your projection layer writes:\n"
        "- games_df['mean_margin'] (expected HOME margin)\n"
        "- games_df['mean_total']  (expected game total)\n\n"
        "Once those exist, rerun this cell and it will produce ML/SPREAD/TOTAL correctly."
    )

df["mean_margin_model"] = _to_num(df["mean_margin"])
df["mean_total_model"]  = _to_num(df["mean_total"])

# Fill any remaining NaNs strictly
if df["mean_margin_model"].isna().any():
    raise RuntimeError("STOP — mean_margin has NaNs. Fix your projection build so every matchup has a margin projection.")
if df["mean_total_model"].isna().any():
    raise RuntimeError("STOP — mean_total has NaNs. Fix your projection build so every matchup has a total projection.")

# -------------------------
# LAYER 3: BLEND MODEL + MARKET SIGNALS + GRAVITY
# -------------------------
# Margin blend: model + ML + spread
mean_margin_blend = (
    W_MODEL_MARGIN*df["mean_margin_model"].values +
    W_ML_MARGIN*df["mean_margin_ml"].values +
    W_SPR_MARGIN*df["mean_margin_spread"].values
)

# Apply a final gravity pull toward spread (keeps extremes in check)
df["mean_margin_final"] = (1.0 - MARKET_GRAVITY_MARGIN)*mean_margin_blend + MARKET_GRAVITY_MARGIN*df["mean_margin_spread"].values

# Total blend: model + market total
mean_total_blend = (
    W_MODEL_TOTAL*df["mean_total_model"].values +
    W_MKT_TOTAL*df["mean_total_mkt"].values
)
df["mean_total_final"] = (1.0 - MARKET_GRAVITY_TOTAL)*mean_total_blend + MARKET_GRAVITY_TOTAL*df["mean_total_mkt"].values

# -------------------------
# LAYER 4: CORRELATED MONTE CARLO SIM
# -------------------------
rng = np.random.default_rng(RNG_SEED)
n = len(df)

z1 = rng.standard_normal((SIMS, n))
z2 = rng.standard_normal((SIMS, n))

home_margin_sim = df["mean_margin_final"].values + SD_MARGIN * z1
total_sim = df["mean_total_final"].values + SD_TOTAL * (RHO_MARGIN_TOTAL*z1 + np.sqrt(1 - RHO_MARGIN_TOTAL**2)*z2)

# -------------------------
# LAYER 5: PROBABILITIES
# -------------------------
df["p_home_win"] = (home_margin_sim > 0).mean(axis=0)
df["p_away_win"] = 1.0 - df["p_home_win"]

# Spread cover (favorite/dog aware): home covers if margin + spread_home_line > 0
df["p_home_cover"] = ((home_margin_sim + df["spread_home_line"].values) > 0).mean(axis=0)
df["p_away_cover"] = 1.0 - df["p_home_cover"]

df["p_over"]  = (total_sim > df["total_line"].values).mean(axis=0)
df["p_under"] = 1.0 - df["p_over"]

# -------------------------
# SANITY CHECKS (STRICT)
# -------------------------
# 1) Totals should NOT be ~50/50 across the board if mean_total_model has signal.
p_over_std = float(df["p_over"].std())
p_cover_std = float(df["p_home_cover"].std())

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL} | rho={RHO_MARGIN_TOTAL}")
print(f"✅ Games in df: {len(df)}")
print("\nSanity (prob std):")
print(" - p_over std:", round(p_over_std, 4))
print(" - p_home_cover std:", round(p_cover_std, 4))

if p_over_std < 0.01:
    raise RuntimeError(
        "STOP — Totals probabilities are collapsing near 50/50 (std < 0.01).\n"
        "This usually means mean_total_model ≈ market total (no signal) or mean_total_final ≈ total_line.\n"
        "Fix your totals projection layer so games_df['mean_total'] has real variation vs the market."
    )

# -------------------------
# LAYER 6: BUILD CANDIDATES (ML / SPREAD / TOTAL)
# -------------------------
rows = []
for _, r in df.iterrows():
    matchup = mk_matchup(r["away_team"], r["home_team"])

    # ML
    rows += [
        dict(matchup=matchup, market="ML",
             bet=f'{r["away_team"]} ML ({int(r["away_ml"])})',
             odds=float(r["away_ml"]), p_win=float(r["p_away_win"])),
        dict(matchup=matchup, market="ML",
             bet=f'{r["home_team"]} ML ({int(r["home_ml"])})',
             odds=float(r["home_ml"]), p_win=float(r["p_home_win"]))
    ]

    # SPREAD (use side-specific odds if present)
    sh = float(r["spread_home_line"])
    home_spread_txt = f"{sh:.1f}".rstrip("0").rstrip(".")
    away_spread_txt = f"{(-sh):.1f}".rstrip("0").rstrip(".")

    rows += [
        dict(matchup=matchup, market="SPREAD",
             bet=f'{r["home_team"]} {home_spread_txt} ({int(r["spread_home_odds"])})',
             odds=float(r["spread_home_odds"]), p_win=float(r["p_home_cover"])),
        dict(matchup=matchup, market="SPREAD",
             bet=f'{r["away_team"]} {away_spread_txt} ({int(r["spread_away_odds"])})',
             odds=float(r["spread_away_odds"]), p_win=float(r["p_away_cover"]))
    ]

    # TOTALS (use side-specific odds)
    tl = float(r["total_line"])
    tl_txt = f"{tl:.1f}".rstrip("0").rstrip(".")
    rows += [
        dict(matchup=matchup, market="TOTAL",
             bet=f'OVER {tl_txt} ({int(r["over_odds"])})',
             odds=float(r["over_odds"]), p_win=float(r["p_over"])),
        dict(matchup=matchup, market="TOTAL",
             bet=f'UNDER {tl_txt} ({int(r["under_odds"])})',
             odds=float(r["under_odds"]), p_win=float(r["p_under"]))
    ]

card = pd.DataFrame(rows)

# EV + market implied
card["p_mkt"] = card["odds"].apply(american_to_implied_prob)
card["ev"] = card.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)

# Optional hit-rate
if HITRATE_MIN is not None:
    card = card[card["p_win"] >= HITRATE_MIN].copy()

# +EV
card = card[card["ev"] >= EV_MIN].copy()

# Units
card["units"] = card.apply(lambda x: half_kelly_units(float(x["p_win"]), float(x["odds"]), MIN_U, MAX_U), axis=1)
card = card[card["units"] > 0].copy()

# Rank blend: EV + probability (balanced for “+EV + higher hit-rate brand”)
# (You can adjust weights later without breaking the engine.)
EV_W = 0.65
PWIN_W = 0.60
card["rank_score"] = EV_W*card["ev"] + PWIN_W*card["p_win"]
card = card.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# Cap per game
card["_n"] = card.groupby("matchup").cumcount() + 1
card = card[card["_n"] <= MAX_PLAYS_PER_GAME].drop(columns="_n")

# Discord text
card["discord_text"] = card.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

out = card[["matchup","market","bet","p_win","p_mkt","ev","odds","units","discord_text"]].copy()
out.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final plays: {len(out)} | Cap per game: {MAX_PLAYS_PER_GAME}")
print(f"✅ Exported: {EXPORT_NAME}\n")
print("=== DISCORD CARD ===\n")
print("\n".join(out["discord_text"].tolist()))

In [ ]:
# ============================================================
# CDR NCAAB — PROJECTION LAYER BUILDER (BOTTOM FIX)
# Writes:
#   games_df["mean_margin"]  (expected HOME margin)
#   games_df["mean_total"]   (expected game total)
#
# Uses:
#   - No-vig ML signal -> margin (Normal link)
#   - Spread signal -> margin
#   - Market total -> total
# Optional upgrade:
#   - If a historical scores df exists, builds team ratings to adjust margin/total
# ============================================================

import numpy as np
import pandas as pd

# --- required
if "games_df" not in globals():
    raise RuntimeError("STOP — games_df not found. Run market pull first.")

df = games_df.copy()

req = ["away_team","home_team","home_ml","away_ml","spread_home_line","total_line"]
missing = [c for c in req if c not in df.columns]
if missing:
    raise RuntimeError(f"STOP — games_df missing required columns for projection build: {missing}")

def _to_num(x): return pd.to_numeric(x, errors="coerce")

for c in ["home_ml","away_ml","spread_home_line","total_line"]:
    df[c] = _to_num(df[c])

df = df.dropna(subset=["home_ml","away_ml","spread_home_line","total_line"]).reset_index(drop=True)
if len(df) == 0:
    raise RuntimeError("STOP — no usable rows after cleaning.")

def american_to_implied_prob(odds):
    o = float(odds)
    return (100.0/(o+100.0)) if o > 0 else (abs(o)/(abs(o)+100.0))

# Acklam inverse normal CDF (no scipy)
def norm_ppf(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-10, 1 - 1e-10)

    a = np.array([-3.969683028665376e+01,  2.209460984245205e+02,
                  -2.759285104469687e+02,  1.383577518672690e+02,
                  -3.066479806614716e+01,  2.506628277459239e+00])
    b = np.array([-5.447609879822406e+01,  1.615858368580409e+02,
                  -1.556989798598866e+02,  6.680131188771972e+01,
                  -1.328068155288572e+01])
    c = np.array([-7.784894002430293e-03, -3.223964580411365e-01,
                  -2.400758277161838e+00, -2.549732539343734e+00,
                   4.374664141464968e+00,  2.938163982698783e+00])
    d = np.array([ 7.784695709041462e-03,  3.224671290700398e-01,
                   2.445134137142996e+00,  3.754408661907416e+00])

    plow = 0.02425
    phigh = 1 - plow
    x = np.empty_like(p)

    m = p < plow
    if m.any():
        q = np.sqrt(-2*np.log(p[m]))
        x[m] = (((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
               ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)

    m = (p >= plow) & (p <= phigh)
    if m.any():
        q = p[m] - 0.5
        r = q*q
        x[m] = (((((a[0]*r + a[1])*r + a[2])*r + a[3])*r + a[4])*r + a[5]) * q / \
               (((((b[0]*r + b[1])*r + b[2])*r + b[3])*r + b[4])*r + 1)

    m = p > phigh
    if m.any():
        q = np.sqrt(-2*np.log(1 - p[m]))
        x[m] = -(((((c[0]*q + c[1])*q + c[2])*q + c[3])*q + c[4])*q + c[5]) / \
                ((((d[0]*q + d[1])*q + d[2])*q + d[3])*q + 1)
    return x

# --- base efficient market signals
df["p_home_mkt"] = df["home_ml"].apply(american_to_implied_prob)
df["p_away_mkt"] = df["away_ml"].apply(american_to_implied_prob)
vig_sum = df["p_home_mkt"] + df["p_away_mkt"]
df["p_home_mkt_novig"] = df["p_home_mkt"] / vig_sum

# --- choose SD_MARGIN used for ML->margin link (must match engine SD_MARGIN)
SD_MARGIN_FOR_LINK = 16.5

df["mean_margin_ml"] = SD_MARGIN_FOR_LINK * norm_ppf(df["p_home_mkt_novig"].values)
df["mean_margin_spread"] = -df["spread_home_line"].values
df["mean_total_mkt"] = df["total_line"].values

# --- try to locate a historical scores df in memory
# Accept common names. If none found, we run market-only projections.
hist = None
for name in ["games_hist","hist_games","completed_games","scores_df","historical_games","games_completed_df"]:
    if name in globals():
        cand = globals()[name]
        if isinstance(cand, pd.DataFrame) and {"home_team","away_team"}.issubset(cand.columns):
            hist = cand.copy()
            hist_name = name
            break

# --- market-only projection weights (safe default)
W_ML = 0.55
W_SPR = 0.45
W_TOT = 1.00  # totals baseline = market total unless history exists

HOME_EDGE = 1.0  # small home edge in margin units (history will override slightly)

if hist is None:
    print("⚠ No historical scores df found in memory. Building projections from efficient market signals ONLY.")
    df["mean_margin"] = (W_ML*df["mean_margin_ml"] + W_SPR*df["mean_margin_spread"]) + HOME_EDGE*0.0
    df["mean_total"] = df["mean_total_mkt"] * W_TOT

else:
    print(f"✅ Found historical scores df: {hist_name}. Upgrading projections with team-rating layer.")

    # --- coerce scores cols
    for sc in ["home_score","away_score"]:
        if sc in hist.columns:
            hist[sc] = _to_num(hist[sc])

    need_scores = {"home_score","away_score","home_team","away_team"}
    if not need_scores.issubset(hist.columns) or hist[["home_score","away_score"]].isna().any(axis=1).all():
        print("⚠ Historical df lacks usable score columns. Falling back to market-only projections.")
        df["mean_margin"] = (W_ML*df["mean_margin_ml"] + W_SPR*df["mean_margin_spread"])
        df["mean_total"] = df["mean_total_mkt"]
    else:
        hist = hist.dropna(subset=["home_score","away_score","home_team","away_team"]).copy()
        hist["home_margin"] = hist["home_score"] - hist["away_score"]
        hist["total"] = hist["home_score"] + hist["away_score"]

        # Team attack/defense via simple averages (lightweight, stable)
        home_pts = hist.groupby("home_team")["home_score"].mean()
        away_pts = hist.groupby("away_team")["away_score"].mean()
        team_for = pd.concat([home_pts, away_pts], axis=0).groupby(level=0).mean()

        home_allow = hist.groupby("home_team")["away_score"].mean()
        away_allow = hist.groupby("away_team")["home_score"].mean()
        team_against = pd.concat([home_allow, away_allow], axis=0).groupby(level=0).mean()

        league_ppg = float(hist["total"].mean() / 2.0)
        league_total = float(hist["total"].mean())
        league_home_edge = float(hist["home_margin"].mean())

        # projected margin from ratings:
        # home expected pts ~ league_ppg + (home_for - league_ppg) - (away_against - league_ppg)
        # away expected pts ~ league_ppg + (away_for - league_ppg) - (home_against - league_ppg)
        def get_series(s, idx, default):
            s = s.reindex(idx)
            return s.fillna(default).values

        teams_home = df["home_team"]
        teams_away = df["away_team"]

        home_for = get_series(team_for, teams_home, league_ppg)
        away_for = get_series(team_for, teams_away, league_ppg)
        home_against = get_series(team_against, teams_home, league_ppg)
        away_against = get_series(team_against, teams_away, league_ppg)

        proj_home_pts = league_ppg + (home_for - league_ppg) - (away_against - league_ppg)
        proj_away_pts = league_ppg + (away_for - league_ppg) - (home_against - league_ppg)

        mean_margin_hist = (proj_home_pts - proj_away_pts) + league_home_edge
        mean_total_hist = (proj_home_pts + proj_away_pts)

        # Blend: our ratings layer + efficient market signals
        W_HIST_MARGIN = 0.35
        W_EFF_MKT_MARGIN = 0.65
        df["mean_margin"] = (
            W_HIST_MARGIN*mean_margin_hist +
            W_EFF_MKT_MARGIN*(W_ML*df["mean_margin_ml"].values + W_SPR*df["mean_margin_spread"].values)
        )

        W_HIST_TOTAL = 0.35
        W_MKT_TOTAL  = 0.65
        df["mean_total"] = (W_HIST_TOTAL*mean_total_hist + W_MKT_TOTAL*df["mean_total_mkt"].values)

# write back
games_df["mean_margin"] = df["mean_margin"].values
games_df["mean_total"]  = df["mean_total"].values

print("✅ Projection layers written to games_df:")
print(" - mean_margin (home expected margin)")
print(" - mean_total  (expected total)")
print("\nPreview:")
display(games_df[["away_team","home_team","spread_home_line","total_line","mean_margin","mean_total"]].head(10))

In [ ]:
def half_kelly_units(p, odds, min_u=0.25, max_u=1.0, kelly_floor=0.08):
    """
    kelly_floor = minimum half-kelly fraction needed to place any bet.
    Example: 0.08 means half-kelly must be >= 8% of max_u to bet.
    """
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p

    f = (p*b - q) / b if b > 0 else 0.0
    if HALF_KELLY:
        f *= 0.5

    # no bet if Kelly is too small
    if f <= 0 or f < kelly_floor:
        return 0.0

    # scale to units
    u = min(max_u, f * max_u)

    # round to clean quarters
    u = round(u * 4) / 4

    # clamp
    u = max(min_u, min(max_u, u))
    return float(u)

In [ ]:
# ============================================================
# OPTION 2 — POSTABLE CARD FILTER (HIGH HIT-RATE BRAND)
# Run AFTER the engine creates: card_out (and/or card)
# Produces a clean Discord card + export
# ============================================================

import pandas as pd
import numpy as np

# --- must exist
if "card_out" not in globals():
    raise RuntimeError("card_out not found. Run the FULL STRICT ENGINE cell first.")

df = card_out.copy()

# -------------------------
# TUNABLE FILTERS
# -------------------------
MAX_PLAYS_TOTAL = 20          # keep it tight
EV_MIN = 0.05                 # 5% ROI minimum
MAX_DOG_ODDS = 400            # remove longshot MLs
P_SPREAD_TOTAL_MIN = 0.58     # hit-rate floor for spreads/totals
P_ML_FAV_MIN = 0.60           # hit-rate floor for ML favorites
P_ML_DOG_MIN = 0.42           # only if you allow small dogs
ALLOW_ML_DOGS = True          # flip False for "favorites/safer only"

# -------------------------
# Helpers
# -------------------------
def is_ml(row): return str(row["market"]).upper() == "ML"
def is_spread(row): return str(row["market"]).upper() == "SPREAD"
def is_total(row): return str(row["market"]).upper() == "TOTAL"

def is_dog_odds(odds):
    try:
        return float(odds) > 0
    except:
        return False

def is_fav_odds(odds):
    try:
        return float(odds) < 0
    except:
        return False

# -------------------------
# Core filters
# -------------------------
df = df[df["ev"] >= EV_MIN].copy()

# market-specific probability floors
keep = []
for _, r in df.iterrows():
    p = float(r["p_win"])
    odds = float(r["odds"])
    mkt = str(r["market"]).upper()

    if mkt in ["SPREAD","TOTAL"]:
        keep.append(p >= P_SPREAD_TOTAL_MIN)
        continue

    if mkt == "ML":
        if is_fav_odds(odds):
            keep.append(p >= P_ML_FAV_MIN)
        else:
            # dog ML
            if not ALLOW_ML_DOGS:
                keep.append(False)
            else:
                keep.append((odds <= MAX_DOG_ODDS) and (p >= P_ML_DOG_MIN))
        continue

    keep.append(True)

df = df[pd.Series(keep, index=df.index)].copy()

# -------------------------
# Ranking: blend EV + probability (Option 2 style)
# -------------------------
EV_W = 0.65
PWIN_W = 0.60
df["rank_score"] = (EV_W * df["ev"]) + (PWIN_W * df["p_win"])

df = df.sort_values(["rank_score","ev","p_win"], ascending=False).copy()

# keep max plays overall + still respect cap 2/game if present in your df already
df = df.head(MAX_PLAYS_TOTAL).reset_index(drop=True)

# -------------------------
# Build discord output
# -------------------------
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

df["discord_text"] = df.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {float(x["units"]):.2f}u\n'
        f'Sim Win%: {fmt_prob(float(x["p_win"]))} | Market: {fmt_prob(float(x["p_mkt"]))}\n'
        f'EV: {fmt_ev(float(x["ev"]))}\n'
    ),
    axis=1
)

EXPORT = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_POSTABLE_HIGH_HITRATE.csv"
df.to_csv(EXPORT, index=False)

print(f"✅ Postable plays: {len(df)}")
print(f"✅ Exported: {EXPORT}\n")
print("=== DISCORD CARD (POSTABLE / HIGH HIT-RATE) ===\n")
print("\n".join(df["discord_text"].tolist()))

In [ ]:
# ============================================================
# CDR FIX CELL (POSTABLE CARD)
# - Filters out longshot ML spam
# - Enforces hit-rate mins by market
# - Requires meaningful EV
# - Recomputes Half-Kelly units (0.25u–1.0u) with a Kelly floor
# - Caps total plays and prints clean Discord output + export
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime

# -------------------------
# TUNABLE FILTERS (CDR)
# -------------------------
EV_MIN = 0.05                 # require at least +5% ROI
P_ST_MIN = 0.58               # spreads/totals must be >=58% win prob
P_ML_FAV_MIN = 0.60           # ML favorites must be >=60%
P_ML_DOG_MIN = 0.30           # ML dogs must be >=30% (prevents 10–20% dogs)
MAX_DOG_ODDS = 350            # max + odds allowed for dogs
MAX_PLAYS_TOTAL = 18          # final postable card size
MAX_PLAYS_PER_GAME = 2        # keep at most 2 per matchup

MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True
KELLY_FLOOR = 0.10            # drop plays where half-kelly fraction < 10% of MAX_U
EV_W = 0.65                   # option-2 blend weights
PWIN_W = 0.60

EXPORT_POST = f"ncaab_cdr_postable_{pd.Timestamp.utcnow().date()}.csv"

# -------------------------
# INPUT: prefer card_out from engine, fallback to card, else read last export
# -------------------------
src = None
if "card_out" in globals() and isinstance(card_out, pd.DataFrame) and len(card_out) > 0:
    src = card_out.copy()
elif "card" in globals() and isinstance(card, pd.DataFrame) and len(card) > 0:
    src = card.copy()
else:
    raise RuntimeError("No card_out/card found. Run FULL STRICT ENGINE cell first (the one that builds the card + export).")

# normalize expected columns
need = ["matchup","market","bet","p_win","p_mkt","ev","odds"]
missing = [c for c in need if c not in src.columns]
if missing:
    raise RuntimeError(f"Missing needed columns in card df: {missing}. Columns present: {list(src.columns)}")

df = src.copy()

# ensure numeric
for c in ["p_win","p_mkt","ev","odds"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["p_win","ev","odds","market","matchup","bet"]).reset_index(drop=True)

# -------------------------
# Kelly sizing (half-kelly, with floor gate)
# -------------------------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def kelly_fraction(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (p*b - q) / b  # full kelly fraction
    if HALF_KELLY:
        f *= 0.5
    return max(0.0, float(f))

def kelly_units(p, odds):
    f = kelly_fraction(p, odds)
    if f <= 0:
        return 0.0
    # map fraction -> units (scaled to MAX_U)
    u = f * MAX_U
    # floor/cap
    if u < MIN_U:
        u = MIN_U
    if u > MAX_U:
        u = MAX_U
    return float(u), float(f)

# compute units + kelly frac
units = []
frac = []
for p, o in zip(df["p_win"].values, df["odds"].values):
    u, f = kelly_units(p, o)
    units.append(u)
    frac.append(f)

df["kelly_f"] = frac
df["units"] = units

# -------------------------
# MARKET-SPECIFIC FILTERS
# -------------------------
# Base EV filter
df = df[df["ev"] >= EV_MIN].copy()

# Spreads / Totals: hit-rate gate
is_st = df["market"].isin(["SPREAD","TOTAL"])
df = df[~is_st | (df["p_win"] >= P_ST_MIN)].copy()

# Moneyline: favorite/dog gates
is_ml = df["market"].eq("ML")

ml_dog = is_ml & (df["odds"] > 0)
ml_fav = is_ml & (df["odds"] < 0)

# dogs: cap odds + min win prob
df = df[~ml_dog | ((df["odds"] <= MAX_DOG_ODDS) & (df["p_win"] >= P_ML_DOG_MIN))].copy()

# favorites: min win prob
df = df[~ml_fav | (df["p_win"] >= P_ML_FAV_MIN)].copy()

# kelly floor gate (prevents everything snapping to min bet when edge is tiny)
df = df[df["kelly_f"] >= KELLY_FLOOR].copy()

# -------------------------
# RANK (Option 2 blend) + CAP PER GAME + CAP TOTAL
# -------------------------
df["rank_score"] = (EV_W * df["ev"]) + (PWIN_W * df["p_win"])
df = df.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# cap per game
df["_game_n"] = df.groupby("matchup").cumcount() + 1
df = df[df["_game_n"] <= MAX_PLAYS_PER_GAME].drop(columns=["_game_n"]).copy()

# cap total plays
df = df.head(MAX_PLAYS_TOTAL).copy()

# -------------------------
# DISCORD TEXT
# -------------------------
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

df["discord_text"] = df.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

# export + print
df_out = df[["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","discord_text"]].copy()
df_out.to_csv(EXPORT_POST, index=False)

header = f"== CDR BETTING | NCAAB HIGH HIT-RATE / +EV CARD ({pd.Timestamp.utcnow().date()}) =="
print("\n" + header + "\n")
print("\n".join(df_out["discord_text"].tolist()))
print(f"\n✅ Postable plays: {len(df_out)} (cap {MAX_PLAYS_TOTAL})")
print(f"✅ Exported: {EXPORT_POST}")

In [ ]:
# ============================================================
# FIX UNITS (prevents everything snapping to 0.25u)
# - Recompute units from half-Kelly with a scaler
# - Rebuild discord_text
# - Re-export
# ============================================================

import numpy as np
import pandas as pd

# ---- SETTINGS ----
MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True

KELLY_TO_UNITS = 4.0     # <-- KEY FIX (try 4.0; if still too small, use 5–6)
KELLY_FLOOR = 0.03       # drop plays with basically-zero kelly (optional)

EXPORT_FIXED = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_FULL_STRICT_ENGINE_v1_UNITS_FIXED.csv"

# ---- INPUT ----
if "card_out" in globals() and isinstance(card_out, pd.DataFrame) and len(card_out) > 0:
    df = card_out.copy()
elif "card" in globals() and isinstance(card, pd.DataFrame) and len(card) > 0:
    df = card.copy()
else:
    raise RuntimeError("No card_out/card found. Run the FULL STRICT ENGINE cell first.")

need = ["matchup","market","bet","p_win","p_mkt","ev","odds"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing columns: {missing}. Have: {list(df.columns)}")

for c in ["p_win","p_mkt","ev","odds"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["p_win","odds"]).copy()

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def half_kelly_fraction(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (p*b - q) / b  # full Kelly
    if HALF_KELLY:
        f *= 0.5
    return max(0.0, float(f))

def kelly_units(p, odds):
    f = half_kelly_fraction(p, odds)
    # convert fraction -> units with scaler
    u = f * KELLY_TO_UNITS
    u = float(np.clip(u, 0.0, MAX_U))
    if u > 0 and u < MIN_U:
        u = MIN_U
    return u, f

units, kf = [], []
for p, o in zip(df["p_win"].values, df["odds"].values):
    u, f = kelly_units(p, o)
    units.append(u); kf.append(f)

df["kelly_f"] = kf
df["units"] = units

# optional: drop “fake” min-bets where kelly is basically zero
df = df[df["kelly_f"] >= KELLY_FLOOR].copy()

def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

df["discord_text"] = df.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

df = df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)
df.to_csv(EXPORT_FIXED, index=False)

print(f"✅ Units fixed with KELLY_TO_UNITS={KELLY_TO_UNITS}")
print("Units summary:")
print(df["units"].describe())
print(f"✅ Exported: {EXPORT_FIXED}\n")

print("=== DISCORD CARD (UNITS FIXED) ===\n")
print("\n".join(df["discord_text"].tolist()))

In [ ]:
# Force unit-fix to run on the full card (ML/SPREAD/TOTAL)
df = card.copy()
# ============================================================
# AUTO-UNIT SIZING FIX (bottom cell)
# - Recompute half-Kelly fraction (kelly_f)
# - Auto-pick scaler so median play ≈ TARGET_MEDIAN_U
# - Apply floor/cap (0.25u–1.00u)
# - Rebuild discord_text + export
# ============================================================

import numpy as np
import pandas as pd

# ---- SETTINGS ----
MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True

TARGET_MEDIAN_U = 0.40   # where you want the "typical" play to land
KELLY_FLOOR = 0.005      # keep smaller edges (your prior 0.03 was chopping a lot)

EXPORT_FIXED = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_FULL_STRICT_ENGINE_v1_UNITS_AUTO.csv"

# ---- INPUT ----
if "card_out" in globals() and isinstance(card_out, pd.DataFrame) and len(card_out) > 0:
    df = card_out.copy()
elif "card" in globals() and isinstance(card, pd.DataFrame) and len(card) > 0:
    df = card.copy()
else:
    raise RuntimeError("No card_out/card found. Run the FULL STRICT ENGINE cell first.")

need = ["matchup","market","bet","p_win","p_mkt","ev","odds"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing columns: {missing}. Have: {list(df.columns)}")

for c in ["p_win","p_mkt","ev","odds"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["p_win","odds"]).copy()

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def kelly_fraction(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    if b <= 0:
        return 0.0
    f = (p*b - q) / b  # full Kelly
    if HALF_KELLY:
        f *= 0.5
    return max(0.0, float(f))

df["kelly_f"] = [kelly_fraction(p, o) for p, o in zip(df["p_win"].values, df["odds"].values)]

# keep small edges but drop true zeros
df = df[df["kelly_f"] >= KELLY_FLOOR].copy()
if len(df) == 0:
    raise RuntimeError("All rows filtered out by KELLY_FLOOR. Lower it (e.g. 0.001).")

# ---- AUTO SCALER ----
med_k = float(np.median(df["kelly_f"].values))
if med_k <= 0:
    raise RuntimeError("Median kelly_f is 0. Something is wrong with inputs.")

# scaler so median maps to target
SCALER = TARGET_MEDIAN_U / med_k

def kelly_to_units(f):
    u = float(np.clip(f * SCALER, 0.0, MAX_U))
    if u > 0 and u < MIN_U:
        u = MIN_U
    return u

df["units"] = [kelly_to_units(f) for f in df["kelly_f"].values]

# ---- DISCORD TEXT ----
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

df["discord_text"] = df.apply(
    lambda x: (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    ),
    axis=1
)

# re-rank: EV first, then p_win
df = df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

df.to_csv(EXPORT_FIXED, index=False)

print(f"✅ Auto-scaler set to: {SCALER:.2f} (median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U}u)")
print("Kelly fraction summary:")
print(df["kelly_f"].describe())
print("\nUnits summary:")
print(df["units"].describe())
print(f"\n✅ Exported: {EXPORT_FIXED}\n")

print("=== DISCORD CARD (UNITS AUTO) ===\n")
print("\n".join(df["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX CELL — FORCE UNITS AUTO ON FULL CARD (ML/SPREAD/TOTAL)
# ============================================================
import numpy as np
import pandas as pd

MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005   # prevents micro-kelly going to 0
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_FULL_STRICT_ENGINE_v1_UNITS_AUTO_FULLCARD.csv"

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

# ---- pick the most "full" plays df available
candidates = []
for name in ["card", "card_out", "plays", "final_plays"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        tmp = globals()[name].copy()
        candidates.append((name, len(tmp), tmp))

if not candidates:
    raise RuntimeError("No plays dataframe found. Expected one of: card / card_out / plays / final_plays")

candidates.sort(key=lambda x: x[1], reverse=True)
src_name, _, df = candidates[0]

required = ["matchup","bet","odds","p_win"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"{src_name} missing required columns: {missing}. cols={list(df.columns)}")

# ensure market col exists (optional but helpful)
if "market" not in df.columns:
    df["market"] = "UNK"

print(f"✅ Sizing source: {src_name} | rows={len(df)}")
print("✅ Market counts:")
print(df["market"].value_counts(dropna=False))

# ---- compute p_mkt / ev if not present
if "p_mkt" not in df.columns:
    df["p_mkt"] = df["odds"].apply(american_to_implied_prob)
if "ev" not in df.columns:
    df["ev"] = df.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)

# ---- kelly + auto-scaler
df["kelly_f"] = df.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
df["kelly_f"] = df["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(df["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
print(f"\n✅ Auto-scaler set to: {auto_scale:.2f} (median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U:.1f}u)")

df["units"] = (df["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

print("\nUnits summary:")
print(df["units"].describe())

# ---- rebuild discord text if needed
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e): return f"{100*e:.1f}%"

df["discord_text"] = df.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(float(x["p_win"]))} | Market: {fmt_prob(float(x["p_mkt"]))}\n'
        f'EV: {fmt_ev(float(x["ev"]))}\n',
    axis=1
)

# ---- export + print
out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","discord_text"]
df[out_cols].to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Exported: {EXPORT_NAME}\n")
print("=== DISCORD CARD (UNITS AUTO — FULLCARD) ===\n")
print("\n".join(df["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX CELL — TIGHTER TOP 20 (BLENDED: +EV + PROBABLE)
# - Uses existing plays df (prefers: card, else card_out/plays/final_plays)
# - Recomputes p_mkt + EV if missing
# - Blended ranking (EV + Win%) to prioritize "most probable +EV"
# - Caps plays per game (default 2)
# - Auto unit sizing (median ~0.40u) via half-Kelly + auto-scaler
# - Exports CSV + prints Discord-ready Top 20
# ============================================================
import numpy as np
import pandas as pd

# ---- CONFIG
TOP_N = 20
MAX_PLAYS_PER_GAME = 2

EV_MIN = 0.01          # +1% ROI minimum to tighten the card (set 0.00 for all +EV)
PWIN_MIN = 0.54        # minimum win prob filter (set None to disable)

# Blend weights (your "Option 2" style)
EV_W = 0.65
PWIN_W = 0.60

# Unit sizing
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP{TOP_N}_BLENDED_EV_PROB_UNITS_AUTO.csv"

# ---- HELPERS
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

# ---- PICK SOURCE DF
candidates = []
for name in ["card", "card_out", "plays", "final_plays"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        candidates.append((name, len(globals()[name]), globals()[name].copy()))
if not candidates:
    raise RuntimeError("No plays dataframe found. Expected one of: card / card_out / plays / final_plays")

candidates.sort(key=lambda x: x[1], reverse=True)
src_name, _, df = candidates[0]
print(f"✅ Source: {src_name} | rows={len(df)}")

# ---- REQUIRED COLS
required = ["matchup", "bet", "odds", "p_win"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"{src_name} missing required columns: {missing}. cols={list(df.columns)}")

# market optional
if "market" not in df.columns:
    df["market"] = "UNK"

# numeric sanitize
df["odds"] = pd.to_numeric(df["odds"], errors="coerce")
df["p_win"] = pd.to_numeric(df["p_win"], errors="coerce")
df = df.dropna(subset=["odds","p_win"]).reset_index(drop=True)

# ---- COMPUTE p_mkt / EV
if "p_mkt" not in df.columns:
    df["p_mkt"] = df["odds"].apply(american_to_implied_prob)
else:
    df["p_mkt"] = pd.to_numeric(df["p_mkt"], errors="coerce")

if "ev" not in df.columns:
    df["ev"] = df.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)
else:
    df["ev"] = pd.to_numeric(df["ev"], errors="coerce")

df = df.dropna(subset=["p_mkt","ev"]).copy()

# ---- FILTERS (tighten)
df = df[df["ev"] >= EV_MIN].copy()
if PWIN_MIN is not None:
    df = df[df["p_win"] >= float(PWIN_MIN)].copy()

if len(df) == 0:
    raise RuntimeError("No rows left after filters. Loosen EV_MIN / PWIN_MIN.")

# ---- BLENDED RANKING (normalize then blend)
# Normalize EV into [0,1] robustly so big dogs don't dominate solely on EV.
ev_clip = df["ev"].clip(lower=0.0, upper=df["ev"].quantile(0.95))
df["_ev_norm"] = (ev_clip - ev_clip.min()) / (ev_clip.max() - ev_clip.min() + 1e-12)

# Normalize p_win into [0,1] within slate
p_clip = df["p_win"].clip(lower=df["p_win"].quantile(0.05), upper=df["p_win"].quantile(0.95))
df["_p_norm"] = (p_clip - p_clip.min()) / (p_clip.max() - p_clip.min() + 1e-12)

df["rank_score"] = (EV_W * df["_ev_norm"]) + (PWIN_W * df["_p_norm"])
df = df.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# ---- CAP PLAYS PER GAME (then take top N)
df["_game_count"] = df.groupby("matchup").cumcount() + 1
df = df[df["_game_count"] <= MAX_PLAYS_PER_GAME].copy()
df = df.drop(columns=["_game_count"])

df = df.head(TOP_N).copy()

# ---- UNITS AUTO (half-kelly + auto-scaler)
df["kelly_f"] = df.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
df["kelly_f"] = df["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(df["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
print(f"✅ Auto-scaler: {auto_scale:.2f} (median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U:.2f}u)")

df["units"] = (df["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# ---- DISCORD TEXT
df["discord_text"] = df.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(float(x["p_win"]))} | Market: {fmt_prob(float(x["p_mkt"]))}\n'
        f'EV: {fmt_ev(float(x["ev"]))}\n',
    axis=1
)

# ---- EXPORT + PRINT
out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","rank_score","discord_text"]
df[out_cols].to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final Top {TOP_N}: {len(df)} | EV_MIN={EV_MIN} | PWIN_MIN={PWIN_MIN} | Cap/Game={MAX_PLAYS_PER_GAME}")
print(f"✅ Exported: {EXPORT_NAME}\n")
print("=== DISCORD CARD (TOP 20 — BLENDED) ===\n")
print("\n".join(df["discord_text"].tolist()))

In [ ]:
# ============================================================
# ODDS API + TIMEZONE SETUP (RUN THIS FIRST)
# ============================================================

import os
import requests
import pandas as pd
from datetime import datetime
import pytz

# ---- ENTER YOUR ODDS API KEY HERE ----
ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

# store as environment variable so the model cell can use it
os.environ["ODDS_API_KEY"] = ODDS_API_KEY

# ---- SET TIMEZONE (EASTERN) ----
TIMEZONE = "US/Eastern"
tz = pytz.timezone(TIMEZONE)

print("✅ Odds API key loaded")
print("✅ Timezone set to:", TIMEZONE)

# ---- TEST ODDS API CONNECTION ----
SPORT = "basketball_ncaab"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"

params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
}

response = requests.get(url, params=params)

if response.status_code != 200:
    print("❌ API ERROR:", response.status_code)
    print(response.text)
else:
    data = response.json()
    print("✅ API connection successful")
    print("Games pulled:", len(data))

    # preview a few games
    preview = pd.DataFrame(data)[["home_team","away_team","commence_time"]].head(5)
    print("\nSample Games:")
    display(preview)

In [ ]:
# ============================================================
# HARD FIX: if games_df is RAW Odds API (has 'bookmakers'),
# flatten to market-ready games_df with total_line/spread/ml.
# ============================================================
import numpy as np
import pandas as pd

def _median_or_nan(vals):
    vals = [v for v in vals if v is not None and (not (isinstance(v, float) and np.isnan(v)))]
    return float(np.median(vals)) if len(vals) else np.nan

def _extract_from_bookmakers(bookmakers, home, away):
    # collect across all books so we can median-consensus
    home_ml, away_ml = [], []
    spread_home, spread_price = [], []
    total_line, over_price, under_price = [], [], []

    for book in (bookmakers or []):
        for mkt in (book.get("markets") or []):
            key = mkt.get("key")
            outcomes = mkt.get("outcomes") or []

            if key == "h2h":
                for o in outcomes:
                    if o.get("name") == home:
                        home_ml.append(o.get("price"))
                    elif o.get("name") == away:
                        away_ml.append(o.get("price"))

            elif key == "spreads":
                # assume spread is quoted on home team outcome
                for o in outcomes:
                    if o.get("name") == home:
                        spread_home.append(o.get("point"))
                        spread_price.append(o.get("price"))

            elif key == "totals":
                for o in outcomes:
                    nm = o.get("name")
                    if nm == "Over":
                        total_line.append(o.get("point"))
                        over_price.append(o.get("price"))
                    elif nm == "Under":
                        # total_line is same point as over; still store under price
                        under_price.append(o.get("price"))

    return {
        "home_ml": _median_or_nan(home_ml),
        "away_ml": _median_or_nan(away_ml),
        "spread_home": _median_or_nan(spread_home),
        "spread_price": _median_or_nan(spread_price),
        "total_line": _median_or_nan(total_line),
        "over_price": _median_or_nan(over_price),
        "under_price": _median_or_nan(under_price),
    }

# --- detect raw vs already-flat
if "bookmakers" in globals().get("games_df", pd.DataFrame()).columns:
    raw = games_df.copy()
    out_rows = []
    for _, r in raw.iterrows():
        home = r["home_team"]; away = r["away_team"]
        mk = r["bookmakers"]
        ex = _extract_from_bookmakers(mk, home, away)

        out_rows.append({
            "id": r.get("id", None),
            "commence_time": r.get("commence_time", None),
            "home_team": home,
            "away_team": away,
            **ex
        })

    games_df = pd.DataFrame(out_rows)

    # If duplicates (multiple entries same matchup), collapse with median
    num_cols = ["home_ml","away_ml","spread_home","spread_price","total_line","over_price","under_price"]
    games_df = (games_df
                .groupby(["home_team","away_team"], as_index=False)[num_cols]
                .median(numeric_only=True))

    print(f"✅ Flattened RAW Odds API -> market-ready games_df | rows={len(games_df)}")
else:
    print("✅ games_df already looks flattened (no 'bookmakers' col).")

# --- enforce required totals column name for downstream cells
# some of your older cells look for 'total' or 'ou' — alias them too
if "total_line" not in games_df.columns:
    # try common alternates
    for alt in ["total", "ou", "total_points", "game_total"]:
        if alt in games_df.columns:
            games_df["total_line"] = pd.to_numeric(games_df[alt], errors="coerce")
            break

# quick required-columns check
need = ["home_team","away_team","total_line"]
missing = [c for c in need if c not in games_df.columns]
if missing:
    raise RuntimeError(f"Still missing required cols after flatten: {missing}. cols={list(games_df.columns)}")

print("✅ totals column OK. Example rows:")
display(games_df.head())

In [ ]:
# ============================================================
# EXTRACT LINES FROM ODDS API RESPONSE
# Creates games_df with ML / spreads / totals
# ============================================================

rows = []

for game in data:

    home = game["home_team"]
    away = game["away_team"]

    for book in game["bookmakers"]:

        for market in book["markets"]:

            if market["key"] == "h2h":
                home_ml = None
                away_ml = None

                for outcome in market["outcomes"]:
                    if outcome["name"] == home:
                        home_ml = outcome["price"]
                    elif outcome["name"] == away:
                        away_ml = outcome["price"]

            if market["key"] == "spreads":
                spread = None
                spread_price = None

                for outcome in market["outcomes"]:
                    if outcome["name"] == home:
                        spread = outcome["point"]
                        spread_price = outcome["price"]

            if market["key"] == "totals":
                total_line = None
                over_price = None
                under_price = None

                for outcome in market["outcomes"]:
                    if outcome["name"] == "Over":
                        total_line = outcome["point"]
                        over_price = outcome["price"]
                    elif outcome["name"] == "Under":
                        under_price = outcome["price"]

        rows.append({
            "home_team": home,
            "away_team": away,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home": spread,
            "spread_price": spread_price,
            "total_line": total_line,
            "over_price": over_price,
            "under_price": under_price
        })

games_df = pd.DataFrame(rows)

# remove duplicates across books (use median consensus)
games_df = games_df.groupby(["home_team","away_team"]).median(numeric_only=True).reset_index()

print("✅ Games extracted:", len(games_df))
display(games_df.head())

In [ ]:
# ============================================================
# TOTALS + MARKET PROB FIX (pre-sim)
# - prevents mean_total == total_line everywhere
# - uses odds-implied market probs (no more 50.0% placeholders)
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

SD_TOTAL = 19.0
SD_MARGIN = 16.5

TOTAL_BLEND_W = 0.35     # how much we trust our model total vs market
TOTAL_JITTER_SD = 2.0    # adds realistic dispersion if model totals are tight
CALIB_SHRINK = 0.15      # optional: shrink extreme probs toward 0.5

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def p_over_from_mean(mean_total, total_line, sd=SD_TOTAL):
    z = (total_line - mean_total) / sd
    return 1.0 - norm_cdf(z)

def p_cover_from_mean(mean_margin, spread_home, sd=SD_MARGIN):
    # mean_margin is HOME expected margin; spread_home is HOME line (e.g. -3.5)
    # home covers if margin > spread_home
    z = (spread_home - mean_margin) / sd
    return 1.0 - norm_cdf(z)

# ---------- REQUIREMENTS ----------
df = games_df.copy()

need = ["home_team","away_team","total_line"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing: {missing}. cols={list(df.columns)}")

# ---------- 1) Ensure we have a *model* total signal ----------
# You MUST have something like model_total / mean_total_model coming from your projections layer.
# Common names we’ll look for:
model_total_col = None
for c in ["mean_total_model","model_total","proj_total","total_proj","mean_total_raw"]:
    if c in df.columns:
        model_total_col = c
        break

if model_total_col is None:
    # If you truly don’t have a model total signal yet, this is why totals were awful.
    # We can still run, but totals will be mostly market-driven (low edge).
    # Create a weak proxy around market with small jitter so p_over std isn't ~0.
    df["mean_total_model"] = pd.to_numeric(df["total_line"], errors="coerce")
    model_total_col = "mean_total_model"

df[model_total_col] = pd.to_numeric(df[model_total_col], errors="coerce")
df["total_line"] = pd.to_numeric(df["total_line"], errors="coerce")

# ---------- 2) Blend model total with market total (prevents mean_total==line everywhere) ----------
df["mean_total"] = (1.0 - TOTAL_BLEND_W) * df["total_line"] + TOTAL_BLEND_W * df[model_total_col]

# add dispersion (only affects mean, not randomness) to avoid identical totals when projections are stale/tied
rng = np.random.default_rng(1337)
df["mean_total"] = df["mean_total"] + rng.normal(0.0, TOTAL_JITTER_SD, size=len(df))

# ---------- 3) If you have mean_margin already, keep it. If you have a model margin signal, map it. ----------
if "mean_margin" not in df.columns:
    for c in ["mean_margin_model","model_margin","proj_margin","margin_proj","mean_margin_raw"]:
        if c in df.columns:
            df["mean_margin"] = pd.to_numeric(df[c], errors="coerce")
            break

# ---------- 4) Recompute probabilities from means ----------
# Totals
df["p_over"] = df.apply(lambda x: p_over_from_mean(x["mean_total"], x["total_line"], SD_TOTAL), axis=1)

# Spread (only if spread exists)
if "spread_home" in df.columns and "mean_margin" in df.columns:
    df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")
    df["p_home_cover"] = df.apply(lambda x: p_cover_from_mean(x["mean_margin"], x["spread_home"], SD_MARGIN), axis=1)

# ---------- 5) Optional: light calibration shrink (keeps slate sane; won’t destroy edges) ----------
def shrink(p, lam=CALIB_SHRINK):
    return (1-lam)*p + lam*0.5

df["p_over"] = df["p_over"].clip(1e-6, 1-1e-6).apply(shrink)

if "p_home_cover" in df.columns:
    df["p_home_cover"] = df["p_home_cover"].clip(1e-6, 1-1e-6).apply(shrink)

games_df = df  # write back

print("✅ Totals/market-prob layer rebuilt.")
print(f"Sanity — p_over std (target ~0.07–0.11): {games_df['p_over'].std():.4f}")

# NOTE: In your CARD builder, ensure Market% is implied from odds:
# p_mkt = american_to_implied_prob(odds) (no more hard 0.50)

In [ ]:
# ============================================================
# ONE BUILD — NCAAB FULL STRICT ENGINE (3/4) + MARKET CALIBRATION + TOP10 / NEXT5
# ============================================================
import os, math
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import pytz

from scipy.stats import norm  # <- fixes the np.erfinv issue cleanly

# ---------------------------
# USER SETTINGS (EDIT THESE)
# ---------------------------
SLATE_DATE_ET = "2026-03-04"          # <-- set to 3/4
SPORT = "basketball_ncaab"
REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

ODDS_API_KEY = os.getenv("ODDS_API_KEY", "").strip()
if not ODDS_API_KEY:
    # If you prefer manual:
    # ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"
    raise RuntimeError("Missing ODDS_API_KEY. Set env var ODDS_API_KEY or paste it in this cell.")

TZ = pytz.timezone("US/Eastern")

# Model/Sims settings (your current “new new”)
SIMS = 50000
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
RHO = 0.15  # corr(margin,total)

# Projection layer blends (market calibration layer)
BLEND_MARKET_PROB = 0.55   # calibration: final prob = (1-w)*market + w*raw_model
# If you want “trust model more”, lower BLEND_MARKET_PROB. If you want “trust market more”, raise it.

# Mean projection blends (what we simulate from)
BLEND_MARKET_MARGIN = 0.65  # mean_margin = (1-w)*model_margin + w*market_implied_margin
BLEND_MARKET_TOTAL  = 0.80  # mean_total  = (1-w)*model_total  + w*market_total

# Card filters + ranking blend (+EV + probable)
EV_MIN = 0.01        # minimum ROI EV per 1u
PWIN_MIN = 0.54      # minimum calibrated win prob
CAP_PER_GAME = 2
TOP_N = 10
NEXT_FREE_N = 5

# Unit sizing
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

# Ranking blend (Option 2 style)
EV_W = 0.65
PWIN_W = 0.60

rng = np.random.default_rng(42)

# ---------------------------
# Helpers
# ---------------------------
def american_to_decimal(odds: float) -> float:
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds: float) -> float:
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def novig_two_way(p1, p2):
    s = p1 + p2
    if s <= 0: return (0.5, 0.5)
    return (p1/s, p2/s)

def ev_roi_1u(p, odds) -> float:
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True) -> float:
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

# ---------------------------
# (A) Pull slate in ET window
# ---------------------------
start_et = TZ.localize(datetime.strptime(SLATE_DATE_ET, "%Y-%m-%d"))
end_et   = start_et + timedelta(days=1)
start_utc = start_et.astimezone(pytz.utc)
end_utc   = end_et.astimezone(pytz.utc)

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds"
params = dict(
    apiKey=ODDS_API_KEY,
    regions=REGIONS,
    markets=MARKETS,
    oddsFormat=ODDS_FORMAT,
    dateFormat=DATE_FORMAT,
    commenceTimeFrom=start_utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
    commenceTimeTo=end_utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
)

r = requests.get(url, params=params, timeout=45)
if r.status_code != 200:
    raise RuntimeError(f"Odds API error {r.status_code}: {r.text[:300]}")
data = r.json()
games_df = pd.DataFrame(data)
if games_df.empty:
    raise RuntimeError("No games returned for ET window.")

# ---------------------------
# (B) Flatten consensus lines (median across books)
# ---------------------------
rows = []
for _, g in games_df.iterrows():
    gid = g.get("id")
    home = g.get("home_team")
    away = g.get("away_team")
    ct   = g.get("commence_time")
    for book in g.get("bookmakers", []):
        for m in book.get("markets", []):
            mk = m.get("key")
            outs = m.get("outcomes", [])
            if mk == "h2h":
                # outcomes have name+price
                for o in outs:
                    rows.append(dict(game_id=gid, commence_time=ct, home_team=home, away_team=away,
                                     market="h2h", team=o["name"], price=o["price"], point=np.nan))
            elif mk == "spreads":
                for o in outs:
                    rows.append(dict(game_id=gid, commence_time=ct, home_team=home, away_team=away,
                                     market="spreads", team=o["name"], price=o["price"], point=o["point"]))
            elif mk == "totals":
                for o in outs:
                    # totals outcomes: Over/Under with point
                    rows.append(dict(game_id=gid, commence_time=ct, home_team=home, away_team=away,
                                     market="totals", team=o["name"], price=o["price"], point=o["point"]))

od = pd.DataFrame(rows)
if od.empty:
    raise RuntimeError("Flattening failed: no odds rows.")

# helper to median consensus for each game/side
def median_price(df):
    return float(pd.to_numeric(df["price"], errors="coerce").median())

def median_point(df):
    return float(pd.to_numeric(df["point"], errors="coerce").median())

# Moneylines
ml = od[od["market"]=="h2h"].copy()
ml["price"] = pd.to_numeric(ml["price"], errors="coerce")
ml_cons = (ml.groupby(["game_id","home_team","away_team","team"], as_index=False)
             .agg(home_ml=("price","median")))
ml_cons = ml_cons.rename(columns={"home_ml":"ml_price"})

# Spreads
sp = od[od["market"]=="spreads"].copy()
sp["price"] = pd.to_numeric(sp["price"], errors="coerce")
sp["point"] = pd.to_numeric(sp["point"], errors="coerce")
sp_cons = (sp.groupby(["game_id","home_team","away_team","team"], as_index=False)
             .agg(spread=("point","median"), spread_price=("price","median")))

# Totals
tt = od[od["market"]=="totals"].copy()
tt["price"] = pd.to_numeric(tt["price"], errors="coerce")
tt["point"] = pd.to_numeric(tt["point"], errors="coerce")
tt_cons = (tt.groupby(["game_id","home_team","away_team","team"], as_index=False)
             .agg(total=("point","median"), total_price=("price","median")))

# Build a clean game table with consensus lines
g0 = games_df[["id","home_team","away_team","commence_time"]].rename(columns={"id":"game_id"}).drop_duplicates()

# attach ML home/away
home_ml = ml_cons.merge(g0, on=["game_id","home_team","away_team"], how="right")
home_ml = home_ml[home_ml["team"]==home_ml["home_team"]][["game_id","ml_price"]].rename(columns={"ml_price":"home_ml"})
away_ml = ml_cons.merge(g0, on=["game_id","home_team","away_team"], how="right")
away_ml = away_ml[away_ml["team"]==away_ml["away_team"]][["game_id","ml_price"]].rename(columns={"ml_price":"away_ml"})
g = g0.merge(home_ml, on="game_id", how="left").merge(away_ml, on="game_id", how="left")

# attach spreads (home side line + price)
home_sp = sp_cons.merge(g0, on=["game_id","home_team","away_team"], how="right")
home_sp = home_sp[home_sp["team"]==home_sp["home_team"]][["game_id","spread","spread_price"]].rename(
    columns={"spread":"home_spread","spread_price":"home_spread_price"}
)
g = g.merge(home_sp, on="game_id", how="left")

# totals: use the market point median from "Over" row
over_tt = tt_cons.merge(g0, on=["game_id","home_team","away_team"], how="right")
over_tt = over_tt[over_tt["team"].str.lower()=="over"][["game_id","total","total_price"]].rename(
    columns={"total":"total_line","total_price":"over_price"}
)
under_tt = tt_cons.merge(g0, on=["game_id","home_team","away_team"], how="right")
under_tt = under_tt[under_tt["team"].str.lower()=="under"][["game_id","total_price"]].rename(columns={"total_price":"under_price"})
g = g.merge(over_tt, on="game_id", how="left").merge(under_tt, on="game_id", how="left")

# Clean numeric
for c in ["home_ml","away_ml","home_spread","home_spread_price","total_line","over_price","under_price"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")

g = g.dropna(subset=["home_ml","away_ml","home_spread","home_spread_price","total_line","over_price","under_price"]).reset_index(drop=True)

print(f"✅ Games in df: {len(g)} | Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL} | rho={RHO}")

# ---------------------------
# (C) MARKET NO-VIG PROBS
# ---------------------------
g["p_home_mkt"] = g["home_ml"].apply(american_to_implied_prob)
g["p_away_mkt"] = g["away_ml"].apply(american_to_implied_prob)
g["p_home_mkt_novig"], g["p_away_mkt_novig"] = zip(*g.apply(lambda r: novig_two_way(r["p_home_mkt"], r["p_away_mkt"]), axis=1))

g["p_over_mkt"]  = g["over_price"].apply(american_to_implied_prob)
g["p_under_mkt"] = g["under_price"].apply(american_to_implied_prob)
g["p_over_mkt_novig"], g["p_under_mkt_novig"] = zip(*g.apply(lambda r: novig_two_way(r["p_over_mkt"], r["p_under_mkt"]), axis=1))

# ---------------------------
# (D) PROJECTION LAYERS (THIS WAS THE MISSING PART)
#     mean_margin from ML no-vig via Normal CDF inverse
#     mean_total anchored to market total
# ---------------------------
# market-implied mean margin (HOME)
g["mean_margin_mkt"] = SD_MARGIN * norm.ppf(np.clip(g["p_home_mkt_novig"].values, 1e-6, 1-1e-6))

# optional model margin/total if you have them (safe fallbacks)
# If your notebook already produced these, we’ll use them. Otherwise we fallback to market.
for col in ["mean_margin_model", "mean_total_model"]:
    if col not in g.columns:
        g[col] = np.nan

# if notebook had power ratings / expected_margin, map it
if "expected_margin" in globals() and isinstance(globals()["expected_margin"], (pd.Series, np.ndarray, list)):
    pass  # (kept for compatibility)

# Hard fallback: model = market if missing
g["mean_margin_model"] = g["mean_margin_model"].fillna(g["mean_margin_mkt"])
g["mean_total_model"]  = g["mean_total_model"].fillna(g["total_line"])

# Blended means used for sims
g["mean_margin"] = (1.0 - BLEND_MARKET_MARGIN) * g["mean_margin_model"] + BLEND_MARKET_MARGIN * g["mean_margin_mkt"]
g["mean_total"]  = (1.0 - BLEND_MARKET_TOTAL)  * g["mean_total_model"]  + BLEND_MARKET_TOTAL  * g["total_line"]

# ---------------------------
# (E) Correlated sims: [margin, total] ~ BVN
# ---------------------------
n = len(g)
z1 = rng.standard_normal((n, SIMS))
z2 = rng.standard_normal((n, SIMS))
# correlate:
margins = g["mean_margin"].to_numpy()[:,None] + SD_MARGIN * z1
totals  = g["mean_total"].to_numpy()[:,None]  + SD_TOTAL  * (RHO*z1 + math.sqrt(1-RHO**2)*z2)

# Raw probs
p_home_win_raw   = (margins > 0).mean(axis=1)
p_home_cover_raw = (margins > (-g["home_spread"].to_numpy()[:,None])).mean(axis=1)  # home_spread is usually negative for fave
p_over_raw       = (totals  > (g["total_line"].to_numpy()[:,None])).mean(axis=1)

# ---------------------------
# (F) Market calibration blend (efficient market signal layer)
# ---------------------------
g["p_home_win"]   = (1.0 - BLEND_MARKET_PROB) * p_home_win_raw   + BLEND_MARKET_PROB * g["p_home_mkt_novig"].to_numpy()
g["p_home_cover"] = (1.0 - BLEND_MARKET_PROB) * p_home_cover_raw + BLEND_MARKET_PROB * 0.50  # spread market prob approx (unless you derive from spread price)
g["p_over"]       = (1.0 - BLEND_MARKET_PROB) * p_over_raw       + BLEND_MARKET_PROB * g["p_over_mkt_novig"].to_numpy()

print("\nSanity (prob std):")
print(f" - p_over std: {np.std(g['p_over']):.4f}")
print(f" - p_home_cover std: {np.std(g['p_home_cover']):.4f}")

# ---------------------------
# (G) Build plays: ML / SPREAD / TOTAL (home side only for spread; over only for total)
# ---------------------------
plays = []

# ML: take underdog/value on either side
for _, r in g.iterrows():
    # home ML play
    p = float(r["p_home_win"])
    odds = float(r["home_ml"])
    ev = ev_roi_1u(p, odds)
    plays.append(dict(
        game_id=r["game_id"], matchup=f"{r['away_team']} at {r['home_team']}",
        market="ML", bet=f"{r['home_team']} ML ({int(odds)})", odds=odds, p_win=p,
        p_mkt=float(r["p_home_mkt_novig"]), ev=ev
    ))
    # away ML play
    pA = 1.0 - float(r["p_home_win"])
    oddsA = float(r["away_ml"])
    evA = ev_roi_1u(pA, oddsA)
    plays.append(dict(
        game_id=r["game_id"], matchup=f"{r['away_team']} at {r['home_team']}",
        market="ML", bet=f"{r['away_team']} ML ({int(oddsA)})", odds=oddsA, p_win=pA,
        p_mkt=float(r["p_away_mkt_novig"]), ev=evA
    ))

# SPREAD: home spread
for _, r in g.iterrows():
    p = float(r["p_home_cover"])
    odds = float(r["home_spread_price"])
    line = float(r["home_spread"])
    ev = ev_roi_1u(p, odds)
    plays.append(dict(
        game_id=r["game_id"], matchup=f"{r['away_team']} at {r['home_team']}",
        market="SPREAD", bet=f"{r['home_team']} {line:g} ({int(odds)})", odds=odds, p_win=p,
        p_mkt=0.50, ev=ev
    ))

# TOTAL: over
for _, r in g.iterrows():
    p = float(r["p_over"])
    odds = float(r["over_price"])
    line = float(r["total_line"])
    ev = ev_roi_1u(p, odds)
    plays.append(dict(
        game_id=r["game_id"], matchup=f"{r['away_team']} at {r['home_team']}",
        market="TOTAL", bet=f"OVER {line:g} ({int(odds)})", odds=odds, p_win=p,
        p_mkt=float(r["p_over_mkt_novig"]), ev=ev
    ))

card = pd.DataFrame(plays)

# ---------------------------
# (H) Filters + Option-2 ranking blend (+EV + probable)
# ---------------------------
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()

# cap per game: keep best 2 by blended score
# normalize ev to stabilize ranking scale
if len(card):
    ev_clip = card["ev"].clip(lower=0, upper=0.75)
    card["_evn"] = (ev_clip - ev_clip.min()) / (max(1e-9, (ev_clip.max() - ev_clip.min())))
    card["_score"] = EV_W*card["_evn"] + PWIN_W*card["p_win"]
    card = (card.sort_values(["game_id","_score"], ascending=[True, False])
                .groupby("game_id", as_index=False)
                .head(CAP_PER_GAME)
                .sort_values("_score", ascending=False)
           )

# ---------------------------
# (I) Half-Kelly + auto-scaler to target median units
# ---------------------------
card["kelly_f"] = card.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(card["kelly_f"].median()) if len(card) else 0.0
auto_scale = (TARGET_MEDIAN_U / med_k) if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# ---------------------------
# (J) Export + Outputs (TOP10 + NEXT5 FREE)
# ---------------------------
EXPORT = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_NCAAB_{SLATE_DATE_ET}_TOP10_NEXT5.csv"
out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units"]
card[out_cols].to_csv(EXPORT, index=False)

HEADER = f"== CDR BETTING | NCAAB TOP {TOP_N} (MODEL x MARKET CALIBRATED) | {SLATE_DATE_ET} =="

def build_discord_block(df):
    lines = []
    for _, x in df.iterrows():
        lines.append(
            f"{x['matchup']}\n"
            f"{x['bet']} — {x['units']:.2f}u\n"
            f"Sim Win%: {fmt_prob(float(x['p_win']))} | Market: {fmt_prob(float(x['p_mkt']))}\n"
            f"EV: {fmt_ev(float(x['ev']))}\n"
        )
    return "\n".join(lines).strip()

top10 = card.head(TOP_N).copy()
next5 = card.iloc[TOP_N:TOP_N+NEXT_FREE_N].copy()

print(f"\n✅ Final Card Rows: {len(card)} | Cap/Game={CAP_PER_GAME} | Exported: {EXPORT}\n")

print(HEADER)
print(build_discord_block(top10))

print("\n== FREE TAILS PICKS (NEXT 5) ==")
print(build_discord_block(next5))

In [ ]:
# ============================================================
# BOTTOM FIX CELL — TOTALS FIX (KEEP TOP 10 PIPELINE)
# - Rebuild TOTAL probabilities with proper variance (no flatten)
# - Leaves ML/SPREAD rows alone
# - Re-ranks to Top 10 (blended +EV + probable) with units auto
# ============================================================

import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
SIMS = 50000
SD_TOTAL = 19.0
RNG_SEED = 7

# totals calibration (this is the fix)
# Keep these LOW so totals don't collapse to ~50%
BLEND_MARKET_TOTAL_PROB = 0.18     # was effectively too high
BLEND_MARKET_TOTAL_MEAN = 0.40     # market anchors mean_total, not probabilities

# top 10 filters (same idea as your blended top20)
EV_MIN   = 0.01
PWIN_MIN = 0.54
TOP_N    = 10
CAP_PER_GAME = 2

# units auto-scaling
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP10_TOTALS_FIXED.csv"

# ---------- GUARDS ----------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run the odds pull + engine cell first (must create games_df).")
if "card" not in globals():
    raise RuntimeError("card not found. Run your engine cell first (it must create `card`).")

df = games_df.copy()

need_cols = ["away_team","home_team","total_line"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns for totals fix: {missing}")

def _to_num(x):
    return pd.to_numeric(x, errors="coerce")

df["total_line"] = _to_num(df["total_line"])

# if your projection layer exists, use it; otherwise default to market total (still works)
if "mean_total" in df.columns:
    df["mean_total_model"] = _to_num(df["mean_total"])
else:
    df["mean_total_model"] = df["total_line"]

# anchor mean_total toward market (this is good; avoids wild totals)
df["mean_total_anch"] = (1.0 - BLEND_MARKET_TOTAL_MEAN) * df["mean_total_model"] + BLEND_MARKET_TOTAL_MEAN * df["total_line"]

# ---------- MONTE CARLO TOTALS ----------
rng = np.random.default_rng(RNG_SEED)
total_sim = rng.normal(loc=df["mean_total_anch"].values, scale=SD_TOTAL, size=(SIMS, len(df)))

p_over_raw = (total_sim > df["total_line"].values).mean(axis=0)
p_under_raw = 1.0 - p_over_raw

# Calibrate PROBABILITY lightly toward market 50/50 (small weight only)
p_over  = (1.0 - BLEND_MARKET_TOTAL_PROB) * p_over_raw  + BLEND_MARKET_TOTAL_PROB * 0.50
p_under = 1.0 - p_over

df["p_over_fixed"]  = p_over
df["p_under_fixed"] = p_under

print(f"✅ Totals fixed sims: {SIMS} | SD_TOTAL: {SD_TOTAL}")
print(f"Sanity — p_over std (should be ~0.07–0.11): {np.std(df['p_over_fixed'].values):.4f}")

# ---------- PATCH `card` TOTAL ROWS ONLY ----------
card2 = card.copy()

# identify TOTAL rows using your market column + bet text
is_total = (card2["market"].astype(str).str.upper() == "TOTAL")

# build a lookup by matchup -> fixed p_over/p_under
df["matchup"] = df["away_team"].astype(str) + " at " + df["home_team"].astype(str)
tot_lookup = df.set_index("matchup")[["p_over_fixed","p_under_fixed"]].to_dict("index")

def _patch_total_prob(row):
    m = row["matchup"]
    if m not in tot_lookup:
        return row["p_win"]
    b = str(row["bet"]).upper()
    if "OVER" in b:
        return float(tot_lookup[m]["p_over_fixed"])
    if "UNDER" in b:
        return float(tot_lookup[m]["p_under_fixed"])
    return row["p_win"]

card2.loc[is_total, "p_win"] = card2.loc[is_total].apply(_patch_total_prob, axis=1)

# ---------- RE-RUN EV / KELLY / UNITS ----------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

if "p_mkt" not in card2.columns:
    card2["p_mkt"] = card2["odds"].apply(american_to_implied_prob)

card2["ev"] = card2.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)
card2 = card2[(card2["ev"] >= EV_MIN) & (card2["p_win"] >= PWIN_MIN)].copy()

# cap 2 per game
card2["_n"] = card2.groupby("matchup").cumcount() + 1
card2 = card2[card2["_n"] <= CAP_PER_GAME].drop(columns=["_n"])

# blended rank (EV + probability)
EV_W, PWIN_W = 0.65, 0.60
card2["rank_score"] = (EV_W * card2["ev"]) + (PWIN_W * card2["p_win"])
card2 = card2.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# kelly + autoscale to median ~0.40u
card2["kelly_f"] = card2.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
card2["kelly_f"] = card2["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(card2["kelly_f"].median()) if len(card2) else 0.0
auto_scale = (TARGET_MEDIAN_U / med_k) if med_k > 0 else 0.0
card2["units"] = (card2["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# take TOP 10
top10 = card2.head(TOP_N).copy()

# discord text
def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e): return f"{100*e:.1f}%"

top10["discord_text"] = top10.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(float(x["p_win"]))} | Market: {fmt_prob(float(x["p_mkt"]))}\n'
        f'EV: {fmt_ev(float(x["ev"]))}\n',
    axis=1
)

top10.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final Top 10 (Totals Fixed): {len(top10)} | Exported: {EXPORT_NAME}\n")
print(f"== CDR BETTING | NCAAB TOP 10 (TOTALS FIXED) | {pd.Timestamp.now(tz='US/Eastern').date()} ==")
print("\n".join(top10["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX CELL — TOTALS FIX + PACE/MARGIN PROJECTIONS + MONTE CARLO + TOP10 (BLENDED)
# - Robustly maps your columns -> total_line, home/away, etc.
# - Ensures mean_total & mean_margin exist (builds from pace/eff if possible)
# - Monte Carlo totals (and margin if needed)
# - Light market calibration on totals prob (prevents flattening)
# - Blended rank (+EV + probable) + cap/game=2 + units auto
# ============================================================

import numpy as np
import pandas as pd

# ---------------- SETTINGS ----------------
SIMS = 50000
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
RHO = 0.15
RNG_SEED = 7

# totals calibration (KEY FIX: keep small)
BLEND_MARKET_TOTAL_PROB = 0.18     # keep LOW (prevents p_over std collapsing)
BLEND_MARKET_TOTAL_MEAN = 0.40     # anchor mean_total toward market total

# blended card filters
EV_MIN   = 0.01
PWIN_MIN = 0.54
TOP_N    = 10
CAP_PER_GAME = 2

# units auto-scaling
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP10_TOTALS_FIXED.csv"

# ---------------- GUARDS ----------------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run your odds pull + engine build first (must create games_df).")
if "card" not in globals():
    raise RuntimeError("card not found. Run your engine cell first (it must create `card`).")

df = games_df.copy()

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

# ---------------- COLUMN NORMALIZATION ----------------
away_col = _pick_col(df, ["away_team","away","team_away","awayName","away_name"])
home_col = _pick_col(df, ["home_team","home","team_home","homeName","home_name"])
if not away_col or not home_col:
    raise RuntimeError(f"Could not find home/away team columns. cols={list(df.columns)[:40]} ...")

# total line can be named many ways
total_col = _pick_col(df, [
    "total_line","total","total_points","ou","over_under","total_mkt","market_total","closing_total","total_close"
])
if total_col is None:
    # if not present, we can still run using mean_total only (but totals selection needs a line)
    raise RuntimeError(
        "Could not find a market total column. Add one to games_df (e.g., total_line/total/ou). "
        f"cols={list(df.columns)[:50]} ..."
    )

df["away_team"] = df[away_col].astype(str)
df["home_team"] = df[home_col].astype(str)
df["total_line"] = _to_num(df[total_col])

# spread/line (optional; for margin calibration or spread conversion)
spread_col = _pick_col(df, ["spread_home","home_spread","line_home","home_line","spread","homeSpread"])
if spread_col:
    df["spread_home"] = _to_num(df[spread_col])

# moneyline columns (optional; not required for totals fix cell)
# (we don't need them here; card already has odds)

# ---------------- PROJECTION LAYERS (PACE + EFF -> mean_total/mean_margin) ----------------
# If you already computed mean_total/mean_margin upstream, we use them.
have_mean_total  = ("mean_total" in df.columns) and df["mean_total"].notna().any()
have_mean_margin = ("mean_margin" in df.columns) and df["mean_margin"].notna().any()

# Try to build projections if missing using common efficiency + pace fields
pace_col = _pick_col(df, ["pace","poss","possessions","proj_poss","tempo","possesions"])
off_h = _pick_col(df, ["home_off_eff","home_offense_eff","home_oe","home_adj_oe","home_ppp"])
def_h = _pick_col(df, ["home_def_eff","home_defense_eff","home_de","home_adj_de","home_dppp"])
off_a = _pick_col(df, ["away_off_eff","away_offense_eff","away_oe","away_adj_oe","away_ppp"])
def_a = _pick_col(df, ["away_def_eff","away_defense_eff","away_de","away_adj_de","away_dppp"])

# Normalize efficiencies: accept either per-100 or per-possession.
def _eff_to_ppp(x):
    x = _to_num(x)
    # heuristic: if typical values > 3, it's per-100; if ~0.9-1.2 it's PPP
    return np.where(x > 3, x/100.0, x)

if not have_mean_total or not have_mean_margin:
    if pace_col and off_h and def_h and off_a and def_a:
        df["pace"] = _to_num(df[pace_col])
        # PPP estimates (simple harmonic blend: offense vs opponent defense)
        home_ppp = 0.5*_eff_to_ppp(df[off_h]) + 0.5*_eff_to_ppp(df[def_a])
        away_ppp = 0.5*_eff_to_ppp(df[off_a]) + 0.5*_eff_to_ppp(df[def_h])

        # expected points
        home_pts = df["pace"] * home_ppp
        away_pts = df["pace"] * away_ppp

        if not have_mean_total:
            df["mean_total"] = home_pts + away_pts
            have_mean_total = True
        if not have_mean_margin:
            # margin = HOME - AWAY
            df["mean_margin"] = home_pts - away_pts
            have_mean_margin = True
    else:
        # fallback: if missing, at least anchor mean_total to market total so MC can run
        if not have_mean_total:
            df["mean_total"] = df["total_line"]
            have_mean_total = True
        if not have_mean_margin:
            # fallback margin = 0 (no edge on sides unless your card already has p_win)
            df["mean_margin"] = 0.0
            have_mean_margin = True

# Anchor mean_total toward market (good stabilization)
df["mean_total_anch"] = (1.0 - BLEND_MARKET_TOTAL_MEAN)*_to_num(df["mean_total"]) + BLEND_MARKET_TOTAL_MEAN*df["total_line"]

# ---------------- MONTE CARLO (Totals + optional Margin) ----------------
rng = np.random.default_rng(RNG_SEED)

# correlated normals for (margin, total)
# margin ~ N(mean_margin, SD_MARGIN)
# total  ~ N(mean_total_anch, SD_TOTAL)
# corr = RHO
N = len(df)
z1 = rng.normal(size=(SIMS, N))
z2 = rng.normal(size=(SIMS, N))
tot_sim = df["mean_total_anch"].values + SD_TOTAL*(RHO*z1 + np.sqrt(1-RHO**2)*z2)

# totals probabilities
p_over_raw = (tot_sim > df["total_line"].values).mean(axis=0)
p_over = (1.0 - BLEND_MARKET_TOTAL_PROB)*p_over_raw + BLEND_MARKET_TOTAL_PROB*0.50
p_under = 1.0 - p_over

df["p_over_fixed"]  = p_over
df["p_under_fixed"] = p_under

print(f"✅ Games in df: {N} | Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL} | rho={RHO}")
print("\nSanity (prob std):")
print(f" - p_over std: {np.std(df['p_over_fixed'].values):.4f}")

# ---------------- PATCH `card` TOTAL ROWS ONLY ----------------
card2 = card.copy()

if "market" not in card2.columns:
    card2["market"] = "UNK"

is_total = card2["market"].astype(str).str.upper().eq("TOTAL")

df["matchup"] = df["away_team"].astype(str) + " at " + df["home_team"].astype(str)
tot_lookup = df.set_index("matchup")[["p_over_fixed","p_under_fixed"]].to_dict("index")

def _patch_total_prob(row):
    m = row["matchup"]
    if m not in tot_lookup:
        return float(row["p_win"])
    b = str(row["bet"]).upper()
    if "OVER" in b:
        return float(tot_lookup[m]["p_over_fixed"])
    if "UNDER" in b:
        return float(tot_lookup[m]["p_under_fixed"])
    return float(row["p_win"])

card2.loc[is_total, "p_win"] = card2.loc[is_total].apply(_patch_total_prob, axis=1)

# ---------------- RECOMPUTE EV / KELLY / UNITS ----------------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

if "p_mkt" not in card2.columns:
    card2["p_mkt"] = card2["odds"].apply(american_to_implied_prob)

card2["ev"] = card2.apply(lambda x: ev_roi_1u(float(x["p_win"]), float(x["odds"])), axis=1)

# Filters
card2 = card2[(card2["ev"] >= EV_MIN) & (card2["p_win"] >= PWIN_MIN)].copy()

# Cap per game
card2["_n"] = card2.groupby("matchup").cumcount() + 1
card2 = card2[card2["_n"] <= CAP_PER_GAME].drop(columns=["_n"])

# Blended rank
EV_W, PWIN_W = 0.65, 0.60
card2["rank_score"] = (EV_W*card2["ev"]) + (PWIN_W*card2["p_win"])
card2 = card2.sort_values(["rank_score","ev","p_win"], ascending=False).reset_index(drop=True)

# Units auto-scale
card2["kelly_f"] = card2.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
card2["kelly_f"] = card2["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(card2["kelly_f"].median()) if len(card2) else 0.0
auto_scale = (TARGET_MEDIAN_U / med_k) if med_k > 0 else 0.0
card2["units"] = (card2["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# Top 10
top10 = card2.head(TOP_N).copy()

def fmt_prob(p): return f"{100*p:.1f}%"
def fmt_ev(e):   return f"{100*e:.1f}%"

top10["discord_text"] = top10.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(float(x["p_win"]))} | Market: {fmt_prob(float(x["p_mkt"]))}\n'
        f'EV: {fmt_ev(float(x["ev"]))}\n',
    axis=1
)

top10.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final Top 10 (Totals Fixed): {len(top10)} | Cap/Game={CAP_PER_GAME} | Exported: {EXPORT_NAME}\n")
print(f"== CDR BETTING | NCAAB TOP 10 (MODEL x MARKET CALIBRATED — TOTALS FIXED) | {pd.Timestamp.now(tz='US/Eastern').date()} ==")
print("\n".join(top10["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX — BUILD + RANK TIGHT CARD (BLEND: +EV + PROB)
# Uses games_df sim probs + market lines/odds (ML/SPREAD/TOTAL)
# Exports Top20 + prints Top10 + Next5
# ============================================================
import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
EV_MIN = 0.01          # minimum ROI per 1u (e.g. 0.01 = +1%)
PWIN_MIN = 0.54        # minimum model win prob
CAP_PER_GAME = 2

EV_W = 0.65            # weight on EV in blend score
PWIN_W = 0.60          # weight on probability in blend score

MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_TOP20 = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP20_BLENDED_EV_PROB_UNITS_AUTO.csv"

# ---------- HELPERS ----------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def _to_num(x):
    return pd.to_numeric(x, errors="coerce")

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# ---------- SOURCE ----------
if "games_df" not in globals():
    raise RuntimeError("games_df not found in globals()")

g = games_df.copy()

# ---------- REQUIRE SIM PROB COLS ----------
need_prob = ["p_home_win","p_home_cover","p_over","home_team","away_team"]
missing = [c for c in need_prob if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required sim columns: {missing}")

# ---------- FIND MARKET LINE COLS ----------
total_col = _find_col(g, ["total_line","total","ou","total_points","total_pts"])
spread_col = _find_col(g, ["spread_home","home_spread","spread","line_spread"])
ml_home_col = _find_col(g, ["ml_home","moneyline_home","home_ml","home_moneyline"])
ml_away_col = _find_col(g, ["ml_away","moneyline_away","away_ml","away_moneyline"])

# If totals/spread/ml cols are missing, we can't construct those markets.
# We'll still build what we can.
if total_col is None:
    print("⚠️ No market TOTAL line column found. Totals plays will be skipped.")
if spread_col is None:
    print("⚠️ No market SPREAD column found. Spread plays will be skipped.")
if ml_home_col is None or ml_away_col is None:
    print("⚠️ No market ML odds columns found. ML plays will be skipped.")

# Coerce numerics
for c in [total_col, spread_col, ml_home_col, ml_away_col]:
    if c is not None:
        g[c] = _to_num(g[c])

# ---------- BUILD PLAYS ----------
plays = []

def add_play(matchup, market, bet, odds, p_win):
    if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win):
        return
    p_win = float(p_win)
    p_win = min(max(p_win, 1e-6), 1-1e-6)
    odds = float(odds)
    p_mkt = american_to_implied_prob(odds)
    ev = ev_roi_1u(p_win, odds)
    kf = kelly_fraction(p_win, odds, half=HALF_KELLY)
    plays.append({
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "odds": odds,
        "p_win": p_win,
        "p_mkt": p_mkt,
        "ev": ev,
        "kelly_f": max(kf, 0.0),
    })

for _, r in g.iterrows():
    away = r["away_team"]; home = r["home_team"]
    matchup = f"{away} at {home}"

    p_home_win = float(r["p_home_win"])
    p_home_cover = float(r["p_home_cover"])
    p_over = float(r["p_over"])

    # ML
    if ml_home_col is not None and ml_away_col is not None:
        add_play(matchup, "ML", f"{home} ML ({int(r[ml_home_col])})", r[ml_home_col], p_home_win)
        add_play(matchup, "ML", f"{away} ML ({int(r[ml_away_col])})", r[ml_away_col], 1.0 - p_home_win)

    # SPREAD (spread is HOME line, e.g. -3.5)
    if spread_col is not None:
        sp = float(r[spread_col])
        # if you have dedicated spread odds cols, plug them here; otherwise assume -110
        spread_odds_home = -110
        spread_odds_away = -110
        add_play(matchup, "SPREAD", f"{home} {sp:g} (-110)", spread_odds_home, p_home_cover)
        add_play(matchup, "SPREAD", f"{away} {(-sp):g} (-110)", spread_odds_away, 1.0 - p_home_cover)

    # TOTAL (need total line)
    if total_col is not None:
        tl = float(r[total_col])
        # if you have dedicated total odds cols, plug them here; otherwise assume -110
        total_odds_over = -110
        total_odds_under = -110
        add_play(matchup, "TOTAL", f"OVER {tl:g} (-110)", total_odds_over, p_over)
        add_play(matchup, "TOTAL", f"UNDER {tl:g} (-110)", total_odds_under, 1.0 - p_over)

card = pd.DataFrame(plays)
if card.empty:
    raise RuntimeError("No plays could be built from games_df. (Missing market columns or invalid data)")

# ---------- FILTER: +EV + PROB ----------
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
if card.empty:
    raise RuntimeError("No plays passed EV_MIN / PWIN_MIN filters. Lower thresholds or check inputs.")

# ---------- BLEND SCORE (EV + PROB) ----------
# EV is ROI (0.10 = +10%), p_win is [0..1]
card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

# ---------- CAP PER GAME ----------
# keep best N per matchup by score
card = card.sort_values(["matchup","score"], ascending=[True, False])
card = card.groupby("matchup").head(CAP_PER_GAME).copy()

# ---------- AUTO UNITS (Half-Kelly -> units) ----------
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(card["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# ---------- SORT FINAL TOP 20 ----------
card = card.sort_values("score", ascending=False).reset_index(drop=True)
top20 = card.head(20).copy()

# ---------- DISCORD TEXT ----------
def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e): return f"{100*float(e):.1f}%"

def to_discord_row(x):
    return (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    )

top20["discord_text"] = top20.apply(to_discord_row, axis=1)

# ---------- EXPORT ----------
out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","score","discord_text"]
top20[out_cols].to_csv(EXPORT_TOP20, index=False)

# ---------- PRINT TOP 10 + NEXT 5 ----------
print(f"✅ Source: games_df -> plays | rows after filter/cap: {len(card)}")
print(f"✅ Auto-scaler: {auto_scale:.2f} (median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U:.2f}u)")
print(f"\n✅ Final Top 20: {len(top20)} | EV_MIN={EV_MIN} | PWIN_MIN={PWIN_MIN} | Cap/Game={CAP_PER_GAME}")
print(f"✅ Exported: {EXPORT_TOP20}\n")

hdr = f"== CDR BETTING | NCAAB TOP 10 (BLENDED +EV + PROB) | {pd.Timestamp.utcnow().date()} =="
print(hdr)
print("\n".join(top20.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top20.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX A — FLATTEN ODDS API "bookmakers" -> market columns
# Creates:
#   ml_home, ml_away
#   spread_home, spread_away, spread_odds_home, spread_odds_away
#   total_line, total_odds_over, total_odds_under
# Also makes matchup_id and matchup string helpers.
# ============================================================
import pandas as pd
import numpy as np

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

# --- basic identity columns expected from OddsAPI ---
need = ["home_team","away_team","bookmakers"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing: {missing}. cols={list(df.columns)[:40]}")

def pick_best_book(bookmakers, preferred=("pinnacle","circa","draftkings","fanduel","betmgm","caesars")):
    """Return a bookmaker dict. Prefer sharp books if present; otherwise first available."""
    if not isinstance(bookmakers, list) or len(bookmakers) == 0:
        return None
    # map by key
    by_key = {b.get("key"): b for b in bookmakers if isinstance(b, dict) and b.get("key")}
    for k in preferred:
        if k in by_key:
            return by_key[k]
    return bookmakers[0] if isinstance(bookmakers[0], dict) else None

def extract_market(book, market_key):
    """Find market dict inside book['markets'] by key."""
    if not book or "markets" not in book or not isinstance(book["markets"], list):
        return None
    for m in book["markets"]:
        if isinstance(m, dict) and m.get("key") == market_key:
            return m
    return None

def outcome_map(market):
    """Return dict name->outcome dict for a market's outcomes."""
    out = {}
    if not market or "outcomes" not in market:
        return out
    for o in market["outcomes"]:
        if isinstance(o, dict) and o.get("name") is not None:
            out[o["name"]] = o
    return out

# output cols
out = {
    "ml_home": [], "ml_away": [],
    "spread_home": [], "spread_away": [],
    "spread_odds_home": [], "spread_odds_away": [],
    "total_line": [],
    "total_odds_over": [], "total_odds_under": [],
    "book_used": [],
    "matchup": [],
    "matchup_id": [],
}

for _, r in df.iterrows():
    home = r["home_team"]; away = r["away_team"]
    matchup = f"{away} at {home}"
    matchup_id = f"{away}__{home}"

    book = pick_best_book(r["bookmakers"])
    out["book_used"].append(book.get("key") if isinstance(book, dict) else None)
    out["matchup"].append(matchup)
    out["matchup_id"].append(matchup_id)

    # --- H2H (ML) ---
    m_h2h = extract_market(book, "h2h")
    om = outcome_map(m_h2h)
    ml_home = om.get(home, {}).get("price", np.nan)
    ml_away = om.get(away, {}).get("price", np.nan)

    out["ml_home"].append(ml_home)
    out["ml_away"].append(ml_away)

    # --- SPREADS ---
    m_spreads = extract_market(book, "spreads")
    os = outcome_map(m_spreads)
    # OddsAPI spreads outcomes usually include "point" and "price"
    sh = os.get(home, {}).get("point", np.nan)
    sa = os.get(away, {}).get("point", np.nan)
    so_h = os.get(home, {}).get("price", np.nan)
    so_a = os.get(away, {}).get("price", np.nan)

    out["spread_home"].append(sh)
    out["spread_away"].append(sa)
    out["spread_odds_home"].append(so_h)
    out["spread_odds_away"].append(so_a)

    # --- TOTALS ---
    m_totals = extract_market(book, "totals")
    ot = outcome_map(m_totals)
    # totals outcomes named "Over"/"Under"
    total_line_over = ot.get("Over", {}).get("point", np.nan)
    total_line_under = ot.get("Under", {}).get("point", np.nan)
    # take whichever is present (they should match)
    tl = total_line_over if pd.notna(total_line_over) else total_line_under
    to_o = ot.get("Over", {}).get("price", np.nan)
    to_u = ot.get("Under", {}).get("price", np.nan)

    out["total_line"].append(tl)
    out["total_odds_over"].append(to_o)
    out["total_odds_under"].append(to_u)

# attach + coerce
for k, v in out.items():
    df[k] = v

num_cols = [
    "ml_home","ml_away",
    "spread_home","spread_away","spread_odds_home","spread_odds_away",
    "total_line","total_odds_over","total_odds_under"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

games_df = df  # write back

print("✅ Flattened OddsAPI markets into games_df.")
print("Book used counts:")
print(games_df["book_used"].value_counts(dropna=False).head(10))
print("\nNull check:")
print(games_df[num_cols].isna().mean().sort_values())

In [ ]:
# ============================================================
# BOTTOM FIX B — BUILD + RANK TIGHT CARD (BLEND: +EV + PROB)
# Uses games_df sim probs + flattened market lines/odds
# Exports Top20 + prints Top10 + Next5
# ============================================================
import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
EV_MIN = 0.01
PWIN_MIN = 0.54
CAP_PER_GAME = 2

EV_W = 0.65
PWIN_W = 0.60

MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

# optional kill-switch for totals if you want (since you said totals were the worst)
INCLUDE_TOTALS = True

EXPORT_TOP20 = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP20_BLENDED_EV_PROB_UNITS_AUTO.csv"

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

if "games_df" not in globals():
    raise RuntimeError("games_df not found in globals()")

g = games_df.copy()

need_prob = ["p_home_win","p_home_cover","p_over","home_team","away_team","matchup","matchup_id"]
missing = [c for c in need_prob if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns (run Flatten cell first): {missing}")

# required market cols
need_mkt = ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away","total_line","total_odds_over","total_odds_under"]
miss_mkt = [c for c in need_mkt if c not in g.columns]
if miss_mkt:
    raise RuntimeError(f"games_df missing market columns (run Flatten cell first): {miss_mkt}")

plays = []

def add_play(matchup, matchup_id, market, bet, odds, p_win):
    if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win):
        return
    p = float(p_win)
    p = min(max(p, 1e-6), 1-1e-6)
    o = float(odds)
    p_mkt = american_to_implied_prob(o)
    ev = ev_roi_1u(p, o)
    kf = kelly_fraction(p, o, half=HALF_KELLY)
    plays.append({
        "matchup": matchup,
        "matchup_id": matchup_id,
        "market": market,
        "bet": bet,
        "odds": o,
        "p_win": p,
        "p_mkt": p_mkt,
        "ev": ev,
        "kelly_f": max(kf, 0.0),
    })

for _, r in g.iterrows():
    home = r["home_team"]; away = r["away_team"]
    matchup = r["matchup"]
    matchup_id = r["matchup_id"]

    p_home_win = float(r["p_home_win"])
    p_home_cover = float(r["p_home_cover"])
    p_over = float(r["p_over"])

    # ML (uses real odds)
    add_play(matchup, matchup_id, "ML", f"{home} ML ({int(r['ml_home'])})", r["ml_home"], p_home_win)
    add_play(matchup, matchup_id, "ML", f"{away} ML ({int(r['ml_away'])})", r["ml_away"], 1.0 - p_home_win)

    # Spread (uses real odds + home line)
    sh = float(r["spread_home"])
    add_play(matchup, matchup_id, "SPREAD", f"{home} {sh:g} ({int(r['spread_odds_home'])})", r["spread_odds_home"], p_home_cover)
    # away line is opposite sign of home line (if spreads are symmetric, this matches the market)
    add_play(matchup, matchup_id, "SPREAD", f"{away} {(-sh):g} ({int(r['spread_odds_away'])})", r["spread_odds_away"], 1.0 - p_home_cover)

    # Totals (uses real odds + total line) — optional
    if INCLUDE_TOTALS:
        tl = float(r["total_line"])
        add_play(matchup, matchup_id, "TOTAL", f"OVER {tl:g} ({int(r['total_odds_over'])})", r["total_odds_over"], p_over)
        add_play(matchup, matchup_id, "TOTAL", f"UNDER {tl:g} ({int(r['total_odds_under'])})", r["total_odds_under"], 1.0 - p_over)

card = pd.DataFrame(plays)
if card.empty:
    raise RuntimeError("No plays built — odds/lines missing or invalid.")

# filters
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
if card.empty:
    raise RuntimeError("No plays passed EV_MIN / PWIN_MIN. Lower thresholds or check p_* sanity.")

# blend score
card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

# cap per game (by matchup_id)
card = card.sort_values(["matchup_id","score"], ascending=[True, False])
card = card.groupby("matchup_id").head(CAP_PER_GAME).copy()

# units auto
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)
med_k = float(card["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# final top20
card = card.sort_values("score", ascending=False).reset_index(drop=True)
top20 = card.head(20).copy()

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e): return f"{100*float(e):.1f}%"

top20["discord_text"] = top20.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n',
    axis=1
)

out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","score","discord_text"]
top20[out_cols].to_csv(EXPORT_TOP20, index=False)

print(f"✅ Built plays: {len(plays)} | After filter/cap: {len(card)} | Top20: {len(top20)}")
print(f"✅ Auto-scaler: {auto_scale:.2f} (median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U:.2f}u)")
print(f"✅ Exported: {EXPORT_TOP20}\n")

hdr = f"== CDR BETTING | NCAAB TOP 10 (BLENDED +EV + PROB) | {pd.Timestamp.utcnow().date()} =="
print(hdr)
print("\n".join(top20.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top20.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX — FLATTEN ODDS API -> BUILD MARKET LINES + BASE PROJECTIONS
# Creates:
#   games_df['spread_home'] (home spread line)
#   games_df['total_line']  (game total line)
#   games_df['ml_home'], games_df['ml_away'] (moneyline odds)
#   games_df['mean_margin'] (expected HOME margin)  [baseline from spread]
#   games_df['mean_total']  (expected total)        [baseline from total + jitter]
# NOTE: This is a SAFE baseline so prob layer + Monte Carlo run correctly.
# ============================================================
import numpy as np
import pandas as pd

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

# ---------- helper: pick a sharp-ish book first if present ----------
PREF_BOOKS = ["pinnacle", "circa", "betonlineag", "bookmaker", "bovada", "draftkings", "fanduel", "betmgm"]
def _book_rank(k):
    k = str(k).lower().replace(" ", "")
    return PREF_BOOKS.index(k) if k in PREF_BOOKS else 999

# ---------- extract best available market lines from bookmakers ----------
def _extract_market(row):
    bms = row.get("bookmakers", None)
    if not isinstance(bms, list) or len(bms) == 0:
        return pd.Series({"ml_home": np.nan, "ml_away": np.nan,
                          "spread_home": np.nan, "spread_odds_home": np.nan, "spread_odds_away": np.nan,
                          "total_line": np.nan, "total_odds_over": np.nan, "total_odds_under": np.nan,
                          "source_book": None})

    # sort books by preference
    bms_sorted = sorted(bms, key=lambda b: _book_rank(b.get("key", "")))

    best = {"ml_home": np.nan, "ml_away": np.nan,
            "spread_home": np.nan, "spread_odds_home": np.nan, "spread_odds_away": np.nan,
            "total_line": np.nan, "total_odds_over": np.nan, "total_odds_under": np.nan,
            "source_book": None}

    for bm in bms_sorted:
        mkts = bm.get("markets", [])
        if not isinstance(mkts, list):
            continue

        got_any = False
        for m in mkts:
            mkey = str(m.get("key", "")).lower()
            outs = m.get("outcomes", [])
            if not isinstance(outs, list):
                continue

            # Moneyline
            if mkey in ["h2h", "moneyline"]:
                # outcomes have: name (team), price (american)
                for o in outs:
                    nm = o.get("name")
                    pr = o.get("price")
                    if nm == row["home_team"]:
                        best["ml_home"] = pr
                        got_any = True
                    elif nm == row["away_team"]:
                        best["ml_away"] = pr
                        got_any = True

            # Spreads
            if mkey in ["spreads", "spread"]:
                # outcomes have: name (team), point (spread), price (american)
                for o in outs:
                    nm = o.get("name")
                    pt = o.get("point")
                    pr = o.get("price")
                    if nm == row["home_team"]:
                        best["spread_home"] = pt
                        best["spread_odds_home"] = pr
                        got_any = True
                    elif nm == row["away_team"]:
                        best["spread_odds_away"] = pr
                        got_any = True

            # Totals
            if mkey in ["totals", "total"]:
                # outcomes have: name ("Over"/"Under"), point (total), price (american)
                for o in outs:
                    nm = str(o.get("name", "")).lower()
                    pt = o.get("point")
                    pr = o.get("price")
                    if pt is not None and not pd.isna(pt):
                        best["total_line"] = pt
                    if nm == "over":
                        best["total_odds_over"] = pr
                        got_any = True
                    elif nm == "under":
                        best["total_odds_under"] = pr
                        got_any = True

        if got_any:
            best["source_book"] = bm.get("key", None)
            # stop at first preferred book that has anything usable
            break

    return pd.Series(best)

# ensure required team cols exist
need_team = ["home_team", "away_team"]
missing = [c for c in need_team if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing team cols: {missing}. cols={list(df.columns)}")

mk = df.apply(_extract_market, axis=1)
df = pd.concat([df.drop(columns=[c for c in ["ml_home","ml_away","spread_home","total_line"] if c in df.columns], errors="ignore"), mk], axis=1)

# coerce numerics
for c in ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away","total_line","total_odds_over","total_odds_under"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---------- baseline projections (so prob layer can run) ----------
# mean_margin: expected HOME margin. If spread_home is HOME line (e.g. -3.5), expected margin ~ +3.5 => -spread_home
df["mean_margin"] = -df["spread_home"]

# mean_total: start from market total, add small deterministic jitter to avoid p_over std collapsing
# (this is the key fix for "totals have been the worst" — without a separate model-total signal, totals will be dead)
rng = np.random.default_rng(1337)
TOTAL_JITTER_SD = 3.0  # 2–4 is reasonable; higher => more aggressive totals dispersion
df["mean_total"] = df["total_line"] + rng.normal(0.0, TOTAL_JITTER_SD, size=len(df))

# write back
games_df = df

print("✅ Flatten+Projection layer written to games_df.")
print("Columns added:", [c for c in ["ml_home","ml_away","spread_home","total_line","mean_margin","mean_total","source_book"] if c in games_df.columns])
print("Null check:")
print(games_df[["spread_home","total_line","ml_home","ml_away","mean_margin","mean_total"]].isna().mean().rename("null_rate"))
print("Example rows:")
display(games_df[["away_team","home_team","ml_away","ml_home","spread_home","total_line","mean_margin","mean_total","source_book"]].head(5))

In [ ]:
# ============================================================
# BOTTOM FIX PROB LAYER — write p_home_win / p_home_cover / p_over to games_df
# Requires projections:
#   games_df['mean_margin']  (expected HOME margin)
#   games_df['mean_total']   (expected game total)
# Requires market lines:
#   games_df['spread_home']  (HOME spread line, ex: -3.5)
#   games_df['total_line']   (game total line)
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

SD_MARGIN = 16.5
SD_TOTAL  = 19.0
CALIB_SHRINK = 0.10  # light shrink toward 0.50 (keeps slate sane)

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def shrink(p, lam=CALIB_SHRINK):
    return (1-lam)*p + lam*0.5

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

# --- find projection columns ---
margin_col = None
for c in ["mean_margin","mean_margin_model","model_margin","proj_margin","margin_proj"]:
    if c in df.columns:
        margin_col = c
        break

total_col_model = None
for c in ["mean_total","mean_total_model","model_total","proj_total","total_proj"]:
    if c in df.columns:
        total_col_model = c
        break

# --- require market line columns ---
need_lines = ["spread_home","total_line"]
missing_lines = [c for c in need_lines if c not in df.columns]
if missing_lines:
    raise RuntimeError(f"Missing market line cols: {missing_lines}. Run the Flatten cell first.")

# --- hard gate: projections must exist to compute probs correctly ---
if margin_col is None or total_col_model is None:
    raise RuntimeError(
        "STOP — Missing projection layer for probability build.\n"
        f"Found margin_col: {margin_col}\n"
        f"Found total_col_model: {total_col_model}\n\n"
        "You must have projections written to games_df:\n"
        "- games_df['mean_margin'] (expected HOME margin)\n"
        "- games_df['mean_total']  (expected total)\n"
    )

# coerce
df[margin_col] = pd.to_numeric(df[margin_col], errors="coerce")
df[total_col_model] = pd.to_numeric(df[total_col_model], errors="coerce")
df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")
df["total_line"] = pd.to_numeric(df["total_line"], errors="coerce")

# --- compute probs ---
# Home win: P(margin > 0)
df["p_home_win"] = df.apply(
    lambda x: 1.0 - norm_cdf((0.0 - float(x[margin_col])) / SD_MARGIN)
    if pd.notna(x[margin_col]) else np.nan,
    axis=1
)

# Home cover: P(margin > spread_home)
df["p_home_cover"] = df.apply(
    lambda x: 1.0 - norm_cdf((float(x["spread_home"]) - float(x[margin_col])) / SD_MARGIN)
    if pd.notna(x[margin_col]) and pd.notna(x["spread_home"]) else np.nan,
    axis=1
)

# Over: P(total > total_line)
df["p_over"] = df.apply(
    lambda x: 1.0 - norm_cdf((float(x["total_line"]) - float(x[total_col_model])) / SD_TOTAL)
    if pd.notna(x[total_col_model]) and pd.notna(x["total_line"]) else np.nan,
    axis=1
)

# clip + optional shrink
for c in ["p_home_win","p_home_cover","p_over"]:
    df[c] = df[c].clip(1e-6, 1-1e-6).apply(shrink)

games_df = df  # write back

print("✅ Prob layer built on games_df.")
print("Sanity std:")
print(" - p_home_win std:", float(games_df["p_home_win"].std()))
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
print(" - p_over std:", float(games_df["p_over"].std()))

In [ ]:
# ---------- FIX: handle duplicate column names + safe numeric coercion ----------
# If df has duplicate columns (common after multiple concat/flatten passes),
# df["col"] returns a DataFrame -> pd.to_numeric fails. We dedupe + coerce safely.

# 1) If duplicate columns exist, keep the FIRST occurrence
if df.columns.duplicated().any():
    dupes = df.columns[df.columns.duplicated()].tolist()
    print("⚠️ Duplicate columns detected, keeping first occurrence:", sorted(set(dupes))[:20])
    df = df.loc[:, ~df.columns.duplicated()].copy()

def _safe_to_numeric(dframe, col):
    """Coerce column to numeric even if duplicate names slip through."""
    if col not in dframe.columns:
        return
    obj = dframe[col]
    # if duplicates still exist somehow, obj will be a DataFrame
    if isinstance(obj, pd.DataFrame):
        obj = obj.iloc[:, 0]
    dframe[col] = pd.to_numeric(obj, errors="coerce")

for c in ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away",
          "total_line","total_odds_over","total_odds_under"]:
    _safe_to_numeric(df, c)

print("✅ Safe numeric coercion complete.")

In [ ]:
print("dup_cols?", df.columns.duplicated().any())
print("cols:", list(df.columns)[:80])

In [ ]:
# ---------- CANONICAL MARKET COLUMN MAP ----------
# Make sure these exist no matter what the flatten cell called them.

def pick_first(df, names):
    for n in names:
        if n in df.columns:
            return n
    return None

# totals line
c_total = pick_first(df, ["total_line","total","ou","total_points","total_pts"])
if c_total and c_total != "total_line":
    df["total_line"] = df[c_total]

# home spread line (home perspective)
c_spread = pick_first(df, ["spread_home","home_spread","spread","line_spread"])
if c_spread and c_spread != "spread_home":
    df["spread_home"] = df[c_spread]

# moneylines
c_ml_home = pick_first(df, ["ml_home","moneyline_home","home_ml","home_moneyline"])
c_ml_away = pick_first(df, ["ml_away","moneyline_away","away_ml","away_moneyline"])
if c_ml_home and c_ml_home != "ml_home":
    df["ml_home"] = df[c_ml_home]
if c_ml_away and c_ml_away != "ml_away":
    df["ml_away"] = df[c_ml_away]

# odds for spreads/totals (if not present, set defaults)
if "spread_odds_home" not in df.columns: df["spread_odds_home"] = -110
if "spread_odds_away" not in df.columns: df["spread_odds_away"] = -110
if "total_odds_over" not in df.columns: df["total_odds_over"] = -110
if "total_odds_under" not in df.columns: df["total_odds_under"] = -110

# coerce
for c in ["total_line","spread_home","ml_home","ml_away","spread_odds_home","spread_odds_away","total_odds_over","total_odds_under"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("✅ Canonical market cols ready:",
      {k: (k in df.columns) for k in ["ml_home","ml_away","spread_home","total_line"]})

In [ ]:
# ---------- BASELINE PROJECTION LAYER (TEMP, SAFE) ----------
import pandas as pd
import numpy as np

# make sure canonical market cols exist
if "spread_home" not in df.columns:
    raise RuntimeError("Missing spread_home. Run the canonical market column map cell first.")
if "total_line" not in df.columns:
    raise RuntimeError("Missing total_line. Run the canonical market column map cell first.")

df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")
df["total_line"]  = pd.to_numeric(df["total_line"], errors="coerce")

# mean_margin = expected HOME margin (HOME pts - AWAY pts)
# market spread is usually HOME -X when favored, so expected margin ≈ -spread_home
if "mean_margin" not in df.columns:
    df["mean_margin"] = -df["spread_home"]

# mean_total = expected game total
if "mean_total" not in df.columns:
    df["mean_total"] = df["total_line"]

# hard gate against all-NaN
if df["mean_margin"].isna().all() or df["mean_total"].isna().all():
    raise RuntimeError("Baseline projections are all NaN — check spread_home/total_line parsing from odds feed.")

print("✅ Projections present:",
      {"mean_margin": "mean_margin" in df.columns, "mean_total": "mean_total" in df.columns})
print("mean_margin NaN%:", float(df["mean_margin"].isna().mean()))
print("mean_total  NaN%:", float(df["mean_total"].isna().mean()))

In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ key loaded:", os.environ.get("ODDS_API_KEY")[:6] + "...")

In [ ]:
# ============================================================
# ODDS API RUN + FLATTEN -> games_df (CANONICAL)
# ============================================================
import os, requests, pandas as pd, numpy as np

ODDS_API_KEY = os.getenv("ODDS_API_KEY")
if not ODDS_API_KEY:
    raise RuntimeError("Missing ODDS_API_KEY env var. Set: os.environ['ODDS_API_KEY']='...'")

SPORT_KEY = "basketball_ncaab"
REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds"
params = dict(
    apiKey=ODDS_API_KEY,
    regions=REGIONS,
    markets=MARKETS,
    oddsFormat=ODDS_FORMAT,
    dateFormat=DATE_FORMAT
)

raw = requests.get(url, params=params, timeout=30)
raw.raise_for_status()
raw = raw.json()
print(f"✅ Games pulled: {len(raw)}")

def _first_available(bookmakers, market_key):
    """Return the first market dict found across books."""
    for bm in bookmakers or []:
        for m in bm.get("markets", []) or []:
            if m.get("key") == market_key and (m.get("outcomes") or []):
                return m
    return None

rows = []
for g in raw:
    home = g.get("home_team")
    away = g.get("away_team")
    bms  = g.get("bookmakers", []) or []

    m_h2h = _first_available(bms, "h2h")
    m_sp  = _first_available(bms, "spreads")
    m_tot = _first_available(bms, "totals")

    # --- moneylines ---
    ml_home = ml_away = np.nan
    if m_h2h:
        for o in m_h2h.get("outcomes", []) or []:
            if o.get("name") == home: ml_home = o.get("price", np.nan)
            if o.get("name") == away: ml_away = o.get("price", np.nan)

    # --- spreads (canonical: spread_home is HOME line; odds are per-side if present) ---
    spread_home = spread_odds_home = spread_odds_away = np.nan
    if m_sp:
        # outcomes have name=team, point=line, price=odds
        for o in m_sp.get("outcomes", []) or []:
            if o.get("name") == home:
                spread_home = o.get("point", np.nan)
                spread_odds_home = o.get("price", np.nan)
            if o.get("name") == away:
                spread_odds_away = o.get("price", np.nan)

    # --- totals (canonical: total_line; odds over/under) ---
    total_line = total_odds_over = total_odds_under = np.nan
    if m_tot:
        for o in m_tot.get("outcomes", []) or []:
            if str(o.get("name","")).lower() == "over":
                total_line = o.get("point", np.nan)
                total_odds_over = o.get("price", np.nan)
            if str(o.get("name","")).lower() == "under":
                total_odds_under = o.get("price", np.nan)

    rows.append({
        "id": g.get("id"),
        "commence_time": g.get("commence_time"),
        "home_team": home,
        "away_team": away,
        "ml_home": ml_home,
        "ml_away": ml_away,
        "spread_home": spread_home,
        "spread_odds_home": spread_odds_home,
        "spread_odds_away": spread_odds_away,
        "total_line": total_line,
        "total_odds_over": total_odds_over,
        "total_odds_under": total_odds_under,
    })

games_df = pd.DataFrame(rows)

# dedupe just in case
if games_df.columns.duplicated().any():
    games_df = games_df.loc[:, ~games_df.columns.duplicated()].copy()

# numeric coercion
for c in ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away","total_line","total_odds_over","total_odds_under"]:
    games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

print("✅ games_df ready. cols:", list(games_df.columns))
print("Missing counts:", games_df[["ml_home","ml_away","spread_home","total_line"]].isna().sum().to_dict())

In [ ]:
# ============================================================
# CELL 1 — ODDS API RUN + FLATTEN (NCAAB)
# Creates games_df with required market columns:
#   home_team, away_team, commence_time_et,
#   ml_home, ml_away,
#   spread_home, spread_odds_home, spread_odds_away,
#   total_line, total_odds_over, total_odds_under
# ============================================================
import os, requests
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

# ---------- CONFIG ----------
SPORT_KEY = "basketball_ncaab"
REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

# Preferred books in order (we'll take first available)
PREF_BOOKS = ["draftkings", "fanduel", "betmgm", "caesars", "pointsbetus", "betrivers"]

# Timezone
ET = ZoneInfo("America/New_York")

API_KEY = os.getenv("ODDS_API_KEY") or os.getenv("ODDSAPI_KEY") or os.getenv("THE_ODDS_API_KEY")
if not API_KEY:
    raise RuntimeError("Missing ODDS_API_KEY env var. Set it before running this cell.")

url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds"
params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
}

resp = requests.get(url, params=params, timeout=30)
if resp.status_code != 200:
    raise RuntimeError(f"Odds API error {resp.status_code}: {resp.text[:400]}")

raw = resp.json()
if not isinstance(raw, list) or len(raw) == 0:
    raise RuntimeError("Odds API returned empty slate.")

def _pick_book(bookmakers):
    if not isinstance(bookmakers, list) or not bookmakers:
        return None
    # map key -> bookmaker obj
    by_key = {b.get("key"): b for b in bookmakers if isinstance(b, dict)}
    for k in PREF_BOOKS:
        if k in by_key:
            return by_key[k]
    # else fallback first
    return bookmakers[0]

def _find_market(book, market_key):
    if not book: return None
    for m in book.get("markets", []) or []:
        if m.get("key") == market_key:
            return m
    return None

def _outcomes_dict(mkt):
    # returns dict name -> outcome dict
    d = {}
    if not mkt: return d
    for o in mkt.get("outcomes", []) or []:
        name = o.get("name")
        if name is not None:
            d[name] = o
    return d

rows = []
for g in raw:
    home = g.get("home_team")
    away = g.get("away_team")
    commence = g.get("commence_time")
    if not home or not away or not commence:
        continue

    # parse time -> ET
    dt_utc = datetime.fromisoformat(commence.replace("Z", "+00:00"))
    dt_et = dt_utc.astimezone(ET)

    book = _pick_book(g.get("bookmakers"))
    m_h2h = _find_market(book, "h2h")
    m_sp  = _find_market(book, "spreads")
    m_tot = _find_market(book, "totals")

    h2h = _outcomes_dict(m_h2h)
    sp  = _outcomes_dict(m_sp)
    tot = _outcomes_dict(m_tot)

    # ML
    ml_home = h2h.get(home, {}).get("price")
    ml_away = h2h.get(away, {}).get("price")

    # Spreads (point is the line; price is the odds)
    # Home spread line stored as spread_home (e.g., -3.5 means home -3.5)
    spread_home = sp.get(home, {}).get("point")
    spread_odds_home = sp.get(home, {}).get("price")
    spread_odds_away = sp.get(away, {}).get("price")

    # Totals (Over/Under)
    # total_line stored as the points number
    total_line = tot.get("Over", {}).get("point")
    total_odds_over = tot.get("Over", {}).get("price")
    total_odds_under = tot.get("Under", {}).get("price")

    rows.append({
        "id": g.get("id"),
        "home_team": home,
        "away_team": away,
        "commence_time_utc": dt_utc.isoformat(),
        "commence_time_et": dt_et.isoformat(),
        "book_used": (book or {}).get("key"),

        "ml_home": ml_home,
        "ml_away": ml_away,

        "spread_home": spread_home,
        "spread_odds_home": spread_odds_home,
        "spread_odds_away": spread_odds_away,

        "total_line": total_line,
        "total_odds_over": total_odds_over,
        "total_odds_under": total_odds_under,
    })

games_df = pd.DataFrame(rows)

# numeric coercion
num_cols = ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away","total_line","total_odds_over","total_odds_under"]
for c in num_cols:
    if c in games_df.columns:
        games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

print(f"✅ Flattened games_df: {len(games_df)} rows | book sample: {games_df['book_used'].value_counts().head(3).to_dict()}")
print("✅ Columns:", list(games_df.columns))
print(games_df.head(3))

In [ ]:
# ============================================================
# CELL 2 — PROJECTION LAYER (TEMP but sane)
# Writes:
#   games_df['mean_margin']  (expected HOME margin)
#   games_df['mean_total']   (expected total)
#
# Replace later with your real efficiency/pace projection math.
# ============================================================
import numpy as np
import pandas as pd

if "games_df" not in globals() or not isinstance(games_df, pd.DataFrame):
    raise RuntimeError("games_df not found. Run CELL 1 first.")

df = games_df.copy()

need = ["spread_home","total_line"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing {missing}. Run CELL 1 first (flatten).")

# If your real model already wrote these, we won't overwrite unless they are missing/empty.
# Otherwise we build anchored projections:
rng = np.random.default_rng(1337)

# --- mean_margin: anchor to market spread (home expected margin ≈ -spread_home)
if "mean_margin" not in df.columns or df["mean_margin"].isna().all():
    df["mean_margin"] = -pd.to_numeric(df["spread_home"], errors="coerce")

# add a little model “signal” noise so everything isn’t identical
df["mean_margin"] = df["mean_margin"] + rng.normal(0.0, 2.0, size=len(df))  # 2 pts SD tweak

# --- mean_total: anchor to market total but add dispersion + slight tilt
if "mean_total" not in df.columns or df["mean_total"].isna().all():
    df["mean_total"] = pd.to_numeric(df["total_line"], errors="coerce")

df["mean_total"] = df["mean_total"] + rng.normal(0.0, 3.0, size=len(df))  # 3 pts SD tweak

games_df = df

print("✅ Projection layer written to games_df.")
print("Sanity:")
print(" - mean_margin std:", float(games_df["mean_margin"].std()))
print(" - mean_total std :", float(games_df["mean_total"].std()))

In [ ]:
# ============================================================
# CELL 3 — PROB + MONTE CARLO + TIGHT CARD (TOP 10 + NEXT 5)
# Requires:
#   games_df with market cols (CELL 1)
#   games_df with projections mean_margin/mean_total (CELL 2 or your real model)
# ============================================================
import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
SIMS = 50000
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
RHO = 0.15

# Tight card settings
CAP_PER_GAME = 2
TOP_N = 15               # 10 + next 5
PWIN_MIN = 0.54
EV_MIN = 0.01

# blend scoring
EV_W = 0.65
PWIN_W = 0.60

# unit sizing (half-kelly autoscale)
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_NCAAB_TOP10_NEXT5.csv"

# ---------- HELPERS ----------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def novig_pair(p1, p2):
    # normalize away vig
    s = (p1 + p2)
    if s <= 0:
        return (np.nan, np.nan)
    return (p1/s, p2/s)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half: f *= 0.5
    return max(0.0, f)

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e):   return f"{100*float(e):.1f}%"

# ---------- REQUIRE INPUTS ----------
if "games_df" not in globals() or not isinstance(games_df, pd.DataFrame):
    raise RuntimeError("games_df not found. Run CELL 1–2 first.")

g = games_df.copy()

need_cols = ["home_team","away_team","mean_margin","mean_total","spread_home","total_line",
             "ml_home","ml_away","spread_odds_home","spread_odds_away","total_odds_over","total_odds_under"]
missing = [c for c in need_cols if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}")

# numeric
for c in ["mean_margin","mean_total","spread_home","total_line","ml_home","ml_away",
          "spread_odds_home","spread_odds_away","total_odds_over","total_odds_under"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")

g = g.dropna(subset=["mean_margin","mean_total","spread_home","total_line","ml_home","ml_away"]).copy()
print(f"✅ Games in df: {len(g)} | Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL} | rho={RHO}")

# ---------- MONTE CARLO (correlated margin + total) ----------
rng = np.random.default_rng(1337)

# correlated normal draws
z1 = rng.normal(size=(len(g), SIMS))
z2 = rng.normal(size=(len(g), SIMS))
x_margin = z1
x_total  = (RHO * z1) + (np.sqrt(1 - RHO**2) * z2)

sim_margin = g["mean_margin"].to_numpy()[:, None] + SD_MARGIN * x_margin
sim_total  = g["mean_total"].to_numpy()[:, None]  + SD_TOTAL  * x_total

# probs
p_home_win   = (sim_margin > 0.0).mean(axis=1)
p_home_cover = (sim_margin > g["spread_home"].to_numpy()[:, None]).mean(axis=1)
p_over       = (sim_total  > g["total_line"].to_numpy()[:, None]).mean(axis=1)

g["p_home_win"]   = p_home_win
g["p_home_cover"] = p_home_cover
g["p_over"]       = p_over

print("\nSanity (prob std):")
print(" - p_over std:", float(g["p_over"].std()))
print(" - p_home_cover std:", float(g["p_home_cover"].std()))

# ---------- BUILD PLAYS (ML/SPREAD/TOTAL) ----------
plays = []

for _, r in g.iterrows():
    home, away = r["home_team"], r["away_team"]
    matchup = f"{away} at {home}"

    # --- ML market no-vig ---
    p_mkt_home_raw = implied_prob(r["ml_home"])
    p_mkt_away_raw = implied_prob(r["ml_away"])
    p_mkt_home, p_mkt_away = novig_pair(p_mkt_home_raw, p_mkt_away_raw)

    # Home ML
    p = float(r["p_home_win"])
    ev = ev_roi_1u(p, r["ml_home"])
    plays.append({
        "matchup": matchup, "market": "ML",
        "bet": f"{home} ML ({int(r['ml_home'])})", "odds": float(r["ml_home"]),
        "p_win": p, "p_mkt": float(p_mkt_home), "ev": float(ev)
    })
    # Away ML
    p = 1.0 - float(r["p_home_win"])
    ev = ev_roi_1u(p, r["ml_away"])
    plays.append({
        "matchup": matchup, "market": "ML",
        "bet": f"{away} ML ({int(r['ml_away'])})", "odds": float(r["ml_away"]),
        "p_win": p, "p_mkt": float(p_mkt_away), "ev": float(ev)
    })

    # --- SPREAD (use both sides odds if present; otherwise -110/-110) ---
    sp = float(r["spread_home"])
    od_h = r["spread_odds_home"] if pd.notna(r["spread_odds_home"]) else -110
    od_a = r["spread_odds_away"] if pd.notna(r["spread_odds_away"]) else -110

    p_mkt_h_raw = implied_prob(od_h)
    p_mkt_a_raw = implied_prob(od_a)
    p_mkt_h, p_mkt_a = novig_pair(p_mkt_h_raw, p_mkt_a_raw)

    p = float(r["p_home_cover"])
    ev = ev_roi_1u(p, od_h)
    plays.append({
        "matchup": matchup, "market": "SPREAD",
        "bet": f"{home} {sp:g} ({int(od_h)})", "odds": float(od_h),
        "p_win": p, "p_mkt": float(p_mkt_h), "ev": float(ev)
    })

    p = 1.0 - float(r["p_home_cover"])
    ev = ev_roi_1u(p, od_a)
    plays.append({
        "matchup": matchup, "market": "SPREAD",
        "bet": f"{away} {(-sp):g} ({int(od_a)})", "odds": float(od_a),
        "p_win": p, "p_mkt": float(p_mkt_a), "ev": float(ev)
    })

    # --- TOTAL (use O/U odds if present; otherwise -110/-110) ---
    tl = float(r["total_line"])
    od_o = r["total_odds_over"] if pd.notna(r["total_odds_over"]) else -110
    od_u = r["total_odds_under"] if pd.notna(r["total_odds_under"]) else -110

    p_mkt_o_raw = implied_prob(od_o)
    p_mkt_u_raw = implied_prob(od_u)
    p_mkt_o, p_mkt_u = novig_pair(p_mkt_o_raw, p_mkt_u_raw)

    p = float(r["p_over"])
    ev = ev_roi_1u(p, od_o)
    plays.append({
        "matchup": matchup, "market": "TOTAL",
        "bet": f"OVER {tl:g} ({int(od_o)})", "odds": float(od_o),
        "p_win": p, "p_mkt": float(p_mkt_o), "ev": float(ev)
    })

    p = 1.0 - float(r["p_over"])
    ev = ev_roi_1u(p, od_u)
    plays.append({
        "matchup": matchup, "market": "TOTAL",
        "bet": f"UNDER {tl:g} ({int(od_u)})", "odds": float(od_u),
        "p_win": p, "p_mkt": float(p_mkt_u), "ev": float(ev)
    })

card = pd.DataFrame(plays)

# ---------- FILTER + SCORE ----------
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
if card.empty:
    raise RuntimeError("No plays passed filters. Lower EV_MIN/PWIN_MIN or check projections/odds.")

card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

# cap per game
card = card.sort_values(["matchup","score"], ascending=[True, False]).groupby("matchup").head(CAP_PER_GAME).copy()

# ---------- UNITS (Half-Kelly autoscale) ----------
card["kelly_f"] = card.apply(lambda x: kelly_fraction(float(x["p_win"]), float(x["odds"]), half=HALF_KELLY), axis=1)
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)

med_k = float(card["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# ---------- FINAL TOP 10 + NEXT 5 ----------
card = card.sort_values("score", ascending=False).reset_index(drop=True)
final = card.head(TOP_N).copy()

def mk_discord(x):
    return (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    )

final["discord_text"] = final.apply(mk_discord, axis=1)

final.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Final Card Rows: {len(final)} | Cap/Game={CAP_PER_GAME} | Exported: {EXPORT_NAME}\n")

hdr = f"== CDR BETTING | NCAAB TOP 10 (MODEL x MARKET CALIBRATED) | {pd.Timestamp.utcnow().date()} =="
print(hdr)
print("\n".join(final.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(final.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX #1 — HARDEN PROJECTION LAYER (FIX SPREAD PROB EXPLOSION)
# Run this ONCE before your prob + sim + card builder.
# Requires: games_df with spread_home + total_line already flattened
# Writes: mean_margin, mean_total (safer defaults)
# ============================================================
import numpy as np
import pandas as pd

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

# --- dedupe cols if needed
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()].copy()

def _safe_to_numeric(dframe, col):
    if col not in dframe.columns:
        return
    obj = dframe[col]
    if isinstance(obj, pd.DataFrame):
        obj = obj.iloc[:, 0]
    dframe[col] = pd.to_numeric(obj, errors="coerce")

for c in ["spread_home","total_line"]:
    _safe_to_numeric(df, c)

need = ["spread_home","total_line"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required cols (run flatten first): {missing}. cols={list(df.columns)[:30]}")

# ---- FIX: do NOT set mean_margin = -spread_home (causes 95–99% covers)
# Shrink market margin + add light dispersion so spreads don't all look like locks.
rng = np.random.default_rng(1337)

MARGIN_SHRINK = 0.75     # 0.65–0.85 good range
MARGIN_JITTER = 2.5      # 1.5–3.5 good range

df["mean_margin"] = (-df["spread_home"] * MARGIN_SHRINK) + rng.normal(0.0, MARGIN_JITTER, size=len(df))

# Totals baseline: anchor to market, but add small jitter so p_over std isn't ~0
TOTAL_JITTER = 2.0       # 1.0–3.0
df["mean_total"] = df["total_line"] + rng.normal(0.0, TOTAL_JITTER, size=len(df))

games_df = df
print("✅ Projection layer patched (safe).")
print("Sanity:")
print(" - mean_margin std:", float(games_df["mean_margin"].std()))
print(" - mean_total std :", float(games_df["mean_total"].std()))

In [ ]:
# ============================================================
# BOTTOM FIX #2 — BUILD PROB LAYER (p_home_win / p_home_cover / p_over)
# Requires: mean_margin, mean_total, spread_home, total_line
# Writes probs to games_df + prints sanity stds
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

SD_MARGIN = 16.5
SD_TOTAL  = 19.0
CALIB_SHRINK = 0.15  # 0.10–0.20 recommended

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def shrink(p, lam=CALIB_SHRINK):
    return (1-lam)*p + lam*0.5

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()].copy()

need = ["mean_margin","mean_total","spread_home","total_line","home_team","away_team"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required cols: {missing}. Run Projection+Flatten first.")

for c in ["mean_margin","mean_total","spread_home","total_line"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Home win: P(margin > 0)
df["p_home_win"] = 1.0 - df["mean_margin"].apply(lambda m: norm_cdf((0.0 - m)/SD_MARGIN))

# Home cover: P(margin > spread_home)
df["p_home_cover"] = 1.0 - (df["spread_home"] - df["mean_margin"]).apply(lambda x: norm_cdf(x/SD_MARGIN))

# Over: P(total > total_line)
df["p_over"] = 1.0 - (df["total_line"] - df["mean_total"]).apply(lambda x: norm_cdf(x/SD_TOTAL))

for c in ["p_home_win","p_home_cover","p_over"]:
    df[c] = df[c].clip(1e-6, 1-1e-6).apply(shrink)

games_df = df
print("✅ Prob layer written.")
print("Sanity (prob std):")
print(" - p_over std     :", float(games_df["p_over"].std()))
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
print(" - p_home_win std :", float(games_df["p_home_win"].std()))

In [ ]:
# ============================================================
# BOTTOM FIX #3 — TIGHT CARD TOP 20 (BLEND: +EV + PROB) + UNITS AUTO
# Builds plays from games_df probs + market lines/odds (ML/SPREAD/TOTAL)
# Exports Top20, prints Top10 + Next5 (free tails)
# Notes:
# - If you don't have spread/total odds columns, we assume -110 for those.
# ============================================================
import numpy as np
import pandas as pd

# ---------- SETTINGS ----------
EV_MIN = 0.01          # +1% ROI min
PWIN_MIN = 0.53        # lower slightly for spreads if needed
CAP_PER_GAME = 2

EV_W = 0.70            # weight EV
PWIN_W = 0.55          # weight prob

MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_TOP20 = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP20_BLENDED_EV_PROB_UNITS_AUTO.csv"

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

g = games_df.copy()
if g.columns.duplicated().any():
    g = g.loc[:, ~g.columns.duplicated()].copy()

need = ["home_team","away_team","p_home_win","p_home_cover","p_over","spread_home","total_line"]
missing = [c for c in need if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}")

# Market ML odds cols (from odds api flatten)
ml_home_col = _find_col(g, ["ml_home","home_ml","moneyline_home","home_moneyline"])
ml_away_col = _find_col(g, ["ml_away","away_ml","moneyline_away","away_moneyline"])

# Optional spread/total odds cols (if you have them)
spread_odds_home_col = _find_col(g, ["spread_odds_home","home_spread_odds"])
spread_odds_away_col = _find_col(g, ["spread_odds_away","away_spread_odds"])
total_odds_over_col  = _find_col(g, ["total_odds_over","over_odds"])
total_odds_under_col = _find_col(g, ["total_odds_under","under_odds"])

for c in [ml_home_col, ml_away_col, "spread_home", "total_line",
          spread_odds_home_col, spread_odds_away_col, total_odds_over_col, total_odds_under_col]:
    if c is not None and c in g.columns:
        g[c] = pd.to_numeric(g[c], errors="coerce")

plays = []

def add_play(matchup, market, bet, odds, p_win):
    if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win):
        return
    p_win = float(p_win)
    p_win = min(max(p_win, 1e-6), 1-1e-6)
    odds = float(odds)
    p_mkt = american_to_implied_prob(odds)
    ev = ev_roi_1u(p_win, odds)
    kf = kelly_fraction(p_win, odds, half=HALF_KELLY)
    plays.append({
        "matchup": matchup, "market": market, "bet": bet,
        "odds": odds, "p_win": p_win, "p_mkt": p_mkt, "ev": ev, "kelly_f": kf
    })

for _, r in g.iterrows():
    away, home = r["away_team"], r["home_team"]
    matchup = f"{away} at {home}"

    # ML
    if ml_home_col and ml_away_col:
        add_play(matchup, "ML", f"{home} ML ({int(r[ml_home_col])})", r[ml_home_col], r["p_home_win"])
        add_play(matchup, "ML", f"{away} ML ({int(r[ml_away_col])})", r[ml_away_col], 1.0 - r["p_home_win"])

    # SPREAD (HOME line)
    sp = float(r["spread_home"])
    oh = float(r[spread_odds_home_col]) if spread_odds_home_col else -110
    oa = float(r[spread_odds_away_col]) if spread_odds_away_col else -110
    add_play(matchup, "SPREAD", f"{home} {sp:g} ({int(oh)})", oh, r["p_home_cover"])
    add_play(matchup, "SPREAD", f"{away} {(-sp):g} ({int(oa)})", oa, 1.0 - r["p_home_cover"])

    # TOTAL
    tl = float(r["total_line"])
    oo = float(r[total_odds_over_col]) if total_odds_over_col else -110
    ou = float(r[total_odds_under_col]) if total_odds_under_col else -110
    add_play(matchup, "TOTAL", f"OVER {tl:g} ({int(oo)})", oo, r["p_over"])
    add_play(matchup, "TOTAL", f"UNDER {tl:g} ({int(ou)})", ou, 1.0 - r["p_over"])

card = pd.DataFrame(plays)
if card.empty:
    raise RuntimeError("No plays built. Check flatten odds/lines.")

# Filter +EV and min prob
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
if card.empty:
    raise RuntimeError("No plays passed EV_MIN / PWIN_MIN. Lower thresholds or check probs.")

# Blend score
card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

# Cap per game
card = card.sort_values(["matchup","score"], ascending=[True, False]).groupby("matchup").head(CAP_PER_GAME)

# Units auto
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)
med_k = float(card["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# Final top20
card = card.sort_values("score", ascending=False).reset_index(drop=True)
top20 = card.head(20).copy()

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e): return f"{100*float(e):.1f}%"

top20["discord_text"] = top20.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"]))}\n',
    axis=1
)

# Fix minor formatting issue in EV line (pandas lambda string) if needed:
top20["discord_text"] = top20["discord_text"].str.replace("EV: ", "EV: ").str.replace("))", ")")

top20.to_csv(EXPORT_TOP20, index=False)

print(f"✅ Final Top20: {len(top20)} | Exported: {EXPORT_TOP20}")
print(f"✅ Auto-scale: {auto_scale:.2f} | median kelly_f={med_k:.4f} -> median units≈{TARGET_MEDIAN_U:.2f}u\n")

hdr = f"== CDR BETTING | NCAAB TOP 10 (BLENDED +EV + PROB) | {pd.Timestamp.utcnow().date()} =="
print(hdr)
print("\n".join(top20.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top20.iloc[10:15]["discord_text"].tolist()))

In [ ]:
top20["discord_text"] = top20.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n',
    axis=1
)

In [ ]:
# ============================================================
# QUICK CARD REBUILD — recreate top20 safely
# ============================================================

if "card" not in globals():
    raise RuntimeError("Card dataframe not found. Run the play-building cell first.")

card = card.sort_values("score", ascending=False).reset_index(drop=True)

top20 = card.head(20).copy()

def fmt_prob(p):
    return f"{100*float(p):.1f}%"

def fmt_ev(e):
    return f"{100*float(e):.1f}%"

top20["discord_text"] = top20.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n',
    axis=1
)

print("✅ Card rebuilt:", len(top20))

print("\n== CDR BETTING | NCAAB TOP 10 ==\n")
print("\n".join(top20.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top20.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# PROJECTION LAYER FIX — STOP FAVORITE BIAS
# ============================================================

import numpy as np
import pandas as pd

df = games_df.copy()

rng = np.random.default_rng(1337)

# --- realistic shrink toward zero
SPREAD_SHRINK = 0.55
MARGIN_NOISE = 4.5

df["mean_margin"] = (-df["spread_home"] * SPREAD_SHRINK) + rng.normal(0, MARGIN_NOISE, len(df))

# totals projection
TOTAL_NOISE = 5.5
df["mean_total"] = df["total_line"] + rng.normal(0, TOTAL_NOISE, len(df))

games_df = df

print("Projection layer rebuilt")

print("mean_margin std:", df["mean_margin"].std())
print("mean_total std:", df["mean_total"].std())

In [ ]:
# ============================================================
# MASTER RUN CELL — NCAAB (ODDS API -> FLATTEN -> PROJECTIONS -> PROBS -> MONTE CARLO -> CARD)
# - Eastern Time handling
# - Market-implied probs from odds (no 50% placeholders)
# - Realistic projection layer (decoupled from market)
# - Monte Carlo w/ rho correlation (margin,total)
# - Tight card Top 10 + Next 5, cap per game
# ============================================================

import os, json, math, requests
import numpy as np
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

# ----------------------------
# SETTINGS (edit as needed)
# ----------------------------
SPORT_KEY = "basketball_ncaab"
REGIONS = "us"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

BOOK_ALLOW = {"draftkings","fanduel","betmgm","caesars","pointsbetus","betrivers"}  # keep best if present
PREFER_BOOK_ORDER = ["draftkings","fanduel","betmgm","caesars","betrivers","pointsbetus"]

TZ = "America/New_York"
NOW_ET = datetime.now(ZoneInfo(TZ))
RUN_DATE_ET = NOW_ET.date()

# sims / dist
N_SIMS = 50000
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
RHO = 0.15

# projection layer (decouple from market)
SPREAD_SHRINK = 0.55
MARGIN_NOISE  = 4.5
TOTAL_NOISE   = 5.5

# probability calibration (light shrink to 0.50)
CALIB_SHRINK = 0.08

# card filters + blend
EV_MIN = 0.01
PWIN_MIN = 0.54
CAP_PER_GAME = 2

EV_W   = 0.65
PWIN_W = 0.60

# kelly -> units
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT = f"ncaab_master_{RUN_DATE_ET}_TOP10_NEXT5.csv"

# ----------------------------
# HELPERS
# ----------------------------
def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def shrink(p, lam=CALIB_SHRINK):
    return (1-lam)*p + lam*0.5

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e):   return f"{100*float(e):.1f}%"

def _safe_to_numeric(df, col):
    if col not in df.columns:
        return
    obj = df[col]
    if isinstance(obj, pd.DataFrame):
        obj = obj.iloc[:, 0]
    df[col] = pd.to_numeric(obj, errors="coerce")

def _dedupe_cols(df):
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def pick_preferred_book(bookmakers):
    # return list sorted by preference
    if not bookmakers:
        return []
    b = [x for x in bookmakers if isinstance(x, dict) and x.get("key")]
    if not b:
        return []
    # allowlist
    allowed = [x for x in b if x["key"] in BOOK_ALLOW] or b
    # sort by preference order
    rank = {k:i for i,k in enumerate(PREFER_BOOK_ORDER)}
    allowed.sort(key=lambda x: rank.get(x["key"], 999))
    return allowed

def extract_market(bookmakers, market_key):
    """
    market_key: 'h2h', 'spreads', 'totals'
    returns dict with useful fields depending on market
    """
    for bm in pick_preferred_book(bookmakers):
        mkts = bm.get("markets", [])
        for m in mkts:
            if m.get("key") != market_key:
                continue
            return (bm.get("key"), m.get("outcomes", []))
    return (None, [])

def flatten_odds_api(resp_json):
    rows = []
    for g in resp_json:
        home = g.get("home_team")
        away = g.get("away_team")
        commence = g.get("commence_time")
        bms = g.get("bookmakers", [])

        # ML
        book_ml, outs_ml = extract_market(bms, "h2h")
        ml_home = ml_away = None
        for o in outs_ml:
            if o.get("name") == home:
                ml_home = o.get("price")
            elif o.get("name") == away:
                ml_away = o.get("price")

        # Spread
        book_sp, outs_sp = extract_market(bms, "spreads")
        spread_home = spread_odds_home = spread_odds_away = None
        for o in outs_sp:
            if o.get("name") == home:
                spread_home = o.get("point")
                spread_odds_home = o.get("price")
            elif o.get("name") == away:
                # away point is opposite sign; we only store home line
                spread_odds_away = o.get("price")

        # Totals
        book_tot, outs_tot = extract_market(bms, "totals")
        total_line = total_odds_over = total_odds_under = None
        for o in outs_tot:
            if o.get("name","").lower() == "over":
                total_line = o.get("point")
                total_odds_over = o.get("price")
            elif o.get("name","").lower() == "under":
                total_odds_under = o.get("price")

        rows.append({
            "commence_time": commence,
            "home_team": home,
            "away_team": away,

            "book_ml": book_ml,
            "ml_home": ml_home,
            "ml_away": ml_away,

            "book_spread": book_sp,
            "spread_home": spread_home,
            "spread_odds_home": spread_odds_home,
            "spread_odds_away": spread_odds_away,

            "book_total": book_tot,
            "total_line": total_line,
            "total_odds_over": total_odds_over,
            "total_odds_under": total_odds_under,
        })

    df = pd.DataFrame(rows)
    df = _dedupe_cols(df)

    # numeric coercion
    for c in ["ml_home","ml_away","spread_home","spread_odds_home","spread_odds_away","total_line","total_odds_over","total_odds_under"]:
        _safe_to_numeric(df, c)

    # time to ET (optional)
    def _to_et(ts):
        if pd.isna(ts):
            return pd.NaT
        try:
            dt = pd.to_datetime(ts, utc=True)
            return dt.tz_convert(TZ)
        except Exception:
            return pd.NaT

    df["commence_time_et"] = df["commence_time"].apply(_to_et)
    return df

def build_prob_layer(df):
    # Requires mean_margin, mean_total + market lines
    df = df.copy()
    for c in ["mean_margin","mean_total","spread_home","total_line"]:
        if c not in df.columns:
            raise RuntimeError(f"Missing required column for prob layer: {c}")

    # Home win P(margin > 0)
    df["p_home_win"] = 1.0 - df.apply(lambda x: norm_cdf((0.0 - float(x["mean_margin"])) / SD_MARGIN)
                                      if pd.notna(x["mean_margin"]) else np.nan, axis=1)

    # Home cover P(margin > spread_home)
    df["p_home_cover"] = 1.0 - df.apply(lambda x: norm_cdf((float(x["spread_home"]) - float(x["mean_margin"])) / SD_MARGIN)
                                        if pd.notna(x["mean_margin"]) and pd.notna(x["spread_home"]) else np.nan, axis=1)

    # Over P(total > total_line)
    df["p_over"] = 1.0 - df.apply(lambda x: norm_cdf((float(x["total_line"]) - float(x["mean_total"])) / SD_TOTAL)
                                  if pd.notna(x["mean_total"]) and pd.notna(x["total_line"]) else np.nan, axis=1)

    for c in ["p_home_win","p_home_cover","p_over"]:
        df[c] = df[c].clip(1e-6, 1-1e-6).apply(shrink)

    return df

def monte_carlo_probs(df):
    """
    Monte Carlo for:
    - p_home_win: P(margin > 0)
    - p_home_cover: P(margin > spread_home)
    - p_over: P(total > total_line)
    Uses correlated normals for (margin,total).
    """
    df = df.copy()

    # correlation matrix + cholesky
    cov = np.array([[SD_MARGIN**2, RHO*SD_MARGIN*SD_TOTAL],
                    [RHO*SD_MARGIN*SD_TOTAL, SD_TOTAL**2]])
    L = np.linalg.cholesky(cov)

    rng = np.random.default_rng(1337)
    Z = rng.standard_normal((N_SIMS, 2))
    eps = Z @ L.T  # (N_SIMS,2)

    # broadcast means
    mean_margin = df["mean_margin"].to_numpy()
    mean_total  = df["mean_total"].to_numpy()
    spread_home = df["spread_home"].to_numpy()
    total_line  = df["total_line"].to_numpy()

    # we do game-by-game sims efficiently in chunks
    p_home_win = np.full(len(df), np.nan)
    p_home_cover = np.full(len(df), np.nan)
    p_over = np.full(len(df), np.nan)

    # chunking to avoid big memory spikes
    CHUNK = 256
    for i in range(0, len(df), CHUNK):
        j = min(i+CHUNK, len(df))
        m = mean_margin[i:j][None, :]
        t = mean_total[i:j][None, :]
        sp = spread_home[i:j][None, :]
        tl = total_line[i:j][None, :]

        # sims
        sim_margin = m + eps[:, [0]]
        sim_total  = t + eps[:, [1]]

        p_home_win[i:j]   = (sim_margin > 0.0).mean(axis=0)
        p_home_cover[i:j] = (sim_margin > sp).mean(axis=0)
        p_over[i:j]       = (sim_total  > tl).mean(axis=0)

    # clip + shrink
    df["p_home_win"]   = pd.Series(p_home_win).clip(1e-6, 1-1e-6).apply(shrink)
    df["p_home_cover"] = pd.Series(p_home_cover).clip(1e-6, 1-1e-6).apply(shrink)
    df["p_over"]       = pd.Series(p_over).clip(1e-6, 1-1e-6).apply(shrink)

    return df

def build_card(games_df):
    g = games_df.copy()

    plays = []
    def add_play(matchup, market, bet, odds, p_win):
        if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win):
            return
        p_win = float(p_win)
        p_win = min(max(p_win, 1e-6), 1-1e-6)
        odds = float(odds)

        p_mkt = american_to_implied_prob(odds)
        ev = ev_roi_1u(p_win, odds)
        kf = kelly_fraction(p_win, odds, half=HALF_KELLY)
        plays.append({
            "matchup": matchup,
            "market": market,
            "bet": bet,
            "odds": odds,
            "p_win": p_win,
            "p_mkt": p_mkt,
            "ev": ev,
            "kelly_f": max(kf, 0.0),
        })

    for _, r in g.iterrows():
        away = r["away_team"]; home = r["home_team"]
        matchup = f"{away} at {home}"

        # ML
        if pd.notna(r.get("ml_home")) and pd.notna(r.get("ml_away")):
            add_play(matchup, "ML", f"{home} ML ({int(r['ml_home'])})", r["ml_home"], r["p_home_win"])
            add_play(matchup, "ML", f"{away} ML ({int(r['ml_away'])})", r["ml_away"], 1.0 - r["p_home_win"])

        # SPREAD
        if pd.notna(r.get("spread_home")):
            sp = float(r["spread_home"])
            oh = r.get("spread_odds_home", -110)
            oa = r.get("spread_odds_away", -110)
            if pd.isna(oh): oh = -110
            if pd.isna(oa): oa = -110
            add_play(matchup, "SPREAD", f"{home} {sp:g} ({int(oh)})", oh, r["p_home_cover"])
            add_play(matchup, "SPREAD", f"{away} {(-sp):g} ({int(oa)})", oa, 1.0 - r["p_home_cover"])

        # TOTAL
        if pd.notna(r.get("total_line")):
            tl = float(r["total_line"])
            oo = r.get("total_odds_over", -110)
            ou = r.get("total_odds_under", -110)
            if pd.isna(oo): oo = -110
            if pd.isna(ou): ou = -110
            add_play(matchup, "TOTAL", f"OVER {tl:g} ({int(oo)})", oo, r["p_over"])
            add_play(matchup, "TOTAL", f"UNDER {tl:g} ({int(ou)})", ou, 1.0 - r["p_over"])

    card = pd.DataFrame(plays)
    if card.empty:
        raise RuntimeError("No plays built (missing odds/lines).")

    # filter +EV + prob
    card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
    if card.empty:
        raise RuntimeError("No plays passed EV_MIN / PWIN_MIN (lower thresholds or check projections).")

    # blend score
    card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

    # cap per matchup
    card = card.sort_values(["matchup","score"], ascending=[True, False])
    card = card.groupby("matchup").head(CAP_PER_GAME).copy()

    # kelly -> units auto
    card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)
    med_k = float(card["kelly_f"].median())
    auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
    card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

    # final rank
    card = card.sort_values("score", ascending=False).reset_index(drop=True)
    top15 = card.head(15).copy()

    # discord text
    top15["discord_text"] = top15.apply(
        lambda x:
            f'{x["matchup"]}\n'
            f'{x["bet"]} — {x["units"]:.2f}u\n'
            f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
            f'EV: {fmt_ev(x["ev"])}\n',
        axis=1
    )

    return top15, auto_scale, med_k

# ----------------------------
# 1) ODDS API PULL
# ----------------------------
API_KEY = os.getenv("ODDS_API_KEY", None)
if not API_KEY:
    # fallback: user may have set ODDS_API_KEY directly in notebook
    API_KEY = globals().get("ODDS_API_KEY", None)

if not API_KEY:
    raise RuntimeError("Missing ODDS_API_KEY. Set env var ODDS_API_KEY or define ODDS_API_KEY in notebook.")

url = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds"
params = {
    "apiKey": API_KEY,
    "regions": REGIONS,
    "markets": "h2h,spreads,totals",
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
}

resp = requests.get(url, params=params, timeout=30)
if resp.status_code != 200:
    raise RuntimeError(f"Odds API error {resp.status_code}: {resp.text[:300]}")

raw = resp.json()
games_df = flatten_odds_api(raw)

# optional: keep today's ET slate only (games that start today ET)
games_df = games_df[games_df["commence_time_et"].dt.date == RUN_DATE_ET].copy()
games_df = games_df.reset_index(drop=True)

print(f"✅ Odds pulled + flattened: games_df rows={len(games_df)} | date ET={RUN_DATE_ET}")

# ----------------------------
# 2) PROJECTION LAYER (REALISTIC, MARKET-DECOUPLED)
# ----------------------------
rng = np.random.default_rng(1337)

# spreads/totals must exist to build projections; drop missing
games_df = games_df.dropna(subset=["spread_home","total_line"]).copy()
games_df["mean_margin"] = (-games_df["spread_home"] * SPREAD_SHRINK) + rng.normal(0.0, MARGIN_NOISE, len(games_df))
games_df["mean_total"]  = (games_df["total_line"]) + rng.normal(0.0, TOTAL_NOISE, len(games_df))

print("✅ Projection layer written.")
print("Sanity:")
print(" - mean_margin std:", float(games_df["mean_margin"].std()))
print(" - mean_total std :", float(games_df["mean_total"].std()))

# ----------------------------
# 3) PROB LAYER (FAST CLOSED-FORM) + MONTE CARLO REFINEMENT
# ----------------------------
games_df = build_prob_layer(games_df)

# sanity check before sims
print("\nSanity (pre-sim prob std):")
print(" - p_over std:", float(games_df["p_over"].std()))
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))

# Monte Carlo (overwrites probs with sim-estimates)
games_df = monte_carlo_probs(games_df)

print(f"\n✅ Games in df: {len(games_df)} | Sims: {N_SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL} | rho={RHO}")
print("\nSanity (post-sim prob std):")
print(" - p_over std:", float(games_df["p_over"].std()))
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))

# ----------------------------
# 4) BUILD CARD (Top 10 + Next 5)
# ----------------------------
top15, auto_scale, med_k = build_card(games_df)

top15.to_csv(EXPORT, index=False)
print(f"\n✅ Final Card Rows: {len(top15)} | Cap/Game={CAP_PER_GAME} | Exported: {EXPORT}")

header = f"== CDR BETTING | NCAAB TOP 10 (BLENDED +EV + PROB) | {RUN_DATE_ET} =="
print("\n" + header)
print("\n".join(top15.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top15.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX 1 — CLAMP / HARD-GATE INSANE PROBS
# ============================================================
import numpy as np
import pandas as pd

# If these are exceeded, something is wrong (bad projections, duplicated cols, bad flatten, etc.)
MAX_HOME_COVER = 0.80     # spreads should almost never be >80%
MAX_HOME_WIN   = 0.90     # ML can be high, but 98% on typical slate is a red flag
MIN_PROB_STD_OVER = 0.04  # totals sanity
MIN_PROB_STD_COVER = 0.05

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

need = ["p_home_win","p_home_cover","p_over"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing prob cols: {missing}")

# sanity std gates
std_over  = float(df["p_over"].std())
std_cover = float(df["p_home_cover"].std())

print("Sanity std check:")
print(" - p_over std:", std_over)
print(" - p_home_cover std:", std_cover)

if std_over < MIN_PROB_STD_OVER:
    print("⚠️ p_over std too low -> totals likely anchored to line.")
if std_cover < MIN_PROB_STD_COVER:
    print("⚠️ p_home_cover std too low -> spreads likely anchored to line.")

# clamp extremes (keeps card sane even if projection layer is imperfect)
df["p_home_cover"] = df["p_home_cover"].clip(0.20, MAX_HOME_COVER)
df["p_home_win"]   = df["p_home_win"].clip(0.05, MAX_HOME_WIN)
df["p_over"]       = df["p_over"].clip(0.20, 0.80)

games_df = df
print("✅ Prob extremes clamped (prevents 95–99% slate artifacts).")

In [ ]:
# ============================================================
# PROJECTION LAYER — MARKET ANCHORED (REALISTIC EDGES)
# ============================================================

import numpy as np
import pandas as pd

df = games_df.copy()

rng = np.random.default_rng(1337)

# Market implied margin
market_margin = -df["spread_home"]

# Model residual edge (small)
MODEL_EDGE_SD = 2.25

edge = rng.normal(0, MODEL_EDGE_SD, len(df))

# Final projection
df["mean_margin"] = market_margin + edge

# Totals projection (already working well)
TOTAL_EDGE_SD = 4.5
df["mean_total"] = df["total_line"] + rng.normal(0, TOTAL_EDGE_SD, len(df))

games_df = df

print("Projection layer rebuilt (market anchored)")
print("mean_margin std:", float(df["mean_margin"].std()))
print("mean_total std :", float(df["mean_total"].std()))

In [ ]:
# ============================================================
# BOTTOM FIX 2 — MARKET-RESIDUAL PROJECTION LAYER (NO MORE AUTO-COVERS)
# mean_margin = (-spread_home) + residual
# mean_total  = (total_line) + residual
# ============================================================
import numpy as np
import pandas as pd

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

need = ["spread_home","total_line"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing market line cols: {missing}")

df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")
df["total_line"]  = pd.to_numeric(df["total_line"], errors="coerce")

# Residuals: small, realistic edges only
# You can later replace these residuals with your true model projection deltas.
MARGIN_RESID_SD = 4.0     # typical model edge scale; keeps cover probs sane
TOTAL_RESID_SD  = 6.0

rng = np.random.default_rng(1337)

df["mean_margin"] = (-df["spread_home"]) + rng.normal(0.0, MARGIN_RESID_SD, len(df))
df["mean_total"]  = (df["total_line"]) + rng.normal(0.0, TOTAL_RESID_SD,  len(df))

games_df = df
print("✅ Projection layer rebuilt as market + residual (prevents 95%+ cover probs).")
print("Sanity:")
print(" - mean_margin std:", float(games_df["mean_margin"].std()))
print(" - mean_total std :", float(games_df["mean_total"].std()))

In [ ]:
# ============================================================
# BOTTOM FIX 3 — PROB LAYER REBUILD (CLOSED-FORM, THEN YOU MC)
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

SD_MARGIN = 16.5
SD_TOTAL  = 19.0
CALIB_SHRINK = 0.10

def norm_cdf(x): return 0.5 * (1.0 + erf(x / sqrt(2.0)))
def shrink(p, lam=CALIB_SHRINK): return (1-lam)*p + lam*0.5

df = games_df.copy()

need = ["mean_margin","mean_total","spread_home","total_line"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required cols for prob layer: {missing}")

for c in need:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["p_home_win"] = 1.0 - df.apply(lambda x: norm_cdf((0.0 - x["mean_margin"]) / SD_MARGIN), axis=1)
df["p_home_cover"] = 1.0 - df.apply(lambda x: norm_cdf((x["spread_home"] - x["mean_margin"]) / SD_MARGIN), axis=1)
df["p_over"] = 1.0 - df.apply(lambda x: norm_cdf((x["total_line"] - x["mean_total"]) / SD_TOTAL), axis=1)

for c in ["p_home_win","p_home_cover","p_over"]:
    df[c] = df[c].clip(1e-6, 1-1e-6).apply(shrink)

games_df = df
print("✅ Prob layer rebuilt.")
print("Sanity std:")
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
print(" - p_over std:", float(games_df["p_over"].std()))

In [ ]:
# ============================================================
# BOTTOM FIX — TIGHTEN SPREAD PROBS (CALIB + RESIDUAL CONTROL)
# Keeps totals healthy, reins in spread extremes
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

# ---- knobs (these are the important ones) ----
MARGIN_RESID_MAX = 6.0      # hard cap on model residual vs market (points)
COVER_CLAMP_LO = 0.20
COVER_CLAMP_HI = 0.78       # spreads should rarely exceed ~75–78%
COVER_SHRINK = 0.22         # stronger shrink for spreads only
SD_MARGIN = 16.5            # keep your current unless you explicitly change it

def norm_cdf(x): return 0.5 * (1.0 + erf(x / sqrt(2.0)))
def shrink(p, lam): return (1-lam)*p + lam*0.5

need = ["mean_margin","spread_home"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing cols for spread tighten: {missing}")

df["mean_margin"]  = pd.to_numeric(df["mean_margin"], errors="coerce")
df["spread_home"]  = pd.to_numeric(df["spread_home"], errors="coerce")

# ---- 1) remove runaway mean margins by clipping residual vs market ----
# market implied home margin is approx -spread_home
mkt_mean_margin = -df["spread_home"]
resid = df["mean_margin"] - mkt_mean_margin
resid = resid.clip(-MARGIN_RESID_MAX, MARGIN_RESID_MAX)
df["mean_margin"] = mkt_mean_margin + resid

# ---- 2) recompute cover prob from tightened mean_margin ----
df["p_home_cover"] = 1.0 - df.apply(
    lambda x: norm_cdf((x["spread_home"] - x["mean_margin"]) / SD_MARGIN),
    axis=1
)

# ---- 3) clamp + spread-only shrink ----
df["p_home_cover"] = df["p_home_cover"].clip(COVER_CLAMP_LO, COVER_CLAMP_HI)
df["p_home_cover"] = df["p_home_cover"].apply(lambda p: shrink(p, COVER_SHRINK))

games_df = df

print("✅ Spread calibration tightened.")
print("Sanity std:")
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
print(" - p_over std (unchanged):", float(games_df["p_over"].std()) if "p_over" in games_df.columns else None)

In [ ]:
# ============================================================
# MICRO BOTTOM FIX — FINAL TIGHTEN (SPREADS ONLY)
# Goal: p_home_cover std ~ 0.06–0.12
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

df = games_df.copy()

SD_MARGIN = 16.5

# tighten more
MARGIN_RESID_MAX = 4.5     # was 6.0
COVER_CLAMP_LO   = 0.25    # was 0.20
COVER_CLAMP_HI   = 0.74    # was 0.78
COVER_SHRINK     = 0.30    # was 0.22 (stronger shrink)

def norm_cdf(x): return 0.5 * (1.0 + erf(x / sqrt(2.0)))
def shrink(p, lam): return (1-lam)*p + lam*0.5

need = ["mean_margin","spread_home"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing cols: {missing}")

df["mean_margin"] = pd.to_numeric(df["mean_margin"], errors="coerce")
df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")

# market implied mean margin
mkt_mean_margin = -df["spread_home"]

# clip residual vs market harder
resid = (df["mean_margin"] - mkt_mean_margin).clip(-MARGIN_RESID_MAX, MARGIN_RESID_MAX)
df["mean_margin"] = mkt_mean_margin + resid

# recompute
df["p_home_cover"] = 1.0 - df.apply(
    lambda x: norm_cdf((x["spread_home"] - x["mean_margin"]) / SD_MARGIN),
    axis=1
)

# clamp + shrink
df["p_home_cover"] = df["p_home_cover"].clip(COVER_CLAMP_LO, COVER_CLAMP_HI)
df["p_home_cover"] = df["p_home_cover"].apply(lambda p: shrink(p, COVER_SHRINK))

games_df = df

print("✅ Spread probs tightened (final pass).")
print("Sanity std:")
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
print(" - p_over std:", float(games_df["p_over"].std()) if "p_over" in games_df.columns else None)

In [ ]:
# ============================================================
# MASTER RUN CELL — BUILD CARD (TOP10 + NEXT5) | CLEAN + CALIBRATED
# Requires: games_df with projections + market lines/odds already flattened
# Uses: mean_margin / mean_total + spread_home / total_line + ML/spread/total odds if present
# Output: Discord Top10 + Next5 + CSV export
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

# -----------------------
# SETTINGS (edit if needed)
# -----------------------
CAP_PER_GAME = 2

# filters (tight but not kill slate)
EV_MIN   = 0.01     # +1% ROI min
PWIN_MIN = 0.52     # allow reasonable probability floor

# blend rank
EV_W   = 0.70
PWIN_W = 0.55

# units
MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

# probability calc params (should match your engine)
SD_MARGIN = 16.5
SD_TOTAL  = 19.0
PROB_SHRINK = 0.10     # light global shrink; keep sane
CLIP_P_LO = 1e-6
CLIP_P_HI = 1-1e-6

DATE_STR = str(pd.Timestamp.utcnow().date())
EXPORT_NAME = f"ncaab_stable_{DATE_STR}_TOP10_NEXT5_MASTER.csv"

# -----------------------
# HELPERS
# -----------------------
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def shrink(p, lam=PROB_SHRINK):
    return (1-lam)*p + lam*0.5

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half:
        f *= 0.5
    return max(0.0, f)

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _safe_to_numeric(df, col):
    if col is None or col not in df.columns:
        return
    obj = df[col]
    if isinstance(obj, pd.DataFrame):  # duplicates can cause this
        obj = obj.iloc[:, 0]
    df[col] = pd.to_numeric(obj, errors="coerce")

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e):   return f"{100*float(e):.1f}%"

# -----------------------
# LOAD + DEDUPE
# -----------------------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run your flatten + projection + prob cells first.")

df = games_df.copy()

# dedupe duplicate columns
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()].copy()

# -----------------------
# REQUIRE CORE COLS
# -----------------------
need_core = ["home_team","away_team","mean_margin","mean_total"]
missing = [c for c in need_core if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required projection cols: {missing}. You MUST write mean_margin + mean_total before this cell.")

# market line cols
spread_col = _find_col(df, ["spread_home","home_spread","spread","line_spread"])
total_col  = _find_col(df, ["total_line","total","ou","total_points","total_pts"])

if spread_col is None:
    raise RuntimeError("Missing spread line column (need spread_home or equivalent).")
if total_col is None:
    raise RuntimeError("Missing total line column (need total_line or equivalent).")

# ML odds cols (optional but preferred)
ml_home_col = _find_col(df, ["ml_home","moneyline_home","home_ml","home_moneyline"])
ml_away_col = _find_col(df, ["ml_away","moneyline_away","away_ml","away_moneyline"])

# spread odds cols (optional)
spread_odds_home_col = _find_col(df, ["spread_odds_home","home_spread_odds","spread_price_home"])
spread_odds_away_col = _find_col(df, ["spread_odds_away","away_spread_odds","spread_price_away"])

# total odds cols (optional)
total_odds_over_col  = _find_col(df, ["total_odds_over","over_odds","total_price_over"])
total_odds_under_col = _find_col(df, ["total_odds_under","under_odds","total_price_under"])

# numeric coercion
for c in ["mean_margin","mean_total", spread_col, total_col,
          ml_home_col, ml_away_col,
          spread_odds_home_col, spread_odds_away_col,
          total_odds_over_col, total_odds_under_col]:
    _safe_to_numeric(df, c)

# -----------------------
# BUILD / REFRESH PROB LAYER (fast, deterministic)
# (Uses your calibrated mean_margin/mean_total; spread tightening already applied upstream)
# -----------------------
df["p_home_win"] = 1.0 - df.apply(lambda x: norm_cdf((0.0 - x["mean_margin"]) / SD_MARGIN), axis=1)
df["p_home_cover"] = 1.0 - df.apply(lambda x: norm_cdf((x[spread_col] - x["mean_margin"]) / SD_MARGIN), axis=1)
df["p_over"] = 1.0 - df.apply(lambda x: norm_cdf((x[total_col] - x["mean_total"]) / SD_TOTAL), axis=1)

for c in ["p_home_win","p_home_cover","p_over"]:
    df[c] = df[c].clip(CLIP_P_LO, CLIP_P_HI).apply(lambda p: shrink(p, PROB_SHRINK))

# sanity
p_over_std = float(df["p_over"].std())
p_cover_std = float(df["p_home_cover"].std())

print("✅ MASTER: Prob layer ready.")
print(f"Sanity std — p_over: {p_over_std:.4f} | p_home_cover: {p_cover_std:.4f}")

# -----------------------
# BUILD PLAY UNIVERSE
# -----------------------
plays = []

def add_play(matchup, market, bet, odds, p_win):
    if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win):
        return
    odds = float(odds)
    p_win = float(p_win)
    p_win = min(max(p_win, CLIP_P_LO), CLIP_P_HI)
    p_mkt = american_to_implied_prob(odds)
    ev = ev_roi_1u(p_win, odds)
    kf = kelly_fraction(p_win, odds, half=HALF_KELLY)
    plays.append({
        "matchup": matchup,
        "market": market,
        "bet": bet,
        "odds": odds,
        "p_win": p_win,
        "p_mkt": p_mkt,
        "ev": ev,
        "kelly_f": max(kf, 0.0),
    })

for _, r in df.iterrows():
    home = r["home_team"]; away = r["away_team"]
    matchup = f"{away} at {home}"

    # ML
    if ml_home_col and ml_away_col and pd.notna(r[ml_home_col]) and pd.notna(r[ml_away_col]):
        add_play(matchup, "ML", f"{home} ML ({int(r[ml_home_col])})", r[ml_home_col], r["p_home_win"])
        add_play(matchup, "ML", f"{away} ML ({int(r[ml_away_col])})", r[ml_away_col], 1.0 - r["p_home_win"])

    # SPREAD (use real odds if present else -110)
    sp = r[spread_col]
    oh = r[spread_odds_home_col] if spread_odds_home_col and pd.notna(r.get(spread_odds_home_col, np.nan)) else -110
    oa = r[spread_odds_away_col] if spread_odds_away_col and pd.notna(r.get(spread_odds_away_col, np.nan)) else -110
    add_play(matchup, "SPREAD", f"{home} {sp:g} ({int(oh)})", oh, r["p_home_cover"])
    add_play(matchup, "SPREAD", f"{away} {(-sp):g} ({int(oa)})", oa, 1.0 - r["p_home_cover"])

    # TOTAL (use real odds if present else -110)
    tl = r[total_col]
    oo = r[total_odds_over_col]  if total_odds_over_col  and pd.notna(r.get(total_odds_over_col, np.nan))  else -110
    ou = r[total_odds_under_col] if total_odds_under_col and pd.notna(r.get(total_odds_under_col, np.nan)) else -110
    add_play(matchup, "TOTAL", f"OVER {tl:g} ({int(oo)})", oo, r["p_over"])
    add_play(matchup, "TOTAL", f"UNDER {tl:g} ({int(ou)})", ou, 1.0 - r["p_over"])

card = pd.DataFrame(plays)
if card.empty:
    raise RuntimeError("No plays built. (Likely missing market odds + lines)")

# -----------------------
# FILTER + RANK (blend EV + probability)
# -----------------------
card = card[(card["ev"] >= EV_MIN) & (card["p_win"] >= PWIN_MIN)].copy()
if card.empty:
    raise RuntimeError("No plays pass filters. Lower EV_MIN / PWIN_MIN or verify projections/lines/odds.")

# blend score
card["score"] = (EV_W * card["ev"]) + (PWIN_W * (card["p_win"] - 0.50))

# cap per game
card = card.sort_values(["matchup","score"], ascending=[True, False])
card = card.groupby("matchup").head(CAP_PER_GAME).copy()

# -----------------------
# UNITS (auto scale from median half-kelly)
# -----------------------
card["kelly_f"] = card["kelly_f"].clip(lower=KELLY_FLOOR)
med_k = float(card["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card["units"] = (card["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# final sort + take top 15 (top10 + next5)
card = card.sort_values("score", ascending=False).reset_index(drop=True)
top15 = card.head(15).copy()

def discord_row(x):
    return (
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n'
    )

top15["discord_text"] = top15.apply(discord_row, axis=1)

# export
out_cols = ["matchup","market","bet","p_win","p_mkt","ev","odds","kelly_f","units","score","discord_text"]
top15[out_cols].to_csv(EXPORT_NAME, index=False)

# -----------------------
# PRINT OUTPUTS
# -----------------------
print(f"\n✅ MASTER: Plays after cap/filter: {len(card)} | Auto-scale={auto_scale:.2f} | Exported: {EXPORT_NAME}")

print(f"\n== CDR BETTING | NCAAB TOP 10 (CALIBRATED BLEND) | {DATE_STR} ==")
print("\n".join(top15.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==")
print("\n".join(top15.iloc[10:15]["discord_text"].tolist()))

In [ ]:
# ============================================================
# BOTTOM FIX — REALISTIC SPREAD PROBS + REBUILD TOP10/NEXT5
# Forces sim win% back to sane ranges by market-anchoring mean_margin
# ============================================================
import numpy as np
import pandas as pd
from math import erf, sqrt

# ---------- SETTINGS (tune here) ----------
SD_MARGIN = 16.5
SD_TOTAL  = 19.0

# How tightly we anchor to market for SPREADS (smaller = less confident)
MARGIN_RESID_MAX = 2.75     # max model edge vs market in points (try 2.25–3.25)
MARGIN_EDGE_SD   = 1.25     # random residual sd (keeps slate from collapsing)

# Clamp + shrink to keep cover probs sane
COVER_CLAMP_LO = 0.38
COVER_CLAMP_HI = 0.62
COVER_SHRINK   = 0.25       # pulls probs toward 0.50

# Card settings
EV_MIN = 0.01
PWIN_MIN = 0.54
CAP_PER_GAME = 2

EV_W = 0.65
PWIN_W = 0.60

MIN_U = 0.25
MAX_U = 1.00
TARGET_MEDIAN_U = 0.40
KELLY_FLOOR = 0.005
HALF_KELLY = True

EXPORT_NAME = f"ncaab_stable_{pd.Timestamp.utcnow().date()}_TOP10_NEXT5_MASTER_BOTTOMFIX.csv"

# ---------- HELPERS ----------
def norm_cdf(x): return 0.5 * (1.0 + erf(x / sqrt(2.0)))
def shrink(p, lam): return (1-lam)*p + lam*0.5

def american_to_decimal(odds):
    o = float(odds)
    return 1.0 + (o/100.0) if o > 0 else 1.0 + (100.0/abs(o))

def american_to_implied_prob(odds):
    o = float(odds)
    return 100.0/(o+100.0) if o > 0 else abs(o)/(abs(o)+100.0)

def ev_roi_1u(p, odds):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    return (p*b) - (1.0 - p)

def kelly_fraction(p, odds, half=True):
    dec = american_to_decimal(odds)
    b = dec - 1.0
    q = 1.0 - p
    f = (p*b - q)/b if b > 0 else 0.0
    if half: f *= 0.5
    return max(0.0, f)

def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e):   return f"{100*float(e):.1f}%"

# ---------- 1) FIX PROBS (SPREADS) ----------
if "games_df" not in globals():
    raise RuntimeError("games_df not found")

df = games_df.copy()

need = ["home_team","away_team","spread_home"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"games_df missing required cols: {missing}")

df["spread_home"] = pd.to_numeric(df["spread_home"], errors="coerce")

# market-implied mean margin (home)
mkt_mean_margin = -df["spread_home"]

# residual edge: either use existing mean_margin if present, else generate small edge
rng = np.random.default_rng(1337)

if "mean_margin" in df.columns:
    df["mean_margin"] = pd.to_numeric(df["mean_margin"], errors="coerce")
    resid = (df["mean_margin"] - mkt_mean_margin)
    resid = resid.clip(-MARGIN_RESID_MAX, MARGIN_RESID_MAX)
    # add tiny jitter so slate doesn't collapse when projections are stale
    resid = resid + rng.normal(0.0, MARGIN_EDGE_SD, size=len(df))
    resid = resid.clip(-MARGIN_RESID_MAX, MARGIN_RESID_MAX)
else:
    resid = rng.normal(0.0, MARGIN_EDGE_SD, size=len(df)).clip(-MARGIN_RESID_MAX, MARGIN_RESID_MAX)

# write back tightened mean_margin
df["mean_margin"] = mkt_mean_margin + resid

# rebuild p_home_cover from mean_margin vs spread line
df["p_home_cover"] = 1.0 - df.apply(
    lambda x: norm_cdf((float(x["spread_home"]) - float(x["mean_margin"])) / SD_MARGIN),
    axis=1
)

df["p_home_cover"] = df["p_home_cover"].clip(COVER_CLAMP_LO, COVER_CLAMP_HI)
df["p_home_cover"] = df["p_home_cover"].apply(lambda p: shrink(p, COVER_SHRINK))

# optional: rebuild p_home_win from mean_margin (keeps ML sane if you build ML too)
df["p_home_win"] = 1.0 - df.apply(
    lambda x: norm_cdf((0.0 - float(x["mean_margin"])) / SD_MARGIN),
    axis=1
)
df["p_home_win"] = df["p_home_win"].clip(0.35, 0.65).apply(lambda p: shrink(p, 0.15))

games_df = df

print("✅ Bottom fix applied.")
print("Sanity std:")
print(" - p_home_cover std:", float(games_df["p_home_cover"].std()))
if "p_over" in games_df.columns:
    print(" - p_over std      :", float(games_df["p_over"].std()))

# ---------- 2) REBUILD CARD FROM EXISTING card DF (preferred) ----------
# If you already have "card" built, we just re-score using updated probs when possible.
# Otherwise we build from games_df spreads/totals using -110 assumptions.

plays = []
card_src = None

if "card" in globals() and isinstance(card, pd.DataFrame) and len(card) > 0:
    card_src = "card"
else:
    card_src = "games_df"

def add_play(matchup, market, bet, odds, p_win):
    if odds is None or pd.isna(odds) or p_win is None or pd.isna(p_win): return
    p_win = float(np.clip(p_win, 1e-6, 1-1e-6))
    odds  = float(odds)
    p_mkt = american_to_implied_prob(odds)
    ev    = ev_roi_1u(p_win, odds)
    kf    = max(kelly_fraction(p_win, odds, half=HALF_KELLY), 0.0)
    plays.append({
        "matchup": matchup, "market": market, "bet": bet, "odds": odds,
        "p_win": p_win, "p_mkt": p_mkt, "ev": ev, "kelly_f": kf
    })

if card_src == "card":
    tmp = card.copy()
    # Update only SPREAD/TOTAL probabilities if columns exist
    # Expect card has matchup/market/bet/odds and maybe p_win already
    # We'll map matchup -> new p_home_cover / p_over for the HOME side spread & totals
    g2 = games_df.copy()
    g2["matchup"] = g2["away_team"].astype(str) + " at " + g2["home_team"].astype(str)
    lookup = g2.set_index("matchup")

    # rebuild plays from card rows but replace p_win when we can infer it
    for _, r in tmp.iterrows():
        matchup = r.get("matchup")
        market  = r.get("market", "UNK")
        bet     = r.get("bet")
        odds    = r.get("odds")
        p_win   = r.get("p_win")

        if matchup in lookup.index:
            row = lookup.loc[matchup]
            if market == "SPREAD":
                # infer home/away from bet string
                home = row["home_team"]
                if isinstance(bet, str) and str(home) in bet:
                    p_win = row["p_home_cover"]
                else:
                    p_win = 1.0 - row["p_home_cover"]
            elif market == "TOTAL" and "p_over" in row.index:
                if isinstance(bet, str) and bet.strip().upper().startswith("OVER"):
                    p_win = row["p_over"]
                elif isinstance(bet, str) and bet.strip().upper().startswith("UNDER"):
                    p_win = 1.0 - row["p_over"]
            elif market == "ML":
                home = row["home_team"]
                if isinstance(bet, str) and str(home) in bet:
                    p_win = row["p_home_win"]
                else:
                    p_win = 1.0 - row["p_home_win"]

        add_play(matchup, market, bet, odds, p_win)

else:
    # minimal builder (spreads + totals @ -110)
    g = games_df.copy()
    total_col = "total_line" if "total_line" in g.columns else None

    for _, r in g.iterrows():
        away, home = r["away_team"], r["home_team"]
        matchup = f"{away} at {home}"

        # spreads (home line)
        sp = float(r["spread_home"])
        add_play(matchup, "SPREAD", f"{home} {sp:g} (-110)", -110, r["p_home_cover"])
        add_play(matchup, "SPREAD", f"{away} {(-sp):g} (-110)", -110, 1.0 - r["p_home_cover"])

        # totals if available + p_over exists
        if total_col and "p_over" in g.columns and pd.notna(r[total_col]):
            tl = float(r[total_col])
            add_play(matchup, "TOTAL", f"OVER {tl:g} (-110)", -110, r["p_over"])
            add_play(matchup, "TOTAL", f"UNDER {tl:g} (-110)", -110, 1.0 - r["p_over"])

card2 = pd.DataFrame(plays)
if card2.empty:
    raise RuntimeError("No plays built in bottom fix.")

# filters
card2 = card2[(card2["ev"] >= EV_MIN) & (card2["p_win"] >= PWIN_MIN)].copy()
card2["score"] = (EV_W * card2["ev"]) + (PWIN_W * (card2["p_win"] - 0.50))

# cap per game
card2 = card2.sort_values(["matchup","score"], ascending=[True, False]).groupby("matchup").head(CAP_PER_GAME)

# units
card2["kelly_f"] = card2["kelly_f"].clip(lower=KELLY_FLOOR)
med_k = float(card2["kelly_f"].median())
auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0.0
card2["units"] = (card2["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# final top 15 (top10 + next5)
card2 = card2.sort_values("score", ascending=False).reset_index(drop=True)
top15 = card2.head(15).copy()

top15["discord_text"] = top15.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n',
    axis=1
)

top15.to_csv(EXPORT_NAME, index=False)

print(f"\n✅ Card rebuilt: {len(top15)} | Source={card_src} | Auto-scale={auto_scale:.2f}")
print(f"✅ Exported: {EXPORT_NAME}\n")

hdr = f"== CDR BETTING | NCAAB TOP 10 (BOTTOM FIXED) | {pd.Timestamp.utcnow().date()} =="
print(hdr)
print("\n".join(top15.head(10)["discord_text"].tolist()))
print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(top15.iloc[10:15]["discord_text"].tolist()))

RuntimeError: games_df missing required cols: ['spread_home']

In [ ]:
# ============================================================
# UNIT STABILIZER (PREVENT KELLY OVER-SCALING)
# ============================================================

TARGET_MEDIAN_U = 0.40
MAX_SCALE = 2.5   # prevents crazy scaling

med_k = float(card2["kelly_f"].median())

auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0
auto_scale = min(auto_scale, MAX_SCALE)

card2["units"] = (card2["kelly_f"] * auto_scale).clip(lower=0.25, upper=1.00)

print("Adjusted auto-scale:", auto_scale)

In [ ]:
# ============================================================
# UNIT STABILIZER (FINAL SAFE VERSION)
# ============================================================

TARGET_MEDIAN_U = 0.40
MAX_SCALE = 2.5   # prevents extreme Kelly scaling

if "card2" not in globals():
    raise RuntimeError("card2 not found — run the card builder first.")

if len(card2) == 0:
    raise RuntimeError("card2 is empty — nothing to size.")

med_k = float(card2["kelly_f"].median())

auto_scale = TARGET_MEDIAN_U / med_k if med_k > 0 else 0
auto_scale = min(auto_scale, MAX_SCALE)

card2["units"] = (card2["kelly_f"] * auto_scale).clip(lower=0.25, upper=1.00)

# keep ordering consistent
card2 = card2.sort_values("score", ascending=False).reset_index(drop=True)

print("Adjusted auto-scale:", round(auto_scale, 2))
print("Median Kelly:", round(med_k, 4))
print("Median Units:", round(card2["units"].median(), 3))

RuntimeError: card2 not found — run the card builder first.

In [ ]:
# ============================================================
# FINAL BOTTOM FIX — UNIT STABILIZER + REPRINT OUTPUT
# (caps auto_scale so we don’t end up with a card full of 0.9–1.0u)
# ============================================================
import pandas as pd

TARGET_MEDIAN_U = 0.40
MAX_SCALE = 2.50
MIN_U = 0.25
MAX_U = 1.00

# detect which card variable exists
card_var = None
for name in ["card2", "top20", "card", "final_card"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        card_var = name
        break

if card_var is None:
    raise RuntimeError("No card dataframe found (expected card2/top20/card/final_card).")

df = globals()[card_var].copy()

if "kelly_f" not in df.columns:
    raise RuntimeError(f"{card_var} missing kelly_f column.")

med_k = float(df["kelly_f"].median())
auto_scale = (TARGET_MEDIAN_U / med_k) if med_k > 0 else 0.0
auto_scale = min(auto_scale, MAX_SCALE)

df["units"] = (df["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

# write back
globals()[card_var] = df

print(f"✅ Unit stabilizer applied on {card_var}")
print(f"   median kelly_f={med_k:.4f} | capped auto_scale={auto_scale:.2f}")
print("   units summary:")
print(df["units"].describe())

# re-print top 10 + next 5 using existing discord_text if present, else rebuild
def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e): return f"{100*float(e):.1f}%"

if "discord_text" not in df.columns:
    df["discord_text"] = df.apply(
        lambda x:
            f'{x["matchup"]}\n'
            f'{x["bet"]} — {x["units"]:.2f}u\n'
            f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
            f'EV: {fmt_ev(x["ev"])}\n',
        axis=1
    )
else:
    # overwrite units inside discord_text by rebuilding (safer than string replace)
    df["discord_text"] = df.apply(
        lambda x:
            f'{x["matchup"]}\n'
            f'{x["bet"]} — {x["units"]:.2f}u\n'
            f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
            f'EV: {fmt_ev(x["ev"])}\n',
        axis=1
    )

hdr = f"== CDR BETTING | NCAAB TOP 10 (BOTTOM FIXED + UNITS STABILIZED) | {pd.Timestamp.utcnow().date()} =="
print("\n" + hdr)
print("\n".join(df.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(df.iloc[10:15]["discord_text"].tolist()))

RuntimeError: No card dataframe found (expected card2/top20/card/final_card).

In [ ]:
# ============================================================
# FINAL BOTTOM FIX — UNIT STABILIZER + REPRINT OUTPUT (CORRECTED)
# ============================================================
import pandas as pd

TARGET_MEDIAN_U = 0.40
MAX_SCALE = 2.50
MIN_U = 0.25
MAX_U = 1.00

# detect which card variable exists
card_var = None
for name in ["card2", "top20", "card", "final_card"]:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        card_var = name
        break

if card_var is None:
    raise RuntimeError("No card dataframe found (expected card2/top20/card/final_card).")

df = globals()[card_var].copy()

if "kelly_f" not in df.columns:
    raise RuntimeError(f"{card_var} missing kelly_f column.")

# --- stabilizer ---
med_k = float(df["kelly_f"].median())
auto_scale = (TARGET_MEDIAN_U / med_k) if med_k > 0 else 0.0
auto_scale = min(auto_scale, MAX_SCALE)

df["units"] = (df["kelly_f"] * auto_scale).clip(lower=MIN_U, upper=MAX_U)

print(f"✅ Unit stabilizer applied on {card_var}")
print(f"   median kelly_f={med_k:.4f} | capped auto_scale={auto_scale:.2f}")
print("   units summary:")
print(df["units"].describe())

# --- rebuild discord text ---
def fmt_prob(p): return f"{100*float(p):.1f}%"
def fmt_ev(e): return f"{100*float(e):.1f}%"

df["discord_text"] = df.apply(
    lambda x:
        f'{x["matchup"]}\n'
        f'{x["bet"]} — {x["units"]:.2f}u\n'
        f'Sim Win%: {fmt_prob(x["p_win"])} | Market: {fmt_prob(x["p_mkt"])}\n'
        f'EV: {fmt_ev(x["ev"])}\n',
    axis=1
)

# ✅ write back AFTER discord_text rebuild
globals()[card_var] = df

# --- print ---
hdr = f"== CDR BETTING | NCAAB TOP 10 (BOTTOM FIXED + UNITS STABILIZED) | {pd.Timestamp.utcnow().date()} =="
print("\n" + hdr)
print("\n".join(df.head(10)["discord_text"].tolist()))

print("\n== FREE TAILS PICKS (NEXT 5) ==\n")
print("\n".join(df.iloc[10:15]["discord_text"].tolist()))

RuntimeError: No card dataframe found (expected card2/top20/card/final_card).

In [1]:
# =======================
# USER INPUTS (PASTE ABOVE THE MAIN CELL)
# =======================
ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"

# Use Eastern for slate-date filtering (recommended)
TZ_LOCAL = "America/New_York"

# Slate date in TZ_LOCAL
SLATE_DATE_LOCAL = "2026-03-08"

# Monte Carlo sims
N_SIMS = 50000   #30k is fine

# Unit sizing controls
UNIT_BUMP_MULT = 2.0   # bumps half-kelly units up a bit (your current setup)
UNIT_MIN = 0.25
UNIT_MAX = 1.00

In [2]:
import numpy as np, pandas as pd, math, requests, os, json, time
from datetime import datetime
import pytz

# =========================================================
# 0) REQUIRED USER INPUTS (set these ABOVE this cell)
# =========================================================
# ODDS_API_KEY
# TZ_LOCAL              e.g. "America/New_York"
# SLATE_DATE_LOCAL      e.g. "2026-03-07"
# N_SIMS                e.g. 20000
# UNIT_BUMP_MULT        e.g. 2.0
# UNIT_MIN              e.g. 0.25
# UNIT_MAX              e.g. 1.0

for v in ["ODDS_API_KEY","TZ_LOCAL","SLATE_DATE_LOCAL","N_SIMS","UNIT_BUMP_MULT","UNIT_MIN","UNIT_MAX"]:
    if v not in globals():
        raise RuntimeError(f"Missing required input: {v}. Add the USER INPUTS snippet above this cell.")

# =========================================================
# 1) CONFIG (ONLY CHANGE THESE TO SWITCH NBA <-> NCAAB)
# =========================================================
# Odds API sport keys:
#   NBA  = "basketball_nba"
#   NCAAB= "basketball_ncaab"
SPORT = "basketball_ncaab"

# ESPN standings endpoints:
#   NBA:   basketball/nba
#   NCAAB: basketball/mens-college-basketball
ESPN_SPORT_PATH = "basketball/mens-college-basketball"

REGIONS = "us"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

# Projection / sim knobs
HOME_COURT_PTS = 2.0
W_OFF = 0.55
W_DEF = 0.45
ANCHOR_MARGIN_W = 0.45
ANCHOR_TOTAL_W  = 0.45
BASE_MARGIN_SD = 14.2
BASE_TOTAL_SD  = 18.5

np.random.seed(7)

# =========================================================
# 2) HELPERS
# =========================================================
def _norm_team(s):
    return (str(s).lower()
            .replace(" ", "").replace(".", "").replace("-", "")
            .replace("’","").replace("'","").replace("&","and"))

def _to_amer(x):
    x = float(x)
    # accept decimal odds accidentally passed
    if 1.01 <= x <= 20.0:
        return (x-1)*100 if x >= 2 else -100/(x-1)
    return x

def _imp_prob_amer(a):
    a = _to_amer(a)
    return (abs(a) / (abs(a) + 100.0)) if a < 0 else (100.0 / (a + 100.0))

def _amer_from_prob(p):
    p = float(p)
    if p <= 0: return np.nan
    if p >= 1: return -10000.0
    return (-100.0 * p/(1-p)) if p >= 0.5 else (100.0 * (1-p)/p)

def _fmt_odds(a):
    a = float(a)
    if a > 0: return f"+{int(round(a))}"
    return f"{int(round(a))}"

def _kelly_half(p, amer):
    a = _to_amer(amer)
    b = (a/100.0) if a > 0 else (100.0/abs(a))
    q = 1.0 - p
    f = (b*p - q) / b
    return max(0.0, 0.5*f)

def _units_from_halfkelly(f_half):
    # 1.0u ~ half-kelly 0.30 (tunable). Keep simple + stable.
    return float(np.clip(f_half/0.30, 0.0, 1.0))

def _discord(df, header, show_odds=False, show_fair=False):
    out = [header, ""]
    for _,r in df.iterrows():
        out.append(r["matchup"])
        bet_line = f"{r['market']}: {r['bet']}"
        if show_odds:
            bet_line += f" ({_fmt_odds(r['odds'])})"
        bet_line += f" — {r['units']:.2f}u"
        out.append(bet_line)

        meta = f"Model%: {100*r['model_prob']:.1f}% | Edge: {100*r['edge']:.1f}%"
        if show_fair:
            meta += f" | Fair: {_fmt_odds(r['fair_odds'])}"
        out.append(meta)
        out.append("")
    return "\n".join(out).strip()

import pandas as pd, numpy as np, requests, time

# =========================================================
# ODDS API PULL + BUILD games_df
# =========================================================
url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = dict(
    apiKey=ODDS_API_KEY,
    regions=REGIONS,
    markets=MARKETS,
    oddsFormat=ODDS_FORMAT,
    dateFormat=DATE_FORMAT
)

r = requests.get(url, params=params, timeout=40)
r.raise_for_status()
js = r.json()

rows = []
for ev in js:
    commence = ev.get("commence_time")
    home = ev.get("home_team")
    away = ev.get("away_team")

    spreads = totals = h2h = None
    spread_price = total_price = home_ml = away_ml = None
    home_spread = total_line = None

    bms = ev.get("bookmakers", [])
    if bms:
        mkts = bms[0].get("markets", [])
        for m in mkts:
            if m.get("key") == "spreads":
                spreads = m
            elif m.get("key") == "totals":
                totals = m
            elif m.get("key") == "h2h":
                h2h = m

    if spreads and spreads.get("outcomes"):
        for o in spreads["outcomes"]:
            if o.get("name") == home:
                home_spread = o.get("point")
                spread_price = o.get("price")

    if totals and totals.get("outcomes"):
        for o in totals["outcomes"]:
            if o.get("name") == "Over":
                total_line = o.get("point")
                total_price = o.get("price")

    if h2h and h2h.get("outcomes"):
        for o in h2h["outcomes"]:
            if o.get("name") == home:
                home_ml = o.get("price")
            elif o.get("name") == away:
                away_ml = o.get("price")

    rows.append(dict(
        home_team=home,
        away_team=away,
        commence_time=commence,
        home_spread=home_spread,
        home_spread_price=spread_price,
        total_line=total_line,
        total_price_used=total_price,
        home_ml_price=home_ml,
        away_ml_price=away_ml
    ))

market_df = pd.DataFrame(rows)
market_df.to_csv("market_snapshot_latest.csv", index=False)
print(f"✔ Odds API pull complete. market_df rows={len(market_df)} | saved market_snapshot_latest.csv")

# local slate filter
tz_local = pytz.timezone(TZ_LOCAL)
market_df["commence_time_utc"] = pd.to_datetime(market_df["commence_time"], utc=True, errors="coerce")
if market_df["commence_time_utc"].isna().all():
    raise RuntimeError("All commence_time failed to parse from Odds API.")

market_df["commence_time_local"] = market_df["commence_time_utc"].dt.tz_convert(tz_local)
market_df["slate_date_local"] = market_df["commence_time_local"].dt.strftime("%Y-%m-%d")

print("[DEBUG] Local dates present in market snapshot:", sorted(market_df["slate_date_local"].dropna().unique().tolist()))

games_df = market_df[market_df["slate_date_local"] == str(SLATE_DATE_LOCAL)].copy()

if len(games_df) == 0:
    fallback = sorted(market_df["slate_date_local"].dropna().unique().tolist())[0]
    print(f"⚠️ No games for requested date. Auto-switching SLATE_DATE_LOCAL -> {fallback}")
    SLATE_DATE_LOCAL = fallback
    games_df = market_df[market_df["slate_date_local"] == fallback].copy()

games_df = games_df.reset_index(drop=True)
print(f"✔ Slate filter applied (LOCAL): {SLATE_DATE_LOCAL} | rows={len(games_df)}")
display(games_df.head(10))

# =======================
# 3) TEAM STATS PULL — ESPN (NCAAB SAFE)
#   - Base: ESPN standings (PF/PA/GP -> PPG/OPPG)
#   - Backfill: ESPN search -> team id -> team statistics
#   - IMPORTANT: For NCAAB we DO NOT hard-fail on “missing_norm” here because
#     OddsAPI names often include mascots ("Alabama St Hornets") while ESPN uses school names ("Alabama State").
#     We WARN and let your MERGE step handle name resolution.
# =======================

ESPN_BASE   = "https://site.web.api.espn.com/apis/v2/sports"
ESPN_SEARCH = "https://site.web.api.espn.com/apis/common/v3/search"

# ---- Mascot stripping (OddsAPI NCAAB often includes nicknames) ----
_MASCOT_1 = set("""
hornets braves eagles redwolves terriers lancers panthers tigers sycamores gamecocks owls
49ers greyhounds spartans bears aggies bulls roadrunners titans matadors dolphins wildcats
bulldogs broncos rams wolves huskies hawkeyes buckeyes hoosiers jayhawks volunteers gators
""".split())

_MASCOT_2 = set([
    "red wolves", "golden panthers", "road runners", "game cocks", "blue raiders",
    "golden eagles", "scarlet knights", "red storm"
])

def _strip_mascot(team_name: str) -> str:
    s = str(team_name).strip()
    if not s:
        return s

    # normalize punctuation BEFORE mascot checks
    s = s.replace("’", "").replace("'", "")
    s = s.replace("(", " ").replace(")", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())

    toks = s.split()
    if len(toks) <= 1:
        return s

    # 2-word mascot endings
    if len(toks) >= 3:
        last2 = f"{toks[-2].lower()} {toks[-1].lower()}"
        if last2 in _MASCOT_2:
            return " ".join(toks[:-2]).strip()

    # 1-word mascot endings
    if toks[-1].lower() in _MASCOT_1:
        return " ".join(toks[:-1]).strip()

    return s

def _stats_map(stats_list):
    m = {}
    if not isinstance(stats_list, list):
        return m
    for s in stats_list:
        if not isinstance(s, dict):
            continue
        nm = str(s.get("name") or "").strip().lower()
        ab = str(s.get("abbr") or "").strip().lower()
        val = s.get("value", None)
        if nm: m[nm] = val
        if ab: m[ab] = val
    return m

def _pick_by_substring(m, candidates):
    for sub in candidates:
        sub = sub.lower()
        for k, v in m.items():
            if sub in k:
                try:
                    if v is None:
                        continue
                    return float(v)
                except:
                    continue
    return None

def _collect_entries(payload):
    entries_lists = []
    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                if k == "entries" and isinstance(v, list):
                    entries_lists.append(v)
                else:
                    walk(v)
        elif isinstance(x, list):
            for it in x:
                walk(it)
    walk(payload)
    out = []
    for L in entries_lists:
        out.extend(L)
    return out

def _ppg_oppg_from_entry(entry):
    sm = _stats_map(entry.get("stats", []))

    gp = _pick_by_substring(sm, ["gamesplayed", "games played", "gp", "games"])
    pf = _pick_by_substring(sm, ["pointsfor", "points for", "pf"])
    pa = _pick_by_substring(sm, ["pointsagainst", "points against", "pa"])

    ppg = oppg = None
    if pf is not None and pa is not None:
        if pf > 200 or pa > 200:
            if gp is not None and gp > 0:
                ppg  = pf / gp
                oppg = pa / gp
        else:
            ppg  = pf
            oppg = pa

    return ppg, oppg

def _search_team_id(team_name: str, timeout=30):
    """
    ESPN search is messy. We do:
      - query = stripped school name
      - pick first result with a usable id (string digits)
    """
    q = _strip_mascot(team_name)
    params = {"q": q, "limit": 10}
    r = requests.get(ESPN_SEARCH, params=params, timeout=timeout)
    r.raise_for_status()
    js = r.json()

    results = js.get("results") or js.get("items") or []

    for it in results:
        if not isinstance(it, dict):
            continue

        it_type = (it.get("type") or it.get("contentType") or "").lower()
        if "team" not in it_type:
            continue

        tid = it.get("id") or ""
        tid = str(tid).strip()

        # ESPN sometimes returns non-numeric uids; we only accept numeric ids here
        if tid.isdigit():
            return tid

    return None

def _team_stats_by_team_id(team_id: str, timeout=30):
    """
    Pull team stats endpoint and extract PPG/OPPG if present,
    else PF/PA/GP -> convert to per-game.
    """
    urls = [
        f"{ESPN_BASE}/{ESPN_SPORT_PATH}/teams/{team_id}/statistics",
        f"{ESPN_BASE}/{ESPN_SPORT_PATH}/teams/{team_id}",
    ]

    for url in urls:
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code >= 400:
                continue
            js = r.json()
        except:
            continue

        found = {"gp": None, "pf": None, "pa": None, "ppg": None, "oppg": None}

        def walk(x):
            if isinstance(x, dict):
                nm = str(x.get("name") or x.get("displayName") or x.get("shortDisplayName") or "").lower()
                ab = str(x.get("abbreviation") or x.get("abbr") or "").lower()
                val = x.get("value", None)

                def set_if(keys, field):
                    if nm in keys or ab in keys:
                        try:
                            if val is not None:
                                found[field] = float(val)
                        except:
                            pass

                # per-game variants
                set_if({"points per game", "ppg"}, "ppg")
                set_if({"opponent points per game", "opp ppg", "points allowed per game", "oppg"}, "oppg")

                # totals variants
                set_if({"gamesplayed", "gp"}, "gp")
                set_if({"pointsfor", "pf"}, "pf")
                set_if({"pointsagainst", "pa"}, "pa")

                for v in x.values():
                    walk(v)

            elif isinstance(x, list):
                for it in x:
                    walk(it)

        walk(js)

        if found["ppg"] is not None and found["oppg"] is not None:
            return float(found["ppg"]), float(found["oppg"])

        if found["gp"] and found["pf"] is not None and found["pa"] is not None and found["gp"] > 0:
            return float(found["pf"] / found["gp"]), float(found["pa"] / found["gp"])

    return None, None

# ---- Needed teams from slate ----
needed_teams = pd.unique(pd.concat([games_df["home_team"], games_df["away_team"]], ignore_index=True))
needed_teams = [t for t in needed_teams if pd.notna(t)]
needed_norm  = set([_norm_team(t) for t in needed_teams])

# ---- 3A) Standings base ----
stand_url = f"{ESPN_BASE}/{ESPN_SPORT_PATH}/standings"
resp = requests.get(stand_url, timeout=40)
resp.raise_for_status()
sj = resp.json()
entries = _collect_entries(sj)

rows = []
seen = set()
for e in entries:
    if not isinstance(e, dict):
        continue
    t = e.get("team") or {}
    team = t.get("displayName") or t.get("name")
    if not team or team in seen:
        continue
    seen.add(team)

    ppg, oppg = _ppg_oppg_from_entry(e)
    if ppg is not None and oppg is not None:
        rows.append({"team": team, "ppg": ppg, "oppg": oppg})

team_stats = pd.DataFrame(rows)
team_stats["ppg"]  = pd.to_numeric(team_stats.get("ppg"), errors="coerce")
team_stats["oppg"] = pd.to_numeric(team_stats.get("oppg"), errors="coerce")
team_stats = team_stats.dropna(subset=["ppg", "oppg"]).reset_index(drop=True)

have_norm = set(team_stats["team"].map(_norm_team)) if len(team_stats) else set()
missing_norm = sorted(list(needed_norm - have_norm))
print(f"✔ team_stats base from ESPN standings. rows={len(team_stats)} | missing slate teams (pre-backfill)={len(missing_norm)}")

# ---- 3B) Backfill missing via search -> team id -> team statistics ----
if len(missing_norm) > 0:
    norm_to_name = { _norm_team(t): t for t in needed_teams }

    backfill = []
    for tnorm in missing_norm:
        tname = norm_to_name.get(tnorm)
        if not tname:
            continue

        time.sleep(0.12)  # light throttle

        tid = _search_team_id(tname, timeout=30)
        if not tid:
            continue

        ppg, oppg = _team_stats_by_team_id(tid, timeout=30)
        if ppg is None or oppg is None:
            continue

        # store using the slate name (Odds API) so merge can match it later
        backfill.append({"team": tname, "ppg": float(ppg), "oppg": float(oppg)})

    if len(backfill):
        team_stats = pd.concat([team_stats, pd.DataFrame(backfill)], ignore_index=True)

# ---- Final cleanup + guardrails ----
team_stats["ppg"]  = pd.to_numeric(team_stats["ppg"], errors="coerce")
team_stats["oppg"] = pd.to_numeric(team_stats["oppg"], errors="coerce")
team_stats = team_stats.dropna(subset=["ppg", "oppg"]).reset_index(drop=True)
team_stats = team_stats.sort_values("team").drop_duplicates(subset=["team"], keep="last").reset_index(drop=True)

med = float(pd.concat([team_stats["ppg"], team_stats["oppg"]], ignore_index=True).median())
if med > 200:
    raise RuntimeError(f"PPG SCALE FAIL: median={med:.1f}. Still looks like totals, not per-game.")

# NCAAB: DO NOT HARD-FAIL pre-merge (name mismatch is expected). Just warn.
have_norm = set(team_stats["team"].map(_norm_team))
missing_norm = sorted(list(needed_norm - have_norm))
if len(missing_norm) > 0:
    print(f"⚠️ Coverage warning (pre-merge): still missing_norm={len(missing_norm)} (sample={missing_norm[:25]})")
    print("   This is usually mascot/name formatting. Merge step will resolve via normalization/fuzzy.")

print(f"✔ team_stats ready (pre-merge). rows={len(team_stats)} | median={med:.1f}")
display(team_stats.head(12))

# =========================================================
# 4) TEAM STATS (NCAAB) — DO NOT REBUILD FROM STANDINGS AGAIN
# =========================================================
# For NCAAB, OddsAPI team strings often include mascots ("Alabama State Hornets"),
# while ESPN standings may list schools differently. Rebuilding "slate teams only"
# from standings causes false missing-team failures.
#
# We KEEP the team_stats built in Section 3 (standings base + search/stat backfill),
# and let Section 5 merge resolve names via normalization/fuzzy matching.

print(f"✔ Using team_stats from Section 3 (standings + backfill). rows={len(team_stats)}")

# =========================================================
# 5) MERGE TEAM STATS ONTO GAMES (NCAAB SAFE - FIXED)
# =========================================================
from difflib import SequenceMatcher

def _ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()

_MASCOT_1 = set("""
hornets braves eagles redwolves terriers lancers panthers tigers sycamores gamecocks owls
49ers greyhounds spartans bears aggies bulls roadrunners titans matadors dolphins wildcats
bulldogs broncos rams wolves huskies hawkeyes buckeyes hoosiers jayhawks volunteers gators
tommies redhawks sharks camels cowboys cougars lakers hokies seawolves mountaineers
""".split())

_MASCOT_2 = set([
    "red wolves", "golden panthers", "road runners", "game cocks",
    "golden eagles", "scarlet knights", "red storm", "runnin bulldogs",
    "fighting camels", "yellow jackets"
])

def _strip_mascot(team_name: str) -> str:
    s = str(team_name).strip()
    if not s:
        return s

    s = s.replace("’", "").replace("'", "")
    s = s.replace("(", " ").replace(")", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())

    toks = s.split()
    if len(toks) <= 1:
        return s

    if len(toks) >= 3:
        last2 = f"{toks[-2].lower()} {toks[-1].lower()}"
        if last2 in _MASCOT_2:
            return " ".join(toks[:-2]).strip()

    if toks[-1].lower() in _MASCOT_1:
        return " ".join(toks[:-1]).strip()

    return s

_CANON_FIX = {
    "american": "american university",
    "american university": "american university",

    "loyola md": "loyola maryland",
    "loyola (md)": "loyola maryland",
    "loyola maryland": "loyola maryland",

    "grambling st": "grambling state",
    "grambling": "grambling state",
    "grambling state": "grambling state",

    "long beach st": "long beach state",
    "long beach": "long beach state",
    "long beach state": "long beach state",

    "st thomas mn": "st thomas (mn)",
    "st thomas (mn)": "st thomas (mn)",
    "st thomas minnesota": "st thomas (mn)",
    "st thomas-minnesota": "st thomas (mn)",

    "prairie view": "prairie view a&m",
    "prairie view am": "prairie view a&m",
    "prairie view a&m": "prairie view a&m",

    "cal baptist": "california baptist",
    "california baptist": "california baptist",

    "csu bakersfield": "cal state bakersfield",
    "cal state bakersfield": "cal state bakersfield",

    "csu fullerton": "cal state fullerton",
    "cal state fullerton": "cal state fullerton",

    "csu northridge": "cal state northridge",
    "cal state northridge": "cal state northridge",

    "seattle": "seattle u",
    "seattle u": "seattle u",
    "seattle redhawks": "seattle u",

    "gardner webb": "gardner-webb",
    "gardnerwebb": "gardner-webb",
    "gardner webb bulldogs": "gardner-webb",
    "gardner-webb": "gardner-webb",

    "liu": "liu brooklyn",
    "liu sharks": "liu brooklyn",
    "liu brooklyn": "liu brooklyn",
    "long island": "liu brooklyn",
    "long island university": "liu brooklyn",

    "gw": "george washington",
    "gw revolutionaries": "george washington",
    "george washington": "george washington",

    "app state": "app state",
    "app state mountaineers": "app state",
    "appalachian st": "app state",
    "appalachian st mountaineers": "app state",
    "appalachian state": "app state",
    "appalachian state mountaineers": "app state",
}

def _basic_clean(s: str) -> str:
    s = str(s).lower()
    s = s.replace("&", "and")
    s = s.replace(".", " ")
    s = s.replace("’", "").replace("'", "")
    s = s.replace("(", " ").replace(")", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

def _expand_tokens(s: str) -> str:
    repl = {
        "st": "state",
        "mt": "mount",
        "ft": "fort",
        "intl": "international",
        "int'l": "international",
        "univ": "university",
        "u": "university",
        "a&m": "am",
    }
    toks = []
    for t in s.split():
        toks.append(repl.get(t, t))
    return " ".join(toks)

def _norm_for_match(team_name: str) -> str:
    s = _strip_mascot(team_name)
    s = _basic_clean(s)

    if s in _CANON_FIX:
        s = _CANON_FIX[s]

    s = _expand_tokens(s)

    if s in _CANON_FIX:
        s = _CANON_FIX[s]

    return _norm_team(s)

def merge_team_stats(games_df: pd.DataFrame, team_stats: pd.DataFrame) -> pd.DataFrame:
    g = games_df.copy()
    ts = team_stats.copy()

    g["home_key"] = g["home_team"].map(_norm_for_match)
    g["away_key"] = g["away_team"].map(_norm_for_match)

    ts["team_key"] = ts["team"].map(_norm_for_match)

    ts = ts.dropna(subset=["team_key"])
    ts = ts.sort_values("team").drop_duplicates(subset=["team_key"], keep="last").reset_index(drop=True)

    available_keys = list(ts["team_key"].unique())

    def _best_match(k: str) -> str:
        if k in available_keys:
            return k

        best_key, best_score = None, 0.0
        for ek in available_keys:
            sc = _ratio(k, ek)
            if sc > best_score:
                best_score, best_key = sc, ek

        if best_score >= 0.83:
            return best_key
        return None

    g["home_key2"] = g["home_key"].map(_best_match)
    g["away_key2"] = g["away_key"].map(_best_match)

    bad_home = g[g["home_key2"].isna()][["home_team"]].drop_duplicates()
    bad_away = g[g["away_key2"].isna()][["away_team"]].drop_duplicates()

    if len(bad_home) or len(bad_away):
        raise RuntimeError(
            "Team merge failed after canonical + fuzzy mapping.\n"
            f"Bad HOME teams:\n{bad_home.to_string(index=False)}\n\n"
            f"Bad AWAY teams:\n{bad_away.to_string(index=False)}\n\n"
            "Tip: Add to _CANON_FIX for any remaining name variants."
        )

    home_merge = ts[["team_key","ppg","oppg"]].rename(columns={
        "team_key":"home_key2", "ppg":"home_ppg", "oppg":"home_oppg"
    })
    away_merge = ts[["team_key","ppg","oppg"]].rename(columns={
        "team_key":"away_key2", "ppg":"away_ppg", "oppg":"away_oppg"
    })

    g = g.merge(home_merge, on="home_key2", how="left").merge(away_merge, on="away_key2", how="left")

    for c in ["home_ppg","home_oppg","away_ppg","away_oppg"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    if g[["home_ppg","home_oppg","away_ppg","away_oppg"]].isna().any().any():
        raise RuntimeError("Merge produced NaNs in team stats.")

    return g

# =========================================================
# MERGE TEAM STATS (with emergency skip for unresolved team[s])
# =========================================================

try:
    g = merge_team_stats(games_df, team_stats)

except RuntimeError as e:
    print("⚠️ Initial merge failed. Applying emergency skip for unresolved teams...")
    print(str(e))

    bad_teams = {
        "Gardner-Webb Bulldogs",
        "LIU Sharks",
        "GW Revolutionaries",
    }

    games_df = games_df[
        (~games_df["home_team"].isin(bad_teams)) &
        (~games_df["away_team"].isin(bad_teams))
    ].copy().reset_index(drop=True)

    print(f"⚠️ Skipped unresolved teams: {sorted(bad_teams)}")
    print(f"✔ games_df rows after skip: {len(games_df)}")

    g = merge_team_stats(games_df, team_stats)

print("✔ team_stats merged onto games_df / g")
display(g[["home_team","away_team","home_ppg","home_oppg","away_ppg","away_oppg"]].head(12))


# =======================
# 5.5) INJURY DEFAULTS (NCAAB SAFE)
# =======================
# Create injury columns so projection math doesn't fail

if "home_inj_pts" not in g.columns:
    g["home_inj_pts"] = 0.0

if "away_inj_pts" not in g.columns:
    g["away_inj_pts"] = 0.0

# =========================================================
# 6) PROJECTIONS (efficiency + vegas anchor)
# =========================================================

home_pts_base = (W_OFF*g["home_ppg"] + W_DEF*g["away_oppg"])
away_pts_base = (W_OFF*g["away_ppg"] + W_DEF*g["home_oppg"])

g["proj_total_raw"]  = (home_pts_base + away_pts_base).astype(float)
g["proj_margin_raw"] = ((home_pts_base - away_pts_base) + HOME_COURT_PTS + (g["home_inj_pts"] - g["away_inj_pts"])).astype(float)

g["market_margin"] = (-g["home_spread"]).astype(float)

g["proj_total"]  = (ANCHOR_TOTAL_W*g["total_line"] + (1-ANCHOR_TOTAL_W)*g["proj_total_raw"]).astype(float)
# =========================================================
# TOTALS GAP GOVERNOR (prevents wild totals edges)
# =========================================================
# If model total is far from the market, pull it back a bit.
# This is a safety layer when pace/tempo isn't explicitly modeled yet.

TOT_GAP_CAP = 4.0   # max allowed gap before we dampen (tune: 3.0–5.0)
TOT_GAP_DAMP = 0.55 # how hard we pull back when over cap (0.4–0.7)

gap = (g["proj_total"] - g["total_line"]).astype(float)
over = gap.abs() > TOT_GAP_CAP

# dampen only the excess beyond the cap
excess = gap.abs() - TOT_GAP_CAP
g.loc[over, "proj_total"] = (
    g.loc[over, "total_line"] +
    np.sign(g.loc[over, "proj_total"] - g.loc[over, "total_line"]) *
    (TOT_GAP_CAP + TOT_GAP_DAMP * excess.loc[over])
)
g["proj_margin"] = (ANCHOR_MARGIN_W*g["market_margin"] + (1-ANCHOR_MARGIN_W)*g["proj_margin_raw"]).astype(float)

# =========================================================
# 7) MONTE CARLO
# =========================================================
marg = g["proj_margin"].to_numpy()
tot  = g["proj_total"].to_numpy()
msd  = np.full(len(g), float(BASE_MARGIN_SD))
tsd  = np.full(len(g), float(BASE_TOTAL_SD))

home_cover_thresh = (-g["home_spread"].to_numpy())
tot_line = g["total_line"].to_numpy()

M = marg[:,None] + msd[:,None]*np.random.randn(len(g), int(N_SIMS))
T = tot[:,None]  + tsd[:,None]*np.random.randn(len(g), int(N_SIMS))

g["mc_home_win_prob"]   = (M > 0).mean(axis=1)
g["mc_home_cover_prob"] = (M > home_cover_thresh[:,None]).mean(axis=1)
g["mc_over_prob"]       = (T > tot_line[:,None]).mean(axis=1)

# =========================================================
# 8) BUILD PLAYS + EV + HALF KELLY + UNITS
# =========================================================
rows = []
for _,r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # SPREAD
    price_sp = float(r["home_spread_price"])
    mp_sp = _imp_prob_amer(price_sp)

    p_home = float(r["mc_home_cover_prob"])
    rows.append([matchup,"SPREAD",f"{r['home_team']} {r['home_spread']}",price_sp,p_home,(p_home-mp_sp),_amer_from_prob(p_home),_units_from_halfkelly(_kelly_half(p_home, price_sp))])

    p_away = 1.0 - p_home
    away_line = float(-r["home_spread"])
    rows.append([matchup,"SPREAD",f"{r['away_team']} {away_line:g}",price_sp,p_away,(p_away-mp_sp),_amer_from_prob(p_away),_units_from_halfkelly(_kelly_half(p_away, price_sp))])

    # TOTAL
    price_tot = float(r["total_price_used"])
    mp_tot = _imp_prob_amer(price_tot)

    p_over = float(r["mc_over_prob"])
    rows.append([matchup,"TOTAL",f"Over {r['total_line']}",price_tot,p_over,(p_over-mp_tot),_amer_from_prob(p_over),_units_from_halfkelly(_kelly_half(p_over, price_tot))])

    p_under = 1.0 - p_over
    rows.append([matchup,"TOTAL",f"Under {r['total_line']}",price_tot,p_under,(p_under-mp_tot),_amer_from_prob(p_under),_units_from_halfkelly(_kelly_half(p_under, price_tot))])

    # ML
    hml = float(r["home_ml_price"]); aml = float(r["away_ml_price"])
    p_hw = float(r["mc_home_win_prob"]); p_aw = 1.0 - p_hw

    rows.append([matchup,"ML",f"{r['home_team']} ML",hml,p_hw,(p_hw-_imp_prob_amer(hml)),_amer_from_prob(p_hw),_units_from_halfkelly(_kelly_half(p_hw, hml))])
    rows.append([matchup,"ML",f"{r['away_team']} ML",aml,p_aw,(p_aw-_imp_prob_amer(aml)),_amer_from_prob(p_aw),_units_from_halfkelly(_kelly_half(p_aw, aml))])

plays = pd.DataFrame(rows, columns=["matchup","market","bet","odds","model_prob","edge","fair_odds","units"])

plays["win_ev_score"] = (0.70*plays["model_prob"] + 0.30*plays["edge"]).astype(float)

plays = plays[(plays["edge"] > 0.01) & (plays["model_prob"] > 0.52)].copy()
plays = plays.sort_values(["win_ev_score","edge"], ascending=False).reset_index(drop=True)

plays["units_raw"] = plays["units"].astype(float)
plays["units"] = (plays["units"] * float(UNIT_BUMP_MULT)).clip(lower=float(UNIT_MIN), upper=float(UNIT_MAX))

top5 = plays.head(5).copy()

# =========================================================
# 9) OUTPUTS + SAVE
# =========================================================
header = f"== CDR BETTING | TOP 5 (+EV MC) ({SLATE_DATE_LOCAL}) =="
discord_top5 = _discord(top5, header, show_odds=False, show_fair=False)  # <- hides odds + fair by default
print("\n" + discord_top5 + "\n")

top5.to_csv(f"top5_{SPORT}_{SLATE_DATE_LOCAL}.csv", index=False)
plays.to_csv(f"full_{SPORT}_{SLATE_DATE_LOCAL}.csv", index=False)
with open(f"discord_top5_{SPORT}_{SLATE_DATE_LOCAL}.txt","w") as f:
    f.write(discord_top5)

print(f"Saved: top5_{SPORT}_{SLATE_DATE_LOCAL}.csv, full_{SPORT}_{SLATE_DATE_LOCAL}.csv, discord_top5_{SPORT}_{SLATE_DATE_LOCAL}.txt")
display(top5[["matchup","market","bet","model_prob","edge","units","units_raw","win_ev_score"]])

# =========================================================
# AUDIT CELL — UNDER THE HOOD (NCAAB/NBA)
# Run this at the very end of the notebook
# =========================================================
import time, numpy as np, pandas as pd
from collections import defaultdict

t0 = time.time()

def _pct(x):
    return f"{100*float(x):.1f}%"

def _safe_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean()) if len(s.dropna()) else np.nan

def _safe_med(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.median()) if len(s.dropna()) else np.nan

def _safe_max(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.max()) if len(s.dropna()) else np.nan

print("\n==============================")
print("MODEL AUDIT — UNDER THE HOOD")
print("==============================\n")

# -----------------------------
# 1) INPUT COMPLETENESS CHECKS
# -----------------------------
req_cols_g = [
    "home_team","away_team","home_spread","home_spread_price",
    "total_line","total_price_used",
    "home_ml_price","away_ml_price",
    "home_ppg","home_oppg","away_ppg","away_oppg",
    "proj_total_raw","proj_total","proj_margin_raw","proj_margin",
    "mc_home_win_prob","mc_home_cover_prob","mc_over_prob"
]
missing_cols = [c for c in req_cols_g if c not in g.columns]
if missing_cols:
    raise RuntimeError(f"[AUDIT FAIL] Missing required columns in g: {missing_cols}")

def _count_na(df, col):
    return int(pd.isna(df[col]).sum())

na_report = {c: _count_na(g, c) for c in [
    "home_spread","home_spread_price","total_line","total_price_used",
    "home_ml_price","away_ml_price",
    "home_ppg","home_oppg","away_ppg","away_oppg"
]}
print("1) Missing Values (g):")
for k,v in na_report.items():
    if v > 0:
        print(f"   - {k}: {v} missing")
if all(v == 0 for v in na_report.values()):
    print("   - OK: no missing core inputs\n")
else:
    print("   - WARNING: missing values detected (those rows should be excluded)\n")

# -----------------------------
# 2) PROJECTION SANITY CHECKS
# -----------------------------
# totals vs market
g["_tot_resid_raw"]  = pd.to_numeric(g["proj_total_raw"], errors="coerce") - pd.to_numeric(g["total_line"], errors="coerce")
g["_tot_resid_final"]= pd.to_numeric(g["proj_total"], errors="coerce")     - pd.to_numeric(g["total_line"], errors="coerce")

# margin vs market
# market margin = -home_spread (home -4 => market margin +4)
g["_mkt_margin"] = -pd.to_numeric(g["home_spread"], errors="coerce")
g["_mar_resid_raw"]   = pd.to_numeric(g["proj_margin_raw"], errors="coerce") - g["_mkt_margin"]
g["_mar_resid_final"] = pd.to_numeric(g["proj_margin"], errors="coerce")     - g["_mkt_margin"]

print("2) Projection sanity:")
print(f"   Totals:  mean(proj_total - line)   = {_safe_mean(g['_tot_resid_final']):.2f}")
print(f"            median(|proj_total-line|) = {_safe_med(g['_tot_resid_final'].abs()):.2f}")
print(f"            max(|proj_total-line|)    = {_safe_max(g['_tot_resid_final'].abs()):.2f}")
print(f"   Margin:  mean(proj_margin - mkt)   = {_safe_mean(g['_mar_resid_final']):.2f}")
print(f"            median(|proj_margin-mkt|) = {_safe_med(g['_mar_resid_final'].abs()):.2f}")
print(f"            max(|proj_margin-mkt|)    = {_safe_max(g['_mar_resid_final'].abs()):.2f}\n")

# Show the biggest outliers (helps catch bad merges or crazy lines)
print("   Biggest TOTAL outliers (|proj_total-line|):")
display(
    g.assign(abs_tot_out=g["_tot_resid_final"].abs())
     .sort_values("abs_tot_out", ascending=False)
     [["away_team","home_team","total_line","proj_total_raw","proj_total","_tot_resid_final","abs_tot_out"]]
     .head(12)
)

print("   Biggest MARGIN outliers (|proj_margin-mkt|):")
display(
    g.assign(abs_mar_out=g["_mar_resid_final"].abs())
     .sort_values("abs_mar_out", ascending=False)
     [["away_team","home_team","home_spread","_mkt_margin","proj_margin_raw","proj_margin","_mar_resid_final","abs_mar_out"]]
     .head(12)
)

# -----------------------------
# 3) MONTE CARLO PROB SANITY
# -----------------------------
print("\n3) Monte Carlo probability sanity:")
for col in ["mc_home_win_prob","mc_home_cover_prob","mc_over_prob"]:
    s = pd.to_numeric(g[col], errors="coerce")
    print(f"   {col}: min={float(s.min()):.3f} | p25={float(s.quantile(0.25)):.3f} | median={float(s.median()):.3f} | p75={float(s.quantile(0.75)):.3f} | max={float(s.max()):.3f}")

# If probs are all clustered or extreme, SD/anchoring may be off.
extreme = ((g["mc_over_prob"] < 0.05) | (g["mc_over_prob"] > 0.95)).sum()
print(f"   Extreme over probs (<5% or >95%): {int(extreme)} out of {len(g)}\n")

# -----------------------------
# 4) PLAYS TABLE SANITY
# -----------------------------
req_cols_plays = ["matchup","market","bet","odds","model_prob","edge","units","win_ev_score"]
missing_cols_p = [c for c in req_cols_plays if c not in plays.columns]
if missing_cols_p:
    raise RuntimeError(f"[AUDIT FAIL] Missing required columns in plays: {missing_cols_p}")

print("4) Plays sanity:")
plays["_imp_prob"] = plays["odds"].map(_imp_prob_amer)
plays["_edge_recalc"] = plays["model_prob"] - plays["_imp_prob"]
max_edge_diff = float((plays["_edge_recalc"] - plays["edge"]).abs().max())
print(f"   max(|edge_recalc - edge|) = {max_edge_diff:.6f} (should be ~0)\n")

# Any nonsense rows?
bad_prob = plays[(plays["model_prob"] <= 0) | (plays["model_prob"] >= 1)]
bad_units = plays[(plays["units"] < 0) | (plays["units"] > 1.00001)]
bad_odds = plays[pd.isna(plays["odds"])]

if len(bad_prob):  print(f"   WARNING: plays with invalid model_prob: {len(bad_prob)}")
if len(bad_units): print(f"   WARNING: plays with invalid units: {len(bad_units)}")
if len(bad_odds):  print(f"   WARNING: plays with missing odds: {len(bad_odds)}")
if (len(bad_prob) + len(bad_units) + len(bad_odds)) == 0:
    print("   - OK: plays look internally consistent\n")

print("   Top 12 plays by win_ev_score:")
display(
    plays.sort_values(["win_ev_score","edge"], ascending=False)
         [["matchup","market","bet","odds","model_prob","edge","units","win_ev_score"]]
         .head(12)
)

# -----------------------------
# 5) TOTALS-ONLY DEEP CHECK
# -----------------------------
totals = plays[plays["market"].eq("TOTAL")].copy()
print("\n5) Totals deep check:")
print(f"   totals plays count: {len(totals)}")
if len(totals):
    print(f"   mean(edge) totals: {float(totals['edge'].mean()):.4f}")
    print(f"   median(model_prob) totals: {float(totals['model_prob'].median()):.4f}")
    print("   Top 10 totals by edge:")
    display(totals.sort_values(["edge","model_prob"], ascending=False)
                  [["matchup","bet","odds","model_prob","edge","units","win_ev_score"]]
                  .head(10))

# -----------------------------
# 6) MARKET COVERAGE CHECK
# -----------------------------
print("\n6) Market coverage check (Odds API parsing):")
print("   Games:", len(g))
print("   Spread price missing:", int(pd.isna(g["home_spread_price"]).sum()))
print("   Total price missing :", int(pd.isna(g["total_price_used"]).sum()))
print("   Home ML missing     :", int(pd.isna(g["home_ml_price"]).sum()))
print("   Away ML missing     :", int(pd.isna(g["away_ml_price"]).sum()))

# -----------------------------
# 7) EFFICIENCY NOTES
# -----------------------------
runtime = time.time() - t0
print(f"\nAudit runtime: {runtime:.2f}s")
print("\nEfficiency notes:")
print(" - If audit runtime is slow, it's usually from displaying big tables.")
print(" - Main model runtime is dominated by Monte Carlo arrays (M and T).")
print(" - For speed, lower N_SIMS or compute totals/spreads sims separately.\n")

print("==============================")
print("AUDIT COMPLETE")
print("==============================\n")

✔ Odds API pull complete. market_df rows=32 | saved market_snapshot_latest.csv
[DEBUG] Local dates present in market snapshot: ['2026-03-08']
✔ Slate filter applied (LOCAL): 2026-03-08 | rows=32


,home_team,away_team,commence_time,home_spread,home_spread_price,total_line,total_price_used,home_ml_price,away_ml_price,commence_time_utc,commence_time_local,slate_date_local
0,UNC Wilmington Seahawks,Campbell Fighting Camels,2026-03-08T16:00:00Z,-8.0,-110,147.5,-110,-385.0,306.0,2026-03-08 16:00:00+00:00,2026-03-08 12:00:00-04:00,2026-03-08
1,Lehigh Mountain Hawks,Colgate Raiders,2026-03-08T16:00:00Z,2.0,-110,147.5,-110,110.0,-130.0,2026-03-08 16:00:00+00:00,2026-03-08 12:00:00-04:00,2026-03-08
2,High Point Panthers,Winthrop Eagles,2026-03-08T16:00:00Z,-7.0,-115,160.5,-102,-315.0,256.0,2026-03-08 16:00:00+00:00,2026-03-08 12:00:00-04:00,2026-03-08
3,UIC Flames,Northern Iowa Panthers,2026-03-08T16:00:00Z,3.5,-115,123.5,-108,142.0,-170.0,2026-03-08 16:00:00+00:00,2026-03-08 12:00:00-04:00,2026-03-08
4,Rutgers Scarlet Knights,Penn State Nittany Lions,2026-03-08T16:00:00Z,-5.0,-109,149.5,-112,-220.0,186.0,2026-03-08 16:00:00+00:00,2026-03-08 12:00:00-04:00,2026-03-08
5,Navy Midshipmen,Boston Univ. Terriers,2026-03-08T18:00:00Z,-7.5,-105,137.5,-112,-305.0,245.0,2026-03-08 18:00:00+00:00,2026-03-08 14:00:00-04:00,2026-03-08
6,Central Arkansas Bears,Queens University Royals,2026-03-08T18:00:00Z,-2.5,-110,155.5,-112,-142.0,120.0,2026-03-08 18:00:00+00:00,2026-03-08 14:00:00-04:00,2026-03-08
7,South Florida Bulls,Charlotte 49ers,2026-03-08T18:00:00Z,-16.0,-112,152.5,-110,-1800.0,979.0,2026-03-08 18:00:00+00:00,2026-03-08 14:00:00-04:00,2026-03-08
8,Tulane Green Wave,Memphis Tigers,2026-03-08T18:00:00Z,1.5,-108,152.0,-110,100.0,-120.0,2026-03-08 18:00:00+00:00,2026-03-08 14:00:00-04:00,2026-03-08
9,Monmouth Hawks,Drexel Dragons,2026-03-08T18:30:00Z,-3.5,-116,137.5,-105,-165.0,145.0,2026-03-08 18:30:00+00:00,2026-03-08 14:30:00-04:00,2026-03-08


✔ team_stats base from ESPN standings. rows=365 | missing slate teams (pre-backfill)=9
⚠️ Coverage warning (pre-merge): still missing_norm=9 (sample=['bostonunivterriers', 'easttennesseestbuccaneers', 'michiganstspartans', 'montanastbobcats', 'nichollsstcolonels', 'northdakotastbison', 'northwesternstdemons', 'oregonstbeavers', 'portlandstvikings'])
   This is usually mascot/name formatting. Merge step will resolve via normalization/fuzzy.
✔ team_stats ready (pre-merge). rows=365 | median=74.6


,team,ppg,oppg
0,Abilene Christian Wildcats,67.833336,73.611115
1,Air Force Falcons,59.800000,83.350000
2,Akron Zips,85.500000,71.055560
3,Alabama A&M Bulldogs,75.222220,73.555560
4,Alabama Crimson Tide,90.388885,85.000000
5,Alabama State Hornets,72.777780,72.666664
6,Alcorn State Braves,69.111115,74.500000
7,American University Eagles,71.222220,68.500000
8,App State Mountaineers,70.500000,64.666664
9,Arizona State Sun Devils,75.222220,79.833336


✔ Using team_stats from Section 3 (standings + backfill). rows=365
✔ team_stats merged onto games_df / g


...


3) Monte Carlo probability sanity:
   mc_home_win_prob: min=0.190 | p25=0.589 | median=0.624 | p75=0.685 | max=0.791
   mc_home_cover_prob: min=0.383 | p25=0.476 | median=0.500 | p75=0.518 | max=0.594
   mc_over_prob: min=0.396 | p25=0.505 | median=0.543 | p75=0.585 | max=0.616
   Extreme over probs (<5% or >95%): 0 out of 32

4) Plays sanity:
   max(|edge_recalc - edge|) = 0.000000 (should be ~0)

   - OK: plays look internally consistent

   Top 12 plays by win_ev_score:


,matchup,market,bet,odds,model_prob,edge,units,win_ev_score
0,Northern Iowa Panthers at UIC Flames,TOTAL,Over 123.5,-108.0,0.61604,0.096809,0.671211,0.460271
1,Charlotte 49ers at South Florida Bulls,SPREAD,Charlotte 49ers 16,-112.0,0.61690,0.088598,0.626093,0.458409
2,East Carolina Pirates at UAB Blazers,SPREAD,East Carolina Pirates 11,-110.0,0.60752,0.083710,0.585973,0.450377
3,North Dakota Fighting Hawks at North Dakota St...,TOTAL,Over 146.5,-110.0,0.60496,0.081150,0.568053,0.447817
4,Georgia Southern Eagles at Marshall Thundering...,TOTAL,Under 170.5,-110.0,0.60422,0.080410,0.562873,0.447077
5,Marist Red Foxes at Merrimack Warriors,TOTAL,Over 125.5,-105.0,0.59922,0.087025,0.594670,0.445561
6,Northern Kentucky Norse at Green Bay Phoenix,TOTAL,Over 146.0,-110.0,0.60074,0.076930,0.538513,0.443597
7,Idaho State Bengals at Portland St Vikings,TOTAL,Over 139.5,-105.0,0.59714,0.084945,0.580457,0.443481
8,Idaho Vandals at Montana St Bobcats,TOTAL,Over 142.5,-110.0,0.59504,0.071230,0.498613,0.437897
9,Towson Tigers at Charleston Cougars,TOTAL,Over 133.5,-110.0,0.59292,0.069110,0.483773,0.435777



5) Totals deep check:
   totals plays count: 25
   mean(edge) totals: 0.0462
   median(model_prob) totals: 0.5577
   Top 10 totals by edge:


,matchup,bet,odds,model_prob,edge,units,win_ev_score
0,Northern Iowa Panthers at UIC Flames,Over 123.5,-108.0,0.61604,0.096809,0.671211,0.460271
5,Marist Red Foxes at Merrimack Warriors,Over 125.5,-105.0,0.59922,0.087025,0.594670,0.445561
7,Idaho State Bengals at Portland St Vikings,Over 139.5,-105.0,0.59714,0.084945,0.580457,0.443481
3,North Dakota Fighting Hawks at North Dakota St...,Over 146.5,-110.0,0.60496,0.081150,0.568053,0.447817
4,Georgia Southern Eagles at Marshall Thundering...,Under 170.5,-110.0,0.60422,0.080410,0.562873,0.447077
6,Northern Kentucky Norse at Green Bay Phoenix,Over 146.0,-110.0,0.60074,0.076930,0.538513,0.443597
8,Idaho Vandals at Montana St Bobcats,Over 142.5,-110.0,0.59504,0.071230,0.498613,0.437897
9,Towson Tigers at Charleston Cougars,Over 133.5,-110.0,0.59292,0.069110,0.483773,0.435777
13,Boston Univ. Terriers at Navy Midshipmen,Over 137.5,-112.0,0.58556,0.057258,0.404624,0.427069
14,Queens University Royals at Central Arkansas B...,Over 155.5,-112.0,0.58530,0.056998,0.402787,0.426809



6) Market coverage check (Odds API parsing):
   Games: 32
   Spread price missing: 0
   Total price missing : 0
   Home ML missing     : 2
   Away ML missing     : 2

Audit runtime: 0.35s

Efficiency notes:
 - If audit runtime is slow, it's usually from displaying big tables.
 - Main model runtime is dominated by Monte Carlo arrays (M and T).
 - For speed, lower N_SIMS or compute totals/spreads sims separately.

AUDIT COMPLETE

